# 🔱 PosAIdon — Augmented Naval Assistant

**1er prix du hackathon SPASEA (Naval Group / DGA), décembre 2025.** Équipe PosAIdon : Sei Bayle, Juliette Faurie, Charbel Dib, Axel Gruyez.

Détection de *dark vessels* — des navires qui coupent leur AIS ou adoptent une trajectoire anormale — par fusion de trois sources : positions AIS, détections radiofréquence (Unseenlabs) et éphémérides satellites (Prométhée, CNES). Le résultat est un poste de travail opérateur : carte tactique, rejeu de trajectoires, scoring d'anomalies, assistant conversationnel et rapport PDF.

> ### ⚠️ Notebook non exécutable en l'état
> Les jeux de données du hackathon sont **confidentiels et non redistribuables**. Les fichiers Parquet attendus (`AIS_VERSION_FINAL.parquet`, `RF_VERSION_FINAL.parquet`, `ephemerides_Japetus_Mediterranee.parquet`) ne sont pas inclus dans ce dépôt, et aucun accès ne peut être fourni.
>
> Ce notebook est une **archive de démonstration** : il documente ce que l'équipe a construit en 24 h et conserve les sorties d'exécution d'origine. Il n'est pas destiné à être relancé.

### Ce que l'équipe a construit

| Bloc | Contenu |
|---|---|
| **Ingestion** | Authentification OAuth2 Dawex, récupération et consolidation des trois flux (AIS, RF, éphémérides) |
| **Carte tactique** | 35+ couches maritimes Folium (EMODnet, NOAA, GEBCO, ZEE…), superposition AIS / RF / emprises satellites |
| **Détection** | 25 tools d'analyse : coupures AIS, profils de vitesse, zones blanches, signatures RF non standard, estimation de destination |
| **Agent IA** | Agent LangChain / LangGraph exposant ces tools, interrogeable en langage naturel |
| **Interface** | Tableau de bord ipywidgets : filtres, timeline animée, fiche navire, export PDF |

Le notebook part du *starter pack* fourni aux équipes par l'organisation ; la section « Les tools de l'agent IA » et tout ce qui suit sont le travail de l'équipe.

---

# Configuration initiale et chargement des données



Configurations de base, récupération des fichiers, paramètrages API et création de la carte interactive


**Installation des dépendances**

In [ ]:
#!pip install -U langchain langchain-community langchain-openai langchain-core python-dotenv
#!pip install -U langgraph
#%pip install -q pandas plotly ipywidgets folium geopy nest_asyncio copernicusmarine xarray netCDF4 shapely duckdb owslib



**Import des librairies Python primordiales**

In [2]:
import requests
import http.client
import json
import time
import os
import gzip
import io
import pandas as pd
from pandas import json_normalize
import numpy as np
import re
import folium
from folium.plugins import HeatMap

from typing import Optional
from math import radians, sin, cos, sqrt, atan2, asin, degrees

import numpy as np
import pandas as pd
from dotenv import load_dotenv


**Gestion ID et Clés API**

In [3]:
load_dotenv()

hostname = "platform.space-data-marketplace.eu"
organisation_id = "6924c0947402bd34c25c988c"
transaction_id_ais = "dbfbe2e084444386a6302ce2b73cff22"
file_access_id_ais = "9516e5ab03284975b560a2091bb0597d"
transaction_id_rf = "c52e85167dae44be9b944a8aa3162f3e"
file_access_id_rf = "ddb2b055350742d0bb07db0aabcf409f"
transaction_id_eph = "96f64cb4e0d9405e8542654259a30842"
file_access_id_eph = "13558391ceb340938ebddf50cee1e000"
OPENWEATHER_BASE_URL = 'http://api.openweathermap.org/data/2.5/weather'
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1/chat/completions'
USE_LOCAL_CHATBOT = False

client_id = os.getenv("DAWEX_CLIENT_ID")
client_secret = os.getenv("DAWEX_CLIENT_SECRET")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

**Récupération du token d'accès DAWEX**

In [4]:
url = f"https://{hostname}/api/auth/oauth2/token"

data = {
    "grant_type": "client_credentials"
}

headers = {
    "Content-Type": "application/x-www-form-urlencoded"
}

response = requests.post(url, data=data, headers=headers, auth=(client_id, client_secret))

if response.status_code == 200:
    token_info = response.json()
    access_token = token_info.get("access_token")
    print("✅ Token récupéré avec succès !")
    print("Access Token :", access_token)
else:
    print(f"❌ Erreur {response.status_code}: {response.text}")


❌ Erreur 401: {"error":"invalid_client"}


**Requête de download sur DAWEX du fichier de type parquet contenant les données AIS**

In [ ]:
conn = http.client.HTTPSConnection(hostname)

path = f"/api/participant/organisations/{organisation_id}/acquiredDataProductFilesDownloadRequests"

payload = json.dumps({
    "acquiredDataProductFilesTransactionId": transaction_id_ais,
    "acquiredDataProductFileAccessIds": [file_access_id_ais]
})

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

conn.request("POST", path, body=payload, headers=headers)
res = conn.getresponse()
data = res.read()

print("Status:", res.status, res.reason)
print(data.decode("utf-8"))

# décoder et parser le JSON
response_json = json.loads(data.decode("utf-8"))

# stocker l'id dans une variable
if "id" in response_json:
    download_request_id = response_json["id"]
    print("Download request ID:", download_request_id)
else:
    print("Download request already exists or failed to create.")

Status: 201 Created
{"id":"6932b78afd991b722bf33029","organisationId":"6924c0947402bd34c25c988c","acquiredDataProductFilesTransactionId":"dbfbe2e084444386a6302ce2b73cff22","acquiredDataProductFileAccessIds":["9516e5ab03284975b560a2091bb0597d"],"percentageCompletion":0,"downloadStatus":"INITIATED","downloadFilename":"AIS_VERSION_FINAL.parquet","createdDate":"2025-12-05T10:44:26.078219981Z"}
Download request ID: 6932b78afd991b722bf33029


**Download sur DAWEX du fichier de type parquet contenant les données AIS**

In [ ]:
# === Endpoints et headers ===
status_url = f"https://{hostname}/api/participant/organisations/{organisation_id}/acquiredDataProductFilesDownloadRequests/{download_request_id}"
content_url = f"{status_url}/content"
headers = {"Authorization": f"Bearer {access_token}"}

# === Paramètres de backoff ===
delay = 5        # Première attente en secondes
max_delay = 120  # Attente max
max_attempts = 20

print("⏳ Vérification de l’état du téléchargement...")

for attempt in range(1, max_attempts + 1):
    response = requests.get(status_url, headers=headers)

    print(f"Tentative {attempt} → HTTP {response.status_code}")

    if response.status_code != 200:
        print("❌ Erreur:", response.text)
        break

    data = response.json()
    status = data.get("downloadStatus")
    percent = data.get("percentageCompletion", 0)
    filename = data.get("downloadFilename", "downloaded_file")

    print(f"Statut: {status} | Progression: {percent}%")

    if status == "COMPLETED":
        print("✅ Fichier prêt :", filename)
        output_path = os.path.join("/content", filename)

        # Télécharger le fichier
        print("⬇️ Téléchargement du fichier...")
        with requests.get(content_url, headers=headers, stream=True) as r:
            r.raise_for_status()
            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

        print(f"✅ Fichier téléchargé avec succès : {output_path}")
        break

    elif status in ("FAILED", "ERROR"):
        print("❌ Le téléchargement a échoué.")
        break

    print(f"⏳ Nouvelle vérification dans {delay} secondes...")
    time.sleep(delay)
    delay = min(delay * 2, max_delay)

else:
    print("⚠️ Temps d’attente maximal atteint sans complétion du téléchargement.")


⏳ Vérification de l’état du téléchargement...
Tentative 1 → HTTP 200
Statut: INITIATED | Progression: 0%
⏳ Nouvelle vérification dans 5 secondes...
Tentative 2 → HTTP 200
Statut: INITIATED | Progression: 0%
⏳ Nouvelle vérification dans 10 secondes...
Tentative 3 → HTTP 200
Statut: INITIATED | Progression: 0%
⏳ Nouvelle vérification dans 20 secondes...
Tentative 4 → HTTP 200
Statut: INITIATED | Progression: 0%
⏳ Nouvelle vérification dans 40 secondes...
Tentative 5 → HTTP 200
Statut: INITIATED | Progression: 0%
⏳ Nouvelle vérification dans 80 secondes...
Tentative 6 → HTTP 200
Statut: COMPLETED | Progression: 100%
✅ Fichier prêt : AIS_VERSION_FINAL.parquet
⬇️ Téléchargement du fichier...
✅ Fichier téléchargé avec succès : /content/AIS_VERSION_FINAL.parquet


**Requête de download sur DAWEX du fichier de type parquet contenant les données RF**

In [ ]:
conn = http.client.HTTPSConnection(hostname)

path = f"/api/participant/organisations/{organisation_id}/acquiredDataProductFilesDownloadRequests"

payload = json.dumps({
    "acquiredDataProductFilesTransactionId": transaction_id_rf,
    "acquiredDataProductFileAccessIds": [file_access_id_rf]
})

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

conn.request("POST", path, body=payload, headers=headers)
res = conn.getresponse()
data = res.read()

print("Status:", res.status, res.reason)
print(data.decode("utf-8"))

# décoder et parser le JSON
response_json = json.loads(data.decode("utf-8"))

# stocker l'id dans une variable
if "id" in response_json:
    download_request_id = response_json["id"]
    print("Download request ID:", download_request_id)
else:
    print("Download request already exists or failed to create.")


Status: 201 Created
{"id":"6932b8e2c6578c1626d29b4a","organisationId":"6924c0947402bd34c25c988c","acquiredDataProductFilesTransactionId":"c52e85167dae44be9b944a8aa3162f3e","acquiredDataProductFileAccessIds":["ddb2b055350742d0bb07db0aabcf409f"],"percentageCompletion":0,"downloadStatus":"INITIATED","downloadFilename":"RF_VERSION_FINAL.parquet","createdDate":"2025-12-05T10:50:10.589628884Z"}
Download request ID: 6932b8e2c6578c1626d29b4a


**Download sur DAWEX du fichier de type parquet contenant les données RF**

In [ ]:
# 1. Polling pour vérifier que le téléchargement est prêt
# -----------------------------
status_url = f"https://{hostname}/api/participant/organisations/{organisation_id}/acquiredDataProductFilesDownloadRequests/{download_request_id}"
headers    = {"Authorization": f"Bearer {access_token}"}

delay = 5
max_delay = 120
max_attempts = 20

print("⏳ Vérification de l’état du téléchargement...")
for attempt in range(1, max_attempts+1):
    resp = requests.get(status_url, headers=headers)
    resp.raise_for_status()
    data = resp.json()
    status = data.get("downloadStatus")
    percent = data.get("percentageCompletion", 0)
    print(f"Tentative {attempt} — statut : {status} — {percent}%")

    if status == "COMPLETED":
        print("✅ Le fichier est prêt au téléchargement.")
        filename = data.get("downloadFilename", "download_file")
        break
    elif status in ("ERROR", "FAILED"):
        raise RuntimeError(f"Échec du téléchargement — statut : {status}")

    time.sleep(delay)
    delay = min(delay*2, max_delay)
else:
    raise RuntimeError("⚠️ Délai maximal atteint sans que le téléchargement soit prêt.")

# -----------------------------
# 2. Téléchargement du fichier
# -----------------------------
content_url = f"{status_url}/content"
output_path = os.path.join("/content", filename)
print("⬇️ Téléchargement vers :", output_path)

with requests.get(content_url, headers=headers, stream=True) as r:
    r.raise_for_status()
    with open(output_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print("🎉 Téléchargement terminé :", output_path)


⏳ Vérification de l’état du téléchargement...
Tentative 1 — statut : INITIATED — 0%
Tentative 2 — statut : COMPLETED — 100%
✅ Le fichier est prêt au téléchargement.
⬇️ Téléchargement vers : /content/RF_VERSION_FINAL.parquet
🎉 Téléchargement terminé : /content/RF_VERSION_FINAL.parquet


In [ ]:
conn = http.client.HTTPSConnection(hostname)
file_access_id_corr = "41a0767697304b6ca351a9cf65247288"
path = f"/api/participant/organisations/{organisation_id}/acquiredDataProductFilesDownloadRequests"

payload = json.dumps({
    "acquiredDataProductFilesTransactionId": transaction_id_rf,
    "acquiredDataProductFileAccessIds": [file_access_id_corr]
})

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

conn.request("POST", path, body=payload, headers=headers)
res = conn.getresponse()
data = res.read()

print("Status:", res.status, res.reason)
print(data.decode("utf-8"))

# décoder et parser le JSON
response_json = json.loads(data.decode("utf-8"))

# stocker l'id dans une variable
if "id" in response_json:
    download_request_id = response_json["id"]
    print("Download request ID:", download_request_id)
else:
    print("Download request already exists or failed to create.")


Status: 201 Created
{"id":"6932b8e9fd991b722bf3302a","organisationId":"6924c0947402bd34c25c988c","acquiredDataProductFilesTransactionId":"c52e85167dae44be9b944a8aa3162f3e","acquiredDataProductFileAccessIds":["41a0767697304b6ca351a9cf65247288"],"percentageCompletion":0,"downloadStatus":"INITIATED","downloadFilename":"rf_correlation.parquet","createdDate":"2025-12-05T10:50:17.904344535Z"}
Download request ID: 6932b8e9fd991b722bf3302a


**Requête de download sur DAWEX du fichier de type parquet contenant les données éphémérides**

In [ ]:
conn = http.client.HTTPSConnection(hostname)

path = f"/api/participant/organisations/{organisation_id}/acquiredDataProductFilesDownloadRequests"

payload = json.dumps({
    "acquiredDataProductFilesTransactionId": transaction_id_eph,
    "acquiredDataProductFileAccessIds": [file_access_id_eph]
})

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

conn.request("POST", path, body=payload, headers=headers)
res = conn.getresponse()
data = res.read()

print("Status:", res.status, res.reason)
print(data.decode("utf-8"))

# décoder et parser le JSON
response_json = json.loads(data.decode("utf-8"))

# stocker l'id dans une variable
if "id" in response_json:
    download_request_id = response_json["id"]
    print("Download request ID:", download_request_id)
else:
    print("Download request already exists or failed to create.")

Status: 201 Created
{"id":"6932b8ebfd991b722bf3302b","organisationId":"6924c0947402bd34c25c988c","acquiredDataProductFilesTransactionId":"96f64cb4e0d9405e8542654259a30842","acquiredDataProductFileAccessIds":["13558391ceb340938ebddf50cee1e000"],"percentageCompletion":0,"downloadStatus":"INITIATED","downloadFilename":"ephemerides_Japetus_Mediterranee.parquet","createdDate":"2025-12-05T10:50:19.752046943Z"}
Download request ID: 6932b8ebfd991b722bf3302b


**Download sur DAWEX du fichier de type parquet contenant les données éphémérides**

In [ ]:
# === Endpoints et headers ===
status_url = f"https://{hostname}/api/participant/organisations/{organisation_id}/acquiredDataProductFilesDownloadRequests/{download_request_id}"
content_url = f"{status_url}/content"
headers = {"Authorization": f"Bearer {access_token}"}

# === Paramètres de backoff ===
delay = 5        # Première attente en secondes
max_delay = 120  # Attente max
max_attempts = 20

print("⏳ Vérification de l’état du téléchargement...")

for attempt in range(1, max_attempts + 1):
    response = requests.get(status_url, headers=headers)

    print(f"Tentative {attempt} → HTTP {response.status_code}")

    if response.status_code != 200:
        print("❌ Erreur:", response.text)
        break

    data = response.json()
    status = data.get("downloadStatus")
    percent = data.get("percentageCompletion", 0)
    filename = data.get("downloadFilename", "downloaded_file")

    print(f"Statut: {status} | Progression: {percent}%")

    if status == "COMPLETED":
        print("✅ Fichier prêt :", filename)
        output_path = os.path.join("/content", filename)

        # Télécharger le fichier
        print("⬇️ Téléchargement du fichier...")
        with requests.get(content_url, headers=headers, stream=True) as r:
            r.raise_for_status()
            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

        print(f"✅ Fichier téléchargé avec succès : {output_path}")
        break

    elif status in ("FAILED", "ERROR"):
        print("❌ Le téléchargement a échoué.")
        break

    print(f"⏳ Nouvelle vérification dans {delay} secondes...")
    time.sleep(delay)
    delay = min(delay * 2, max_delay)

else:
    print("⚠️ Temps d’attente maximal atteint sans complétion du téléchargement.")


⏳ Vérification de l’état du téléchargement...
Tentative 1 → HTTP 200
Statut: COMPLETED | Progression: 100%
✅ Fichier prêt : ephemerides_Japetus_Mediterranee.parquet
⬇️ Téléchargement du fichier...
✅ Fichier téléchargé avec succès : /content/ephemerides_Japetus_Mediterranee.parquet


**Imports des modules**

In [ ]:


# 🗂️ Imports de base
print("📦 Chargement des bibliothèques...")

from collections import defaultdict, deque
from datetime import datetime, timedelta
from folium import plugins
from geopy.exc import GeocoderTimedOut
from geopy.geocoders import Nominatim
from google.colab import output
from html import escape
from IPython.display import clear_output, display, HTML
from ipywidgets import fixed, interact, interact_manual, interactive



# Autres imports
import time
import asyncio
import datetime as dt
import difflib
import folium
import ipywidgets as widgets
import math
import nest_asyncio
import plotly.express as px
import plotly.graph_objects as go
import re
import sys
import warnings
import functools
import hashlib
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

nest_asyncio.apply()

print("✅ Imports terminés ")

📦 Chargement des bibliothèques...
✅ Imports terminés 


In [ ]:
#Chargement du contenu des fichiers parquets dans des variables pandas dataframe

df_ais_global=pd.read_parquet("/content/AIS_VERSION_FINAL.parquet")
df_rf_global=pd.read_parquet("/content/RF_VERSION_FINAL.parquet")
df_ephemeris_global = pd.read_parquet("/content/ephemerides_Japetus_Mediterranee.parquet")
df_corr_global=pd.read_parquet("/content/rf_correlation.parquet")


# --- MID → Pays/Territoire ---

MID_TO_COUNTRY = {
    # Europe (2xx)
    201: "Albania",
    202: "Andorra",
    203: "Austria",
    204: "Portugal - Azores",
    205: "Belgium",
    206: "Belarus",
    207: "Bulgaria",
    208: "Vatican City State",
    209: "Cyprus",
    210: "Cyprus",
    211: "Germany",
    212: "Cyprus",
    213: "Georgia",
    214: "Moldova",
    215: "Malta",
    216: "Armenia",
    218: "Germany",
    219: "Denmark",
    220: "Denmark",
    224: "Spain",
    225: "Spain",
    226: "France",
    227: "France",
    228: "France",
    229: "Malta",
    230: "Finland",
    231: "Faroe Islands",
    232: "United Kingdom",
    233: "United Kingdom",
    234: "United Kingdom",
    235: "United Kingdom",
    236: "Gibraltar",
    237: "Greece",
    238: "Croatia",
    239: "Greece",
    240: "Greece",
    241: "Greece",
    242: "Morocco",
    243: "Hungary",
    244: "Netherlands",
    245: "Netherlands",
    246: "Netherlands",
    247: "Italy",
    248: "Malta",
    249: "Malta",
    250: "Ireland",
    251: "Iceland",
    252: "Liechtenstein",
    253: "Luxembourg",
    254: "Monaco",
    255: "Portugal - Madeira",
    256: "Malta",
    257: "Norway",
    258: "Norway",
    259: "Norway",
    261: "Poland",
    262: "Montenegro",
    263: "Portugal",
    264: "Romania",
    265: "Sweden",
    266: "Sweden",
    267: "Slovak Republic",
    268: "San Marino",
    269: "Switzerland",
    270: "Czech Republic",
    271: "Turkey",
    272: "Ukraine",
    273: "Russian Federation",

    # Amérique du Nord & Caraïbes (3xx) – principaux
    303: "United States (Alaska)",
    304: "Antigua and Barbuda",
    305: "Antigua and Barbuda",
    306: "Caribbean Netherlands (NL Antilles)",
    307: "Aruba",
    308: "Bahamas",
    309: "Bahamas",
    310: "Bermuda",
    311: "Bahamas",
    312: "Belize",
    314: "Barbados",
    316: "Canada",
    319: "Cayman Islands",
    321: "Costa Rica",
    325: "Dominican Republic",
    330: "Grenada",
    331: "Greenland",
    332: "Guatemala",
    334: "Honduras",
    336: "Haiti",
    338: "United States",
    339: "Jamaica",
    345: "Mexico",
    348: "Saint Kitts and Nevis",
    350: "Saint Lucia",
    351: "Panama",
    354: "Saint Vincent and the Grenadines",
    356: "Trinidad and Tobago",
    358: "Saint Pierre and Miquelon (FR)",
    359: "El Salvador",
    362: "Costa Rica (legacy allocations)",
    364: "Dominica",
    366: "United States",
    367: "United States",
    368: "United States",
    369: "United States",

    # Sélection Asie / Océanie (4xx–5xx)
    401: "Afghanistan",
    405: "China (PRC)",
    408: "Korea (DPRK)",
    410: "Bhutan",
    412: "China (PRC)",
    413: "China (PRC)",
    414: "China (PRC)",
    416: "Taiwan",
    419: "India",
    422: "Iran (Islamic Republic of)",
    423: "Pakistan",
    425: "Iraq",
    428: "Israel",
    431: "Japan",
    432: "Japan",
    438: "Jordan",
    440: "Korea (Republic of)",
    441: "Korea (Republic of)",
    445: "Sri Lanka",
    450: "Korea (Republic of)",
    453: "Lebanon",
    455: "Maldives",
    457: "Mongolia",
    459: "Nepal",
    466: "Hong Kong (China)",
    470: "Saudi Arabia",
    472: "United Arab Emirates",
    474: "Qatar",
    475: "Syrian Arab Republic",
    477: "Hong Kong (China)",
    478: "Yemen",
    503: "Australia",
    508: "Fiji",
    510: "Micronesia (Federated States of)",
    514: "Papua New Guinea",
    515: "Philippines",
    520: "Solomon Islands",
    523: "Tonga",
    525: "Indonesia",
    529: "Kiribati",
    533: "Malaysia",
    536: "Nauru",
    540: "New Zealand",
    544: "Samoa",
    546: "New Caledonia (France)",
    548: "Papua New Guinea",
    553: "Vanuatu",
    559: "American Samoa (USA)",

    # Afrique & Moyen-Orient (6xx) – principaux
    601: "South Africa",
    603: "Algeria",
    605: "Ethiopia",
    607: "Mauritius (legacy/alt allocations may be 611)",
    608: "Ghana",
    609: "Nigeria",
    610: "Benin",
    611: "Mauritius",
    612: "Mozambique",
    613: "Liberia",
    615: "Mauritania",
    616: "Morocco (alt region)",
    619: "Côte d'Ivoire",
    620: "Senegal",
    622: "Egypt",
    624: "Eritrea",
    626: "Madagascar",
    630: "Equatorial Guinea",
    631: "Guinea",
    632: "Burkina Faso",
    633: "Gabon",
    636: "Tunisia",
    642: "Egypt (alt region)",
    647: "Tanzania",
    657: "Seychelles",
    660: "Comoros",
    664: "Mozambique (alt region)",

    # Amérique du Sud (7xx) – principaux
    701: "Argentina (legacy/alt)",
    710: "Brazil",
    720: "Chile",
    725: "Argentina",
    730: "Colombia",
    735: "Peru",
    740: "Suriname",
    755: "Uruguay",
    775: "Venezuela",
}

# Ajout de la colonne Pays en extrayant le MID du MMSI
print("🔄 Ajout de la colonne Pays...")
df_ais_global['Pays'] = df_ais_global['Mmsi'].astype(str).str[:3].astype(int).map(MID_TO_COUNTRY).fillna('Unknown')

# Affichage des informations sur le DataFrame chargé
print("✅ Fichier parquet chargé avec succès dans df_ais_global")

🔄 Ajout de la colonne Pays...
✅ Fichier parquet chargé avec succès dans df_ais_global


In [ ]:
import pandas as pd

# À adapter avec les VRAIS noms de fichiers si besoin
files = {
    "AIS": "AIS_VERSION_FINAL.parquet",
    "RF": "RF_VERSION_FINAL.parquet",
    "EPHEMERIDES": "ephemerides_Japetus_Mediterranee.parquet",
}

for name, path in files.items():
    print("=" * 100)
    print(f"{name}  |  fichier : {path}")
    print("-" * 100)

    # Lecture parquet
    df = pd.read_parquet(path)

    # Taille du jeu de données
    print(f"\nShape : {df.shape[0]} lignes, {df.shape[1]} colonnes")

    # Colonnes présentes
    print("\nColonnes :")
    print(list(df.columns))

    # Séparation numériques / catégorielles
    num_cols = df.select_dtypes(include="number").columns.tolist()
    cat_cols = df.select_dtypes(exclude="number").columns.tolist()

    # Ranges des variables numériques
    if num_cols:
        print("\nVariables numériques – min / max :")
        display(
            df[num_cols].agg(["min", "max"]).T.rename(columns={"min": "Min", "max": "Max"})
        )
    else:
        print("\nAucune variable numérique détectée.")

    # Valeurs uniques des variables catégorielles
    if cat_cols:
        print("\nVariables catégorielles – valeurs uniques :")
        for col in cat_cols:
            uniques = df[col].dropna().unique()
            if len(uniques) > 30:
                print(f" - {col}: {len(uniques)} valeurs uniques, premières : {uniques[:30]}")
            else:
                print(f" - {col}: {uniques}")
    else:
        print("\nAucune variable catégorielle détectée.")

AIS  |  fichier : AIS_VERSION_FINAL.parquet
----------------------------------------------------------------------------------------------------

Shape : 17170692 lignes, 24 colonnes

Colonnes :
['Mmsi', 'datetime', 'Position_LatitudeDeg', 'Position_LongitudeDeg', 'SogKt', 'Imo', 'Name', 'DimensionAm', 'DimensionBm', 'DimensionCm', 'DimensionDm', 'SourceId', 'NavigationalStatus', 'CogDegrees', 'Destination', 'Eta', 'SpecialManoeuvreIndicator', 'SafetyText', 'ShipTypeLabel', 'NavigationalStatusLabel', 'SourceLabel', 'SpecialManoeuvreLabel', 'Country', 'ShipTypeNormalized']

Variables numériques – min / max :


,Min,Max
Mmsi,2.129930e+06,9.942811e+08
Position_LatitudeDeg,-8.548333e+01,8.886000e+01
Position_LongitudeDeg,-1.804800e+02,1.799523e+02
SogKt,0.000000e+00,1.023000e+02
Imo,0.000000e+00,1.073742e+09
DimensionAm,1.000000e+00,5.100000e+02
DimensionBm,1.000000e+00,5.080000e+02
DimensionCm,1.000000e+00,6.000000e+01
DimensionDm,1.000000e+00,6.300000e+01
SourceId,1.000000e+00,1.000000e+00



Variables catégorielles – valeurs uniques :
 - datetime: 405809 valeurs uniques, premières : <DatetimeArray>
['2024-12-12 14:24:09+00:00', '2024-12-11 06:35:21+00:00',
 '2024-12-11 14:08:14+00:00', '2024-12-12 08:58:57+00:00',
 '2024-12-12 09:14:36+00:00', '2024-12-13 08:46:06+00:00',
 '2024-12-13 12:58:02+00:00', '2024-12-15 14:49:38+00:00',
 '2024-12-16 13:46:03+00:00', '2024-12-16 16:48:06+00:00',
 '2024-12-17 16:22:07+00:00', '2024-12-17 21:49:57+00:00',
 '2024-12-11 14:09:23+00:00', '2024-12-16 19:05:28+00:00',
 '2024-12-16 21:01:20+00:00', '2024-12-16 19:06:08+00:00',
 '2024-12-16 21:01:32+00:00', '2024-12-11 09:29:02+00:00',
 '2024-12-11 09:39:17+00:00', '2024-12-11 09:49:30+00:00',
 '2024-12-11 10:04:47+00:00', '2024-12-11 10:14:57+00:00',
 '2024-12-11 10:25:12+00:00', '2024-12-11 10:35:21+00:00',
 '2024-12-11 10:45:31+00:00', '2024-12-11 11:05:55+00:00',
 '2024-12-15 08:51:09+00:00', '2024-12-15 09:01:18+00:00',
 '2024-12-15 09:11:27+00:00', '2024-12-15 09:21:28+00:00']
Lengt

,Min,Max
latitude_deg,3.152776e+01,3.695511e+01
longitude_deg,2.920789e+01,3.625385e+01
RF_Frequency_MHz,3.028800e+03,9.425700e+03
pulses_duration_ns,6.490000e+01,1.280400e+03
pulses_repetition_frequency_Hz,2.156000e+02,9.475300e+03
waveform,1.699847e+06,3.272908e+06



Variables catégorielles – valeurs uniques :
 - timestamp_utc: <DatetimeArray>
['2024-12-11 10:22:31+00:00', '2024-12-11 13:27:46+00:00',
 '2024-12-11 21:59:45+00:00', '2024-12-12 09:06:09+00:00',
 '2024-12-12 22:06:33+00:00', '2024-12-13 13:06:07+00:00',
 '2024-12-14 00:23:25+00:00', '2024-12-14 08:21:48+00:00',
 '2024-12-15 00:33:42+00:00', '2024-12-15 08:00:57+00:00',
 '2024-12-15 09:36:38+00:00', '2024-12-16 00:05:31+00:00',
 '2024-12-16 13:26:42+00:00', '2024-12-17 13:34:09+00:00']
Length: 14, dtype: datetime64[ns, UTC]
 - accuracy_level: ['HIGH' 'MEDIUM' 'LOW']
EPHEMERIDES  |  fichier : ephemerides_Japetus_Mediterranee.parquet
----------------------------------------------------------------------------------------------------

Shape : 457 lignes, 11 colonnes

Colonnes :
['Source', 'Target', 'StartTime', 'StartLatitude', 'StartLongitude', 'EndTime', 'EndLatitude', 'EndLongitude', 'Duration', 'Distance_km', 'Average_Speedkmh']

Variables numériques – min / max :


,Min,Max
StartLatitude,27.620526,46.357986
StartLongitude,-4.606185,38.352476
EndLatitude,28.050719,46.445847
EndLongitude,-4.721965,38.628101
Duration,20.000000,270.000000
Distance_km,135.406347,1831.814773
Average_Speedkmh,24373.142400,25721.590086



Variables catégorielles – valeurs uniques :
 - Source: ['JAPETUS SSO_1' 'JAPETUS SSO_4' 'JAPETUS SSO_3' 'JAPETUS SSO_2'
 'JAPETUS Inclined_16' 'JAPETUS Inclined_4' 'JAPETUS Inclined_15'
 'JAPETUS Inclined_3' 'JAPETUS Inclined_14' 'JAPETUS Inclined_2'
 'JAPETUS Inclined_13' 'JAPETUS Inclined_1' 'JAPETUS Inclined_8'
 'JAPETUS Inclined_7' 'JAPETUS Inclined_6' 'JAPETUS Inclined_5']
 - Target: ['Mediterranee Ouest' 'Mediterranee Centre' 'Mediterranee Est']
 - StartTime: 439 valeurs uniques, premières : <DatetimeArray>
['2024-12-11 08:47:00', '2024-12-11 09:09:10', '2024-12-11 09:33:00',
 '2024-12-11 09:56:40', '2024-12-11 10:13:00', '2024-12-11 10:20:30',
 '2024-12-11 10:21:40', '2024-12-11 10:37:00', '2024-12-11 10:45:10',
 '2024-12-11 11:01:10', '2024-12-11 11:08:50', '2024-12-11 11:25:50',
 '2024-12-11 11:33:20', '2024-12-11 11:50:30', '2024-12-11 11:59:30',
 '2024-12-11 12:15:20', '2024-12-12 08:50:30', '2024-12-12 09:13:00',
 '2024-12-12 09:36:50', '2024-12-12 09:51:30', '2024-12-12 1

**Système de cache global pour optimiser les performances + Styles CSS harmonisés**

In [ ]:
class DataCache:
    """Cache intelligent pour les calculs coûteux"""

    def __init__(self):
        self.cache = {}
        self.cache_timestamps = {}
        self.cache_ttl_seconds = 300  # 5 minutes par défaut

    def _generate_key(self, *args, **kwargs):
        """Génère une clé de cache unique"""
        key_str = str(args) + str(sorted(kwargs.items()))
        return hashlib.md5(key_str.encode()).hexdigest()

    def get(self, key):
        """Récupère depuis le cache si valide"""
        if key in self.cache:
            timestamp = self.cache_timestamps.get(key)
            if timestamp and (datetime.now() - timestamp).seconds < self.cache_ttl_seconds:
                print(f"⚡ Cache HIT pour {key[:8]}...")
                return self.cache[key]
            else:
                print(f"⏰ Cache EXPIRED pour {key[:8]}...")
                del self.cache[key]
                del self.cache_timestamps[key]
        return None

    def set(self, key, value):
        """Stocke dans le cache"""
        self.cache[key] = value
        self.cache_timestamps[key] = datetime.now()
        print(f"💾 Cache SET pour {key[:8]}... ({len(self.cache)} entrées)")

    def clear(self):
        """Vide tout le cache"""
        count = len(self.cache)
        self.cache.clear()
        self.cache_timestamps.clear()
        print(f"🗑️ Cache vidé ({count} entrées supprimées)")

    def clear_old_entries(self, max_age_seconds=None):
        """Nettoie les entrées expirées"""
        if max_age_seconds is None:
            max_age_seconds = self.cache_ttl_seconds

        now = datetime.now()
        expired_keys = [
            key for key, timestamp in self.cache_timestamps.items()
            if (now - timestamp).seconds > max_age_seconds
        ]

        for key in expired_keys:
            del self.cache[key]
            del self.cache_timestamps[key]

        if expired_keys:
            print(f"🧹 {len(expired_keys)} entrées de cache expirées nettoyées")

# Initialisation du cache global
data_cache = DataCache()

def get_last_positions_cached(date_filter=None, vessel_type='Tous', country='Tous'):
    """
    Récupère les dernières positions avec cache intelligent

    Args:
        date_filter: Date spécifique ou None pour toutes les dates
        vessel_type: Type de navire ou 'Tous'
        country: Pays ou 'Tous'

    Returns:
        DataFrame avec les dernières positions par MMSI
    """
    # Génération de la clé de cache
    cache_key = data_cache._generate_key(
        'last_positions',
        date_filter=str(date_filter),
        vessel_type=vessel_type,
        country=country
    )

    # Vérification du cache
    cached_result = data_cache.get(cache_key)
    if cached_result is not None:
        return cached_result

    # Cache MISS - calcul
    print(f"🔄 Calcul des dernières positions...")
    start_time = time.time()

    if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
        print('❌ df_ais_global non disponible')
        return pd.DataFrame()

    # Filtrage par date si nécessaire
    if date_filter:
        if isinstance(date_filter, str):
            date_filter = pd.to_datetime(date_filter).date()

        if 'date_only' not in df_ais_global.columns:
            df_ais_global['date_only'] = pd.to_datetime(df_ais_global['datetime']).dt.date

        df_to_use = df_ais_global[df_ais_global['date_only'] == date_filter].copy()

        if df_to_use.empty:
            print(f'⚠️ Aucune donnée pour {date_filter}')
            return pd.DataFrame()
    else:
        df_to_use = df_ais_global

    # Calcul des dernières positions par MMSI
    df_last_positions = df_to_use.sort_values('datetime').groupby('Mmsi').tail(1)

    # Filtrage des positions valides
    df_valid = df_last_positions[
        (df_last_positions['Position_LatitudeDeg'].notna()) &
        (df_last_positions['Position_LongitudeDeg'].notna()) &
        (df_last_positions['Position_LatitudeDeg'] != 0) &
        (df_last_positions['Position_LongitudeDeg'] != 0)
    ].copy()

    # Appliquer les filtres de type et pays
    if vessel_type != 'Tous' and 'ShipTypeLabel' in df_valid.columns:
        df_valid = df_valid[df_valid['ShipTypeLabel'] == vessel_type]

    if country != 'Tous' and 'Pays' in df_valid.columns:
        df_valid = df_valid[df_valid['Pays'] == country]

    elapsed = time.time() - start_time
    print(f"✅ {len(df_valid)} positions calculées en {elapsed:.2f}s")

    # Mise en cache
    data_cache.set(cache_key, df_valid)

    return df_valid

# cache pour les positions les plus récentes
_latest_positions_cache = None
_latest_positions_hash = None

def get_latest_vessel_positions(force_refresh=False):
    """
    Retourne les dernières positions de chaque navire.
    Utilise un cache pour éviter les calculs répétés.

    Args:
        force_refresh: Force le recalcul même si le cache est valide

    Returns:
        DataFrame avec les dernières positions par MMSI
    """
    global _latest_positions_cache, _latest_positions_hash

    if 'df_ais_global' not in globals() or df_ais_global is None:
        return pd.DataFrame()

    # Calcul d'un hash des données pour détecter les changements
    current_hash = hash(tuple(df_ais_global.index)) if len(df_ais_global) < 100000 else id(df_ais_global)

    # Si le cache est valide, le retourner
    if not force_refresh and _latest_positions_cache is not None and current_hash == _latest_positions_hash:
        return _latest_positions_cache.copy()

    # Sinon, recalculer
    print("🔄 Calcul des dernières positions...")
    result = df_ais_global.sort_values('datetime').groupby('Mmsi').tail(1)

    # Mise en cache
    _latest_positions_cache = result
    _latest_positions_hash = current_hash

    return result.copy()


# css harmonisé pour toute l'application
MARITIME_CSS = {
    # Couleurs principales
    'primary_color': '#007bff',
    'success_color': '#28a745',
    'warning_color': '#ffc107',
    'danger_color': '#dc3545',
    'info_color': '#17a2b8',
    'dark_color': '#1a1a1a',
    'light_color': '#f8f9fa',

    # Styles de base
    'border_radius': '5px',
    'padding': '10px',
    'font_family': "'Segoe UI', Tahoma, Geneva, Verdana, sans-serif",
    'font_size_normal': '14px',
    'font_size_small': '12px',
    'font_size_large': '16px',

    # Styles spécifiques
    'box_shadow': '0 2px 4px rgba(0,0,0,0.1)',
    'transition': 'all 0.3s ease',
}

def get_styled_html(content, style_type='info', icon=''):
    """
    Génère du HTML avec style harmonisé

    Args:
        content: Contenu HTML ou texte
        style_type: 'info', 'success', 'warning', 'danger', 'neutral'
        icon: Emoji à afficher (optionnel)

    Returns:
        HTML stylé
    """
    style_map = {
        'info': {
            'bg': '#d1ecf1',
            'color': '#0c5460',
            'border': '#bee5eb'
        },
        'success': {
            'bg': '#d4edda',
            'color': '#155724',
            'border': '#c3e6cb'
        },
        'warning': {
            'bg': '#fff3cd',
            'color': '#856404',
            'border': '#ffeaa7'
        },
        'danger': {
            'bg': '#f8d7da',
            'color': '#721c24',
            'border': '#f5c6cb'
        },
        'neutral': {
            'bg': '#f8f9fa',
            'color': '#6c757d',
            'border': '#dee2e6'
        }
    }

    style = style_map.get(style_type, style_map['neutral'])

    return f"""
    <div style='
        padding: {MARITIME_CSS['padding']};
        background: {style['bg']};
        color: {style['color']};
        border: 1px solid {style['border']};
        border-radius: {MARITIME_CSS['border_radius']};
        font-family: {MARITIME_CSS['font_family']};
        font-size: {MARITIME_CSS['font_size_normal']};
        box-shadow: {MARITIME_CSS['box_shadow']};
        margin: 5px 0;
    '>
        {icon + ' ' if icon else ''}{content}
    </div>
    """

def get_popup_style():
    """Retourne le style CSS pour les popups de carte"""
    return f"""
    font-family: {MARITIME_CSS['font_family']};
    font-size: {MARITIME_CSS['font_size_small']};
    min-width: 250px;
    """

def get_button_style(button_type='primary'):
    """Retourne le style CSS pour les boutons"""
    color_map = {
        'primary': MARITIME_CSS['primary_color'],
        'success': MARITIME_CSS['success_color'],
        'warning': MARITIME_CSS['warning_color'],
        'danger': MARITIME_CSS['danger_color'],
    }

    bg_color = color_map.get(button_type, MARITIME_CSS['primary_color'])

    return f"""
    background: {bg_color};
    color: white;
    border: none;
    padding: 5px 10px;
    border-radius: {MARITIME_CSS['border_radius']};
    cursor: pointer;
    font-size: {MARITIME_CSS['font_size_small']};
    transition: {MARITIME_CSS['transition']};
    """

print("✅ Système de cache global initialisé")
print(f"⚙️ TTL du cache: {data_cache.cache_ttl_seconds}s")

✅ Système de cache global initialisé
⚙️ TTL du cache: 300s


**Fonctions de géolocalisation, zones maritimes et API OpenWeather**

In [ ]:
def get_weather_data(lat, lon, api_key):
    """
    Récupère les données météo pour une position donnée
    """
    try:
        url = f'{OPENWEATHER_BASE_URL}?lat={lat}&lon={lon}&appid={api_key}&units=metric&lang=fr'
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()
        else:
            return {'error': f'Erreur API: {response.status_code}'}
    except Exception as e:
        return {'error': f'Erreur de connexion: {str(e)}'}

def format_weather_info(weather_data):
    """
    Formate les informations météo pour l'affichage
    """
    if 'error' in weather_data:
        return f"❌ {weather_data['error']}"
    try:
        temp = weather_data['main']['temp']
        feels_like = weather_data['main']['feels_like']
        humidity = weather_data['main']['humidity']
        pressure = weather_data['main']['pressure']
        description = weather_data['weather'][0]['description'].title()
        wind_speed = weather_data['wind']['speed']
        wind_deg = weather_data['wind'].get('deg', 0)
        weather_info = f'\n        🌡️ **Température**: {temp}°C (ressenti {feels_like}°C)\n        🌤️ **Conditions**: {description}\n        💨 **Vent**: {wind_speed} m/s, {wind_deg}°\n        💧 **Humidité**: {humidity}%\n        📊 **Pression**: {pressure} hPa\n        '
        return weather_info.strip()
    except KeyError as e:
        return f'❌ Données météo incomplètes: {str(e)}'

def get_location_info(lat, lon):
    """
    Récupère les informations de localisation avec zones maritimes précises
    """
    try:
        geolocator = Nominatim(user_agent='boat_tracker_app')
        location = geolocator.reverse(f'{lat}, {lon}', timeout=10, language='fr')
        if location:
            address = location.raw['address']
            country = translate_to_french(address.get('country', 'Inconnu'))
            city = translate_to_french(address.get('city') or address.get('town') or address.get('village') or 'Zone portuaire')
            state = translate_to_french(address.get('state', ''))
            water_body = get_water_body(lat, lon)
            return {'country': country, 'city': city, 'state': state, 'water_body': water_body, 'full_address': translate_to_french(location.address), 'maritime_status': 'Port/Côte'}
        else:
            maritime_zone = get_precise_maritime_zone(lat, lon)
            return {'country': maritime_zone['country'], 'city': maritime_zone['zone_type'], 'state': '', 'water_body': maritime_zone['water_body'], 'full_address': maritime_zone['description'], 'maritime_status': maritime_zone['status']}
    except GeocoderTimedOut:
        return {'country': 'Timeout', 'city': 'Erreur de géolocalisation', 'state': '', 'water_body': 'Inconnu', 'full_address': 'Erreur de connexion', 'maritime_status': 'Erreur'}
    except Exception as e:
        return {'country': 'Erreur', 'city': str(e), 'state': '', 'water_body': 'Inconnu', 'full_address': f'Erreur: {str(e)}', 'maritime_status': 'Erreur'}

def get_precise_maritime_zone(lat, lon):
    """
    Détermine la zone maritime précise selon le droit international
    """
    water_body = get_water_body(lat, lon)
    closest_country_info = find_closest_coastal_country(lat, lon)
    closest_country = closest_country_info['country']
    distance_nm = closest_country_info['distance_nm']
    if distance_nm <= 12:
        zone_type = 'Eaux territoriales'
        country = closest_country
        status = f'Territorial ({distance_nm:.1f} nm de la côte)'
        description = f'Eaux territoriales {closest_country} - {distance_nm:.1f} nm de la côte'
    elif distance_nm <= 24:
        zone_type = 'Zone contiguë'
        country = f'Zone contiguë {closest_country}'
        status = f'Contiguë ({distance_nm:.1f} nm de la côte)'
        description = f'Zone contiguë {closest_country} - {distance_nm:.1f} nm de la côte'
    elif distance_nm <= 200:
        zone_type = 'ZEE'
        country = f'ZEE {closest_country}'
        status = f'ZEE ({distance_nm:.1f} nm de la côte)'
        description = f'Zone Économique Exclusive {closest_country} - {distance_nm:.1f} nm de la côte'
    else:
        zone_type = 'Eaux internationales'
        country = 'Eaux internationales'
        status = f'International ({distance_nm:.1f} nm de la côte)'
        description = f'Eaux internationales - {distance_nm:.1f} nm de {closest_country}'
    return {'country': country, 'zone_type': zone_type, 'water_body': water_body, 'status': status, 'description': description, 'distance_nm': distance_nm, 'closest_country': closest_country}

def find_closest_coastal_country(lat, lon):
    """
    Trouve le pays côtier le plus proche et calcule la distance
    """
    coastal_references = {'France': [(43.2965, 5.3698), (43.7102, 7.262), (42.6976, 2.8954), (43.1258, 6.1266), (43.5528, 3.833)], 'Espagne': [(41.3851, 2.1734), (36.7201, -4.4203), (37.3886, -5.9823), (39.4699, 2.7581), (36.1408, -5.3536)], 'Italie': [(40.8518, 14.2681), (41.9028, 12.4964), (44.4056, 8.9463), (45.4408, 12.3155), (38.1157, 13.3615), (40.6401, 17.9443)], 'Grèce': [(37.9755, 23.7348), (40.6401, 22.9444), (35.5138, 24.018), (39.6243, 19.9217), (36.4414, 25.3656)], 'Turquie': [(41.0082, 28.9784), (36.8969, 30.7133), (38.4192, 27.1287), (36.2012, 36.161)], 'Chypre': [(35.1856, 33.3823), (35.0408, 33.9533)], 'Croatie': [(45.815, 15.9819), (42.6507, 18.0944), (43.5081, 16.4402)], 'Algérie': [(36.7538, 3.0588), (35.6976, -0.6337)], 'Tunisie': [(36.8065, 10.1815), (34.7406, 10.7603)], 'Maroc': [(35.7595, -5.834), (33.5731, -7.5898)], 'Égypte': [(31.2001, 29.9187), (27.2579, 33.8116)], 'Malte': [(35.8997, 14.5146)], 'Monaco': [(43.7384, 7.4246)]}
    min_distance = float('inf')
    closest_country = 'Inconnu'
    for country, points in coastal_references.items():
        for point_lat, point_lon in points:
            distance = haversine_nm(lat, lon, point_lat, point_lon)
            if distance < min_distance:
                min_distance = distance
                closest_country = country
    return {'country': closest_country, 'distance_nm': min_distance}

def haversine_nm(lat1, lon1, lat2, lon2):
    """
    Calcule la distance en milles nautiques entre deux points GPS
    """
    R = 3440.065
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.asin(math.sqrt(a))
    return R * c

def get_water_body(lat, lon):
    """
    Détermine la masse d'eau avec plus de précision
    """
    if 30 <= lat <= 46 and -6 <= lon <= 42:
        if 40 <= lat <= 46 and 3 <= lon <= 20:
            return 'Mer Méditerranée occidentale'
        elif 32 <= lat <= 42 and 15 <= lon <= 30:
            return 'Mer Méditerranée orientale'
        elif 35 <= lat <= 42 and 12 <= lon <= 20:
            return 'Mer Adriatique'
        elif 35 <= lat <= 40 and 19 <= lon <= 30:
            return 'Mer Ionienne'
        elif 35 <= lat <= 42 and 22 <= lon <= 30:
            return 'Mer Égée'
        else:
            return 'Mer Méditerranée'
    elif 40 <= lat <= 48 and 27 <= lon <= 42:
        return 'Mer Noire'
    elif 53 <= lat <= 70 and -5 <= lon <= 30:
        if lat >= 60:
            return 'Mer de Norvège'
        elif 54 <= lat <= 66 and 10 <= lon <= 30:
            return 'Mer Baltique'
        else:
            return 'Mer du Nord'
    elif -60 <= lat <= 70 and -80 <= lon <= 0:
        if 35 <= lat <= 70:
            return 'Océan Atlantique Nord'
        elif 0 <= lat <= 35:
            return 'Océan Atlantique tropical'
        else:
            return 'Océan Atlantique Sud'
    elif -60 <= lat <= 60 and 120 <= lon <= 180:
        return 'Océan Pacifique'
    elif -60 <= lat <= 30 and 20 <= lon <= 120:
        return 'Océan Indien'
    elif lat >= 70:
        return 'Océan Arctique'
    elif lat <= -60:
        return 'Océan Antarctique'
    else:
        return 'Zone maritime'

def translate_to_french(text):
    """
    Traduit les noms de lieux vers le français avec dictionnaire de correspondances
    """
    if not text or text in ['', 'Inconnu', 'En mer', 'Erreur']:
        return text
    translations = {'Ελλάς': 'Grèce', 'Ελλάδα': 'Grèce', 'Greece': 'Grèce', 'Κυψέλη': 'Kypseli', 'Αθήνα': 'Athènes', 'Θεσσαλονίκη': 'Thessalonique', 'Πειραιάς': 'Le Pirée', 'Πάτρα': 'Patras', 'Κρήτη': 'Crète', 'Σαντορίνη': 'Santorin', 'Μύκονος': 'Mykonos', 'Ρόδος': 'Rhodes', 'Κέρκυρα': 'Corfou', 'Επισκοπή': 'Episkopi', 'Επίσκοπι': 'Episkopi', 'Cyprus': 'Chypre', 'Κύπρος': 'Chypre', 'Kıbrıs': 'Chypre', 'Episkopi': 'Episkopi', 'Επισκοπή': 'Episkopi', 'España': 'Espagne', 'Barcelona': 'Barcelone', 'Sevilla': 'Séville', 'Málaga': 'Malaga', 'Valencia': 'Valence', 'Zaragoza': 'Saragosse', 'Italia': 'Italie', 'Roma': 'Rome', 'Milano': 'Milan', 'Napoli': 'Naples', 'Venezia': 'Venise', 'Firenze': 'Florence', 'Genova': 'Gênes', 'Palermo': 'Palerme', 'Türkiye': 'Turquie', 'Turkey': 'Turquie', 'İstanbul': 'Istanbul', 'İzmir': 'Izmir', 'Antalya': 'Antalya', 'المغرب': 'Maroc', 'Morocco': 'Maroc', 'مصر': 'Égypte', 'Egypt': 'Égypte', 'الجزائر': 'Algérie', 'Algeria': 'Algérie', 'تونس': 'Tunisie', 'Tunisia': 'Tunisie', 'Malta': 'Malte', 'Croatia': 'Croatie', 'Hrvatska': 'Croatie', 'Albania': 'Albanie', 'Shqipëria': 'Albanie', 'United States': 'États-Unis', 'United Kingdom': 'Royaume-Uni', 'Germany': 'Allemagne', 'Netherlands': 'Pays-Bas'}
    if text in translations:
        return translations[text]
    result = text
    for original, french in translations.items():
        if original in result:
            result = result.replace(original, french)
    return result

FLAG_EMOJIS = {'France': '🇫🇷', 'Royaume-Uni': '🇬🇧', 'États-Unis': '🇺🇸', 'Japon': '🇯🇵', 'Italie': '🇮🇹', 'Suède': '🇸🇪', 'Finlande': '🇫🇮', 'Bahamas': '🇧🇸', 'Russie': '🇷🇺', 'Pays-Bas': '🇳🇱', 'Hong Kong': '🇭🇰', 'Chine': '🇨🇳', 'Islande': '🇮🇸', 'Belgique': '🇧🇪', 'Espagne': '🇪🇸', 'Allemagne': '🇩🇪', 'Portugal': '🇵🇹', 'Norvège': '🇳🇴', 'Danemark': '🇩🇰', 'Grèce': '🇬🇷', 'Turquie': '🇹🇷', 'Maroc': '🇲🇦', 'Algérie': '🇩🇿', 'Tunisie': '🇹🇳', 'Égypte': '🇪🇬', 'Canada': '🇨🇦', 'Mexique': '🇲🇽', 'Brésil': '🇧🇷', 'Argentine': '🇦🇷', 'Chili': '🇨🇱', 'Australie': '🇦🇺', 'Nouvelle-Zélande': '🇳🇿', 'Afrique du Sud': '🇿🇦', 'Inde': '🇮🇳', 'Corée du Sud': '🇰🇷', 'Thaïlande': '🇹🇭', 'Indonésie': '🇮🇩', 'Philippines': '🇵🇭', 'Singapour': '🇸🇬', 'Malaisie': '🇲🇾', 'Vietnam': '🇻🇳', 'Israël': '🇮🇱', 'Émirats arabes unis': '🇦🇪', 'Arabie saoudite': '🇸🇦', 'United Kingdom': '🇬🇧', 'United States': '🇺🇸', 'Japan': '🇯🇵', 'Italy': '🇮🇹', 'Sweden': '🇸🇪', 'Finland': '🇫🇮', 'Russia': '🇷🇺', 'Netherlands': '🇳🇱', 'China': '🇨🇳', 'Iceland': '🇮🇸', 'Belgium': '🇧🇪', 'Spain': '🇪🇸', 'Germany': '🇩🇪', 'Norway': '🇳🇴', 'Denmark': '🇩🇰', 'Greece': '🇬🇷', 'Turkey': '🇹🇷', 'Morocco': '🇲🇦', 'Algeria': '🇩🇿', 'Tunisia': '🇹🇳', 'Egypt': '🇪🇬', 'Mexico': '🇲🇽', 'Brazil': '🇧🇷', 'Argentina': '🇦🇷', 'Chile': '🇨🇱', 'Australia': '🇦🇺', 'New Zealand': '🇳🇿', 'South Africa': '🇿🇦', 'India': '🇮🇳', 'South Korea': '🇰🇷', 'Thailand': '🇹🇭', 'Indonesia': '🇮🇩', 'Philippines': '🇵🇭', 'Singapore': '🇸🇬', 'Malaysia': '🇲🇾', 'Vietnam': '🇻🇳', 'Israel': '🇮🇱', 'United Arab Emirates': '🇦🇪', 'Saudi Arabia': '🇸🇦', 'En mer': '🌊', 'Inconnu': '🏳️', 'Erreur': '❌'}

def get_flag_emoji(country):
    """Retourne l'emoji du drapeau pour un pays donné"""
    return FLAG_EMOJIS.get(country, '🏳️')

def get_country_flag_smart(country_name):
    """Détecte intelligemment le drapeau d'un pays même avec des noms multilingues"""
    if not country_name or country_name in ['Inconnu', 'En mer', 'Erreur']:
        return get_flag_emoji(country_name)
    country_lower = country_name.lower()
    country_mappings = {'france': '🇫🇷', 'grèce': '🇬🇷', 'greece': '🇬🇷', 'ελλάς': '🇬🇷', 'ελλάδα': '🇬🇷', 'espagne': '🇪🇸', 'spain': '🇪🇸', 'españa': '🇪🇸', 'italie': '🇮🇹', 'italy': '🇮🇹', 'italia': '🇮🇹', 'turquie': '🇹🇷', 'turkey': '🇹🇷', 'türkiye': '🇹🇷', 'chypre': '🇨🇾', 'cyprus': '🇨🇾', 'κύπρος': '🇨🇾', 'croatie': '🇭🇷', 'croatia': '🇭🇷', 'hrvatska': '🇭🇷', 'algérie': '🇩🇿', 'algeria': '🇩🇿', 'الجزائر': '🇩🇿', 'tunisie': '🇹🇳', 'tunisia': '🇹🇳', 'تونس': '🇹🇳', 'maroc': '🇲🇦', 'morocco': '🇲🇦', 'المغرب': '🇲🇦', 'égypte': '🇪🇬', 'egypt': '🇪🇬', 'مصر': '🇪🇬', 'malte': '🇲🇹', 'malta': '🇲🇹', 'albanie': '🇦🇱', 'albania': '🇦🇱', 'monténégro': '🇲🇪', 'montenegro': '🇲🇪', 'eaux internationales': '🌊', 'international': '🌊', 'zee': '🌊', 'zone contiguë': '🌊', 'eaux territoriales': '🌊'}
    for key, flag in country_mappings.items():
        if key in country_lower:
            return flag
    return get_flag_emoji(country_name)

def format_location_info(location_data):
    """Formate les informations de localisation avec zones maritimes précises"""
    country = location_data['country']
    flag = get_country_flag_smart(country)
    if location_data.get('maritime_status') == 'Port/Côte':
        location_info = f"\n        {flag} **Pays**: {country}\n        🏙️ **Ville**: {location_data['city']}\n        🌊 **Zone maritime**: {location_data['water_body']}\n        📍 **Adresse complète**: {location_data['full_address']}\n        "
    else:
        status = location_data.get('maritime_status', '')
        location_info = f"\n        {flag} **Zone**: {country}\n        🌊 **Statut**: {location_data['city']} ({status})\n        🌊 **Masse d'eau**: {location_data['water_body']}\n        📍 **Position**: {location_data['full_address']}\n        "
    return location_info.strip()

**Implémentation des fonctions de chatbot avec OpenRouter et fallback local**

In [ ]:
def query_openrouter(question, context_data, api_key):
    """
    Interroge l'API OpenRouter avec les données contextuelles
    """
    try:
        headers = {'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json', 'HTTP-Referer': 'https://colab.research.google.com', 'X-Title': 'Boat Tracker App'}
        context = f'\n        Tu es un assistant naval expert .\n        Voici les données des bateaux disponibles:\n        {df_ais_global}\n\n        Réponds de manière précise et utile aux questions sur ces bateaux.\n        '
        payload = {'model': "x-ai/grok-4-fast", 'messages': [{'role': 'system', 'content': context}, {'role': 'user', 'content': question}], 'max_tokens': 500, 'temperature': 0.7}
        response = requests.post(OPENROUTER_BASE_URL, headers=headers, json=payload, timeout=30)
        if response.status_code == 200:
            result = response.json()
            return result['choices'][0]['message']['content']
        elif response.status_code == 402:
            return '❌ Erreur 402: Crédit insuffisant sur votre compte OpenRouter. Vérifiez votre solde sur https://openrouter.ai/credits'
        elif response.status_code == 401:
            return '❌ Erreur 401: Clé API OpenRouter invalide. Vérifiez votre clé sur https://openrouter.ai/keys'
        elif response.status_code == 429:
            return '❌ Erreur 429: Limite de taux dépassée. Attendez quelques minutes avant de réessayer.'
        else:
            return f'❌ Erreur API OpenRouter: {response.status_code} - {response.text}'
    except Exception as e:
        return f'❌ Erreur de connexion au chatbot: {str(e)}'

def local_chatbot(question, context_data):
    """
    Chatbot local simple basé sur des règles (alternative à OpenRouter)
    """
    question_lower = question.lower()
    if any((word in question_lower for word in ['combien', 'nombre', 'total'])):
        boat_count = context_data.count('⚪') + context_data.count('🔴')
        return f'Il y a {boat_count} bateaux dans la flotte actuellement trackés.'
    if any((word in question_lower for word in ['type', 'catégorie', 'sorte'])):
        types = ['Cargo', 'Research', 'Fishing', 'Container', 'Cruise', 'Tanker', 'Naval', 'Icebreaker', 'Tourism', 'Yacht']
        return f"Les types de bateaux dans la flotte sont : {', '.join(types)}"
    if any((word in question_lower for word in ['pays', 'pavillon', 'nationalité', 'drapeau'])):
        countries = ['France', 'United Kingdom', 'United States', 'Japan', 'Italy', 'Sweden', 'Finland', 'Bahamas', 'Russia', 'Netherlands', 'Hong Kong', 'China', 'Iceland', 'Belgium']
        return f"Les pavillons représentés sont : {', '.join(set(countries))}"
    if '🔴' in context_data:
        selected_line = [line for line in context_data.split('\n') if '🔴' in line]
        if selected_line:
            boat_info = selected_line[0].replace('🔴', '').strip()
            if any((word in question_lower for word in ['sélectionné', 'choisi', 'actuel', 'rouge'])):
                return f'Le bateau actuellement sélectionné est : {boat_info}'
    if any((word in question_lower for word in ['où', 'position', 'localisation', 'coordonnées'])):
        return 'Les bateaux sont répartis dans le monde entier. Sélectionnez un bateau pour voir sa position exacte et les informations de localisation.'
    if any((word in question_lower for word in ['météo', 'temps', 'température', 'vent'])):
        return 'Les informations météo sont disponibles pour chaque bateau sélectionné. Choisissez un bateau pour voir les conditions météorologiques en temps réel.'
    return f"Je suis un assistant maritime local. Voici ce que je peux vous dire :\n\n{context_data}\n\nPosez-moi des questions sur le nombre de bateaux, leurs types, leurs pavillons, ou sélectionnez un bateau pour plus d'informations."

def create_context_from_boats(df, selected_boat=None):
    """
    Crée un contexte textuel à partir des données des bateaux
    """
    context = 'Données des bateaux:\n'
    for _, boat in df.iterrows():
        marker = '🔴' if selected_boat and boat['boat_name'] == selected_boat else '⚪'
        context += f"{marker} {boat['boat_name']}: {boat['boat_type']}, {boat['flag_country']}, Position: ({boat['latitude']:.4f}, {boat['longitude']:.4f})\n"
    return context

**Fonction pour ajouter les couches maritimes à la carte Folium**

In [ ]:
def add_maritime_layers(folium_map):
    """
    Ajoute TOUTES les couches maritimes avec organisation par catégories
    Les préfixes permettent un regroupement visuel dans le Layer Control

    Catégories:
    📍 BASE - Fonds de carte
    🧭 NAV - Navigation et cartes nautiques
    🌊 BATH - Bathymétrie
    🔌 CABLES - Infrastructures sous-marines
    ⚓ INFRA - Ports et mouillages
    🚢 DENSITY - Densité et trafic
    🛤️ ROUTES - Routes maritimes
    🛡️ ZONES - Zones juridiques et protégées
    🌡️ METEO - Météo et océanographie
    🪨 GEO - Géologie marine

    Args:
        folium_map: objet folium.Map
    """

    # fond de carte

    folium.TileLayer(
        tiles='https://services.arcgisonline.com/arcgis/rest/services/Ocean/World_Ocean_Base/MapServer/tile/{z}/{y}/{x}',
        name='📍 BASE - Ocean (Esri)',
        attr='Esri, GEBCO, NOAA',
        overlay=False,
        control=True,
        show=False
    ).add_to(folium_map)

    folium.TileLayer(
        tiles='https://services.arcgisonline.com/arcgis/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        name='📍 BASE - Satellite (Esri)',
        attr='Esri',
        overlay=False,
        control=True,
        show=True
    ).add_to(folium_map)

    # navigation

    folium.TileLayer(
        tiles='https://services.arcgisonline.com/ArcGIS/rest/services/Specialty/World_Navigation_Charts/MapServer/tile/{z}/{y}/{x}',
        name='🧭 NAV - Cartes Nautiques (Esri)',
        attr='Esri',
        overlay=True,
        control=True,
        opacity=0.7,
        show=False
    ).add_to(folium_map)

    try:
        folium.TileLayer(
            tiles='https://tileservice.charts.noaa.gov/tiles/50000_1/{z}/{x}/{y}.png',
            name='🧭 NAV - Charts NOAA (US)',
            attr='NOAA',
            overlay=True,
            control=True,
            opacity=0.7,
            show=False
        ).add_to(folium_map)
    except:
        pass

    folium.TileLayer(
        tiles='https://tiles.openseamap.org/seamark/{z}/{x}/{y}.png',
        name='🧭 NAV - Marques Marines (OpenSeaMap)',
        attr='OpenSeaMap contributors',
        overlay=True,
        control=True,
        opacity=0.8,
        show=False
    ).add_to(folium_map)

    folium.TileLayer(
        tiles='https://services.arcgisonline.com/arcgis/rest/services/Ocean/World_Ocean_Reference/MapServer/tile/{z}/{y}/{x}',
        name='🧭 NAV - Références Océaniques',
        attr='Esri',
        overlay=True,
        control=True,
        opacity=0.6,
        show=False
    ).add_to(folium_map)

    # bathymétrie

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-bathymetry.eu/wms',
            name='🌊 BATH - Méditerranée (EMODnet) ⭐',
            layers='emodnet:mean',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            attr='EMODnet/Copernicus',
            opacity=0.6,
            show=False,
            version='1.3.0'
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-bathymetry.eu/wms',
            name='🌊 BATH - Globale (EMODnet)',
            layers='emodnet:mean_atlas_land',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            attr='EMODnet',
            opacity=0.6,
            show=False,
            version='1.3.0'
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://www.gebco.net/data_and_products/gebco_web_services/web_map_service/mapserv',
            name='🌊 BATH - GEBCO 2024',
            layers='gebco_latest',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.5,
            attr='GEBCO',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://www.gmrt.org/services/mapserver/wms_merc?',
            name='🌊 BATH - Multi-Résolution (GMRT)',
            layers='topo',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.5,
            attr='GMRT',
            show=False
        ).add_to(folium_map)
    except:
        pass

    # câbles sous-marins

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='🔌 CABLES - Télécom Méditerranée',
            layers='emodnet:telecomcables',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.8,
            attr='EMODnet',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='🔌 CABLES - Électriques Méditerranée',
            layers='emodnet:powercables',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.8,
            attr='EMODnet',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='🔌 CABLES - Tous Types (EMODnet)',
            layers='cables',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.7,
            attr='EMODnet',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.TileLayer(
            tiles='https://submarine-cable-map-2021.telegeography.com/api/v1/tiles/{z}/{x}/{y}.png',
            name='🔌 CABLES - Globaux (TeleGeography)',
            attr='TeleGeography',
            overlay=True,
            control=True,
            opacity=0.75,
            show=False
        ).add_to(folium_map)
    except:
        pass

    # infrastructure portuaire

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='⚓ INFRA - Ports (EMODnet)',
            layers='emodnet:ports',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.7,
            attr='EMODnet',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='⚓ INFRA - Zones de Mouillage',
            layers='emodnet:anchoragearea',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.6,
            attr='EMODnet',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    # densité de navires

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='🚢 DENSITY - Tous Types (EMODnet)',
            layers='emodnet:vesseldensity_all',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.7,
            attr='EMODnet',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='🚢 DENSITY - Cargo',
            layers='emodnet:vesseldensity_cargo',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.7,
            attr='EMODnet',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='🚢 DENSITY - Passagers',
            layers='emodnet:vesseldensity_passenger',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.7,
            attr='EMODnet',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='🚢 DENSITY - Pétroliers',
            layers='emodnet:vesseldensity_tanker',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.7,
            attr='EMODnet',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='🚢 DENSITY - Routes EMSA 2019',
            layers='emsaroutedensity2019',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.65,
            attr='EMSA/EMODnet',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://geoportal.bioinfo.szn.it/arcgis/services/vessel_density/MapServer/WMSServer',
            name='🚢 DENSITY - Méditerranée (SZN)',
            layers='0',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.7,
            attr='SZN',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.TileLayer(
            tiles='https://tiles.arcgis.com/tiles/nzS0F0zdNLvs7nc8/arcgis/rest/services/Global_Ship_Density/MapServer/tile/{z}/{y}/{x}',
            name='🚢 DENSITY - Globale (ArcGIS)',
            attr='ArcGIS',
            overlay=True,
            control=True,
            opacity=0.65,
            show=False
        ).add_to(folium_map)
    except:
        pass

    # routes maritimes

    try:
        folium.TileLayer(
            tiles='https://tiles.arcgis.com/tiles/nzS0F0zdNLvs7nc8/arcgis/rest/services/ShipRoutes/MapServer/tile/{z}/{y}/{x}',
            name='🛤️ ROUTES - Globales (ArcGIS)',
            attr='ArcGIS, CIA',
            overlay=True,
            control=True,
            opacity=0.6,
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='🛤️ ROUTES - Maritimes (EMODnet)',
            layers='sh_rt',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.5,
            attr='EMODnet',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='🛤️ ROUTES - Séparation Trafic',
            layers='emodnet:trafficseparation',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.7,
            attr='EMODnet',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-humanactivities.eu/wms',
            name='🛤️ ROUTES - Zones Trafic Côtier',
            layers='emodnet:inshoretrafficzone',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.6,
            attr='EMODnet',
            version='1.3.0',
            show=False
        ).add_to(folium_map)
    except:
        pass

    # zones juridiques et protégées

    try:
        folium.WmsTileLayer(
            url='https://geo.vliz.be/geoserver/MarineRegions/wms',
            name='🛡️ ZONES - EEZ (200nm)',
            layers='eez',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.4,
            attr='Marine Regions',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://geo.vliz.be/geoserver/MarineRegions/wms',
            name='🛡️ ZONES - Eaux Territoriales (12nm)',
            layers='eez_12nm',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.4,
            attr='Marine Regions',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://geo.vliz.be/geoserver/MarineRegions/wms',
            name='🛡️ ZONES - Aires Marines Protégées',
            layers='mpa',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.5,
            attr='Marine Regions',
            show=False
        ).add_to(folium_map)
    except:
        pass

    # météo et océanographie

    try:
        folium.WmsTileLayer(
            url='https://nowcoast.noaa.gov/arcgis/services/nowcoast/analysis_meteoceanhydro_sfc_rtgsst_time/MapServer/WMSServer',
            name='🌡️ METEO - Température Surface (NOAA)',
            layers='1',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.5,
            attr='NOAA',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://nowcoast.noaa.gov/arcgis/services/nowcoast/wwa_meteoceanhydro_shortduration_hazards_time/MapServer/WMSServer',
            name='🌡️ METEO - Vents (NOAA)',
            layers='3',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.6,
            attr='NOAA',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-physics.eu/wms',
            name='🌡️ METEO - Température Med (EMODnet)',
            layers='sea_water_temperature',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.5,
            attr='EMODnet Physics',
            show=False
        ).add_to(folium_map)
    except:
        pass

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-physics.eu/wms',
            name='🌡️ METEO - Courants Med (EMODnet)',
            layers='sea_water_velocity',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.5,
            attr='EMODnet Physics',
            show=False
        ).add_to(folium_map)
    except:
        pass

    # géologie marine

    try:
        folium.WmsTileLayer(
            url='https://ows.emodnet-geology.eu/wms',
            name='🪨 GEO - Substrat Fond Marin',
            layers='seabed_substrate',
            fmt='image/png',
            transparent=True,
            overlay=True,
            control=True,
            opacity=0.6,
            attr='EMODnet Geology',
            show=False
        ).add_to(folium_map)
    except:
        pass

    print("✅ 35 couches maritimes organisées par catégories")
    return folium_map




**Fonction create_boat_map - Création de la carte interactive avec clustering et sélection de navires**

In [ ]:
def create_boat_map(df, selected_mmsi=None, selected_mmsi_list=None, use_clustering=True, cluster_threshold=100, date_filter=None, show_ephemeris=False):
    """
    Carte maritime UNIFIÉE avec clustering intelligent et sélection single/multi

    Args:
        df: DataFrame avec colonnes mmsi, boat_name, latitude, longitude, boat_type, flag_country
        selected_mmsi: MMSI unique sélectionné (pour compatibilité ancienne)
        selected_mmsi_list: Liste de MMSIs sélectionnés (pour multi-sélection)
        use_clustering: Activer le clustering pour les grandes quantités de navires
        cluster_threshold: Nombre de navires au-delà duquel activer le clustering
        date_filter: Date spécifique à filtrer (pour les points RF)
        show_ephemeris: Afficher les trajectoires satellites (éphémérides)

    Returns:
        Objet folium.Map
    """

    center_lat, center_lon = (38.5, 15.0)
    m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=5,
    tiles=None
    )

    # normaliser la selection
    if selected_mmsi_list is None:
        selected_mmsi_list = []

    if selected_mmsi is not None and selected_mmsi not in selected_mmsi_list:
        selected_mmsi_list.append(selected_mmsi)

    selected_set = {str(mmsi) for mmsi in selected_mmsi_list}

    # palette de couleurs pour la multi-sélection
    color_palette = [
        '#ff0000', '#00ff00', '#0000ff', '#ffff00', '#ff00ff',
        '#00ffff', '#ff8800', '#8800ff', '#00ff88', '#ff0088'
    ]

    mmsi_color_map = {}
    for idx, mmsi in enumerate(selected_mmsi_list):
        mmsi_color_map[str(mmsi)] = color_palette[idx % len(color_palette)]

    # ajouter toutes les couches maritimes
    add_maritime_layers(m)

    # clustering intelligent
    vessel_count = len(df)
    use_clusters = use_clustering and vessel_count > cluster_threshold

    if use_clusters:
        print(f"🗂️ Clustering activé pour {vessel_count} navires")
        from folium import plugins
        marker_cluster = plugins.MarkerCluster(
            name='Navires',
            overlay=True,
            control=False,
            icon_create_function="""
                function(cluster) {
                    var count = cluster.getChildCount();
                    var size = count < 10 ? 'small' : count < 100 ? 'medium' : 'large';
                    return L.divIcon({
                        html: '<div style="background: rgba(0,123,255,0.6); color: white; border-radius: 50%; width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; font-weight: bold; border: 2px solid white;">' + count + '</div>',
                        className: 'marker-cluster marker-cluster-' + size,
                        iconSize: L.point(40, 40)
                    });
                }
            """
        )
        marker_cluster.add_to(m)
        marker_container = marker_cluster
    else:
        print(f"📍 {vessel_count} navires - mode direct (sans clustering)")
        marker_container = m

    # légende pour multi-selection
    if len(selected_set) > 1:
        legend_html = '''
        <div style="position: fixed; bottom: 50px; left: 50px; width: 250px;
                    background-color: rgba(255, 255, 255, 0.9); border: 2px solid #333;
                    border-radius: 8px; padding: 10px; z-index: 9999;
                    font-family: Arial, sans-serif; font-size: 12px;">
            <h4 style="margin: 0 0 8px 0; color: #1a1a1a; font-size: 14px;">
                🚢 Bateaux sélectionnés
            </h4>
        '''

        # Ajouter chaque bateau à la légende
        for mmsi_str in selected_mmsi_list:
            color = mmsi_color_map.get(str(mmsi_str), '#ff0000')
            # Trouver le nom du bateau depuis le DataFrame
            boat_row = df[df['mmsi'].astype(str) == str(mmsi_str)]
            if not boat_row.empty:
                boat_name = str(boat_row.iloc[0]['boat_name']).strip()
                if not boat_name or boat_name.lower() in ['sans nom', 'unknown', 'nan', 'none', '']:
                    boat_name = 'Nom non renseigné'
            else:
                boat_name = 'Nom non renseigné'

            legend_html += f'''
            <div style="margin: 5px 0; display: flex; align-items: center;">
                <span style="display: inline-block; width: 16px; height: 16px;
                            background: {color}; border-radius: 50%; margin-right: 8px;
                            border: 2px solid #fff; box-shadow: 0 0 3px rgba(0,0,0,0.5);"></span>
                <span style="color: #2c2c2c; font-weight: 500;">{boat_name}</span>
            </div>
            '''

        legend_html += '</div>'
        m.get_root().html.add_child(folium.Element(legend_html))

    # création des marqueurs
    selected_markers_added = []

    for _, boat in df.iterrows():
        lat_val = boat['latitude']
        lon_val = boat['longitude']


        if pd.isna(lat_val) or pd.isna(lon_val) or lat_val == 0 or lon_val == 0:
            continue

        lat, lon = float(lat_val), float(lon_val)
        mmsi = str(boat.get('mmsi'))


        name = str(boat.get('boat_name', '')).strip()

        # Si le nom est vide ou invalide, afficher "Nom non renseigné"
        if not name or name.lower() in ['sans nom', 'unknown', 'nan', 'none', '']:
            name = 'Nom non renseigné'

        # récupérer le type de navire
        vessel_type = str(boat.get('boat_type', 'Type non renseigné')).strip()


        if vessel_type.lower() == 'unknown' or vessel_type == '':
            vessel_type = 'Type non renseigné'

        flag_country = boat.get('flag_country', 'Unknown')

        # Vérifier si sélectionné
        is_selected = mmsi in selected_set


        marker_color = mmsi_color_map.get(mmsi, '#ff0000') if is_selected else '#ffffff'

        # Style CSS harmonisé pour le popup
        popup_header_style = f"""
            background: {MARITIME_CSS['dark_color']};
            color: #00ff41;
            padding: 8px;
            margin: -9px -9px 8px -9px;
            font-weight: bold;
            border-radius: {MARITIME_CSS['border_radius']} {MARITIME_CSS['border_radius']} 0 0;
        """

        button_style = get_button_style('success' if is_selected else 'primary')


        name_escaped = name.replace("'", "\\'").replace('"', '\\"').replace('\\', '\\\\')

        popup_content = f"""
        <div style='{get_popup_style()}'>
            <div style='{popup_header_style}'>
                {'🔴 SÉLECTIONNÉ' if is_selected else '⚪ CONTACT'} - {name}
            </div>
            <table style='width: 100%; font-size: {MARITIME_CSS['font_size_small']}; color: #333;'>
                <tr><td><b>MMSI:</b></td><td style='color: {MARITIME_CSS['primary_color']};'>{mmsi}</td></tr>
                <tr><td><b>TYPE:</b></td><td>{vessel_type}</td></tr>
                <tr><td><b>PAVILLON:</b></td><td>{flag_country if flag_country != 'Unknown' else 'Non identifié'}</td></tr>
                <tr><td><b>POS:</b></td><td>{lat:.4f}°N, {lon:.4f}°E</td></tr>
                <tr><td><b>STATUS:</b></td><td style='color: {MARITIME_CSS['success_color']};'>TRACKED</td></tr>
            </table>
        </div>
        """

        # Créer le marqueur
        if is_selected:

            marker = folium.CircleMarker(
                location=[lat, lon],
                radius=12,
                popup=folium.Popup(popup_content, max_width=300),
                tooltip=f'★ {name} | MMSI {mmsi}',
                color='#ffffff',
                weight=2,
                fillColor=marker_color,
                fillOpacity=1.0,
                opacity=1.0
            )
            marker.add_to(m)


            folium.Marker(
                [lat, lon],
                icon=folium.DivIcon(html=f"""
                    <div style='
                        font-family: "Courier New", monospace;
                        font-size: 10px;
                        font-weight: bold;
                        color: {marker_color};
                        text-shadow: 1px 1px 2px #000000;
                        background: rgba(0,0,0,0.7);
                        padding: 2px 4px;
                        border-radius: 3px;
                        white-space: nowrap;
                        transform: translate(-50%, -120%);
                    '>{name}</div>
                """,
                icon_size=(1, 1),
                icon_anchor=(0, 0)
                )
            ).add_to(m)

            selected_markers_added.append(name)

        else:
            marker = folium.CircleMarker(
                location=[lat, lon],
                radius=8,
                popup=folium.Popup(popup_content, max_width=300),
                tooltip=f'● {name} | MMSI {mmsi}',
                color='#333333',
                weight=1,
                fillColor='#ffffff',
                fillOpacity=0.8,
                opacity=0.8
            )
            marker.add_to(marker_container)

    # ajout des points RF
    if 'df_rf_global' in globals() and df_rf_global is not None and not df_rf_global.empty:
        m = add_rf_layer_to_map(m, df_rf_global, date_filter=date_filter)

    # ajout des trajectoires satellites (éphémérides)
    if show_ephemeris:
        print(f"🛰️ Tentative d'ajout des trajectoires satellites...")
        if 'df_ephemeris_global' in globals():
            if df_ephemeris_global is not None and not df_ephemeris_global.empty:
                print(f"   ✅ df_ephemeris_global trouvé: {len(df_ephemeris_global)} lignes")
                m = add_ephemeris_layer_to_map(m, df_ephemeris_global, date_filter=date_filter)
            else:
                print(f"   ⚠️ df_ephemeris_global est vide ou None")
        else:
            print(f"   ❌ df_ephemeris_global n'existe pas dans globals()")

    # contrôles
    folium.LayerControl(position='topright').add_to(m)

    print(f"✅ Carte créée: {vessel_count} navires, {'🗂️ avec clustering' if use_clusters else '📍 sans clustering'}")
    if selected_markers_added:
        print(f"🔴 {len(selected_markers_added)} navire(s) sélectionné(s): {', '.join(selected_markers_added)}")
    if show_ephemeris:
        print(f"🛰️ Trajectoires satellites affichées")

    return m

**Fonction d'ajout des points RF sur la carte**

In [ ]:
def add_rf_layer_to_map(folium_map, df_rf, date_filter=None):
    """
    Ajoute une couche RF à la carte Folium avec icônes distinctes

    Args:
        folium_map: Carte Folium
        df_rf: DataFrame avec les points RF
        date_filter: Date spécifique à filtrer (même logique que pour AIS)
    """
    if df_rf is None or df_rf.empty:
        print("⚠️ Aucune donnée RF disponible")
        return folium_map

    # Filtrage par date si nécessaire (même logique que pour AIS)
    if date_filter:
        if isinstance(date_filter, str):
            date_filter = pd.to_datetime(date_filter).date()

        if 'date_only' not in df_rf.columns:
            df_rf['date_only'] = pd.to_datetime(df_rf['timestamp_utc']).dt.date

        df_rf = df_rf[df_rf['date_only'] == date_filter].copy()

        if df_rf.empty:
            print(f"⚠️ Aucune donnée RF pour {date_filter}")
            return folium_map

    # Créeation d'un FeatureGroup pour les points RF
    rf_group = folium.FeatureGroup(name='📡 Signaux RF', overlay=True, control=True)

    print(f"📡 Traitement de {len(df_rf)} points RF...")

    # Ajout de chaque point RF
    for idx, rf_point in df_rf.iterrows():
        try:
            lat = rf_point['latitude_deg']
            lon = rf_point['longitude_deg']

            # Vérifier que les coordonnées sont valides
            if pd.isna(lat) or pd.isna(lon) or lat == 0 or lon == 0:
                continue

            # Préparer les informations pour le popup

            accuracy = rf_point.get('accuracy_level', 'Non disponible')
            frequency = rf_point.get('RF_Frequency_MHz', 'Non disponible')
            pulse = rf_point.get('pulses_duration_ns', 'Non disponible')
            pulse_repetition = rf_point.get('pulses_repetition_frequency_Hz', 'Non disponible')
            waveform = rf_point.get('waveform', 'Non disponible')

            # Formater le timestamp au format français
            timestamp_raw = rf_point.get('timestamp_utc', None)
            if timestamp_raw is not None and timestamp_raw != 'Non disponible':
                try:
                    # Convertir en datetime si ce n'est pas déjà le cas
                    if isinstance(timestamp_raw, str):
                        timestamp_dt = pd.to_datetime(timestamp_raw)
                    else:
                        timestamp_dt = pd.to_datetime(timestamp_raw)
                    # Formater au format français : 16-12-2024 13:26:41
                    timestamp = timestamp_dt.strftime('%d-%m-%Y %H:%M:%S')
                except Exception:
                    timestamp = str(timestamp_raw)  # En cas d'erreur, garder la valeur originale
            else:
                timestamp = 'Non disponible'

            # Formatage du popup
            popup_content = f"""
            <div style='font-family: Arial; min-width: 280px; color: #1a1a1a;'>
                <div style='background: #ff6b35; color: white; padding: 10px; margin: -10px -10px 10px -10px;
                           border-radius: 5px 5px 0 0; font-weight: bold; text-align: center;'>
                    📡 Signal RF Détecté
                </div>
                <table style='width: 100%; font-size: 12px; margin-top: 5px; line-height: 1.4;'>
                    <tr><td><b>🕐 Date:</b></td><td>{timestamp}</td></tr>
                    <tr><td><b>📶 Accuracy Level:</b></td><td>{accuracy}</td></tr>
                    <tr><td><b>📶 RF Frequency (MHz):</b></td><td>{frequency}</td></tr>
                    <tr><td><b>📶 Pulse Duration (ns):</b></td><td>{pulse}</td></tr>
                    <tr><td><b>📶 Pulse Repetition Frequency (Hz) :</b></td><td>{pulse_repetition}</td></tr>
                    <tr><td><b>📶 Waveform :</b></td><td>{waveform}</td></tr>
                    <tr><td><b>📍 Latitude:</b></td><td>{lat:.4f}°</td></tr>
                    <tr><td><b>📍 Longitude:</b></td><td>{lon:.4f}°</td></tr>
                </table>
                <div style='margin-top: 8px; padding: 5px; background: #f8f9fa; border-radius: 3px; font-size: 11px;'>
                    <b>ℹ️ Source:</b> Données RF Unseen Labs
                </div>
            </div>
            """

            # Créeation d'un marqueur RF sobre avec CircleMarker
            folium.CircleMarker(
                location=[lat, lon],
                radius=6,
                popup=folium.Popup(popup_content, max_width=350),
                tooltip=f'📡 RF: {accuracy} ',
                color='#d97706',
                weight=1.5,
                fillColor='#f97316',
                fillOpacity=0.8,
                opacity=0.9
            ).add_to(rf_group)

        except Exception as e:
            # print(f"⚠️ Erreur avec le point RF {idx}: {e}")
            continue

    # Ajout du groupe RF à la carte
    rf_group.add_to(folium_map)
    rf_count = len([m for m in rf_group._children.values()])
    print(f"✅ {rf_count} points RF ajoutés à la carte")

    return folium_map

**Fonction d'ajout des trajectoire satellites (éphémérides) sur la carte**

In [ ]:
def add_ephemeris_layer_to_map(folium_map, df_ephemeris, date_filter=None):
    """
    Ajoute une couche de trajectoires satellites à la carte Folium

    Args:
        folium_map: Carte Folium
        df_ephemeris: DataFrame avec les données éphémérides
        date_filter: Date spécifique à filtrer (optionnel)

    Returns:
        Carte Folium avec les trajectoires satellites ajoutées
    """
    try:
        if df_ephemeris is None or df_ephemeris.empty:
            print("⚠️ DataFrame éphémérides vide ou None")
            return folium_map

        print(f"🛰️ DataFrame éphémérides reçu: {len(df_ephemeris)} lignes")
        print(f"   Colonnes disponibles: {list(df_ephemeris.columns)}")

        # Filtrage par date si nécessaire
        df_ephemeris_filtered = df_ephemeris.copy()

        if date_filter:
            if isinstance(date_filter, str):
                date_filter = pd.to_datetime(date_filter).date()

            if 'date_only' not in df_ephemeris_filtered.columns:
                df_ephemeris_filtered['date_only'] = pd.to_datetime(df_ephemeris_filtered['StartTime']).dt.date

            df_ephemeris_filtered = df_ephemeris_filtered[df_ephemeris_filtered['date_only'] == date_filter].copy()

            if df_ephemeris_filtered.empty:
                print(f"⚠️ Aucune donnée éphémérides pour la date {date_filter}")
                return folium_map
            else:
                print(f"✅ {len(df_ephemeris_filtered)} trajectoires après filtrage par date")

        # Créeation d'un FeatureGroup pour les trajectoires satellites
        ephemeris_group = folium.FeatureGroup(name='🛰️ Trajectoires Satellites', overlay=True, control=True)

        # Palette de couleurs pour les différents satellites
        satellite_colors = {
            'JAPETUS SSO_1': '#ff0000',
            'JAPETUS SSO_2': '#00ff00',
            'JAPETUS SSO_3': '#0000ff',
            'JAPETUS SSO_4': '#ffff00',
            'JAPETUS Inclined_8': '#ff00ff',
            'JAPETUS Inclined_15': '#00ffff',
            'JAPETUS Inclined_16': '#ff8800',
        }

        # Grouper par satellite pour compter les trajectoires
        satellite_counts = {}
        trajectories_added = 0
        errors_count = 0

        print(f"🛰️ Traitement de {len(df_ephemeris_filtered)} trajectoires satellites...")

        # Ajouter chaque trajectoire
        for idx, row in df_ephemeris_filtered.iterrows():
            try:
                # Coordonnées de départ et d'arrivée
                start_lat = row['StartLatitude']
                start_lon = row['StartLongitude']
                end_lat = row['EndLatitude']
                end_lon = row['EndLongitude']

                # Vérifier que les coordonnées sont valides
                if pd.isna(start_lat) or pd.isna(start_lon) or pd.isna(end_lat) or pd.isna(end_lon):
                    errors_count += 1
                    continue

                if start_lat == 0 or start_lon == 0 or end_lat == 0 or end_lon == 0:
                    errors_count += 1
                    continue

                # Informations du satellite
                satellite_name = row.get('Source', 'Satellite inconnu')
                target = row.get('Target', 'Cible inconnue')
                start_time = pd.to_datetime(row['StartTime'])
                end_time = pd.to_datetime(row['EndTime'])
                duration = row.get('Duration', 0)  # en secondes
                distance_km = row.get('Distance_km', 0)
                speed_kmh = row.get('Average_Speedkmh', 0)

                # Obtenir la couleur pour ce satellite
                color = satellite_colors.get(satellite_name, '#888888')

                # Compter les trajectoires par satellite
                if satellite_name not in satellite_counts:
                    satellite_counts[satellite_name] = 0
                satellite_counts[satellite_name] += 1

                trajectory_coords = [[start_lat, start_lon], [end_lat, end_lon]]

                # Formater les informations pour le popup
                duration_min = duration / 60.0
                popup_content = f"""
                <div style='font-family: Arial; min-width: 280px; color: #1a1a1a;'>
                    <div style='background: {color}; color: white; padding: 10px; margin: -10px -10px 10px -10px;
                              border-radius: 5px 5px 0 0; font-weight: bold; text-align: center;'>
                        🛰️ {satellite_name}
                    </div>
                    <table style='width: 100%; font-size: 12px; margin-top: 5px; line-height: 1.4;'>
                        <tr><td><b>🎯 Cible:</b></td><td>{target}</td></tr>
                        <tr><td><b>🕐 Début:</b></td><td>{start_time.strftime('%d/%m/%Y %H:%M:%S')}</td></tr>
                        <tr><td><b>🕐 Fin:</b></td><td>{end_time.strftime('%d/%m/%Y %H:%M:%S')}</td></tr>
                        <tr><td><b>⏱️ Durée:</b></td><td>{duration_min:.1f} min ({duration}s)</td></tr>
                        <tr><td><b>📏 Distance:</b></td><td>{distance_km:.2f} km</td></tr>
                        <tr><td><b>⚡ Vitesse:</b></td><td>{speed_kmh:.2f} km/h</td></tr>
                        <tr><td><b>📍 Départ:</b></td><td>{start_lat:.4f}°N, {start_lon:.4f}°E</td></tr>
                        <tr><td><b>📍 Arrivée:</b></td><td>{end_lat:.4f}°N, {end_lon:.4f}°E</td></tr>
                    </table>
                    <div style='margin-top: 8px; padding: 5px; background: #f8f9fa; border-radius: 3px; font-size: 11px;'>
                        <b>ℹ️ Source:</b> Données éphémérides CNES
                    </div>
                </div>
                """

                # Créer la ligne de trajectoire
                folium.PolyLine(
                    trajectory_coords,
                    color=color,
                    weight=2.5,
                    opacity=0.7,
                    popup=folium.Popup(popup_content, max_width=350),
                    tooltip=f'🛰️ {satellite_name} - {target} ({duration_min:.1f} min)'
                ).add_to(ephemeris_group)

                # Marqueur de départ
                folium.CircleMarker(
                    location=[start_lat, start_lon],
                    radius=4,
                    popup=folium.Popup(f"""
                        <div style='font-family: Arial; min-width: 200px;'>
                            <div style='background: {color}; color: white; padding: 8px; margin: -9px -9px 8px -9px;
                                      border-radius: 5px 5px 0 0; font-weight: bold; text-align: center;'>
                                🛰️ {satellite_name} - Départ
                            </div>
                            <div style='font-size: 12px; padding: 5px;'>
                                <b>🕐 Heure:</b> {start_time.strftime('%d/%m/%Y %H:%M:%S')}<br>
                                <b>📍 Position:</b> {start_lat:.4f}°N, {start_lon:.4f}°E
                            </div>
                        </div>
                    """, max_width=250),
                    tooltip=f'🛰️ Départ {satellite_name}',
                    color='#ffffff',
                    weight=1.5,
                    fillColor=color,
                    fillOpacity=0.9
                ).add_to(ephemeris_group)

                # Marqueur d'arrivée
                folium.CircleMarker(
                    location=[end_lat, end_lon],
                    radius=4,
                    popup=folium.Popup(f"""
                        <div style='font-family: Arial; min-width: 200px;'>
                            <div style='background: {color}; color: white; padding: 8px; margin: -9px -9px 8px -9px;
                                      border-radius: 5px 5px 0 0; font-weight: bold; text-align: center;'>
                                🛰️ {satellite_name} - Arrivée
                            </div>
                            <div style='font-size: 12px; padding: 5px;'>
                                <b>🕐 Heure:</b> {end_time.strftime('%d/%m/%Y %H:%M:%S')}<br>
                                <b>📍 Position:</b> {end_lat:.4f}°N, {end_lon:.4f}°E<br>
                                <b>⚡ Vitesse:</b> {speed_kmh:.2f} km/h
                            </div>
                        </div>
                    """, max_width=250),
                    tooltip=f'🛰️ Arrivée {satellite_name}',
                    color='#ffffff',
                    weight=1.5,
                    fillColor=color,
                    fillOpacity=0.6
                ).add_to(ephemeris_group)

                trajectories_added += 1

            except Exception as e:
                errors_count += 1
                if errors_count <= 3:
                    print(f"⚠️ Erreur avec trajectoire {idx}: {e}")
                continue

        # Ajout du groupe à la carte
        ephemeris_group.add_to(folium_map)

        # Création de la légende des satellites
        if satellite_counts:
            legend_items = []
            for sat_name, count in sorted(satellite_counts.items()):
                color = satellite_colors.get(sat_name, '#888888')
                legend_items.append(f"""
                    <div style="margin: 5px 0; display: flex; align-items: center;">
                        <span style="display: inline-block; width: 16px; height: 3px;
                                    background: {color}; margin-right: 8px; border-radius: 2px;"></span>
                        <span style="color: white; font-size: 11px;">{sat_name} ({count})</span>
                    </div>
                """)

            legend_html = f"""
            <div style="position: fixed; top: 60px; right: 10px; width: 220px;
                        background: rgba(0, 0, 0, 0.85); border: 2px solid #00ff41;
                        border-radius: 8px; padding: 12px; z-index: 1000;
                        font-family: 'Segoe UI', sans-serif; max-height: 400px; overflow-y: auto;">
                <div style="color: #00ff41; font-weight: bold; margin-bottom: 10px; font-size: 13px;
                           text-align: center; border-bottom: 1px solid rgba(0, 255, 65, 0.3); padding-bottom: 8px;">
                    🛰️ Satellites ({len(satellite_counts)})
                </div>
                {''.join(legend_items)}
            </div>
            """
            folium_map.get_root().html.add_child(folium.Element(legend_html))

        print(f"✅ {trajectories_added} trajectoires satellites ajoutées à la carte")
        if errors_count > 0:
            print(f"⚠️ {errors_count} trajectoires ignorées (données invalides)")

        return folium_map

    except Exception as e:
        print(f"❌ Erreur dans add_ephemeris_layer_to_map: {e}")
        import traceback
        traceback.print_exc()
        return folium_map

**Création des widgets pour la sélection des navires (SelectMultiple)**

In [ ]:
# variables globales
current_boat = None
current_weather = None
current_location = None
last_selected_mmsi = None
last_selected_name = None
selected_mmsi_list = []  # Liste des MMSI sélectionnés

# widgets interface

boat_selector = widgets.SelectMultiple(
    options=[],
    value=(),
    description='🚢 Navires:',
    disabled=False,
    style={
        'description_width': '90px',
        'description_color': '#667eea'
    },
    layout=widgets.Layout(
        width='100%',
        height='180px',
        border='2px solid #667eea',
        border_radius='12px',
        padding='10px',
        background_color='#ffffff',
        box_shadow='0 4px 15px rgba(102, 126, 234, 0.2), inset 0 1px 3px rgba(0,0,0,0.05)'
    )
)

# 🌤️ Widget météo
weather_output = widgets.HTML(
    value="""
    <div style='padding: 18px;
                background: linear-gradient(135deg, #e3f2fd 0%, #f0f8ff 100%);
                border-radius: 12px;
                border-left: 4px solid #2196f3;
                box-shadow: 0 4px 12px rgba(33, 150, 243, 0.15);
                border: 1px solid rgba(33, 150, 243, 0.2);'>
        <div style='display: flex; align-items: center; gap: 10px; color: #01579b; font-weight: 600;'>
            <span style='font-size: 24px;'>🌤️</span>
            <span style='font-size: 14px;'>Météo Marine</span>
        </div>
        <div style='margin-top: 10px; color: #0277bd; font-size: 13px; line-height: 1.6;'>
            Sélectionnez un ou plusieurs bateaux pour afficher les conditions météorologiques
        </div>
    </div>
    """
)

# 🌍 Widget localisation
location_output = widgets.HTML(
    value="""
    <div style='padding: 18px;
                background: linear-gradient(135deg, #e8f5e9 0%, #f0fff0 100%);
                border-radius: 12px;
                border-left: 4px solid #4caf50;
                box-shadow: 0 4px 12px rgba(76, 175, 80, 0.15);
                border: 1px solid rgba(76, 175, 80, 0.2);'>
        <div style='display: flex; align-items: center; gap: 10px; color: #1b5e20; font-weight: 600;'>
            <span style='font-size: 24px;'>🌍</span>
            <span style='font-size: 14px;'>Géolocalisation</span>
        </div>
        <div style='margin-top: 10px; color: #2e7d32; font-size: 13px; line-height: 1.6;'>
            Sélectionnez un ou plusieurs bateaux pour afficher la position et la zone maritime
        </div>
    </div>
    """
)

# 🗺️ Widget sortie carte
map_output = widgets.Output(
    layout=widgets.Layout(
        width='100%',
        border='2px solid #e9ecef',
        border_radius='12px',
        box_shadow='0 4px 15px rgba(0,0,0,0.08)'
    )
)

# 💬 Widget chat input
chat_input = widgets.Text(
    placeholder='💭 Posez votre question sur les navires...',
    description='Question:',
    style={
        'description_width': '70px',
        'description_color': '#667eea'
    },
    layout=widgets.Layout(
        width='100%',
        border='2px solid #667eea',
        border_radius='10px',
        padding='8px',
        box_shadow='0 2px 8px rgba(102, 126, 234, 0.12)'
    )
)

# 🤖 Widget chat output
chat_output = widgets.HTML(
    value="""
    <div style='padding: 18px;
                background: linear-gradient(135deg, #fff9e6 0%, #fffbf0 100%);
                border-radius: 12px;
                border-left: 4px solid #ffc107;
                box-shadow: 0 4px 12px rgba(255, 193, 7, 0.15);
                border: 1px solid rgba(255, 193, 7, 0.2);'>
        <div style='display: flex; align-items: center; gap: 10px; color: #5d4037; font-weight: 600;'>
            <span style='font-size: 24px;'>🤖</span>
            <span style='font-size: 14px;'>Assistant IA Maritime</span>
        </div>
        <div style='margin-top: 10px; color: #6d4c41; font-size: 13px; line-height: 1.6;'>
            Je suis prêt à répondre à vos questions sur les navires, leurs itinéraires et les données maritimes
        </div>
    </div>
    """
)

# 📤 Bouton chat
chat_button = widgets.Button(
    description='📤 Envoyer',
    button_style='success',
    icon='paper-plane',
    layout=widgets.Layout(
        width='140px',
        height='38px',
        border_radius='10px',
        box_shadow='0 3px 10px rgba(102, 126, 234, 0.2)'
    ),
    style={'button_color': '#667eea', 'font_weight': 'bold'}
)

# Bouton pour effacer l'historique
clear_history_button = widgets.Button(
    description='🗑️ Effacer historique',
    button_style='warning',
    tooltip='Efface tout l\'historique de conversation',
    layout=widgets.Layout(width='180px')
)

boat_info_output = widgets.HTML(
    value="<div style='padding:20px;'>Sélectionnez un bateau</div>",
    layout=widgets.Layout(width='100%', height='400px')
)

**Fonctions pour gérer l'affichage de plusieurs navires sélectionnés - Construction HTML et mise à jour automatique**

In [ ]:
def build_multi_boat_info_html(mmsi_list):
    """
    Construit le HTML pour afficher les infos de plusieurs bateaux.

    Args:
        mmsi_list: Liste des MMSIs à afficher

    Returns:
        HTML string avec les informations formatées
    """
    if not mmsi_list:
        return "<div style='padding:20px; text-align:center; color:#6c757d;'>Aucun bateau sélectionné</div>"

    html = f"""
    <div style='padding:15px; background:#f8f9fa; border-radius:8px;'>
        <h3 style='margin-top:0; color:#007bff;'>📊 {len(mmsi_list)} Bateau(x) Sélectionné(s)</h3>
    """

    # Pour chaque bateau, récupérer et afficher ses infos
    for i, mmsi in enumerate(mmsi_list, 1):
        try:
            # Chercher le bateau dans df_ais_global
            vessel_data = df_ais_global[df_ais_global['Mmsi'].astype(str) == str(mmsi)]

            if vessel_data.empty:
                html += f"""
                <div style='padding:10px; margin:10px 0; background:#fff3cd; border-radius:6px;'>
                    ⚠️ <strong>MMSI {mmsi}:</strong> Données non trouvées
                </div>
                """
                continue

            # Prendre la dernière position
            vessel_row = vessel_data.sort_values('datetime').tail(1).iloc[0]

            # Extraire les informations
            name = vessel_row.get('Name', f'MMSI {mmsi}')
            lat = vessel_row.get('Position_LatitudeDeg', 'N/A')
            lon = vessel_row.get('Position_LongitudeDeg', 'N/A')
            speed = vessel_row.get('SogKt', 'N/A')
            course = vessel_row.get('CogDeg', 'N/A')
            ship_type = vessel_row.get('ShipTypeLabel', 'Non déclaré')
            country = vessel_row.get('Pays', 'Non identifié')
            last_update = vessel_row.get('datetime', 'N/A')

            # Formater l'affichage
            html += f"""
            <div style='padding:12px; margin:10px 0; background:white; border-radius:6px;
                        border-left:4px solid #007bff; box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
                <strong style='color:#007bff; font-size:16px;'>{i}. {name}</strong>
                <div style='display:grid; grid-template-columns: 1fr 1fr; gap:8px; margin-top:8px; font-size:13px;'>
                    <div>🔢 <strong>MMSI:</strong> {mmsi}</div>
                    <div>🏷️ <strong>Type:</strong> {ship_type}</div>
                    <div>🏳️ <strong>Pavillon:</strong> {country}</div>
                    <div>⚡ <strong>Vitesse:</strong> {speed} kts</div>
                    <div>📍 <strong>Lat:</strong> {lat:.4f}°</div>
                    <div>📍 <strong>Lon:</strong> {lon:.4f}°</div>
                    <div>🧭 <strong>Cap:</strong> {course}°</div>
                    <div>🕐 <strong>Mise à jour:</strong> {last_update}</div>
                </div>
            </div>
            """

        except Exception as e:
            html += f"""
            <div style='padding:10px; margin:10px 0; background:#f8d7da; border-radius:6px;'>
                ❌ <strong>MMSI {mmsi}:</strong> Erreur - {str(e)}
            </div>
            """

    html += "</div>"
    return html

def update_boat_info_multiple(change):
    """
    Met à jour les informations pour plusieurs bateaux sélectionnés.
    Appelé quand la sélection multiple change.

    Args:
        change: dict avec 'new' contenant le tuple des MMSIs sélectionnés
    """
    global selected_mmsi_list

    try:
        # récupérer les MMSIs sélectionnés
        selected = change.get('new', ())

        if not selected:
            # Aucun bateau sélectionné - afficher message
            boat_info_output.value = """
            <div style='padding:20px; text-align:center; color:#6c757d;'>
                ℹ️ <strong>Aucun bateau sélectionné</strong><br>
                <small>Sélectionnez un ou plusieurs bateaux pour voir les informations</small>
            </div>
            """
            return

        # convertir en liste de strings
        selected_mmsi_list = [str(mmsi) for mmsi in selected]

        print(f"🔄 Mise à jour pour {len(selected_mmsi_list)} bateau(x)...")

        # maj de la carte
        if 'map_output' in globals() and 'df_ais_global' in globals():
            try:
                if 'refresh_ui_from_stream' in globals():
                    refresh_ui_from_stream(selected_mmsi=selected_mmsi_list[0] if selected_mmsi_list else None)
                    print("  ✅ Carte mise à jour")
            except Exception as e:
                pass

        # maj des infos détaillées
        if 'boat_info_output' in globals():
            try:
                info_html = build_multi_boat_info_html(selected_mmsi_list)
                boat_info_output.value = info_html
                print("  ✅ Informations mises à jour")
            except Exception as e:
                print(f"  ⚠️ Erreur mise à jour infos: {e}")
                boat_info_output.value = f"""
                <div style='padding:15px; background:#f8d7da; border-radius:8px; color:#721c24;'>
                    ⚠️ Erreur lors de la mise à jour des informations: {str(e)}
                </div>
                """

        print(f"✅ Mise à jour terminée pour {len(selected_mmsi_list)} bateau(x)")

    except Exception as e:
        print(f"❌ Erreur dans update_boat_info_multiple: {e}")
        import traceback
        traceback.print_exc()

**Création du bouton de validation et widget de statut - Handler on_validate_selection pour valider la sélection des navires**

In [ ]:
# widget de status
boat_selection_info = widgets.HTML(
    value="""
    <div style='padding:12px 15px; background:#f8f9fa; border-radius:8px;
                border-left:4px solid #6c757d; color:#495057; font-size:13px;
                margin:10px 0; box-shadow: 0 2px 6px rgba(0,0,0,0.08);'>
        ℹ️ <strong>Aucune sélection</strong> - Sélectionnez des bateaux puis validez
    </div>
    """,
    layout=widgets.Layout(width='100%')
)

# bouton de validation
boat_validate_btn = widgets.Button(
    description='✅ Valider la sélection',
    button_style='success',
    tooltip='Confirmer la sélection des bateaux',
    layout=widgets.Layout(width='200px', height='38px'),
    icon='check',
    style={'button_color': '#28a745', 'font_weight': 'bold'}
)

# Bouton de désélection
boat_deselect_btn = widgets.Button(
    description='❌ Désélectionner',
    button_style='warning',
    tooltip='Désélectionne tous les navires et affiche la carte sans filtre',
    layout=widgets.Layout(width='180px', height='38px'),
    icon='times',
    style={'button_color': '#ff9800', 'font_weight': 'bold'}
)

def on_deselect_all(button):
    """Désélectionne tous les navires et affiche la carte sans filtre"""
    global selected_mmsi_list, last_selected_mmsi

    # Réinitialiser la sélection
    selected_mmsi_list = []
    last_selected_mmsi = None

    # Réinitialiser le widget de sélection
    if 'boat_selector' in globals():
        boat_selector.value = ()

    if 'weather_output' in globals():
        weather_output.value = """
        <div style='padding: 15px; background: #e8f4f8; border-radius: 8px; border-left: 4px solid #0288d1;
                    color: #01579b; font-weight: 500;'>
            <strong>🌤️ Météo</strong><br>
            Sélectionnez et validez un ou plusieurs bateaux
        </div>
        """

    if 'location_output' in globals():
        location_output.value = """
        <div style='padding: 15px; background: #e8f5e9; border-radius: 8px; border-left: 4px solid #4caf50;
                    color: #1b5e20; font-weight: 500;'>
            <strong>🌍 Localisation</strong><br>
            Sélectionnez et validez un ou plusieurs bateaux
        </div>
        """

    # Mettre à jour l'affichage de sélection
    boat_selection_info.value = """
    <div style='padding:12px 15px; background:#f8f9fa; border-radius:8px;
                border-left:4px solid #6c757d; color:#495057; font-size:13px;
                margin:10px 0; box-shadow: 0 2px 6px rgba(0,0,0,0.08);'>
        ℹ️ <strong>Aucune sélection</strong> - Sélectionnez des bateaux puis validez
    </div>
    """

    # Réinitialiser les filtres aussi
    if 'filter_type_dropdown' in globals():
        filter_type_dropdown.value = 'Tous'
    if 'filter_country_dropdown' in globals():
        filter_country_dropdown.value = 'Tous'
    if 'date_selector' in globals():
        date_selector.value = None

    # Rafraîchir la carte sans sélection ni filtre
    if 'refresh_ui_from_stream' in globals():
        refresh_ui_from_stream(selected_mmsi=None, force_refresh=False)

    print("🔄 Tous les navires désélectionnés - Carte affichée sans filtre")

boat_deselect_btn.on_click(on_deselect_all)

def on_validate_selection(button):
    """Valide et applique la sélection de bateaux"""
    global selected_mmsi_list, last_selected_mmsi

    # Récupérer la sélection actuelle du SelectMultiple
    current_selection = list(boat_selector.value) if boat_selector.value else []

    if not current_selection:
        boat_selection_info.value = """
        <div style='padding:12px 15px; background:#fff3cd; border-radius:8px;
                    border-left:4px solid #ffc107; color:#856404; font-size:13px;
                    margin:10px 0; box-shadow: 0 2px 6px rgba(255, 193, 7, 0.2);'>
            ⚠️ <strong>Aucun bateau sélectionné</strong> - Veuillez sélectionner au moins un bateau
        </div>
        """
        return

    # Mettre à jour la liste globale
    selected_mmsi_list = [str(mmsi) for mmsi in current_selection]

    # Si un seul navire est sélectionné, mettre à jour last_selected_mmsi
    if len(selected_mmsi_list) == 1:
        last_selected_mmsi = selected_mmsi_list[0]
    else:
        last_selected_mmsi = None

    # réinitialiser tous les filtres
    if 'filter_type_dropdown' in globals():
        filter_type_dropdown.value = 'Tous'
    if 'filter_country_dropdown' in globals():
        filter_country_dropdown.value = 'Tous'
    if 'date_selector' in globals():
        date_selector.value = None

    # Mettre à jour le statut des filtres
    if 'filter_status' in globals():
        filter_status.value = """
        <div style='padding:14px 18px; background: linear-gradient(135deg, #fff3cd 0%, #fffbf0 100%);
                    border-radius: 10px; border-left: 4px solid #ffc107;
                    box-shadow: 0 3px 10px rgba(255, 193, 7, 0.15); border: 1px solid #ffecb5;'>
            <div style='display: flex; align-items: center; gap: 10px; color: #664d03; font-weight: 600;'>
                <span style='font-size: 18px;'>ℹ️</span>
                <span style='font-size: 13px;'>Filtres réinitialisés - Affichage des navires sélectionnés uniquement</span>
            </div>
        </div>
        """

    # Message générique de validation
    boat_selection_info.value = f"""
    <div style='padding:12px 15px; background:#d4edda; border-radius:8px;
                border-left:4px solid #28a745; color:#155724; font-size:13px;
                margin:10px 0; box-shadow: 0 2px 6px rgba(40, 167, 69, 0.2);'>
        ✅ <strong>{len(selected_mmsi_list)} bateau(x) validé(s)</strong><br>
        <small style='opacity:0.8;'>Filtres réinitialisés - Mise à jour de la carte en cours...</small>
    </div>
    """

    print(f"✅ Sélection validée: {len(selected_mmsi_list)} bateau(x)")
    print(f"   MMSIs: {selected_mmsi_list}")
    print("🔄 Filtres réinitialisés automatiquement")

    # Déclencher la mise à jour en appelant le handler
    if 'update_boat_info_multiple' in globals():
        update_boat_info_multiple({'new': tuple(current_selection)})
    elif 'update_boat_info_on_validation' in globals():
        update_boat_info_on_validation(None)

# Connecter le bouton
boat_validate_btn.on_click(on_validate_selection)

**Fonction update_boat_info_on_validation - Gestionnaire principal qui met à jour toutes les informations (météo, localisation, carte) après validation de la sélection**

In [ ]:
def update_boat_info_on_validation(button):
    """
    Met à jour météo + localisation + carte UNIQUEMENT après validation
    Cette fonction est déclenchée par le clic sur le bouton de validation
    """
    global current_boat, current_weather, current_location
    global last_selected_mmsi, last_selected_name, selected_mmsi_list

    # Récupérer la sélection actuelle du widget
    current_selection = list(boat_selector.value) if boat_selector.value else []
    selected_mmsi_list = [str(mmsi) for mmsi in current_selection if mmsi is not None]

    # Mettre à jour l'info de sélection
    if not selected_mmsi_list:
        boat_selection_info.value = """
        <div style='padding:10px; background:#fff3e0; border-radius:6px; border-left:4px solid #f57c00;
                    color:#e65100; font-size:13px; font-weight:500; margin:8px 0;'>
            ⚠️ Aucun bateau sélectionné - Sélectionnez au moins un bateau
        </div>
        """

        # Réinitialiser les sorties
        weather_output.value = """
        <div style='padding: 15px; background: #e8f4f8; border-radius: 8px; border-left: 4px solid #0288d1;
                    color: #01579b; font-weight: 500;'>
            <strong>🌤️ Météo</strong><br>
            Sélectionnez et validez un ou plusieurs bateaux
        </div>
        """
        location_output.value = """
        <div style='padding: 15px; background: #e8f5e9; border-radius: 8px; border-left: 4px solid #4caf50;
                    color: #1b5e20; font-weight: 500;'>
            <strong>🌍 Localisation</strong><br>
            Sélectionnez et validez un ou plusieurs bateaux
        </div>
        """

        with map_output:
            clear_output(wait=True)
            try:
                if 'df_ais_global' in globals() and df_ais_global is not None and not df_ais_global.empty:
                    df_last_positions = get_latest_vessel_positions()
                    df_valid = df_last_positions[
                        (df_last_positions['Position_LatitudeDeg'].notna()) &
                        (df_last_positions['Position_LongitudeDeg'].notna()) &
                        (df_last_positions['Position_LatitudeDeg'] != 0) &
                        (df_last_positions['Position_LongitudeDeg'] != 0)
                    ].copy()

                    live_df = pd.DataFrame({
                        'mmsi': df_valid['Mmsi'].astype(str),
                        'boat_name': df_valid['Name'].fillna('Sans nom'),
                        'latitude': df_valid['Position_LatitudeDeg'],
                        'longitude': df_valid['Position_LongitudeDeg'],
                        'boat_type': df_valid['ShipTypeLabel'].fillna('Non déclaré'),
                        'flag_country': df_valid['Pays'].fillna('Inconnu')
                    })
                else:
                    live_df = pd.DataFrame(columns=['mmsi', 'boat_name', 'latitude', 'longitude', 'boat_type', 'flag_country'])
            except Exception:
                live_df = pd.DataFrame(columns=['mmsi', 'boat_name', 'latitude', 'longitude', 'boat_type', 'flag_country'])

            display(create_boat_map(live_df, selected_mmsi=None, date_filter=None))
        return

    print(f"✅ Validation: {len(selected_mmsi_list)} bateau(x) sélectionné(s): {selected_mmsi_list}")

    boat_selection_info.value = f"""
    <div style='padding:12px 15px; background:#d4edda; border-radius:8px;
                border-left:4px solid #28a745; color:#155724; font-size:13px;
                margin:10px 0; box-shadow: 0 2px 6px rgba(40, 167, 69, 0.2);'>
        ✅ <strong>{len(selected_mmsi_list)} bateau(x) validé(s)</strong><br>
        <small style='opacity:0.8;'>Filtres réinitialisés - Mise à jour de la carte en cours...</small>
    </div>
    """

    weather_cards = []
    location_cards = []
    vessel_data_list = []

    # Palette de couleurs pour les cartes
    color_palette = [
        {'bg': '#e3f2fd', 'border': '#1976d2', 'text': '#0d47a1'},
        {'bg': '#e8f5e9', 'border': '#388e3c', 'text': '#1b5e20'},
        {'bg': '#fff3e0', 'border': '#f57c00', 'text': '#e65100'},
        {'bg': '#f3e5f5', 'border': '#7b1fa2', 'text': '#4a148c'},
        {'bg': '#fce4ec', 'border': '#c2185b', 'text': '#880e4f'},
        {'bg': '#e0f2f1', 'border': '#00897b', 'text': '#004d40'},
        {'bg': '#fff9c4', 'border': '#fbc02d', 'text': '#f57f17'},
        {'bg': '#ede7f6', 'border': '#5e35b1', 'text': '#311b92'},
        {'bg': '#e0f7fa', 'border': '#0097a7', 'text': '#006064'},
        {'bg': '#ffebee', 'border': '#d32f2f', 'text': '#b71c1c'}
    ]

    for idx, mmsi_str in enumerate(selected_mmsi_list):
        try:
            # Chercher le navire dans df_ais_global
            if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
                continue

            vessel_row = df_ais_global[df_ais_global['Mmsi'].astype(str) == mmsi_str]
            if vessel_row.empty:
                continue

            # Prendre la dernière position
            vessel_row = vessel_row.sort_values('datetime').tail(1)
            lat = float(vessel_row.iloc[0]['Position_LatitudeDeg'])
            lon = float(vessel_row.iloc[0]['Position_LongitudeDeg'])
            vessel_name = vessel_row.iloc[0]['Name'] or f'MMSI_{mmsi_str}'

            if lat == 0 and lon == 0:
                continue

            vessel_data_list.append({
                'mmsi': mmsi_str,
                'name': vessel_name,
                'lat': lat,
                'lon': lon
            })

            # Couleur de la carte
            colors = color_palette[idx % len(color_palette)]

            # carte météo individuelle
            try:
                w = get_weather_data(lat, lon, OPENWEATHER_API_KEY)
                weather_info = format_weather_info(w)

                weather_card = f"""
                <div style='margin:8px 0; padding:12px; background:{colors['bg']};
                           border-radius:8px; border-left:4px solid {colors['border']};
                           box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
                    <div style='color:{colors['text']}; font-weight:bold; font-size:14px; margin-bottom:8px;'>
                        🚢 {vessel_name} <span style='font-size:11px; font-weight:normal;'>(MMSI: {mmsi_str})</span>
                    </div>
                    <div style='color:{colors['text']}; font-size:13px; line-height:1.6;'>
                        {weather_info}
                    </div>
                </div>
                """
                weather_cards.append(weather_card)

            except Exception as e:
                weather_card = f"""
                <div style='margin:8px 0; padding:12px; background:#ffebee;
                           border-radius:8px; border-left:4px solid #d32f2f;'>
                    <div style='color:#c62828; font-weight:bold; font-size:14px; margin-bottom:4px;'>
                        🚢 {vessel_name}
                    </div>
                    <div style='color:#c62828; font-size:13px;'>
                        ❌ Erreur météo: {str(e)[:50]}
                    </div>
                </div>
                """
                weather_cards.append(weather_card)

            # carte localisation individuelle
            try:
                ldata = get_location_info(lat, lon)
                loc_info = format_location_info(ldata)

                location_card = f"""
                <div style='margin:8px 0; padding:12px; background:{colors['bg']};
                           border-radius:8px; border-left:4px solid {colors['border']};
                           box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
                    <div style='color:{colors['text']}; font-weight:bold; font-size:14px; margin-bottom:8px;'>
                        🚢 {vessel_name} <span style='font-size:11px; font-weight:normal;'>(MMSI: {mmsi_str})</span>
                    </div>
                    <div style='color:{colors['text']}; font-size:13px; line-height:1.6;'>
                        {loc_info}
                    </div>
                </div>
                """
                location_cards.append(location_card)

            except Exception as e:
                location_card = f"""
                <div style='margin:8px 0; padding:12px; background:#ffebee;
                           border-radius:8px; border-left:4px solid #d32f2f;'>
                    <div style='color:#c62828; font-weight:bold; font-size:14px; margin-bottom:4px;'>
                        🚢 {vessel_name}
                    </div>
                    <div style='color:#c62828; font-size:13px;'>
                        ❌ Erreur localisation: {str(e)[:50]}
                    </div>
                </div>
                """
                location_cards.append(location_card)

        except Exception as e:
            print(f"❌ Erreur pour MMSI {mmsi_str}: {e}")
            continue

    # Assemblage des cartes
    if weather_cards:
        weather_html = f"""
        <div style='padding:10px; background:#ffffff; border-radius:8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);'>
            <h3 style='color:#1a1a1a; margin:0 0 12px 0; font-size:16px; font-weight:600;'>
                🌤️ Météo - {len(weather_cards)} bateau(x)
            </h3>
            {''.join(weather_cards)}
        </div>
        """
    else:
        weather_html = """
        <div style='padding:15px; background:#ffebee; border-radius:8px; color:#c62828;'>
            ❌ Aucune donnée météo disponible
        </div>
        """

    if location_cards:
        location_html = f"""
        <div style='padding:10px; background:#ffffff; border-radius:8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);'>
            <h3 style='color:#1a1a1a; margin:0 0 12px 0; font-size:16px; font-weight:600;'>
                🌍 Localisation - {len(location_cards)} bateau(x)
            </h3>
            {''.join(location_cards)}
        </div>
        """
    else:
        location_html = """
        <div style='padding:15px; background:#ffebee; border-radius:8px; color:#c62828;'>
            ❌ Aucune donnée de localisation disponible
        </div>
        """

    weather_output.value = weather_html
    location_output.value = location_html

    # Mettre à jour last_selected pour le premier bateau (compatibilité)
    if vessel_data_list:
        last_selected_mmsi = vessel_data_list[0]['mmsi']
        last_selected_name = vessel_data_list[0]['name']

    # Création de la carte avec tous les bateaux sélectionnés
    with map_output:
        clear_output(wait=True)
        try:
            if 'df_ais_global' in globals() and df_ais_global is not None and not df_ais_global.empty:
                df_last_positions = get_latest_vessel_positions()
                df_valid = df_last_positions[
                    (df_last_positions['Position_LatitudeDeg'].notna()) &
                    (df_last_positions['Position_LongitudeDeg'].notna()) &
                    (df_last_positions['Position_LatitudeDeg'] != 0) &
                    (df_last_positions['Position_LongitudeDeg'] != 0)
                ].copy()

                live_df = pd.DataFrame({
                    'mmsi': df_valid['Mmsi'].astype(str),
                    'boat_name': df_valid['Name'].fillna('Sans nom'),
                    'latitude': df_valid['Position_LatitudeDeg'],
                    'longitude': df_valid['Position_LongitudeDeg'],
                    'boat_type': df_valid['ShipTypeLabel'].fillna('Non déclaré'),
                    'flag_country': df_valid['Pays'].fillna('Inconnu')
                })
            else:
                live_df = pd.DataFrame(columns=['mmsi', 'boat_name', 'latitude', 'longitude', 'boat_type', 'flag_country'])
        except Exception:
            live_df = pd.DataFrame(columns=['mmsi', 'boat_name', 'latitude', 'longitude', 'boat_type', 'flag_country'])

        # Passer la liste de tous les MMSI sélectionnés à create_boat_map
        display(create_boat_map(live_df, selected_mmsi_list=selected_mmsi_list, date_filter=None))


# Connexion du gestionnaire
boat_validate_btn.on_click(update_boat_info_on_validation)


# Développement des fonctions et configuration de l'agent IA


Helper résolution coordonnées navire sélectionné (nom/MMSI → lat/lon)

In [ ]:
def resolve_selection_to_coords(selection_label):
    """
    Retrouve (lat, lon, name) d'un bateau sélectionné en se basant sur df_ais_global (données parquet).
    - selection_label est la valeur du Dropdown (actuellement le nom ou "MMSI NNN...")
    - retourne (lat, lon, name) ou (None, None, None) si non trouvé
    """
    if not selection_label:
        return (None, None, None)


    if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
        return (None, None, None)

    # Recherche par nom
    vessel_row = df_ais_global[df_ais_global['Name'].fillna('').str.strip() == selection_label]
    if not vessel_row.empty:
        # Prendre la dernière position
        vessel_row = vessel_row.sort_values('datetime').tail(1)
        lat = float(vessel_row.iloc[0]['Position_LatitudeDeg'])
        lon = float(vessel_row.iloc[0]['Position_LongitudeDeg'])
        name = vessel_row.iloc[0]['Name'] or ''
        return (lat, lon, name)

    # Recherche par MMSI
    if selection_label.startswith('MMSI '):
        try:
            mmsi_str = selection_label.split('MMSI ', 1)[1]
            vessel_row = df_ais_global[df_ais_global['Mmsi'].astype(str) == mmsi_str]
            if not vessel_row.empty:
                # Prendre la dernière position
                vessel_row = vessel_row.sort_values('datetime').tail(1)
                lat = float(vessel_row.iloc[0]['Position_LatitudeDeg'])
                lon = float(vessel_row.iloc[0]['Position_LongitudeDeg'])
                name = vessel_row.iloc[0]['Name'] or selection_label
                return (lat, lon, name)
        except Exception:
            pass

    return (None, None, None)

**Fonction refresh_ui_from_stream - Rafraîchissement de l'interface utilisateur avec cache optimisé, clustering intelligent et application des filtres (date, type, pays)**

In [ ]:
def refresh_ui_from_stream(selected_mmsi=None, max_ships=100, selected_date=None,
                          vessel_type='Tous', country='Tous', force_refresh=False):
    """
    Rafraîchit l'interface avec cache optimisé et clustering intelligent

    Args:
        selected_mmsi: MMSI du navire sélectionné
        max_ships: Nombre maximum de navires à afficher
        selected_date: Date spécifique à afficher
        vessel_type: Filtre par type de navire
        country: Filtre par pays
        force_refresh: Force le recalcul sans utiliser le cache
    """
    import time
    start_time = time.time()

    if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
        print('❌ Aucune donnée parquet disponible (df_ais_global)')
        return

    # Nettoyer le cache périodiquement
    data_cache.clear_old_entries()

    # utiliser le cache pour df_last_positions
    if force_refresh:
        print("🔄 Rafraîchissement forcé - ignore le cache")
        data_cache.clear()

    df_valid = get_last_positions_cached(
        date_filter=selected_date,
        vessel_type=vessel_type,
        country=country
    )

    if df_valid.empty:
        print(f'❌ Aucune donnée après filtrage')
        with map_output:
            clear_output(wait=True)
            # Récupérer l'état des éphémérides
            show_ephemeris = show_ephemeris_global if 'show_ephemeris_global' in globals() else False
            display(create_boat_map(pd.DataFrame(), selected_mmsi=selected_mmsi, use_clustering=False, date_filter=selected_date, show_ephemeris=show_ephemeris))
        return

    # Limiter le nombre de navires si nécessaire
    if max_ships is not None and len(df_valid) > max_ships:
        print(f"⚠️ Limitation à {max_ships} navires (sur {len(df_valid)} disponibles)")
        if selected_mmsi and str(selected_mmsi) in df_valid['Mmsi'].astype(str).values:
            # S'assurer que le navire sélectionné est inclus
            selected_row = df_valid[df_valid['Mmsi'].astype(str) == str(selected_mmsi)]
            other_rows = df_valid[df_valid['Mmsi'].astype(str) != str(selected_mmsi)]
            df_valid = pd.concat([selected_row, other_rows.head(max_ships - 1)])
        else:
            df_valid = df_valid.head(max_ships)

    # création du dataframe pour l'affichage
    live_df = pd.DataFrame({
        'mmsi': df_valid['Mmsi'].astype(str),
        'boat_name': df_valid['Name'].fillna('').astype(str).str.strip(),
        'latitude': df_valid['Position_LatitudeDeg'],
        'longitude': df_valid['Position_LongitudeDeg'],
        'boat_type': df_valid['ShipTypeLabel'].fillna('Non déclaré').apply(
            lambda x: 'Type non renseigné' if str(x).strip() == 'Unknown' else str(x)
        ),
        'flag_country': df_valid['Pays'].fillna('Inconnu')
    })

    # Nettoyer les noms vides ou invalides et remplacer par "Nom non renseigné"
    live_df['boat_name'] = live_df['boat_name'].replace(['', 'nan', 'None', 'UNKNOWN'], '')
    live_df['boat_name'] = live_df['boat_name'].apply(
        lambda x: 'Nom non renseigné' if not x or x == '' else x
    )

    # créer la carte avec clustering intelligent
    vessel_count = len(live_df)
    use_clustering = vessel_count > 100  # Activer clustering si > 100 navires

    # Récupérer l'état des éphémérides
    show_ephemeris = show_ephemeris_global if 'show_ephemeris_global' in globals() else False

    print(f"🗺️ Création de la carte...")
    map_obj = create_boat_map(
        live_df,
        selected_mmsi=selected_mmsi,
        use_clustering=use_clustering,
        cluster_threshold=100,
        date_filter=selected_date,
        show_ephemeris=show_ephemeris
    )


    # Afficher la carte (compat Colab)
    with map_output:
        clear_output(wait=True)
        display(map_obj)


    # maj du selectmultiple avec les noms
    def labelize(row):
        n = (row['boat_name'] or '').strip()
        mm = row['mmsi']
        # Si le nom est vide ou invalide, afficher "Nom pas renseigné"
        if not n or n.lower() in ['sans nom', 'unknown', 'nan', 'none', ''] or n == 'Nom non renseigné':
            return f'Nom pas renseigné — MMSI {mm}'
        else:
            return f'{n} — MMSI {mm}'

    # Créer une colonne pour le tri : 0 si nom valide, 1 si nom pas renseigné
    live_df['sort_key'] = live_df['boat_name'].apply(
        lambda x: 1 if (not x or str(x).strip().lower() in ['sans nom', 'unknown', 'nan', 'none', ''] or x == 'Nom non renseigné') else 0
    )

    # Trier : d'abord ceux avec nom (sort_key=0), puis ceux sans nom (sort_key=1)
    live_df_sorted = live_df.sort_values(['sort_key', 'boat_name'], ascending=[True, True])

    names = [(labelize(r), r['mmsi']) for _, r in live_df_sorted.iterrows() if pd.notna(r['mmsi'])]

    if 'boat_selector' in globals():
        boat_selector.options = names

    elapsed = time.time() - start_time
    print(f"⏱️ Temps total: {elapsed:.2f}s")

**Fonctions utilitaires - Calcul de distance (Haversine), recherche de navires (vessel_candidates), API météo (OpenWeather, EMODnet) et conversions échelle Beaufort**

In [ ]:
import unicodedata

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = (math.radians(lat1), math.radians(lat2))
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def _norm(s: str) -> str:
    if not s:
        return ''
    s = unicodedata.normalize('NFKD', s)
    s = ''.join((ch for ch in s if not unicodedata.combining(ch)))
    return s.lower().strip()

def vessel_candidates(query: str, k: int=10):
    """Recherche de candidats navires depuis df_ais_global"""

    if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
        return []

    qn = _norm(query or '')
    by_name = []

    # Prendre les dernières positions par MMSI
    df_last_positions = get_latest_vessel_positions()
    df_valid = df_last_positions[
        (df_last_positions['Position_LatitudeDeg'].notna()) &
        (df_last_positions['Position_LongitudeDeg'].notna()) &
        (df_last_positions['Position_LatitudeDeg'] != 0) &
        (df_last_positions['Position_LongitudeDeg'] != 0)
    ].copy()

    for _, row in df_valid.iterrows():
        name = (row.get('Name') or '').strip()
        if not name:
            continue
        nn = _norm(name)
        sub = qn in nn if qn else False
        ratio = difflib.SequenceMatcher(None, qn, nn).ratio() if qn else 0
        score = 1.0 if sub else ratio
        by_name.append((score, str(row['Mmsi']), name))
    by_name.sort(reverse=True, key=lambda x: x[0])
    return [(m, n, s) for s, m, n in by_name[:k]]


def openweather_current(lat: float, lon: float, api_key: str):
    url = 'https://api.openweathermap.org/data/2.5/weather'
    params = {'lat': lat, 'lon': lon, 'appid': api_key, 'units': 'metric', 'lang': 'fr'}
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

def emodnet_depth_sample(lat: float, lon: float):
    wkt = f'POINT({lon} {lat})'
    url = 'https://rest.emodnet-bathymetry.eu/depth_sample'
    r = requests.get(url, params={'geom': wkt}, timeout=10)
    r.raise_for_status()
    return r.json()

def bft_from_ms(v_ms: float) -> int:
    if v_ms is None:
        return None
    bounds = [0.3, 1.6, 3.4, 5.5, 8.0, 10.8, 13.9, 17.2, 20.8, 24.5, 28.5, 32.7]
    for i, t in enumerate(bounds):
        if v_ms < t:
            return i
    return 12

def bft_label(b: int) -> str:
    labels = {0: 'Calme', 1: 'Très légère brise', 2: 'Légère brise', 3: 'Brise légère', 4: 'Petite brise', 5: 'Bonne brise', 6: 'Vent frais', 7: 'Grand frais', 8: 'Coup de vent', 9: 'Fort coup de vent', 10: 'Tempête', 11: 'Violente tempête', 12: 'Ouragan'}
    return labels.get(b, '')

**Fonction _load_archive_points - Charge les points de position d'un navire depuis df_ais_global à partir d'un timestamp donné**

In [ ]:
def _load_archive_points(mmsi: str, since_ts: float):
    """Charge depuis df_ais_global les points >= since_ts pour ce MMSI."""

    if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
        return []

    key = str(mmsi)
    out = []

    # Filtrer les données pour ce MMSI
    vessel_data = df_ais_global[df_ais_global['Mmsi'].astype(str) == key].copy()

    if vessel_data.empty:
        return []

    # Convertir datetime en timestamp si nécessaire
    if not pd.api.types.is_datetime64_any_dtype(vessel_data['datetime']):
        vessel_data['datetime'] = pd.to_datetime(vessel_data['datetime'])

    # Convertir en timestamp Unix
    vessel_data['ts'] = vessel_data['datetime'].astype('int64') // 10**9

    # Filtrer par timestamp
    vessel_data = vessel_data[vessel_data['ts'] >= since_ts]

    # Convertir en format attendu
    for _, row in vessel_data.iterrows():
        rec = {
            'mmsi': key,
            'ts': row['ts'],
            'lat': row.get('Position_LatitudeDeg'),
            'lon': row.get('Position_LongitudeDeg'),
            'sog': row.get('SogKt'),
            'cog': row.get('CogDeg')
        }
        out.append(rec)

    out.sort(key=lambda r: r['ts'])
    return out

**Fonctions d'analyse de trajectoire - Calcul de différence angulaire normalisée et métriques de parcours (longueur, rectitude, virages, vitesse, gaps)**

In [ ]:
def _ang_diff(a, b):
    """Différence angulaire normalisée [-180,180]."""
    if a is None or b is None:
        return None
    d = (a - b + 180) % 360 - 180
    return d

def _path_metrics(points):
    """
    points: liste triée par ts asc de dicts {ts,lat,lon,sog,cog}
    Retourne métriques utiles: longueur parcourue, rectitude, somme des virages, fraction <4kn, gaps, etc.
    """
    if len(points) < 3:
        return {'ok': False, 'reason': 'not_enough_points'}
    total_m = 0.0
    total_turn = 0.0
    below4 = 0
    gaps = []
    first = points[0]
    last = points[-1]
    for i in range(1, len(points)):
        p0, p1 = (points[i - 1], points[i])
        try:
            dm = haversine_m(p0['lat'], p0['lon'], p1['lat'], p1['lon'])
            total_m += dm
        except Exception:
            pass
        dth = _ang_diff(p1.get('cog'), p0.get('cog'))
        if dth is not None:
            total_turn += abs(dth)
        if p1.get('sog') is not None and p1['sog'] < 4.0:
            below4 += 1
        if p1['ts'] - p0['ts'] > 15 * 60:
            gaps.append(p1['ts'] - p0['ts'])
    try:
        gc_m = haversine_m(first['lat'], first['lon'], last['lat'], last['lon'])
        straightness_ratio = total_m / gc_m if gc_m > 1.0 else float('inf')
    except Exception:
        gc_m, straightness_ratio = (None, None)
    frac_below4 = below4 / max(1, len(points))
    duration_min = (last['ts'] - first['ts']) / 60.0
    return {'ok': True, 'n': len(points), 'duration_min': duration_min, 'track_len_m': total_m, 'gc_m': gc_m, 'straightness_ratio': straightness_ratio, 'total_turn_deg': total_turn, 'frac_below4': frac_below4, 'gaps_sec': gaps}

**Dictionnaires de décodage AIS - Conversion MID vers pays et décodage des codes de type de navire selon la norme ITU-R M.1371-5**

In [ ]:
# Mapping MMSI prefix (3 digits) to country
mmsi_to_country = {
    str(k): v for k, v in MID_TO_COUNTRY.items()
}

# Décodage AIS du type d'un navire (based sur le standard ITU-R M.1371-5)
ais_type_decode = {
    # Special craft types (20-29)
    20: "Wing in ground (WIG)",
    21: "Wing in ground (WIG) - Hazardous category A",
    22: "Wing in ground (WIG) - Hazardous category B",
    23: "Wing in ground (WIG) - Hazardous category C",
    24: "Wing in ground (WIG) - Hazardous category D",
    25: "Wing in ground (WIG) - Reserved",
    26: "Wing in ground (WIG) - Reserved",
    27: "Wing in ground (WIG) - Reserved",
    28: "Wing in ground (WIG) - Reserved",
    29: "Wing in ground (WIG) - No additional info",

    # Fishing (30)
    30: "Fishing",

    # Towing/Special ops (31-35)
    31: "Towing",
    32: "Towing: length exceeds 200m or breadth exceeds 25m",
    33: "Dredging or underwater ops",
    34: "Diving ops",
    35: "Military ops",

    # Sailing (36)
    36: "Sailing",

    # Pleasure Craft (37)
    37: "Pleasure Craft",

    # Reserved (38-39)
    38: "Reserved",
    39: "Reserved",

    # High speed craft (40-49)
    40: "High speed craft (HSC)",
    41: "High speed craft (HSC) - Hazardous category A",
    42: "High speed craft (HSC) - Hazardous category B",
    43: "High speed craft (HSC) - Hazardous category C",
    44: "High speed craft (HSC) - Hazardous category D",
    45: "High speed craft (HSC) - Reserved",
    46: "High speed craft (HSC) - Reserved",
    47: "High speed craft (HSC) - Reserved",
    48: "High speed craft (HSC) - Reserved",
    49: "High speed craft (HSC) - No additional info",

    # Pilot Vessel (50)
    50: "Pilot Vessel",

    # Search and Rescue (51)
    51: "Search and Rescue vessel",

    # Tug (52)
    52: "Tug",

    # Port Tender (53)
    53: "Port Tender",

    # Anti-pollution (54)
    54: "Anti-pollution equipment",

    # Law Enforcement (55)
    55: "Law Enforcement",

    # Spare - Local Vessel (56-57)
    56: "Spare - Local Vessel",
    57: "Spare - Local Vessel",

    # Medical Transport (58)
    58: "Medical Transport",

    # Non-combatant ship (59)
    59: "Non-combatant ship (RR Resolution No. 18)",

    # Passenger (60-69)
    60: "Passenger",
    61: "Passenger - Hazardous category A",
    62: "Passenger - Hazardous category B",
    63: "Passenger - Hazardous category C",
    64: "Passenger - Hazardous category D",
    65: "Passenger - Reserved",
    66: "Passenger - Reserved",
    67: "Passenger - Reserved",
    68: "Passenger - Reserved",
    69: "Passenger - No additional info",

    # Cargo (70-79)
    70: "Cargo",
    71: "Cargo - Hazardous category A",
    72: "Cargo - Hazardous category B",
    73: "Cargo - Hazardous category C",
    74: "Cargo - Hazardous category D",
    75: "Cargo - Reserved",
    76: "Cargo - Reserved",
    77: "Cargo - Reserved",
    78: "Cargo - Reserved",
    79: "Cargo - No additional info",

    # Tanker (80-89)
    80: "Tanker",
    81: "Tanker - Hazardous category A",
    82: "Tanker - Hazardous category B",
    83: "Tanker - Hazardous category C",
    84: "Tanker - Hazardous category D",
    85: "Tanker - Reserved",
    86: "Tanker - Reserved",
    87: "Tanker - Reserved",
    88: "Tanker - Reserved",
    89: "Tanker - No additional info",

    # Other types (90-99)
    90: "Other Type",
    91: "Other Type - Hazardous category A",
    92: "Other Type - Hazardous category B",
    93: "Other Type - Hazardous category C",
    94: "Other Type - Hazardous category D",
    95: "Other Type - Reserved",
    96: "Other Type - Reserved",
    97: "Other Type - Reserved",
    98: "Other Type - Reserved",
    99: "Other Type - No additional info",
}

### 🧰 Les tools de l'agent IA

Chaque *tool* est une fonction Python exposée à l'agent LangChain : il choisit lui-même lesquels appeler pour répondre à une question d'opérateur formulée en langage naturel. C'est ici que vit la logique métier du projet — détection, corrélation et aide à la décision.

Le starter pack du hackathon fournissait un seul tool d'exemple (`rechercher_navire`, en fin de section). Les **25 tools** qui suivent ont été développés par l'équipe pendant les 24 h.

> Cette section formait à l'origine une unique cellule de près de 7 000 lignes. Elle est découpée ici par thème, **sans aucune modification du code ni de l'ordre d'exécution**.

#### Socle commun

Helpers géométriques (`haversine_km`), accès à la dernière position AIS d'un navire et table des zones maritimes de référence.

In [ ]:
# 🛠️ TOOL LANGCHAIN SIMPLE - EXEMPLE DE BASE
# Cette cellule contient un exemple très simple de tool LangChain.
# Les équipes devront créer leurs propres tools.
# Un tool LangChain est une fonction Python décorée avec @tool qui permet
# à l'agent IA d'effectuer des actions spécifiques sur les données.

from langchain_core.tools import tool
from typing import Dict, Any

# === TOOL D'EXEMPLE : RECHERCHE BASIQUE D'UN NAVIRE ===
# Ce tool est un exemple très simple pour comprendre comment créer un tool.
# Il permet de rechercher un navire par son MMSI dans les données AIS.
# Les équipes devront créer leurs propres tools pour :
# - Croiser les données AIS et RF
# - Détecter les navires sans AIS
# - Analyser les trajectoires suspectes
# - Identifier les comportements suspects
# - Et bien d'autres cas d'usage...

!pip install wikipedia

import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# Rayon moyen de la Terre en km
EARTH_RADIUS_KM = 6371.0

def haversine_km(lat1, lon1, lat2, lon2):
    """
    Distance great-circle en km entre deux points (lat, lon) en degrés.
    """
    lat1_rad, lon1_rad = radians(lat1), radians(lon1)
    lat2_rad, lon2_rad = radians(lat2), radians(lon2)
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = sin(dlat / 2) ** 2 + cos(lat1_rad) * cos(lat2_rad) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return EARTH_RADIUS_KM * c

def get_last_position_for_vessel(identifier: str):
    """
    Récupère la dernière position AIS d'un navire à partir de son MMSI ou de son nom.

    - si 'identifier' est tout numérique -> interprété comme MMSI
    - sinon -> recherche par Name (contains, insensible à la casse)
    """
    if "df_ais_global" not in globals() or df_ais_global is None or df_ais_global.empty:
        raise ValueError("df_ais_global non disponible ou vide.")

    df = df_ais_global.copy()
    df["datetime_utc"] = pd.to_datetime(df["datetime"], utc=True)

    ident = str(identifier).strip()

    if ident.isdigit():
        df_match = df[df["Mmsi"].astype(str) == ident]
    else:
        df_match = df[df["Name"].astype(str).str.contains(ident, case=False, na=False)]

    if df_match.empty:
        return None

    idx_last = df_match["datetime_utc"].idxmax()
    row = df_match.loc[idx_last]

    return {
        "Mmsi": row.get("Mmsi"),
        "Name": row.get("Name"),
        "Country": row.get("Country"),
        "ShipType": row.get("ShipTypeLabel"),
        "lat": row.get("Position_LatitudeDeg"),
        "lon": row.get("Position_LongitudeDeg"),
        "datetime_utc": row.get("datetime_utc"),
    }


def find_best_sat_pass_for_position(lat: float, lon: float, t_ref, time_window_hours: int = 24):
    """
    Trouve le passage JAPETUS le plus pertinent au-dessus d'une position donnée
    dans une fenêtre temporelle centrée sur t_ref.
    """
    if "df_ephemeris_global" not in globals() or df_ephemeris_global is None or df_ephemeris_global.empty:
        raise ValueError("df_ephemeris_global non disponible ou vide.")

    df_eph = df_ephemeris_global.copy()
    df_eph["StartTime_utc"] = pd.to_datetime(df_eph["StartTime"], utc=True)
    df_eph["EndTime_utc"] = pd.to_datetime(df_eph["EndTime"], utc=True)

    dt_window = pd.Timedelta(hours=time_window_hours)
    df_eph = df_eph[
        (df_eph["EndTime_utc"] >= t_ref - dt_window) &
        (df_eph["StartTime_utc"] <= t_ref + dt_window)
    ]
    if df_eph.empty:
        return None

    # distance moyenne navire <-> segment (Start, End)
    dists = []
    for idx, row in df_eph.iterrows():
        d1 = haversine_km(lat, lon, row["StartLatitude"], row["StartLongitude"])
        d2 = haversine_km(lat, lon, row["EndLatitude"], row["EndLongitude"])
        dists.append((idx, (d1 + d2) / 2))

    if not dists:
        return None

    idx_best, best_dist = min(dists, key=lambda x: x[1])

    # si le passage est vraiment trop loin, on considère que c'est peu pertinent
    if best_dist > 500:  # km
        return None

    best_row = df_eph.loc[idx_best]
    return {
        "Source": best_row.get("Source"),
        "Target": best_row.get("Target"),
        "StartTime": best_row.get("StartTime_utc"),
        "EndTime": best_row.get("EndTime_utc"),
        "Duration": best_row.get("Duration"),
        "Distance_km": best_row.get("Distance_km"),
        "Average_Speedkmh": best_row.get("Average_Speedkmh"),
        "approx_distance_km": best_dist,
    }


def get_combined_contacts_in_zone(
    lat_min: float,
    lat_max: float,
    lon_min: float,
    lon_max: float,
    date: str = None,
    rayon_km: float = 10.0,
    delta_t_minutes: int = 60
) -> dict:
    """
    Combine AIS + RF + éphémérides pour une zone donnée.

    Args:
        lat_min, lat_max, lon_min, lon_max:
            Bornes du rectangle géographique (degrés).
        date:
            Date 'YYYY-MM-DD'. Si None, on travaille sur toute la période.
        rayon_km:
            Rayon max pour associer une détection RF à un navire AIS (km).
        delta_t_minutes:
            Fenêtre temporelle +- en minutes pour associer RF <-> AIS.

    Returns:
        dict avec 3 clés:
            - "ais_vessels": DataFrame des navires AIS dans la zone
            - "rf_detections": DataFrame des détections RF dans la zone
              avec tentative d'association à un navire AIS
            - "sat_passes": DataFrame des passages JAPETUS couvrant la zone
    """
    if "df_ais_global" not in globals() or df_ais_global is None or df_ais_global.empty:
        raise ValueError("df_ais_global indisponible ou vide.")
    if "df_rf_global" not in globals() or df_rf_global is None or df_rf_global.empty:
        raise ValueError("df_rf_global indisponible ou vide.")
    if "df_ephemeris_global" not in globals() or df_ephemeris_global is None or df_ephemeris_global.empty:
        raise ValueError("df_ephemeris_global indisponible ou vide.")

    # Normaliser les bornes
    if lat_min > lat_max:
        lat_min, lat_max = lat_max, lat_min
    if lon_min > lon_max:
        lon_min, lon_max = lon_max, lon_min

    # --------- AIS : navires dans la zone ---------
    df_ais = df_ais_global.copy()
    df_ais["datetime"] = pd.to_datetime(df_ais["datetime"])

    date_param = None if date is None else str(date).strip()
    if date_param:
        df_ais = df_ais[df_ais["datetime"].dt.date.astype(str) == date_param]
    else:
        # on prend la dernière position de chaque navire sur la période
        idx_last = df_ais.groupby("Mmsi")["datetime"].idxmax()
        df_ais = df_ais.loc[idx_last]

    # filtre géographique
    df_ais_zone = df_ais[
        (df_ais["Position_LatitudeDeg"] >= lat_min) &
        (df_ais["Position_LatitudeDeg"] <= lat_max) &
        (df_ais["Position_LongitudeDeg"] >= lon_min) &
        (df_ais["Position_LongitudeDeg"] <= lon_max)
    ].copy()

    # dédupliquer : une ligne par navire (dernière position dans la zone)
    if not df_ais_zone.empty:
        idx_last_zone = df_ais_zone.groupby("Mmsi")["datetime"].idxmax()
        df_ais_zone = df_ais_zone.loc[idx_last_zone].copy()

    # --------- RF : détections dans la zone ---------
    df_rf = df_rf_global.copy()
    df_rf["timestamp_utc"] = pd.to_datetime(df_rf["timestamp_utc"])

    if date_param:
        df_rf = df_rf[df_rf["timestamp_utc"].dt.date.astype(str) == date_param]

    df_rf_zone = df_rf[
        (df_rf["latitude_deg"]  >= lat_min) &
        (df_rf["latitude_deg"]  <= lat_max) &
        (df_rf["longitude_deg"] >= lon_min) &
        (df_rf["longitude_deg"] <= lon_max)
    ].copy()

    # --------- Association RF <-> AIS (approx.) ---------
    if not df_rf_zone.empty and not df_ais_zone.empty:
        assoc_mmsi = []
        assoc_name = []
        assoc_country = []
        assoc_dist_km = []

        dt_window = pd.Timedelta(minutes=delta_t_minutes)
        approx_deg = rayon_km / 111.0  # approx en degrés

        for _, rf_row in df_rf_zone.iterrows():
            t0 = rf_row["timestamp_utc"]
            lat0 = rf_row["latitude_deg"]
            lon0 = rf_row["longitude_deg"]

            # candidats en temps
            cand = df_ais_zone[
                (df_ais_zone["datetime"] >= t0 - dt_window) &
                (df_ais_zone["datetime"] <= t0 + dt_window)
            ]
            if cand.empty:
                assoc_mmsi.append(None)
                assoc_name.append(None)
                assoc_country.append(None)
                assoc_dist_km.append(None)
                continue

            # candidats en position (filtre grossier)
            cand = cand[
                (cand["Position_LatitudeDeg"]  >= lat0 - approx_deg) &
                (cand["Position_LatitudeDeg"]  <= lat0 + approx_deg) &
                (cand["Position_LongitudeDeg"] >= lon0 - approx_deg) &
                (cand["Position_LongitudeDeg"] <= lon0 + approx_deg)
            ]
            if cand.empty:
                assoc_mmsi.append(None)
                assoc_name.append(None)
                assoc_country.append(None)
                assoc_dist_km.append(None)
                continue

            # calcul distance haversine et choix du plus proche
            dists = cand.apply(
                lambda r: haversine_km(
                    lat0, lon0,
                    r["Position_LatitudeDeg"], r["Position_LongitudeDeg"]
                ),
                axis=1
            )
            idx_min = dists.idxmin()
            dist_min = dists.loc[idx_min]

            if dist_min <= rayon_km:
                best = cand.loc[idx_min]
                assoc_mmsi.append(best.get("Mmsi"))
                assoc_name.append(best.get("Name"))
                assoc_country.append(best.get("Country"))
                assoc_dist_km.append(dist_min)
            else:
                assoc_mmsi.append(None)
                assoc_name.append(None)
                assoc_country.append(None)
                assoc_dist_km.append(dist_min)

        df_rf_zone = df_rf_zone.copy()
        df_rf_zone["matched_mmsi"] = assoc_mmsi
        df_rf_zone["matched_name"] = assoc_name
        df_rf_zone["matched_country"] = assoc_country
        df_rf_zone["distance_km_to_ais"] = assoc_dist_km
    else:
        # pas de quoi associer
        df_rf_zone["matched_mmsi"] = None
        df_rf_zone["matched_name"] = None
        df_rf_zone["matched_country"] = None
        df_rf_zone["distance_km_to_ais"] = None

    # --------- Éphémérides : passages couvrant la zone ---------
    df_eph = df_ephemeris_global.copy()
    df_eph["StartTime"] = pd.to_datetime(df_eph["StartTime"])
    df_eph["EndTime"] = pd.to_datetime(df_eph["EndTime"])

    if date_param:
        df_eph = df_eph[df_eph["StartTime"].dt.date.astype(str) == date_param]

    if df_eph.empty:
        df_eph_zone = df_eph
    else:
        # approx : bbox du segment (Start, End) qui intersecte notre zone
        min_lat_seg = df_eph[["StartLatitude", "EndLatitude"]].min(axis=1)
        max_lat_seg = df_eph[["StartLatitude", "EndLatitude"]].max(axis=1)
        min_lon_seg = df_eph[["StartLongitude", "EndLongitude"]].min(axis=1)
        max_lon_seg = df_eph[["StartLongitude", "EndLongitude"]].max(axis=1)

        mask = (
            (max_lat_seg >= lat_min) &
            (min_lat_seg <= lat_max) &
            (max_lon_seg >= lon_min) &
            (min_lon_seg <= lon_max)
        )
        df_eph_zone = df_eph[mask].copy()

    return {
        "ais_vessels": df_ais_zone,
        "rf_detections": df_rf_zone,
        "sat_passes": df_eph_zone
    }


def detecter_forces_pays_dans_zone(
    pays: str,
    lat_min: float,
    lat_max: float,
    lon_min: float,
    lon_max: float,
    date: str = None
) -> pd.DataFrame:
    """
    Détecte la présence de navires dans une zone géographique.

    - Si 'pays' est renseigné (ex: "France"), on filtre sur le pavillon,
      puis on essaie d'identifier les navires appartenant à des forces étatiques
      (militaire, garde-côtes, patrouilleur...).
    - Si 'pays' est vide ou None, on retourne TOUS les navires présents
      dans la zone, sans filtrer par pays ni par type de force.

    Args:
        pays (str): Nom du pays (ex: "France", "Italy").
                    Si vide ("") ou None -> tous pays, tous navires.
        lat_min (float): Latitude minimale de la zone (en degrés).
        lat_max (float): Latitude maximale de la zone (en degrés).
        lon_min (float): Longitude minimale de la zone (en degrés).
        lon_max (float): Longitude maximale de la zone (en degrés).
        date (str, optional): Date au format 'YYYY-MM-DD'.
                              Si None ou "", on utilise la dernière position connue de chaque navire.

    Returns:
        pd.DataFrame: Tableau des navires correspondant, avec leurs dernières positions
                      dans la zone et les principales informations.
    """
    if "df_ais_global" not in globals() or df_ais_global is None or df_ais_global.empty:
        raise ValueError("Les données AIS (df_ais_global) ne sont pas disponibles ou sont vides.")

    # Sécurité sur les bornes
    if lat_min > lat_max:
        lat_min, lat_max = lat_max, lat_min
    if lon_min > lon_max:
        lon_min, lon_max = lon_max, lon_min

    df = df_ais_global.copy()

    # Normalisation de la date
    df["datetime"] = pd.to_datetime(df["datetime"])

    # ✅ Gestion de la date
    date_param = None if date is None else str(date).strip()
    if date_param:
        df["date_str"] = df["datetime"].dt.date.astype(str)
        df = df[df["date_str"] == date_param]
    else:
        # Dernière position connue par navire
        idx_last = df.groupby("Mmsi")["datetime"].idxmax()
        df = df.loc[idx_last]

    if df.empty:
        return pd.DataFrame()

    # ✅ Filtre géographique (rectangle)
    df = df[
        (df["Position_LatitudeDeg"] >= lat_min) &
        (df["Position_LatitudeDeg"] <= lat_max) &
        (df["Position_LongitudeDeg"] >= lon_min) &
        (df["Position_LongitudeDeg"] <= lon_max)
    ]

    if df.empty:
        return pd.DataFrame()

    # ✅ Gestion du pays : filtrage seulement si 'pays' est renseigné
    pays_param = None if pays is None else str(pays).strip()
    if pays_param:
        df["pays_norm"] = df["Country"].astype(str).str.strip().str.lower()
        target_pays = pays_param.lower()
        df = df[df["pays_norm"] == target_pays]

        if df.empty:
            return pd.DataFrame()

        # ✅ Heuristique pour identifier des forces étatiques
        force_keywords = [
            "military", "navy", "naval", "patrol", "patrouille", "law enforcement",
            "coast guard", "guard", "sar", "search and rescue",
            "police", "douane", "customs", "gendarmerie"
        ]

        def est_force_etatique(row) -> bool:
            label = str(row.get("ShipTypeLabel", "")).lower()
            name = str(row.get("Name", "")).lower()
            return any(kw in label or kw in name for kw in force_keywords)

        df["is_state_force"] = df.apply(est_force_etatique, axis=1)
        df = df[df["is_state_force"]]

        if df.empty:
            return pd.DataFrame()

    # Dans tous les cas, on garde une seule ligne par navire (la plus récente dans la zone)
    idx_last_zone = df.groupby("Mmsi")["datetime"].idxmax()
    result = df.loc[idx_last_zone].copy()

    # Nettoyage des colonnes temporaires si elles existent
    for col in ["pays_norm", "date_str", "is_state_force"]:
        if col in result.columns:
            result.drop(columns=[col], inplace=True, errors="ignore")

    # Tri par date décroissante
    result = result.sort_values("datetime", ascending=False)

    return result

from langchain_core.tools import tool

def _normalize_str(s: str) -> str:
    """Minuscule + suppression des accents."""
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = ''.join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )
    return s

def _classer_type_navire_ais(row) -> str:
    """
    Classe un navire AIS en {militaire, commercial, peche, autre}
    à partir de ShipTypeNormalized / ShipTypeLabel.
    """
    label = row.get("ShipTypeNormalized") or row.get("ShipTypeLabel") or ""
    txt = _normalize_str(label)

    # pêche
    if "fish" in txt or "peche" in txt:
        return "peche"

    # militaire / forces étatiques / SAR / law enforcement
    mil_keywords = [
        "military", "navy", "naval", "sar", "search and rescue",
        "anti-pollution", "law enforcement", "patrol",
        "coast guard", "guard", "garde", "police", "douane", "gendarmerie"
    ]
    if any(k in txt for k in mil_keywords):
        return "militaire"

    # commercial large (cargo, tanker, passenger, tug, offshore...)
    com_keywords = [
        "cargo", "tanker", "passenger", "pleasure", "offshore",
        "tug", "dredging", "port tender", "local vessel",
        "high speed craft", "hsc", "other", "liner", "ferry"
    ]
    if any(k in txt for k in com_keywords):
        return "commercial"

    return "autre"


def _standardize_type_request(type_navire: str) -> str:
    """
    Normalise la demande utilisateur :
    'militaire', 'commercial', 'pêche', 'peche', 'tous', etc.
    """
    s = _normalize_str(type_navire)
    if s.startswith("mil"):
        return "militaire"
    if s.startswith("pec") or "peche" in s or "fish" in s:
        return "peche"
    if s.startswith("com") or "mercant" in s:
        return "commercial"
    if s in ("tous", "tout", "all", "*", ""):
        return "tous"
    # fallback
    return s
import unicodedata

#### Rapport opérateur (PDF)

Fiche navire prête à transmettre à l'officier de quart : historique, anomalies, dernière position.

`generer_rapport_pdf`

In [ ]:
# --------- Tool principal ---------

@tool
def generer_rapport_pdf(
    question: str,
    reponse: str,
    titre_rapport: Optional[str] = None,
    nom_fichier: Optional[str] = None,
    format_rapport: str = "pdf",
) -> str:
    """
    Génère un rapport exportable (PDF ou Word) à partir d'une question et de la réponse associée.

    Utilisation prévue :
      - L'assistant répond normalement à une question.
      - Si l'utilisateur demande "fais-en un rapport PDF" ou "fais-en un rapport Word",
        l'assistant rappelle ce tool en lui passant :
          * la question originale
          * le texte de réponse complet
          * éventuellement un titre de rapport
          * le format désiré : "pdf" ou "word"

    Paramètres :
      - question : texte de la question de l'utilisateur (contexte)
      - reponse : texte de la réponse détaillée (analyse, plan, etc.)
      - titre_rapport : titre affiché dans le document (optionnel)
      - nom_fichier : nom de base du fichier sans extension (optionnel).
                      Si None, on fabrique un slug à partir du titre ou de "rapport_navale".
      - format_rapport : "pdf" ou "word"/"docx".
                         Par défaut : "pdf" (compatibilité ascendante).

    Retour :
      - str : message texte indiquant où le fichier a été généré
              (chemin à utiliser côté front pour proposer le téléchargement).
    """
    import os
    import re
    from datetime import datetime

    fmt = (format_rapport or "pdf").lower()
    if fmt == "docx":
        fmt = "word"
    if fmt not in {"pdf", "word"}:
        fmt = "pdf"  # fallback sécurisé

    # --- 1) Préparation du répertoire de sortie ---
    if fmt == "pdf":
        base_dir = "pdf_reports"
    else:
        base_dir = "word_reports"
    os.makedirs(base_dir, exist_ok=True)

    # --- 2) Titre & nom de fichier ---
    if not titre_rapport:
        titre_rapport = "Rapport d'analyse maritime"

    if nom_fichier is None:
        slug_base = re.sub(r"[^a-zA-Z0-9_-]+", "_", titre_rapport).strip("_") or "rapport_navale"
    else:
        slug_base = re.sub(r"[^a-zA-Z0-9_-]+", "_", nom_fichier).strip("_") or "rapport_navale"

    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

    if fmt == "pdf":
        ext = ".pdf"
    else:
        ext = ".docx"

    filename = f"{slug_base}_{timestamp}{ext}"
    filepath = os.path.join(base_dir, filename)

    # --- 3) Génération du document selon le format ---
    if fmt == "pdf":
        # --------- PDF via reportlab ---------
        try:
            from reportlab.lib.pagesizes import A4
            from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
            from reportlab.lib.styles import getSampleStyleSheet
            from reportlab.lib.units import cm
        except ImportError:
            return (
                "❌ Impossible de générer le PDF : la bibliothèque 'reportlab' n'est pas installée. "
                "Installez-la avec : pip install reportlab"
            )

        doc = SimpleDocTemplate(
            filepath,
            pagesize=A4,
            rightMargin=2 * cm,
            leftMargin=2 * cm,
            topMargin=2 * cm,
            bottomMargin=2 * cm,
            title=titre_rapport,
        )

        styles = getSampleStyleSheet()
        story = []

        # Titre
        story.append(Paragraph(titre_rapport, styles["Title"]))
        story.append(Spacer(1, 0.5 * cm))

        # Date
        now_str = datetime.utcnow().strftime("%d/%m/%Y %H:%M UTC")
        story.append(Paragraph(f"Généré le {now_str}", styles["Normal"]))
        story.append(Spacer(1, 0.5 * cm))

        # Question
        story.append(Paragraph("Question initiale :", styles["Heading2"]))
        story.append(Spacer(1, 0.2 * cm))
        story.append(Paragraph(question.replace("\n", "<br/>"), styles["Normal"]))
        story.append(Spacer(1, 0.7 * cm))

        # Réponse
        story.append(Paragraph("Réponse détaillée :", styles["Heading2"]))
        story.append(Spacer(1, 0.2 * cm))
        story.append(Paragraph(reponse.replace("\n", "<br/>"), styles["Normal"]))

        doc.build(story)

        return (
            f"📄 Rapport PDF généré avec succès : {filepath}\n"
            "Vous pouvez le proposer en téléchargement au commandant."
        )

    else:
        # --------- Word (.docx) via python-docx ---------
        try:
            from docx import Document
            from docx.shared import Cm
        except ImportError:
            return (
                "❌ Impossible de générer le rapport Word : la bibliothèque 'python-docx' "
                "n'est pas installée. Installez-la avec : pip install python-docx"
            )

        doc = Document()

        # Marges
        for section in doc.sections:
            section.top_margin = Cm(2)
            section.bottom_margin = Cm(2)
            section.left_margin = Cm(2)
            section.right_margin = Cm(2)

        # Titre
        doc.add_heading(titre_rapport, level=0)

        # Date
        now_str = datetime.utcnow().strftime("%d/%m/%Y %H:%M UTC")
        p_date = doc.add_paragraph()
        p_date.add_run(f"Généré le {now_str}").italic = True

        doc.add_paragraph()  # ligne vide

        # Question
        doc.add_heading("Question initiale :", level=1)
        if question:
            for line in question.split("\n"):
                doc.add_paragraph(line)
        else:
            doc.add_paragraph("(Non fournie)")

        doc.add_paragraph()

        # Réponse
        doc.add_heading("Réponse détaillée :", level=1)
        if reponse:
            for line in reponse.split("\n"):
                doc.add_paragraph(line)
        else:
            doc.add_paragraph("(Non fournie)")

        doc.save(filepath)

        return (
            f"📄 Rapport Word généré avec succès : {filepath}\n"
            "Vous pouvez le proposer en téléchargement au commandant."
        )

#### Filtrage et détection de dark vessels

Filtrage de la flotte par type et par zone, puis scoring du comportement *dark vessel* : coupures AIS, incohérences de trajectoire, signaux RF orphelins.

`filtrer_navires_par_type_zone`, `tool_dark_vessel_behavior`

In [ ]:
@tool
def filtrer_navires_par_type_zone(
    type_navire: str,
    lat_min: float,
    lat_max: float,
    lon_min: float,
    lon_max: float,
    date: str = ""
) -> str:
    """
    Filtre les navires d'une zone par type (militaire, commercial, pêche, etc.) et ajoute, si possible, les vues RF.

    Fonctionnalité :
    - catégorise chaque navire AIS via ses attributs (ShipTypeLabel/Normalized),
    - applique le filtre spatial et temporel demandé puis, si disponible, rapprochements RF ↔ AIS,
    - renvoie deux sections (AIS / RF) pour savoir qui opère dans la zone.

    Args:
        type_navire (str): Type demandé ('militaire', 'commercial', 'pêche', 'tous'...).
        lat_min (float): Latitude minimale en degrés.
        lat_max (float): Latitude maximale en degrés.
        lon_min (float): Longitude minimale en degrés.
        lon_max (float): Longitude maximale en degrés.
        date (str): Date étudiée ('YYYY-MM-DD'). Vide = dernière position connue.

    Returns:
        str: Compte rendu prêt à l'emploi avec sections AIS et RF.
    """
    try:
        if "df_ais_global" not in globals() or df_ais_global is None or df_ais_global.empty:
            return "⚠️ Les données AIS (df_ais_global) ne sont pas disponibles ou sont vides."
        if "df_rf_global" not in globals() or df_rf_global is None or df_rf_global.empty:
            rf_available = False
        else:
            rf_available = True

        # Ordre lat/lon
        if lat_min > lat_max:
            lat_min, lat_max = lat_max, lat_min
        if lon_min > lon_max:
            lon_min, lon_max = lon_max, lon_min

        type_req = _standardize_type_request(type_navire)

        # ---------- AIS : extraction / filtrage ----------
        df_ais = df_ais_global.copy()
        if not pd.api.types.is_datetime64_any_dtype(df_ais["datetime"]):
            df_ais["datetime"] = pd.to_datetime(df_ais["datetime"], utc=True, errors="coerce")
        df_ais = df_ais.dropna(subset=["datetime"])

        date_param = date.strip() if date is not None else ""
        if date_param:
            df_ais = df_ais[df_ais["datetime"].dt.date.astype(str) == date_param]
        else:
            # dernière position par navire
            idx_last = df_ais.groupby("Mmsi")["datetime"].idxmax()
            df_ais = df_ais.loc[idx_last]

        # filtre géographique
        df_ais_zone = df_ais[
            (df_ais["Position_LatitudeDeg"] >= lat_min) &
            (df_ais["Position_LatitudeDeg"] <= lat_max) &
            (df_ais["Position_LongitudeDeg"] >= lon_min) &
            (df_ais["Position_LongitudeDeg"] <= lon_max)
        ].copy()

        if df_ais_zone.empty:
            ais_txt = (
                f"Aucun navire AIS trouvé dans la zone "
                f"[lat {lat_min:.2f}→{lat_max:.2f}, lon {lon_min:.2f}→{lon_max:.2f}]"
                + (f" pour la date {date_param}." if date_param else ".")
            )
        else:
            # classification par type
            df_ais_zone["type_navire"] = df_ais_zone.apply(_classer_type_navire_ais, axis=1)

            if type_req != "tous":
                df_ais_zone = df_ais_zone[df_ais_zone["type_navire"] == type_req]

            if df_ais_zone.empty:
                ais_txt = (
                    f"Aucun navire AIS de type **{type_req}** dans la zone "
                    f"[lat {lat_min:.2f}→{lat_max:.2f}, lon {lon_min:.2f}→{lon_max:.2f}]"
                    + (f" pour la date {date_param}." if date_param else ".")
                )
            else:
                lignes = []
                for _, r in df_ais_zone.sort_values("datetime", ascending=False).iterrows():
                    name = r.get("Name", "Nom inconnu")
                    mmsi = r.get("Mmsi", "N/A")
                    country = r.get("Country", "Pays inconnu")
                    ship_type = r.get("ShipTypeLabel", "Type non renseigné")
                    lat = r.get("Position_LatitudeDeg")
                    lon = r.get("Position_LongitudeDeg")
                    dt = r.get("datetime")
                    lignes.append(
                        f"- {name} (MMSI {mmsi}, {ship_type}, pavillon {country}) "
                        f"à ({lat:.3f}°, {lon:.3f}°) le {dt}"
                    )

                ais_txt = (
                    f"{len(df_ais_zone)} navire(s) AIS "
                    + (f"de type **{type_req}** " if type_req != 'tous' else "")
                    + "dans la zone "
                    f"[lat {lat_min:.2f}→{lat_max:.2f}, lon {lon_min:.2f}→{lon_max:.2f}]"
                    + (f" pour la date {date_param} :\n" if date_param else " :\n")
                    + "\n".join(lignes[:50])
                )

        # ---------- RF : extraction / corrélation éventuelle ----------
        if not rf_available:
            rf_txt = (
                "Les données RF (df_rf_global) ne sont pas disponibles : "
                "impossible de compléter la vue avec les détections RF."
            )
        else:
            df_rf = df_rf_global.copy()
            if not pd.api.types.is_datetime64_any_dtype(df_rf["timestamp_utc"]):
                df_rf["timestamp_utc"] = pd.to_datetime(df_rf["timestamp_utc"], utc=True, errors="coerce")
            df_rf = df_rf.dropna(subset=["timestamp_utc"])

            if date_param:
                df_rf = df_rf[df_rf["timestamp_utc"].dt.date.astype(str) == date_param]

            df_rf_zone = df_rf[
                (df_rf["latitude_deg"] >= lat_min) &
                (df_rf["latitude_deg"] <= lat_max) &
                (df_rf["longitude_deg"] >= lon_min) &
                (df_rf["longitude_deg"] <= lon_max)
            ].copy()

            if df_rf_zone.empty:
                rf_txt = (
                    f"Aucune détection RF UnseenLabs dans la zone "
                    f"[lat {lat_min:.2f}→{lat_max:.2f}, lon {lon_min:.2f}→{lon_max:.2f}]"
                    + (f" pour la date {date_param}." if date_param else ".")
                )
            else:
                # Si on dispose de cand_1_mmsi, on peut projeter le type AIS
                if "cand_1_mmsi" in df_rf_zone.columns and not df_ais_zone.empty:
                    # table MMSI -> type_navire à partir d'AIS (tous types, pas seulement filtrés)
                    df_ais_type = df_ais_global.copy()
                    if not pd.api.types.is_datetime64_any_dtype(df_ais_type["datetime"]):
                        df_ais_type["datetime"] = pd.to_datetime(df_ais_type["datetime"], utc=True, errors="coerce")
                    idx_last_all = df_ais_type.groupby("Mmsi")["datetime"].idxmax()
                    df_ais_type = df_ais_type.loc[idx_last_all].copy()
                    df_ais_type["type_navire"] = df_ais_type.apply(_classer_type_navire_ais, axis=1)
                    type_map = df_ais_type.set_index("Mmsi")["type_navire"]

                    df_rf_zone["mmsi_associe"] = df_rf_zone["cand_1_mmsi"].astype(str)
                    df_rf_zone["type_associe"] = df_rf_zone["cand_1_mmsi"].map(
                        lambda m: type_map.get(m) if not pd.isna(m) and m in type_map.index else None
                    )

                    # filtrage éventuel sur type
                    if type_req != "tous":
                        df_rf_zone_type = df_rf_zone[df_rf_zone["type_associe"] == type_req]
                    else:
                        df_rf_zone_type = df_rf_zone

                    if df_rf_zone_type.empty:
                        rf_txt = (
                            f"{len(df_rf_zone)} détection(s) RF dans la zone, "
                            "mais aucune associée à un navire AIS de type demandé "
                            f"({type_req})."
                        )
                    else:
                        lignes_rf = []
                        for _, r in df_rf_zone_type.sort_values("timestamp_utc").iterrows():
                            t = r.get("timestamp_utc")
                            lat = r.get("latitude_deg")
                            lon = r.get("longitude_deg")
                            acc = r.get("accuracy_level", "UNKNOWN")
                            freq = r.get("RF_Frequency_MHz", None)
                            mmsi = r.get("cand_1_mmsi", None)
                            lignes_rf.append(
                                f"- RF {acc} à {t} ({lat:.3f}°, {lon:.3f}°, ~{freq:.1f} MHz) "
                                f"associée à MMSI {mmsi} (type {r.get('type_associe', 'inconnu')})"
                            )

                        rf_txt = (
                            f"{len(df_rf_zone)} détection(s) RF dans la zone, dont "
                            f"{len(df_rf_zone_type)} associée(s) à des navires "
                            + (f"de type **{type_req}**.\n" if type_req != 'tous' else ":\n")
                            + "\n".join(lignes_rf[:50])
                        )
                else:
                    # pas de corrélation RF↔AIS disponible
                    rf_txt = (
                        f"{len(df_rf_zone)} détection(s) RF UnseenLabs dans la zone "
                        f"[lat {lat_min:.2f}→{lat_max:.2f}, lon {lon_min:.2f}→{lon_max:.2f}], "
                        "mais les données RF disponibles ne permettent pas d'associer un type "
                        "de navire (absence de cand_1_mmsi / corrélation RF-AIS dans df_rf_global)."
                    )

        date_txt = date_param if date_param else "la période couverte par les données"

        resultat = f"""
📍 Filtrage des navires par type dans la zone
[lat {lat_min:.2f}→{lat_max:.2f}, lon {lon_min:.2f}→{lon_max:.2f}]
pour **{date_txt}**.

🎯 Type demandé : **{type_req}**

🚢 **Vue AIS**
{ais_txt}

📡 **Vue RF UnseenLabs**
{rf_txt}

""".strip()

        return resultat

    except Exception as e:
        return f"❌ Erreur dans filtrer_navires_par_type_zone : {e}"

@tool
def tool_dark_vessel_behavior(
    target_mmsi: str,
    days: int = 5,
    min_conf_for_corr: float = 0.5,
    min_dark_gap_hours: float = 1.0,
):
    """
    Analyse le comportement 'dark' d'un navire identifié par son MMSI
    sur les X derniers jours (par défaut 5), en combinant :
      - les positions AIS (df_ais_global)
      - les détections RF corrélées à ce MMSI (df_corr_global = rf_correlation.parquet)

    Logique :
    - On récupère la trajectoire AIS + RF corrélée du navire.
    - On détecte les phases où l'AIS est éteint (gaps dans AIS) alors que
      des détections RF corrélées existent dans ce gap => comportement 'dark'.
    - On résume : nombre de phases dark, durées, distances, etc.

    Hypothèses :
    - df_ais_global contient les données AIS avec colonnes ['Mmsi','datetime','Position_LatitudeDeg','Position_LongitudeDeg']
    - df_corr_global contient rf_correlation.parquet avec au moins :
      ['timestamp_utc','latitude_deg','longitude_deg','cand_1_mmsi','cand_1_confidence']
    """

    import pandas as pd
    import numpy as np

    # --- 1) Vérifications de base ---
    if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
        return {
            "status": "error",
            "message": "Les données AIS (df_ais_global) ne sont pas chargées."
        }
    if 'df_corr_global' not in globals() or df_corr_global is None or df_corr_global.empty:
        return {
            "status": "error",
            "message": "Les données de corrélation RF/AIS (df_corr_global) ne sont pas chargées."
        }

    mmsi_str = str(target_mmsi)

    # --- 2) Sélection des données AIS pour ce MMSI ---
    df_ais = df_ais_global.copy()
    if not pd.api.types.is_datetime64_any_dtype(df_ais['datetime']):
        df_ais['datetime'] = pd.to_datetime(df_ais['datetime'])

    df_ais_vessel = df_ais[df_ais['Mmsi'].astype(str) == mmsi_str].copy()

    # --- 3) Sélection des données de corrélation RF pour ce MMSI ---
    df_corr = df_corr_global.copy()
    if not pd.api.types.is_datetime64_any_dtype(df_corr['timestamp_utc']):
        df_corr['timestamp_utc'] = pd.to_datetime(df_corr['timestamp_utc'])

    # Conversion de la confiance en float
    df_corr['cand_1_confidence'] = pd.to_numeric(
        df_corr.get('cand_1_confidence', np.nan),
        errors='coerce'
    )

    # On ne garde que les corrélations où ce MMSI est en candidat 1
    df_corr_mmsi = df_corr[
        (df_corr.get('cand_1_mmsi', '').astype(str) == mmsi_str) &
        (df_corr['cand_1_confidence'] >= min_conf_for_corr)
    ].copy()

    # --- 4) Fenêtre temporelle (X derniers jours) ---
    # On prend le max temporel entre AIS et RF corrélée pour ce MMSI
    times_candidates = []
    if not df_ais_vessel.empty:
        times_candidates.append(df_ais_vessel['datetime'].max())
    if not df_corr_mmsi.empty:
        times_candidates.append(df_corr_mmsi['timestamp_utc'].max())

    if not times_candidates:
        return {
            "status": "ok",
            "mmsi": mmsi_str,
            "num_detections": 0,
            "message": (
                f"Aucun message AIS ni corrélation RF trouvés pour le MMSI {mmsi_str}."
            )
        }

    t_max = max(times_candidates)
    t_min = t_max - pd.Timedelta(days=days)

    # Filtrage fenêtre
    df_ais_w = df_ais_vessel[df_ais_vessel['datetime'] >= t_min].copy()
    df_corr_w = df_corr_mmsi[df_corr_mmsi['timestamp_utc'] >= t_min].copy()

    # --- 5) Construction de la trajectoire combinée (AIS + RF corrélée) ---
    events = []

    # AIS
    for _, row in df_ais_w.iterrows():
        events.append({
            "timestamp": row['datetime'],
            "source": "AIS",
            "lat": row['Position_LatitudeDeg'],
            "lon": row['Position_LongitudeDeg'],
        })

    # RF corrélée
    for _, row in df_corr_w.iterrows():
        events.append({
            "timestamp": row['timestamp_utc'],
            "source": "RF_CORR",
            "lat": row['latitude_deg'],
            "lon": row['longitude_deg'],
        })

    if not events:
        return {
            "status": "ok",
            "mmsi": mmsi_str,
            "time_window": {"start": t_min, "end": t_max},
            "num_detections": 0,
            "message": (
                f"Aucune détection AIS ni RF corrélée pour le MMSI {mmsi_str} "
                f"dans les {days} derniers jours."
            )
        }

    df_events = pd.DataFrame(events).sort_values('timestamp').reset_index(drop=True)

    # --- 6) Stats de mouvement global (AIS+RF) ---
    coords = df_events[['lat', 'lon']].to_numpy()
    times = df_events['timestamp'].to_numpy()

    distances_km_all = []
    speeds_knots_all = []

    for i in range(1, len(coords)):
        lat1, lon1 = coords[i-1]
        lat2, lon2 = coords[i]
        d_km = haversine_km(lat1, lon1, lat2, lon2)
        delta = times[i] - times[i-1]
        dt_hours = delta.total_seconds() / 3600.0
        if dt_hours > 0:
            v_knots = d_km / dt_hours * 0.539957
            speeds_knots_all.append(v_knots)
        distances_km_all.append(d_km)

    total_distance_km = float(np.nansum(distances_km_all)) if distances_km_all else 0.0
    mean_speed_knots = float(np.nanmean(speeds_knots_all)) if speeds_knots_all else 0.0
    max_speed_knots = float(np.nanmax(speeds_knots_all)) if speeds_knots_all else 0.0

    lat_min, lat_max = df_events['lat'].min(), df_events['lat'].max()
    lon_min, lon_max = df_events['lon'].min(), df_events['lon'].max()

    # --- 7) Détection des phases AIS ON / OFF + DARK (OFF + RF) ---

    dark_phases = []  # liste des phases où AIS est OFF mais RF_CORR présente
    ais_times = sorted(df_ais_w['datetime'].unique()) if not df_ais_w.empty else []

    # a) Gaps entre deux points AIS consécutifs
    for i in range(1, len(ais_times)):
        t1 = ais_times[i-1]
        t2 = ais_times[i]
        gap_h = (t2 - t1).total_seconds() / 3600.0

        if gap_h >= min_dark_gap_hours:
            # RF corrélée dans ce gap ?
            rf_gap = df_corr_w[
                (df_corr_w['timestamp_utc'] > t1) &
                (df_corr_w['timestamp_utc'] < t2)
            ]
            if not rf_gap.empty:
                # Mouvement approx dans ce gap via RF
                coords_gap = rf_gap[['latitude_deg', 'longitude_deg']].to_numpy()
                dist_gap_km = 0.0
                for j in range(1, len(coords_gap)):
                    lat_a, lon_a = coords_gap[j-1]
                    lat_b, lon_b = coords_gap[j]
                    dist_gap_km += haversine_km(lat_a, lon_a, lat_b, lon_b)

                dark_phases.append({
                    "start": t1,
                    "end": t2,
                    "duration_h": gap_h,
                    "num_rf_points": int(len(rf_gap)),
                    "distance_km": float(dist_gap_km),
                    "type": "gap_interne",
                })

    # b) Silence après le dernier AIS (si AIS existe)
    if ais_times:
        last_ais = ais_times[-1]
        rf_after = df_corr_w[df_corr_w['timestamp_utc'] > last_ais]
        if not rf_after.empty:
            t2 = rf_after['timestamp_utc'].max()
            gap_h = (t2 - last_ais).total_seconds() / 3600.0
            if gap_h >= min_dark_gap_hours:
                coords_gap = rf_after[['latitude_deg', 'longitude_deg']].to_numpy()
                dist_gap_km = 0.0
                for j in range(1, len(coords_gap)):
                    lat_a, lon_a = coords_gap[j-1]
                    lat_b, lon_b = coords_gap[j]
                    dist_gap_km += haversine_km(lat_a, lon_a, lat_b, lon_b)

                dark_phases.append({
                    "start": last_ais,
                    "end": t2,
                    "duration_h": gap_h,
                    "num_rf_points": int(len(rf_after)),
                    "distance_km": float(dist_gap_km),
                    "type": "silence_post_AIS",
                })

    # c) Cas extrême : pas d'AIS dans la fenêtre mais RF corrélée
    no_ais_in_window = df_ais_w.empty and not df_corr_w.empty

    # --- 8) Synthèse des phases dark ---
    total_dark_hours = float(sum(p["duration_h"] for p in dark_phases)) if dark_phases else 0.0
    nb_dark_phases = len(dark_phases)

    start_time = df_events['timestamp'].min()
    end_time = df_events['timestamp'].max()

    # --- 9) Narratif "Commandant" ---

    narrative = []

    narrative.append(
        f"👨‍✈️ Commandant,\n\n"
        f"Voici le comportement observé pour le navire MMSI {mmsi_str} "
        f"sur les {days} derniers jours (données AIS + RF corrélée)."
    )

    narrative.append(
        f"\n🕒 Fenêtre temporelle analysée : du {start_time} au {end_time} (UTC)\n"
        f"📡 Messages AIS dans la fenêtre : {len(df_ais_w)}\n"
        f"📡 Detections RF corrélées (cand_1, conf ≥ {min_conf_for_corr}) : {len(df_corr_w)}\n"
    )

    narrative.append(
        f"📏 Distance totale approximative parcourue (AIS + RF) : ~{total_distance_km:.1f} km\n"
        f"⚡ Vitesse moyenne estimée : ~{mean_speed_knots:.1f} nœuds "
        f"(max ~{max_speed_knots:.1f} nœuds)\n"
        f"🌍 Zone d'activité : lat {lat_min:.2f}° à {lat_max:.2f}°, "
        f"lon {lon_min:.2f}° à {lon_max:.2f}°\n"
    )

    if no_ais_in_window:
        narrative.append(
            "\n🚫 Aucun message AIS n'a été reçu pour ce MMSI dans cette fenêtre temporelle, "
            "mais des détections RF corrélées existent.\n"
            "➡ Le navire se comporte comme un dark vessel complet sur cette période."
        )
    else:
        narrative.append(
            f"\n🕳️ Phases détectées où l'AIS était éteint (gap ≥ {min_dark_gap_hours:.1f}h) "
            f"mais où des détections RF corrélées sont présentes : {nb_dark_phases}\n"
            f"⏱️ Durée cumulée estimée en mode 'dark' : ~{total_dark_hours:.1f} heures\n"
        )

        if dark_phases:
            narrative.append("\n📌 Détail des principales phases 'dark' :")
            for i, p in enumerate(dark_phases[:5], 1):
                narrative.append(
                    f"\n   {i}. Du {p['start']} au {p['end']} "
                    f"({p['duration_h']:.1f} h, ~{p['distance_km']:.1f} km, "
                    f"{p['num_rf_points']} points RF, type={p['type']})"
                )

            if nb_dark_phases > 5:
                narrative.append(
                    f"\n   ... et {nb_dark_phases - 5} autres phases 'dark' plus courtes."
                )

    narrative_text = "\n".join(narrative)

    # --- 10) Résultat structuré ---
    result = {
        "status": "ok",
        "mmsi": mmsi_str,
        "time_window": {"start": start_time, "end": end_time},
        "num_events": int(len(df_events)),
        "num_ais_messages": int(len(df_ais_w)),
        "num_rf_corr": int(len(df_corr_w)),
        "total_distance_km": total_distance_km,
        "mean_speed_knots": mean_speed_knots,
        "max_speed_knots": max_speed_knots,
        "area_bbox": {
            "lat_min": lat_min,
            "lat_max": lat_max,
            "lon_min": lon_min,
            "lon_max": lon_max,
        },
        "dark_phases": dark_phases,
        "total_dark_hours": total_dark_hours,
        "n_dark_phases": nb_dark_phases,
        "no_ais_in_window": no_ais_in_window,
        "params": {
            "days": days,
            "min_conf_for_corr": min_conf_for_corr,
            "min_dark_gap_hours": min_dark_gap_hours,
        },
        "narrative": narrative_text,
    }

    return result

#### Imagerie satellite

Recherche d'une acquisition satellite couvrant la position d'un navire.

`rechercher_image_satellite_navire`

In [ ]:
@tool
def rechercher_image_satellite_navire(navire: str, time_window_hours: int = 24) -> str:
    """
    Identifie le meilleur créneau d'imagerie satellite pour un navire indiqué.

    Fonctionnement :
    - récupère la dernière position AIS disponible via MMSI ou nom,
    - recherche le passage JAPETUS le plus pertinent dans ± time_window_hours,
    - rédige un message expliquant la position actuelle et la fenêtre d'acquisition recommandée.

    Args:
        navire (str): Nom du navire ou MMSI numérique à rechercher.
        time_window_hours (int): Fenêtre temporelle (heures) autour de la dernière position AIS.

    Returns:
        str: Briefing texte indiquant coordonnées, horodatage et passage satellite recommandé.
    """
    try:
        # 1) Dernière position AIS
        last_pos = get_last_position_for_vessel(navire)
        if last_pos is None:
            return (
                f"Je n'ai trouvé aucun navire correspondant à '{navire}' dans les données AIS. "
                "Vérifiez le nom ou le MMSI."
            )

        name = last_pos["Name"]
        mmsi = last_pos["Mmsi"]
        country = last_pos["Country"]
        ship_type = last_pos["ShipType"]
        lat = last_pos["lat"]
        lon = last_pos["lon"]
        t_ref = last_pos["datetime_utc"]

        header = (
            f"Dernière position AIS connue pour le navire '{name}' "
            f"(MMSI {mmsi}, type {ship_type}, pavillon {country}) :\n"
            f"- Position : ({lat:.3f}°, {lon:.3f}°)\n"
            f"- Date/heure (UTC) : {t_ref}\n\n"
        )

        # 2) Meilleur passage JAPETUS
        best_pass = find_best_sat_pass_for_position(
            lat=lat,
            lon=lon,
            t_ref=t_ref,
            time_window_hours=time_window_hours
        )

        if best_pass is None:
            return (
                header +
                "Dans la fenêtre de ±"
                f"{time_window_hours} heures autour de cette position, "
                "aucun passage JAPETUS suffisamment proche n'a été identifié pour garantir "
                "une observation directe du navire.\n\n"
                "Je ne peux donc pas proposer d'image satellite précise, mais cette position "
                "peut servir à tasker une acquisition auprès d'un fournisseur d'imagerie."
            )

        sat_txt = (
            "Passage satellite pertinent identifié :\n"
            f"- Satellite : {best_pass['Source']} (cible : {best_pass['Target']})\n"
            f"- Fenêtre de survol : du {best_pass['StartTime']} au {best_pass['EndTime']}\n"
            f"- Durée ~ {best_pass['Duration']} s, distance au sol ~ {best_pass['Distance_km']:.1f} km\n"
            f"- Distance approximative entre la trace du satellite et le navire : "
            f"{best_pass['approx_distance_km']:.1f} km\n\n"
            "Cette fenêtre est un bon candidat pour consulter ou commander une image satellite "
            "où le navire pourrait être visible."
        )

        conclusion = (
            "\n\n⚠️ Remarque : le système n'a pas accès direct aux images, "
            "il identifie simplement la meilleure opportunité de survol à partir des éphémérides JAPETUS "
            "et de la dernière position AIS connue."
        )

        return header + sat_txt + conclusion

    except Exception as e:
        return f"Erreur dans rechercher_image_satellite_navire: {str(e)}"

#### Recherche et localisation

Recherche par zone combinée AIS + RF, et résolution d'une position à partir d'un MMSI ou d'un nom.

`rechercher_navires_zone_combinee`, `rechercher_position_navire`

In [ ]:
@tool
def rechercher_navires_zone_combinee(
    lat_min: float,
    lat_max: float,
    lon_min: float,
    lon_max: float,
    date: str = "",
    rayon_km: float = 10.0,
    delta_t_minutes: int = 60
) -> str:
    """
    Produit une vue fusionnée AIS / RF / passages satellites pour un rectangle géographique.

    Ce tool répond aux questions d'un analyste :
    - Quels navires sont présents dans cette zone et quelles sont leurs caractéristiques ?
    - Les capteurs RF confirment-ils leur présence ou détectent-ils des émetteurs sans AIS ?
    - Des passages JAPETUS couvrent-ils ce secteur autour de la date étudiée ?

    Args:
        lat_min (float): Latitude minimale du rectangle (degrés).
        lat_max (float): Latitude maximale du rectangle (degrés).
        lon_min (float): Longitude minimale du rectangle (degrés).
        lon_max (float): Longitude maximale du rectangle (degrés).
        date (str, optional): Jour à analyser au format 'YYYY-MM-DD'. Vide = dernière position par navire.
        rayon_km (float): Rayon utilisé pour rapprocher une détection RF d'un navire AIS.
        delta_t_minutes (int): Fenêtre temporelle (± minutes) pour associer RF et AIS.

    Returns:
        str: Rapport structuré avec trois sections (AIS, RF, satellites) rédigé pour un briefing.
    """
    try:
        date_param = date.strip() if date is not None else ""
        if date_param == "":
            date_param = None

        combined = get_combined_contacts_in_zone(
            lat_min=lat_min,
            lat_max=lat_max,
            lon_min=lon_min,
            lon_max=lon_max,
            date=date_param,
            rayon_km=rayon_km,
            delta_t_minutes=delta_t_minutes
        )

        df_ais = combined["ais_vessels"]
        df_rf = combined["rf_detections"]
        df_eph = combined["sat_passes"]

        zone_txt = f"[lat {lat_min:.2f} → {lat_max:.2f}, lon {lon_min:.2f} → {lon_max:.2f}]"
        date_txt = date_param if date_param else "la période observée (11–17 décembre 2024)"

        # --- Section AIS ---
        if df_ais.empty:
            ais_txt = "Aucun navire AIS détecté dans cette zone pour " + date_txt + "."
        else:
            lignes_ais = []
            for _, row in df_ais.sort_values("datetime", ascending=False).iterrows():
                name = row.get("Name", "Nom inconnu")
                mmsi = row.get("Mmsi", "N/A")
                ship_type = row.get("ShipTypeLabel", "Type non renseigné")
                country = row.get("Country", "Pays inconnu")
                lat = row.get("Position_LatitudeDeg", None)
                lon = row.get("Position_LongitudeDeg", None)
                dt = row.get("datetime", None)

                lignes_ais.append(
                    f"- {name} (MMSI {mmsi}, {ship_type}, pavillon {country}) "
                    f"à ({lat:.3f}°, {lon:.3f}°) le {dt}"
                )
            ais_txt = (
                f"{len(df_ais)} navire(s) AIS détecté(s) dans la zone {zone_txt} pour {date_txt} :\n"
                + "\n".join(lignes_ais[:50])
            )

        # --- Section RF ---
        if df_rf.empty:
            rf_txt = "Aucune détection RF dans cette zone pour " + date_txt + "."
        else:
            lignes_rf = []
            for _, row in df_rf.sort_values("timestamp_utc").iterrows():
                t = row.get("timestamp_utc", None)
                lat = row.get("latitude_deg", None)
                lon = row.get("longitude_deg", None)
                acc = row.get("accuracy_level", "UNKNOWN")
                freq = row.get("RF_Frequency_MHz", None)
                mmsi = row.get("matched_mmsi", None)
                name = row.get("matched_name", None)
                dist = row.get("distance_km_to_ais", None)

                if mmsi is not None:
                    lignes_rf.append(
                        f"- RF à {t} ({lat:.3f}°, {lon:.3f}°, {acc}, ~{freq:.1f} MHz) "
                        f" associé au navire {name} (MMSI {mmsi}), distance ~{dist:.1f} km"
                    )
                else:
                    lignes_rf.append(
                        f"- RF à {t} ({lat:.3f}°, {lon:.3f}°, {acc}, ~{freq:.1f} MHz), "
                        f"sans navire AIS clairement associé (potentiellement contact dark)."
                    )

            rf_txt = (
                f"{len(df_rf)} détection(s) RF dans la zone {zone_txt} pour {date_txt} "
                f"(rayon association {rayon_km} km, fenêtre ±{delta_t_minutes} min) :\n"
                + "\n".join(lignes_rf[:50])
            )

        # --- Section satellites (JAPETUS) ---
        if df_eph.empty:
            sat_txt = "Aucun passage JAPETUS identifié couvrant cette zone pour " + date_txt + "."
        else:
            lignes_sat = []
            for _, row in df_eph.sort_values("StartTime").iterrows():
                src = row.get("Source", "JAPETUS")
                tgt = row.get("Target", "Zone")
                st  = row.get("StartTime")
                et  = row.get("EndTime")
                dur = row.get("Duration", None)
                dist = row.get("Distance_km", None)

                lignes_sat.append(
                    f"- {src} sur {tgt} de {st} à {et} (durée ~{dur} s, distance au sol ~{dist:.1f} km)"
                )

            sat_txt = (
                f"{len(df_eph)} passage(s) de satellites JAPETUS couvrant (ou traversant) la zone {zone_txt} "
                f"pour {date_txt} :\n"
                + "\n".join(lignes_sat[:50])
            )

        # --- Assemblage final ---
        result = (
            f"📍 Synthèse multi-capteurs pour la zone {zone_txt} ({date_txt})\n\n"
            "🛰️ Satellites / Imagerie (JAPETUS)\n"
            "---------------------------------\n"
            f"{sat_txt}\n\n"
            "🚢 Navires AIS\n"
            "--------------\n"
            f"{ais_txt}\n\n"
            "📡 Détections RF\n"
            "----------------\n"
            f"{rf_txt}\n\n"

        )

        return result

    except Exception as e:
        return f"Erreur dans rechercher_navires_zone_combinee: {str(e)}"

######################### FRONTIERE CHARBEL SEI #########################
#
#
from langchain_core.tools import tool
import pandas as pd
import difflib

# --- Fonction Helper Améliorée ---
def vessel_candidates_ameliore(query: str, df: pd.DataFrame, k: int=10):
    """
    Génère des candidats basés sur la ressemblance (Fuzzy) ET l'inclusion.
    """
    if df is None or df.empty or not query:
        return []

    qn = query.strip().lower()
    candidates = []

    for _, row in df.iterrows():
        name = str(row.get('Name', '')).strip()
        if not name or name.lower() == 'nan':
            continue

        nn = name.lower()

        # Calcul du score
        if qn == nn:
            score = 100.0 # Match exact (Normalement géré avant, mais sécurité)
        elif qn in nn:
            # Le nom contient la recherche (ex: "Yongjinho" dans "37Yongjinho")
            # Score basé sur la longueur : plus le nom est court, plus c'est pertinent
            score = 2.0 - (len(nn) / 100)
        else:
            # Fuzzy match (fautes de frappe)
            score = difflib.SequenceMatcher(None, qn, nn).ratio()

        # On garde si c'est pertinent (Fuzzy > 0.6 ou Inclusion)
        if score > 0.6:
            candidates.append((score, str(row['Mmsi']), name))

    # Tri par score décroissant
    candidates.sort(reverse=True, key=lambda x: x[0])

    # On déduplique les noms pour l'affichage (éviter d'avoir 3 fois le même nom avec MMSI diff)
    seen_names = set()
    unique_candidates = []
    for score, mmsi, name in candidates:
        if name not in seen_names:
            unique_candidates.append((score, mmsi, name))
            seen_names.add(name)

    return unique_candidates[:k]

# --- Le Tool LangChain ---
@tool
def rechercher_position_navire(identifiant: str) -> str:
    """
    Localise un navire à partir d'un MMSI ou d'un nom plus ou moins approximatif.

    L'algorithme enchaîne quatre stratégies :
    - MMSI numérique : retourne immédiatement la dernière position connue et ses métadonnées.
    - Nom exact : fournit la fiche complète ou liste les homonymes à départager.
    - Correspondance partielle : propose des suggestions sans exposer de position sensible.
    - Recherche floue : suggère les meilleures corrections en cas de faute de frappe.

    Args:
        identifiant (str): MMSI ou chaîne libre décrivant le navire (casse et accents ignorés).

    Returns:
        str: Bloc texte prêt à afficher (position, ambiguïtés ou explication claire si rien n'est trouvé).
    """
    if 'df_ais_global' not in globals():
        return "❌ Erreur : Données AIS non chargées."

    # 1. Préparation : Une seule ligne par MMSI (la plus récente)
    df_unique = df_ais_global.sort_values('datetime', ascending=False).drop_duplicates('Mmsi')

    query = str(identifiant).strip()
    query_upper = query.upper()

    # --- ETAPE 1 : RECHERCHE PAR MMSI (Identifiant unique) ---
    if query.isdigit():
        match = df_unique[df_unique['Mmsi'].astype(str) == query]
        if not match.empty:
            return _formater_reponse_position(match.iloc[0])

    # --- ETAPE 2 : RECHERCHE PAR NOM EXACT ---
    # On cherche le navire qui s'appelle STRICTEMENT "Yongjinho" ou "Ari"
    match_exact = df_unique[df_unique['Name'].str.upper() == query_upper]

    if not match_exact.empty:
        # Si on en trouve un (ou plusieurs homonymes exacts)
        if len(match_exact) == 1:
            return _formater_reponse_position(match_exact.iloc[0])
        else:
            # Plusieurs navires s'appellent EXACTEMENT "Ari"
            return _lister_ambiguites(match_exact, query, raison="homonymes exacts")

    # --- ETAPE 3 : RECHERCHE PAR INCLUSION (PARTIAL MATCH) ---
    # C'est ici qu'on gère le cas "Yongjinho" -> "37Yongjinho"
    # On cherche les noms qui CONTIENNENT la requête
    match_partial = df_unique[df_unique['Name'].str.contains(query, case=False, na=False, regex=False)]

    if not match_partial.empty:
        # On ne donne PAS la position directement, on suggère.
        # Même s'il n'y en a qu'un seul (ex: "37Yongjinho"), on demande confirmation.
        return _lister_suggestions(match_partial, query, is_partial=True)

    # --- ETAPE 4 : RECHERCHE FLOUE (TYPOS) ---
    # Si on n'a rien trouvé, peut-être une faute de frappe ?
    candidats = vessel_candidates_ameliore(query, df_unique)

    if candidats:
        # Construction manuelle du message pour les candidats fuzzy
        msg = f"🤔 Je n'ai trouvé aucun navire nommé exactement '{query}'.\nVouliez-vous dire :\n"
        for _, mmsi, name in candidats[:5]:
            msg += f"- **{name}** (MMSI: {mmsi})\n"
        return msg

    return f"❌ Aucun navire trouvé pour '{query}' (ni exact, ni partiel, ni proche)."

# --- Fonctions de formatage (Helpers) ---

def _formater_reponse_position(row):
    return (f"📍 **Position confirmée**\n"
            f"🚢 **Navire :** {row['Name']}\n"
            f"🆔 **MMSI :** {row['Mmsi']}\n"
            f"🌍 **Coord :** {row['Position_LatitudeDeg']:.4f}, {row['Position_LongitudeDeg']:.4f}\n"
            f"🕒 **Relevé :** {row['datetime']}")

def _lister_ambiguites(df, query, raison):
    msg = f"🤔 J'ai trouvé {len(df)} navires qui s'appellent **exactement** '{query}'. Lequel voulez-vous ?\n"
    for _, row in df.head(5).iterrows():
        msg += f"- {row['Name']} (MMSI: {row['Mmsi']}) | {row.get('ShipTypeLabel', 'Inconnu')} | 🏳️ {row.get('Pays', '-')}\n"
    return msg

def _lister_suggestions(df, query, is_partial=False):
    # On prend les 5 premiers résultats pertinents
    top_results = df.head(5)

    if len(df) == 1:
        # Cas unique (ex: Yongjinho -> 37Yongjinho)
        row = df.iloc[0]
        return (f"⚠️ Je n'ai pas trouvé '{query}' exactement.\n"
                f"Par contre, j'ai trouvé : **{row['Name']}** (MMSI: {row['Mmsi']}).\n"
                f"Est-ce celui-là ? Si oui, redemandez-moi sa position avec le nom complet.")

    else:
        # Cas multiple (ex: Ari -> Aria, Arik...)
        msg = f"📋 Plusieurs navires contiennent '{query}'. Précisez votre recherche :\n"
        for _, row in top_results.iterrows():
            msg += f"- **{row['Name']}** (MMSI: {row['Mmsi']})\n"
        if len(df) > 5:
            msg += f"... (et {len(df)-5} autres)"
        return msg

from langchain_core.tools import tool
import pandas as pd
import unicodedata
import math

# --- 1. CONFIGURATION & CONSTANTES ---

# Définition de ce qui est une "Force" (Militaire/État)
CODES_FORCE = [35, 51, 54, 55, 58]
KW_FORCE = ['MILITARY', 'NAVY', 'POLICE', 'PATROL', 'GUARD', 'GOV', 'WARSHIP', 'COAST GUARD']
PFX_FORCE = ['USS ', 'HMS ', 'FS ', 'RFS ', 'PLAN ', 'M/V PATROL', 'NS ']

# Zones géographiques pré-définies (Centres et Rayons en KM)
ZONES_DATA = {
    # --- MÉDITERRANÉE ---
    "mediterranee":            {"lat": 38.0, "lon": 15.0, "rayon": 1800},
    "mediterranee occidentale": {"lat": 43.0, "lon": 8.5,  "rayon": 700},
    "mediterranee orientale":   {"lat": 37.0, "lon": 22.5, "rayon": 800},
    "mer adriatique":          {"lat": 42.0, "lon": 16.0, "rayon": 400},
    "adriatique":              {"lat": 42.0, "lon": 16.0, "rayon": 400},
    "mer ionienne":            {"lat": 37.5, "lon": 19.0, "rayon": 400},
    "mer egee":                {"lat": 38.5, "lon": 26.0, "rayon": 350},
    "egee":                    {"lat": 38.5, "lon": 26.0, "rayon": 350},

    # --- MER NOIRE ---
    "mer noire":               {"lat": 44.0, "lon": 34.5, "rayon": 600},

    # --- EUROPE NORD ---
    "mer de norvege":          {"lat": 65.0, "lon": 5.0,  "rayon": 1000},
    "mer baltique":            {"lat": 60.0, "lon": 20.0, "rayon": 700},
    "mer du nord":             {"lat": 56.0, "lon": 3.0,  "rayon": 600},

    # --- ATLANTIQUE ---
    "atlantique":              {"lat": 35.0, "lon": -40.0, "rayon": 3000},
    "atlantique nord":         {"lat": 52.0, "lon": -30.0, "rayon": 2000},
    "atlantique tropical":     {"lat": 20.0, "lon": -40.0, "rayon": 2000},
    "atlantique sud":          {"lat": -30.0,"lon": -15.0, "rayon": 2500},

    # --- AUTRES OCÉANS ---
    "pacifique":               {"lat": 0.0,   "lon": 150.0, "rayon": 5000},
    "indien":                  {"lat": -15.0, "lon": 70.0,  "rayon": 4000},
    "ocean indien":            {"lat": -15.0, "lon": 70.0,  "rayon": 4000},
    "arctique":                {"lat": 80.0,  "lon": 0.0,   "rayon": 2000},
    "antarctique":             {"lat": -70.0, "lon": 0.0,   "rayon": 2000},

    # --- ZONES PAYS CÔTIERS ---
    "france":                  {"lat": 46.5, "lon": 2.5,   "rayon": 1000},
    "espagne":                 {"lat": 40.0, "lon": -3.5,  "rayon": 800},
    "italie":                  {"lat": 42.0, "lon": 12.5,  "rayon": 800},
    "bretagne":                {"lat": 48.2, "lon": -4.5,  "rayon": 250},
    "manche":                  {"lat": 49.5, "lon": -3.0,  "rayon": 300},
}

# --- 2. FONCTIONS HELPERS ---

def _norm(s: str) -> str:
    """Nettoie le texte : minuscules, sans accents, sans espaces inutiles."""
    if not s:
        return ''
    s = unicodedata.normalize('NFKD', s)
    s = ''.join((ch for ch in s if not unicodedata.combining(ch)))
    return s.lower().strip()

def _haversine_nm(lat1, lon1, lat2, lon2):
    """Calcule la distance en Milles Nautiques (NM) entre deux points."""
    R = 3440.065 # Rayon terre en NM
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

# --- 3. LE TOOL PRINCIPAL ---
def _check_spatial_overlap(lat_target, lon_target, rayon_km, row_sat):
    """
    Vérifie si la zone cible est couverte par la fauchée du satellite.
    On approxime la fauchée par une bounding box autour de la trajectoire Start->End.
    """
    try:
        # Marge de sécurité (Buffer) en degrés (~200km de large pour la fauchée + rayon zone)
        buffer = (rayon_km / 111.0) + 1.5

        lat_min = min(row_sat['StartLatitude'], row_sat['EndLatitude']) - buffer
        lat_max = max(row_sat['StartLatitude'], row_sat['EndLatitude']) + buffer
        lon_min = min(row_sat['StartLongitude'], row_sat['EndLongitude']) - buffer
        lon_max = max(row_sat['StartLongitude'], row_sat['EndLongitude']) + buffer

        # Vérification simple : Est-ce que le centre de la zone est dans la boite du satellite ?
        return (lat_min <= lat_target <= lat_max) and (lon_min <= lon_target <= lon_max)
    except:
        return False

#### Éphémérides et présence stratégique

Prochains passages satellites sur une zone, et présence de navires au voisinage de points stratégiques.

`predire_passages_satellites`, `analyser_presence_strategique`

In [ ]:
@tool
def predire_passages_satellites(zone: str, date_heure_debut: str, date_heure_fin: str) -> str:
    """
    Prédit les passages satellites (Éphémérides) au-dessus d'une zone sur une période donnée.
    Permet de savoir si une zone était surveillée à un moment précis.

    Args:
        zone: Zone géographique (ex: 'Méditerranée' ou '43.0,5.0').
        date_heure_debut: Format 'YYYY-MM-DD HH:MM' (ex: '2024-12-14 08:00').
        date_heure_fin: Format 'YYYY-MM-DD HH:MM' (ex: '2024-12-14 12:00').
    """
    try:
        # A. VÉRIFICATIONS
        if 'df_ephemeris_global' not in globals(): return "❌ Erreur : Données éphémérides non chargées."
        df = df_ephemeris_global.copy()

        # B. PARSING DATE
        try:
            start_dt = pd.to_datetime(date_heure_debut)
            end_dt = pd.to_datetime(date_heure_fin)
        except:
            return f"❌ Format de date invalide. Utilisez 'YYYY-MM-DD HH:MM' (ex: 2024-12-14 10:00)."

        # C. RÉSOLUTION ZONE
        z_clean = _norm(zone)
        z_config = None

        if z_clean in ZONES_DATA: z_config = ZONES_DATA[z_clean]
        else:
            for k in ZONES_DATA:
                if k in z_clean: z_config = ZONES_DATA[k]; break

        if not z_config and "," in zone:
            try: p=zone.replace(" ","").split(","); z_config={"lat":float(p[0]),"lon":float(p[1]),"rayon":200}
            except: pass

        if not z_config: return f"❌ Zone '{zone}' non reconnue."

        lat_c, lon_c, rad_km = z_config['lat'], z_config['lon'], z_config['rayon']

        # D. FILTRAGE TEMPOREL
        # On cherche les satellites dont la fenêtre [StartTime, EndTime] chevauche la demande
        # Conversion des colonnes si nécessaire
        if not pd.api.types.is_datetime64_any_dtype(df['StartTime']):
            df['StartTime'] = pd.to_datetime(df['StartTime'])
        if not pd.api.types.is_datetime64_any_dtype(df['EndTime']):
            df['EndTime'] = pd.to_datetime(df['EndTime'])

        # Overlap: (SatStart <= UserEnd) AND (SatEnd >= UserStart)
        df_time = df[
            (df['StartTime'] <= end_dt) &
            (df['EndTime'] >= start_dt)
        ].copy()

        if df_time.empty:
            return f"🌑 Aucun satellite disponible sur la période {date_heure_debut} - {date_heure_fin} (Filtre temps)."

        # E. FILTRAGE SPATIAL
        passages = []

        for _, row in df_time.iterrows():
            if _check_spatial_overlap(lat_c, lon_c, rad_km, row):

                # Calcul de l'heure exacte de passage au centre de la zone (approx)
                # On prend le milieu de la fenêtre de tir du satellite pour simplifier
                mid_point_time = row['StartTime'] + (row['EndTime'] - row['StartTime']) / 2

                passages.append({
                    "nom": row['Source'],
                    "heure": mid_point_time,
                    "duree": row.get('Duration', 0),
                    "type": "Optique/RF" if "JAPETUS" in str(row['Source']) else "Inconnu"
                })

        # F. RAPPORT
        if not passages:
            return f"⚠️ Aucun satellite ne passe au-dessus de {zone.upper()} sur ce créneau horaire."

        # Tri chronologique
        passages.sort(key=lambda x: x['heure'])

        msg = f"🛰️ **PRÉVISION PASSAGES SATELLITES : {zone.upper()}**\n"
        msg += f"🕒 Créneau : {start_dt.strftime('%d/%m %H:%M')} ➡ {end_dt.strftime('%H:%M')}\n"
        msg += f"📡 **{len(passages)} opportunités de collecte identifiées**\n"
        msg += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"

        for p in passages:
            t_str = p['heure'].strftime('%H:%M:%S')
            msg += f"🛰️ **{p['nom']}**\n"
            msg += f"   ⏰ Heure de survol : **{t_str} UTC**\n"
            msg += f"   ⏳ Fenêtre d'acquisition : ~{int(p['duree'])} sec\n\n"

        return msg

    except Exception as e:
        return f"❌ Erreur critique Tool Satellites : {str(e)}"

@tool
def analyser_presence_strategique(zone: str, pays: str = "TOUS", date: str = "") -> str:
    """
    Analyse la présence des forces navales et civiles d'un ou plusieurs pays dans une zone géographique.

    Deux modes de fonctionnement :
    - Mode "forces d'un pays spécifique" : si 'pays' est renseigné (ex: "FRANCE", "SPAIN").
      -> Recherche les navires appartenant aux forces étatiques de ce(s) pays.
    - Mode "tous les navires dans la zone" : si 'pays' est vide, non fourni, ou égal à 'TOUS'.
      -> Retourne tous les navires présents dans la zone, quelle que soit leur nationalité.

    Args:
        zone (str): Zone géographique à analyser. Deux formats acceptés :
                    - Nom de zone prédéfinie (ex: 'MEDITERRANEE', 'MER_NOIRE')
                    - Coordonnées au format 'lat,lon' (ex: '42.5,6.1')
        pays (str): Nom du ou des pays cibles en ANGLAIS STANDARD et en MAJUSCULES.
                    Exemples de conversion :
                    - 'françaises' → 'FRANCE'
                    - 'espagnols' → 'SPAIN'
                    - 'russes' → 'RUSSIAN FEDERATION'
                    - 'italiennes' → 'ITALY'
                    Pour plusieurs pays, sépare-les par des virgules (ex: 'FRANCE, SPAIN').
                    Si aucun pays n'est précisé ou 'tous', envoie 'TOUS'.
        date (str, optional): Date au format 'YYYY-MM-DD'.
                              Si vide ou non fourni → utilise les dernières positions connues.

    Returns:
        str: Résumé textuel détaillé des navires trouvés avec classification Forces/Civils,
             statistiques par type de navire, et informations de corrélation AIS/RF si disponibles.
    """
    # Vérification des données globales
    if 'df_ais_global' not in globals(): return "❌ Erreur : Données AIS non chargées."

    # 1. RÉSOLUTION DE LA ZONE (Moteur de recherche flou)
    z_clean = _norm(zone)
    z_config = None

    # A. Match exact (ex: "mer noire")
    if z_clean in ZONES_DATA:
        z_config = ZONES_DATA[z_clean]
    else:
        # B. Match partiel (ex: "en méditerranée")
        matches = [k for k in ZONES_DATA.keys() if k in z_clean]
        if matches:
            best_match = max(matches, key=len) # Prend le plus spécifique
            z_config = ZONES_DATA[best_match]

    # C. Fallback Coordonnées
    if not z_config and "," in zone:
        try:
            p = zone.replace(" ","").split(",")
            z_config = {"lat": float(p[0]), "lon": float(p[1]), "rayon": 200}
        except: pass

    if not z_config: return f"❌ Zone '{zone}' non reconnue. Essayez 'Méditerranée' ou des coordonnées."

    # 2. PRÉPARATION DES CIBLES
    cibles = []
    mode_global = False
    p_input = pays.upper().strip()

    if p_input in ["TOUS", "ALL", "GLOBAL", "ANY", ""]:
        mode_global = True
    else:
        # Gestion multi-pays
        raw_list = p_input.replace(" ET ", ",").replace(" AND ", ",").split(",")
        for item in raw_list:
            clean_item = item.strip()
            if clean_item:
                cibles.append(clean_item)

    # 3. SCAN DU DATAFRAME
    # Utilisation de z_config (et non 'z')
    lat, lon, ray_km = z_config['lat'], z_config['lon'], z_config['rayon']

    # Préparation du DataFrame avec gestion de la date
    df = df_ais_global.copy()
    if not pd.api.types.is_datetime64_any_dtype(df['datetime']):
        df['datetime'] = pd.to_datetime(df['datetime'])

    # Filtre par date si spécifiée
    date_param = date.strip() if date is not None else ""
    if date_param:
        df = df[df['datetime'].dt.date.astype(str) == date_param]
        if df.empty:
            return f"❌ Aucune donnée AIS disponible pour la date {date_param}."

    # Optimisation : Tri et déduplication (dernière position par navire)
    df = df.sort_values('datetime', ascending=False).drop_duplicates('Mmsi')

    forces = []
    civils = []

    for _, row in df.iterrows():
        # 1. Filtre Géographique
        # On calcule en NM puis on convertit en KM pour comparer au rayon
        dist_nm = _haversine_nm(lat, lon, row['Position_LatitudeDeg'], row['Position_LongitudeDeg'])
        dist_km = dist_nm * 1.852

        if dist_km > ray_km: continue

        # 2. Filtre Pays
        pays_navire = str(row.get('Pays', '')).upper()

        if not mode_global:
            match = False
            for c in cibles:
                if c in pays_navire:
                    match = True
                    break
            if not match: continue

        # 3. Classification Force vs Civil
        type_lbl = str(row.get('ShipTypeLabel', '')).upper()
        nom = str(row.get('Name', '')).upper()
        code = row.get('ShipType', -1)

        is_force = (code in CODES_FORCE) or \
                   any(k in type_lbl for k in KW_FORCE) or \
                   any(nom.startswith(p) for p in PFX_FORCE)

        # 4. Enrichissement
        try:
            # On suppose que get_precise_maritime_zone est défini ailleurs dans votre code
            desc_geo = get_precise_maritime_zone(row['Position_LatitudeDeg'], row['Position_LongitudeDeg'])['description']
        except:
            desc_geo = f"À {dist_km:.0f} km du centre zone"

        try:
            # On suppose que get_country_flag_smart est défini ailleurs
            flag = get_country_flag_smart(row.get('Pays', ''))
        except:
            flag = "🏳️"

        item = {
            "nom": row['Name'],
            "type": row['ShipTypeLabel'],
            "flag": flag,
            "pos": desc_geo,
            "dist": dist_km
        }

        if is_force: forces.append(item)
        else: civils.append(item)

    # D. GÉNÉRATION DU RAPPORT
    titre = "SITUATION GLOBALE" if mode_global else pays.upper()
    nom_zone = zone.upper() if len(zone) < 20 else "ZONE CIBLE"
    date_txt = f" ({date_param})" if date_param else " (dernières positions)"

    msg = f"📊 **ANALYSE : {titre}**\n"
    msg += f"📍 Zone : {nom_zone} (Rayon {ray_km} km){date_txt}\n"
    msg += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"

    # Bloc Forces
    if forces:
        msg += f"🚨 **FORCES DÉTECTÉES : {len(forces)}**\n"
        forces.sort(key=lambda x: x['dist'])
        for f in forces:
            msg += f"   ⚔️ **{f['nom']}** {f['flag']} ({f['type']})\n"
            msg += f"      └ {f['pos']}\n"
    else:
        msg += f"✅ **FORCES : AUCUNE** présence militaire détectée.\n"

    msg += "\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"

    # Bloc Civils
    if civils:
        msg += f"🚢 **NAVIRES CIVILS : {len(civils)}**\n"
        civils.sort(key=lambda x: x['dist'])
        limit = 5 if mode_global else 8

        for c in civils[:limit]:
            # Sécurisation du type pour le .upper()
            c_type = str(c['type']).upper()
            icon = "🐟" if "FISHING" in c_type else "⛵"

            msg += f"   {icon} **{c['nom']}** {c['flag']} ({c['type']})\n"
            msg += f"      └ {c['pos']}\n"

        if len(civils) > limit:
            msg += f"   ... et {len(civils)-limit} autres navires.\n"
    else:
        msg += f"ℹ️ **CIVILS : AUCUN** navire marchand identifié.\n"

    return msg

#### Anomalies de navigation

Le cœur de la détection : coupures AIS à haute fréquence, profils de vitesse anormaux (dérive, arrêt inexpliqué), analyse comportementale et synthèse des anomalies.

`detecter_coupures_ais_haute_frequence`, `analyser_comportement_avance`, `analyser_profil_vitesse`, `generer_rapport_anomalies_navigation`

In [ ]:
@tool
def detecter_coupures_ais_haute_frequence(zone: str, seuil_heures: float = 2.0) -> str:
    """
    Détecte les navires qui coupent fréquemment leur AIS dans une zone surveillée.

    Étapes principales :
    - normalisation de la zone (prédéfinie ou coordonnées) et filtrage spatial/temps,
    - détection des gaps > seuil_heures sur les trajectoires AIS récentes,
    - élévation du niveau d'alerte si une corroboration RF ou une disparition prolongée est observée.

    Args:
        zone (str): Nom de zone supporté ou coordonnées "lat,lon".
        seuil_heures (float): Durée minimale d'interruption AIS pour lever une alerte.

    Returns:
        str: Liste hiérarchisée des navires suspects avec justification des coupures détectées.
    """
    try:
        # A. INIT & ZONE
        if 'df_ais_global' not in globals(): return "❌ Erreur : Données non chargées."

        z_clean = _norm(zone)
        z_config = None
        if z_clean in ZONES_DATA: z_config = ZONES_DATA[z_clean]
        else:
            for k in ZONES_DATA:
                if k in z_clean or z_clean in k:
                    z_config = ZONES_DATA[k]; break

        if not z_config and "," in zone:
            try: p=zone.replace(" ","").split(","); z_config={"lat":float(p[0]),"lon":float(p[1]),"rayon":200}
            except: pass
        if not z_config: return f"❌ Zone '{zone}' inconnue."

        lat_c, lon_c, rad_km = z_config['lat'], z_config['lon'], z_config['rayon']

        # B. PREP DATA
        df = df_ais_global.copy() # Copie pour ne pas toucher l'original
        if not pd.api.types.is_datetime64_any_dtype(df['datetime']):
            df['datetime'] = pd.to_datetime(df['datetime'])

        # Nettoyage strict
        df = df.dropna(subset=['Position_LatitudeDeg', 'Position_LongitudeDeg'])
        df = df[df['Name'].notna() & (df['Name'] != 'UNKNOWN')]

        dataset_end_time = df['datetime'].max()

        # Filtre Geo global
        deg_lat = rad_km / 111.0
        df_zone = df[
            (df['Position_LatitudeDeg'].between(lat_c - deg_lat, lat_c + deg_lat))
        ].copy()

        if df_zone.empty: return f"ℹ️ Zone vide (aucun navire)."

        # C. ANALYSE
        suspects = []

        for mmsi, group in df_zone.groupby('Mmsi'):
            group = group.sort_values('datetime')
            last_pos = group.iloc[-1]

            # Vérif distance
            if haversine_km(lat_c, lon_c, last_pos['Position_LatitudeDeg'], last_pos['Position_LongitudeDeg']) > rad_km:
                continue

            # 1. ANALYSE DES TROUS HISTORIQUES (ON -> OFF -> ON) -- LE PLUS IMPORTANT
            has_gap = False
            max_gap_duration = 0.0

            if len(group) > 1:
                deltas = group['datetime'].diff().dt.total_seconds() / 3600.0
                max_gap_duration = deltas.max()
                if max_gap_duration > seuil_heures:
                    has_gap = True

            # 2. ANALYSE DU SILENCE ACTUEL (ON -> OFF -> Fin Dataset)
            silence_actuel = (dataset_end_time - last_pos['datetime']).total_seconds() / 3600.0
            is_silent_now = silence_actuel > seuil_heures

            # 3. CORRÉLATION RF (Le Juge de Paix)
            rf_txt = ""
            rf_found = False

            # On ne cherche la RF que si le navire est suspect
            if (is_silent_now or has_gap) and 'df_rf_global' in globals():
                df_rf = globals()['df_rf_global']
                if not pd.api.types.is_datetime64_any_dtype(df_rf['timestamp_utc']):
                     df_rf['timestamp_utc'] = pd.to_datetime(df_rf['timestamp_utc'])

                # RF pendant le silence ou après
                candidates = df_rf[df_rf['timestamp_utc'] > last_pos['datetime']]
                for _, rf in candidates.iterrows():
                    d_rf = haversine_km(last_pos['Position_LatitudeDeg'], last_pos['Position_LongitudeDeg'],
                                         rf['latitude_deg'], rf['longitude_deg'])
                    # Rayon large (100km)
                    if d_rf < 100:
                        rf_txt = f"📡 RF captée à {d_rf:.0f}km"
                        rf_found = True
                        break

            # --- FILTRAGE INTELLIGENT ---

            # Cas A : C'est une sortie de zone probable (Faux Positif)
            # Si silence > 96h (4 jours) ET pas de RF ET pas de gap historique
            if is_silent_now and silence_actuel > 96.0 and not rf_found and not has_gap:
                continue

            # Cas B : C'est un vrai suspect
            if has_gap or (is_silent_now and (rf_found or silence_actuel < 96.0)):

                # Détermination du statut
                if has_gap:
                    statut = "COUPURE INTERMITTENTE"
                    info = f"A éteint AIS pendant **{max_gap_duration:.1f}h** puis rallumé"
                    score = 2000 + max_gap_duration # Priorité Max
                else:
                    statut = "DISPARITION RÉCENTE"
                    info = f"Muet depuis **{silence_actuel:.1f}h**"
                    score = 1000 + silence_actuel
                    if rf_found:
                        statut = "DARK SHIP (CONFIRMÉ RF)"
                        score += 5000

                suspects.append({
                    "nom": last_pos.get('Name', str(mmsi)),
                    "type": last_pos.get('ShipTypeLabel', '-'),
                    "statut": statut,
                    "info": info,
                    "rf": rf_txt,
                    "score": score,
                    "coords": f"{last_pos['Position_LatitudeDeg']:.3f}, {last_pos['Position_LongitudeDeg']:.3f}"
                })

        # E. RAPPORT
        if not suspects:
            return f"✅ Aucun navire suspect détecté (Les silences > {seuil_heures}h semblent être des sorties de zone)."

        suspects.sort(key=lambda x: x['score'], reverse=True)
        top_s = suspects[:15]

        msg = f"📉 **ANALYSE GAP/SILENCE (> {seuil_heures}h) : {zone.upper()}**\n"
        msg += f"🚨 **{len(suspects)} suspects filtrés** (Exclusion des sorties de zone > 96h)\n"
        msg += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"

        for s in top_s:
            icon = "🚨" if "RF" in s['statut'] else "🔴" if "INTERMITTENTE" in s['statut'] else "🟠"
            msg += f"{icon} **{s['nom']}** ({s['type']})\n"
            msg += f"   ⚠️ {s['statut']}\n"
            msg += f"   ⏱️ {s['info']}\n"
            msg += f"   📍 Dernier point : {s['coords']}\n"
            if s['rf']: msg += f"   {s['rf']}\n"
            msg += "\n"

        if len(suspects) > 15: msg += f"... et {len(suspects)-15} autres."

        return msg

    except Exception as e:
        return f"❌ Erreur critique : {str(e)}"

def _safe_int(value):
    """Convertit en int sans planter si NaN ou None"""
    try:
        if pd.isna(value) or math.isnan(value) or np.isinf(value):
            return 0
        return int(value)
    except:
        return 0

# --- 4. FONCTION DE CALCUL ---
def _path_metrics_detailed(points):
    if len(points) < 3: return {'ok': False}

    total_m = 0.0
    total_turn = 0.0
    below4 = 0

    first = points[0]
    last = points[-1]

    for i in range(1, len(points)):
        p0, p1 = points[i - 1], points[i]
        try:
            dist = haversine_km(p0['lat'], p0['lon'], p1['lat'], p1['lon'])
            if not math.isnan(dist): total_m += dist
        except: pass

        # Calcul sécurisé du virage
        turn = _ang_diff(p1.get('cog'), p0.get('cog'))
        if not math.isnan(turn):
            total_turn += abs(turn)

        if p1.get('sog', 0) < 4.0: below4 += 1

    try:
        gc_m = haversine_km(first['lat'], first['lon'], last['lat'], last['lon'])
        if gc_m is None or math.isnan(gc_m) or gc_m < 1.0:
            gc_m = 0.1 # Évite division par zéro

        straightness_ratio = total_m / gc_m
        if math.isnan(straightness_ratio) or np.isinf(straightness_ratio):
            straightness_ratio = 999.0

    except:
        gc_m, straightness_ratio = (0, 1.0)

    # ULTIME SÉCURITÉ : Si total_turn est resté NaN
    if math.isnan(total_turn): total_turn = 0.0

    return {
        'ok': True,
        'total_dist_km': total_m ,
        'net_dist_km': gc_m ,
        'straightness_ratio': straightness_ratio,
        'total_turn_deg': total_turn, # Garanti float valide
        'frac_below4': below4 / max(1, len(points)),
        'start_coords': (first['lat'], first['lon']),
        'end_coords': (last['lat'], last['lon'])
    }

# --- LE TOOL ---
@tool
def analyser_comportement_avance(zone: str, heures_analyse: float = 48.0) -> str:
    """
    Identifie les comportements avancés (patrouille, stationnaire, zig-zag) dans une zone donnée.

    Méthodologie :
    - extrait les trajectoires AIS récentes autour de la zone spécifiée,
    - calcule des métriques géométriques détaillées sur chaque trace (boucles, dispersion, vecteurs),
    - classe les navires selon leur pattern (patrouille, loitering, transit) et génère un résumé opérationnel.

    Args:
        zone (str): Zone nommée ou coordonnées "lat,lon" utilisées pour centrer l'analyse.
        heures_analyse (float): Profondeur temporelle de l'étude (en heures).

    Returns:
        str: Synthèse qualitative listant les navires détectés et leur comportement dominant.
    """
    if 'df_ais_global' not in globals(): return "❌ Erreur données."

    # A. RÉSOLUTION ZONE
    z_clean = _norm(zone)
    z_config = None
    if z_clean in ZONES_DATA: z_config = ZONES_DATA[z_clean]
    elif matches := [k for k in ZONES_DATA.keys() if k in z_clean]: z_config = ZONES_DATA[max(matches, key=len)]

    if not z_config and "," in zone:
        try: p=zone.replace(" ","").split(","); z_config={"lat":float(p[0]),"lon":float(p[1]),"rayon":200}
        except: pass
    if not z_config: return f"❌ Zone '{zone}' non reconnue."

    lat_c, lon_c, rad_km = z_config['lat'], z_config['lon'], z_config['rayon']

    # B. PRÉPARATION DONNÉES
    df = df_ais_global
    if not pd.api.types.is_datetime64_any_dtype(df['datetime']):
        df['datetime'] = pd.to_datetime(df['datetime'])

    date_start = df['datetime'].max() - pd.Timedelta(hours=heures_analyse)
    deg_lat = rad_km / 111.0

    df_zone = df[
        (df['datetime'] >= date_start) &
        (df['Position_LatitudeDeg'].between(lat_c - deg_lat, lat_c + deg_lat))
    ].copy()

    # NETTOYAGE CRITIQUE : On remplace les NaN dans les colonnes clés par 0
    # Cela empêche la propagation des erreurs dans les calculs
    df_zone['SogKt'] = df_zone['SogKt'].fillna(0.0)
    # Vérifiez le nom exact de votre colonne Cap (CogDeg ou CogDegrees ?)
    col_cog = 'CogDegrees' if 'CogDegrees' in df_zone.columns else 'CogDeg'
    if col_cog in df_zone.columns:
        df_zone[col_cog] = df_zone[col_cog].fillna(0.0)

    # C. ANALYSE
    suspects = []

    for mmsi, group in df_zone.groupby('Mmsi'):
        if len(group) < 4: continue

        group = group.sort_values('datetime')
        points_list = []
        for _, r in group.iterrows():
            points_list.append({
                'ts': r['datetime'].timestamp(),
                'lat': r['Position_LatitudeDeg'],
                'lon': r['Position_LongitudeDeg'],
                'sog': r.get('SogKt', 0.0),
                'cog': r.get(col_cog, 0.0) # Utilisation du nom de colonne dynamique
            })

        m = _path_metrics_detailed(points_list)
        if not m['ok']: continue

        last_row = group.iloc[-1]

        # --- CLASSIFICATION INTELLIGENTE ---
        cat = ""
        score = 0

        # CAS 1 : STATIONNAIRE
        if m['total_dist_km'] < 5.0 and m['net_dist_km'] < 2.0:
            continue

        # CAS 2 : LA BOUCLE / PATROUILLE
        elif m['total_dist_km'] > 10.0 and m['net_dist_km'] < 5.0:
            cat = "BOUCLE / RETOUR DÉPART"
            score = 20

        # CAS 3 : ZIG-ZAG / RECHERCHE
        elif m['straightness_ratio'] > 1.5 and m['frac_below4'] < 0.6:
            cat = "TRAJECTOIRE COMPLEXE (Zig-Zag)"
            score = 15

        # CAS 4 : ATTENTE ACTIVE
        elif m['frac_below4'] > 0.8 and m['total_dist_km'] > 5.0:
            cat = "ATTENTE ACTIVE (Loitering)"
            score = 10

        if cat:
            start_str = f"{m['start_coords'][0]:.3f},{m['start_coords'][1]:.3f}"
            end_str = f"{m['end_coords'][0]:.3f},{m['end_coords'][1]:.3f}"

            # Utilisation de _safe_int pour l'affichage final
            turn_display = _safe_int(m['total_turn_deg'])

            suspects.append({
                "nom": last_row.get('Name', 'Inconnu'),
                "type": last_row.get('ShipTypeLabel', '-'),
                "desc": cat,
                "stats": f"Dist. réelle: {m['total_dist_km']:.1f}km | Virages: {turn_display}°",
                "pos_info": f"Départ [{start_str}] -> Fin [{end_str}] (Écart: {m['net_dist_km']:.1f}km)",
                "score": score
            })

    # E. RAPPORT
    if not suspects:
        return f"✅ Trafic standard (Lignes droites ou mouillages fixes)."

    suspects.sort(key=lambda x: x['score'], reverse=True)

    msg = f"🧠 **ANALYSE TRAJECTOIRES : {zone.upper()}**\n"
    msg += f"ℹ️ Détection des boucles et mouvements complexes\n"
    msg += f"🚨 **{len(suspects)} navires d'intérêt**\n"
    msg += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"

    for s in suspects[:8]:
        msg += f"🚢 **{s['nom']}** ({s['type']})\n"
        msg += f"   🔄 **{s['desc']}**\n"
        msg += f"   📊 {s['stats']}\n"
        msg += f"   📍 {s['pos_info']}\n\n"

    return msg

# --- 3. LE TOOL ---

@tool
def analyser_profil_vitesse(zone: str, heures_analyse: float = 48.0) -> str:
    """
    Inspecte les profils de vitesse pour détecter loitering, stop-and-go et autres anomalies cinématiques.

    Le tool :
    - filtre les navires actifs dans la zone/période choisie,
    - calcule moyennes, ratios de basses vitesses et écarts-types pour chaque MMSI,
    - signale les navires dont la signature de vitesse évoque un comportement suspect.

    Args:
        zone (str): Zone nommée ou coordonnées "lat,lon".
        heures_analyse (float): Fenêtre temporelle analysée en heures (48h par défaut).

    Returns:
        str: Rapport listant les navires en alerte avec leurs métriques clefs et dernières positions.
    """
    if 'df_ais_global' not in globals(): return "❌ Erreur données."

    # A. RÉSOLUTION ZONE
    z_clean = _norm(zone)
    z_config = None
    if z_clean in ZONES_DATA: z_config = ZONES_DATA[z_clean]
    elif matches := [k for k in ZONES_DATA.keys() if k in z_clean]: z_config = ZONES_DATA[max(matches, key=len)]

    if not z_config and "," in zone:
        try: p=zone.replace(" ","").split(","); z_config={"lat":float(p[0]),"lon":float(p[1]),"rayon":200}
        except: pass
    if not z_config: return f"❌ Zone '{zone}' non reconnue."

    lat_c, lon_c, rad_km = z_config['lat'], z_config['lon'], z_config['rayon']

    # B. PRÉPARATION DONNÉES
    df = df_ais_global
    if not pd.api.types.is_datetime64_any_dtype(df['datetime']):
        df['datetime'] = pd.to_datetime(df['datetime'])

    date_start = df['datetime'].max() - pd.Timedelta(hours=heures_analyse)
    deg_lat = rad_km / 111.0

    # Filtre Geo + Temps
    df_zone = df[
        (df['datetime'] >= date_start) &
        (df['Position_LatitudeDeg'].between(lat_c - deg_lat, lat_c + deg_lat))
    ].copy()

    if df_zone.empty: return f"ℹ️ Aucun navire dans la zone sur la période."

    # C. ANALYSE PAR NAVIRE
    suspects = []

    for mmsi, group in df_zone.groupby('Mmsi'):
        if len(group) < 5: continue # Pas assez de points pour une stat

        group = group.sort_values('datetime')
        speeds = group['SogKt'].fillna(0)

        # STATS DE BASE
        avg_speed = speeds.mean()
        max_speed = speeds.max()
        std_dev = speeds.std() # Écart-type (Variance)

        # On ignore les navires qui sont restés à quai tout le temps (max < 1 nd)
        if max_speed < 1.0: continue

        last_row = group.iloc[-1]

        # SCÉNARIO 1 : LOITERING (Flânerie suspecte)
        # Navire qui a une vitesse très basse (< 3 nds) pendant longtemps
        # Mais qui n'est PAS à l'arrêt total (moyenne > 0.5)
        low_speed_count = (speeds < 3.0).sum()
        ratio_low = low_speed_count / len(speeds)

        if ratio_low > 0.6 and avg_speed > 0.5 and avg_speed < 4.0:
            # Calcul durée approximative du loitering
            duration_h = (group['datetime'].max() - group['datetime'].min()).total_seconds() / 3600

            suspects.append({
                "nom": last_row['Name'],
                "type": last_row['ShipTypeLabel'],
                "desc": "LOITERING (Dérive lente)",
                "detail": f"Vitesse moy: {avg_speed:.1f} nds | Max: {max_speed:.1f} nds",
                "reason": f"Sous 3 nds pendant {(ratio_low*100):.0f}% du temps ({duration_h:.1f}h)",
                "score": ratio_low * 10
            })
            continue # On classe et on passe

        # SCÉNARIO 2 : STOP & GO (Variation brutale)
        # Écart-type élevé (ex: > 4.0) signifie qu'il alterne vite et lent
        # Ex: Il file à 15 nds, s'arrête, repart.
        if std_dev > 4.0:
            # On vérifie l'amplitude (Max - Min)
            amplitude = max_speed - speeds.min()

            suspects.append({
                "nom": last_row['Name'],
                "type": last_row['ShipTypeLabel'],
                "desc": "STOP & GO (Vitesse instable)",
                "detail": f"Variation: +/- {std_dev:.1f} nds",
                "reason": f"Amplitude : {speeds.min():.1f} ➔ {max_speed:.1f} nds",
                "score": std_dev * 2
            })

    # D. RAPPORT
    if not suspects:
        return f"✅ Profils de vitesse réguliers (Transits constants ou arrêts francs)."

    suspects.sort(key=lambda x: x['score'], reverse=True)

    msg = f"🐇 **ANALYSE VITESSE (Speed Profiling) : {zone.upper()}**\n"
    msg += f"🚨 **{len(suspects)} navires au profil cinétique suspect**\n"
    msg += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"

    for s in suspects[:10]:
        icon = "🐢" if "LOITERING" in s['desc'] else "⚡"

        msg += f"{icon} **{s['nom']}** ({s['type']})\n"
        msg += f"   ⚠️ **{s['desc']}**\n"
        msg += f"   📊 {s['detail']}\n"
        msg += f"   🔎 {s['reason']}\n\n"

    return msg

@tool
def generer_rapport_anomalies_navigation(zone: str, heures_analyse: float = 72.0) -> str:
    """
    Génère un rapport multi-critères sur les navires aux comportements anormaux dans une zone.

    Anomalies couvertes :
    - arrêts prolongés ou loitering hors zone portuaire,
    - changements de cap/vitesse violents (zig-zag, demi-tour) détectés sur la période étudiée.

    Args:
        zone (str): Zone d'intérêt (nom officiel ou coordonnées "lat,lon").
        heures_analyse (float): Durée historique utilisée pour les calculs (heures).

    Returns:
        str: Rapport synthétique prêt à être transmis à un officier de quart.
    """
    if 'df_ais_global' not in globals(): return "❌ Erreur données."

    # A. RÉSOLUTION ZONE
    z_clean = _norm(zone)
    z_config = None
    if z_clean in ZONES_DATA: z_config = ZONES_DATA[z_clean]
    elif matches := [k for k in ZONES_DATA.keys() if k in z_clean]: z_config = ZONES_DATA[max(matches, key=len)]

    if not z_config and "," in zone:
        try: p=zone.replace(" ","").split(","); z_config={"lat":float(p[0]),"lon":float(p[1]),"rayon":200}
        except: pass
    if not z_config: return f"❌ Zone '{zone}' non reconnue."

    lat_c, lon_c, rad_km = z_config['lat'], z_config['lon'], z_config['rayon']

    # B. PRÉPARATION DONNÉES
    df = df_ais_global
    if not pd.api.types.is_datetime64_any_dtype(df['datetime']):
        df['datetime'] = pd.to_datetime(df['datetime'])

    date_start = df['datetime'].max() - pd.Timedelta(hours=heures_analyse)
    deg_lat = rad_km / 111.0

    df_zone = df[
        (df['datetime'] >= date_start) &
        (df['Position_LatitudeDeg'].between(lat_c - deg_lat, lat_c + deg_lat))
    ].copy()

    if df_zone.empty: return f"ℹ️ Aucun navire actif dans la zone sur les dernières {heures_analyse}h."

    # C. ANALYSE COMBINÉE (VITESSE + CAP)
    suspects = []

    for mmsi, group in df_zone.groupby('Mmsi'):
        if len(group) < 5: continue

        group = group.sort_values('datetime')
        last_row = group.iloc[-1]

        # --- ANALYSE 1 : ARRÊTS PROLONGÉS (Vitesse) ---
        speeds = group['SogKt'].fillna(0)
        avg_speed = speeds.mean()

        # Ratio de temps passé à très basse vitesse (< 3 nds)
        # Mais pas à l'arrêt total (on exclut le mouillage portuaire si max < 1)
        if speeds.max() > 1.0:
            low_speed_ratio = (speeds < 3.0).sum() / len(speeds)
        else:
            low_speed_ratio = 0 # Navire à quai

        est_arret_prolonge = False
        duree_arret = 0

        if low_speed_ratio > 0.6 and avg_speed < 4.0:
            est_arret_prolonge = True
            # Estimation durée (Temps total * ratio)
            total_duration = (group['datetime'].max() - group['datetime'].min()).total_seconds() / 3600
            duree_arret = total_duration * low_speed_ratio

        # --- ANALYSE 2 : CHANGEMENTS DE CAP (Trajectoire) ---
        est_cap_soudain = False
        virages_majeurs = 0

        # On ne regarde le cap que si le navire avance (> 2 nds) sinon le cap est fou
        mouv_group = group[group['SogKt'] > 2.0]

        if len(mouv_group) > 3:
            cogs = mouv_group['CogDegrees'].tolist()
            for i in range(1, len(cogs)):
                # Changement brutal > 60° entre deux points
                if _ang_diff(cogs[i], cogs[i-1]) > 60:
                    virages_majeurs += 1

            # Si plus de 2 virages majeurs ou ratio élevé
            if virages_majeurs >= 2:
                est_cap_soudain = True

        # --- CLASSIFICATION FINALE ---
        types_anomalies = []
        score = 0

        if est_arret_prolonge:
            types_anomalies.append(f"🛑 Arrêt prolongé (~{duree_arret:.1f}h)")
            score += 10

        if est_cap_soudain:
            types_anomalies.append(f"↪️ Cap erratique ({virages_majeurs} virages brusques)")
            score += 15

        if types_anomalies:
            suspects.append({
                "nom": last_row['Name'],
                "type": last_row['ShipTypeLabel'],
                "anomalies": types_anomalies,
                "score": score,
                "lat": last_row['Position_LatitudeDeg'],
                "lon": last_row['Position_LongitudeDeg']
            })

    # D. RAPPORT
    if not suspects:
        return f"✅ Trafic régulier. Aucun arrêt suspect ni changement de cap brutal détecté sur {heures_analyse}h."

    suspects.sort(key=lambda x: x['score'], reverse=True)

    msg = f"📉 **RAPPORT ANOMALIES NAVIGATION : {zone.upper()}**\n"
    msg += f"🕒 Période analysée : Dernières {heures_analyse}h\n"
    msg += f"🚨 **{len(suspects)} navires au comportement atypique**\n"
    msg += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"

    for s in suspects[:10]:
        msg += f"🚢 **{s['nom']}** ({s['type']})\n"
        for ano in s['anomalies']:
            msg += f"   ⚠️ {ano}\n"
        msg += f"   📍 Dernière pos : {s['lat']:.3f}, {s['lon']:.3f}\n\n"

    return msg

def _get_cardinal_direction(cog):
    """Convertit un cap (0-360) en direction cardinale (N, NE, E...)"""
    if pd.isna(cog): return "Inconnu"
    dirs = ["N", "NE", "E", "SE", "S", "SO", "O", "NO"]
    ix = round(cog / 45)
    return dirs[ix % 8]

#### Analyse de zone

Tendances de trafic, évaluation de menace, et cartographie des *zones blanches* — les secteurs sans couverture AIS où un navire peut disparaître.

`analyser_tendances_trafic`, `analyser_menaces_zone`, `analyser_et_cartographier_zones_blanches`

In [ ]:
# --- LE TOOL ---

@tool
def analyser_tendances_trafic(zone: str, heures_analyse: float = 72.0) -> str:
    """
    Produit une analyse « Pattern of Life » recensant les tendances de trafic maritime.

    Contenu du rapport :
    - distribution par type de navire et niveau d'activité (à quai vs en mouvement),
    - axes de circulation dominants déduits des caps moyens,
    - statistiques de vitesse moyenne et indicateurs de densité.

    Args:
        zone (str): Zone suivie (nom connue du scénario ou coordonnées "lat,lon").
        heures_analyse (float): Fenêtre d'observation en heures.

    Returns:
        str: Synthèse chiffrée permettant de décrire le trafic normal ou les écarts détectés.
    """
    if 'df_ais_global' not in globals(): return "❌ Erreur données."

    # 1. ZONE
    z_clean = _norm(zone)
    z_config = None
    if z_clean in ZONES_DATA: z_config = ZONES_DATA[z_clean]
    elif matches := [k for k in ZONES_DATA.keys() if k in z_clean]: z_config = ZONES_DATA[max(matches, key=len)]

    if not z_config and "," in zone:
        try: p=zone.replace(" ","").split(","); z_config={"lat":float(p[0]),"lon":float(p[1]),"rayon":200}
        except: pass
    if not z_config: return f"❌ Zone '{zone}' inconnue."

    lat_c, lon_c, rad = z_config['lat'], z_config['lon'], z_config['rayon']

    # 2. DATA
    df = df_ais_global
    if not pd.api.types.is_datetime64_any_dtype(df['datetime']):
        df['datetime'] = pd.to_datetime(df['datetime'])

    start = df['datetime'].max() - pd.Timedelta(hours=heures_analyse)
    deg_lat = rad / 111.0

    # Filtre Geo + Temps
    df_zone = df[
        (df['datetime'] >= start) &
        (df['Position_LatitudeDeg'].between(lat_c - deg_lat, lat_c + deg_lat))
    ].copy()

    if df_zone.empty: return "ℹ️ Aucune donnée de trafic sur la période."

    # 3. STATISTIQUES (Sur navires uniques)
    # On prend la dernière position de chaque navire pour les stats de répartition
    df_unique = df_zone.sort_values('datetime').drop_duplicates('Mmsi', keep='last')

    total_ships = len(df_unique)

    # A. Répartition par Type
    type_counts = df_unique['ShipTypeLabel'].value_counts()
    top_types = type_counts.head(3)

    # B. Analyse des Flux (Directions)
    # On regarde où pointent les étraves
    df_unique['Cardinal'] = df_unique['CogDegrees'].apply(_get_cardinal_direction)
    flow_counts = df_unique['Cardinal'].value_counts()
    main_flow = flow_counts.idxmax()
    main_flow_pct = (flow_counts.max() / total_ships) * 100

    # C. Analyse Vitesse
    avg_speed = df_unique['SogKt'].mean()
    # Navires à l'arrêt vs en route
    stationary = len(df_unique[df_unique['SogKt'] < 1.0])
    moving_pct = ((total_ships - stationary) / total_ships) * 100

    # 4. RAPPORT SYNTHÉTIQUE
    msg = f"📈 **TENDANCES TRAFIC (POL) : {zone.upper()}**\n"
    msg += f"🕒 Période : Dernières {heures_analyse}h\n"
    msg += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"

    msg += f"📊 **DENSITÉ : {total_ships} navires uniques**\n"
    msg += f"   • En mouvement : {moving_pct:.0f}%\n"
    msg += f"   • Au mouillage/quai : {stationary} navires\n\n"

    msg += f"🚢 **COMPOSITION DE LA FLOTTE**\n"
    for type_name, count in top_types.items():
        pct = (count / total_ships) * 100
        icon = "🛢️" if "TANKER" in type_name.upper() else "📦" if "CARGO" in type_name.upper() else "🐟"
        msg += f"   {icon} {type_name}: {count} ({pct:.0f}%)\n"

    msg += f"\n🧭 **DYNAMIQUE DES FLUX**\n"
    msg += f"   • Direction Dominante : **Vers {main_flow}** ({main_flow_pct:.0f}% du trafic)\n"
    msg += f"   • Vitesse Moyenne : {avg_speed:.1f} nœuds\n"

    # Petit insight intelligent
    if main_flow in ["E", "O"] and "MANCHE" in zone.upper():
        msg += "   💡 *Note : Correspond au rail montant/descendant du trafic commercial.*\n"

    return msg

@tool
def analyser_menaces_zone(zone_lat: float, zone_lon: float, rayon_km: float, date_str: str) -> str:
    """
    Croise AIS et RF pour hiérarchiser les menaces autour d'un point donné.

    Processus :
    - isole les signaux RF présents dans le rayon demandé à la date ciblée,
    - confronte ces émissions aux positions AIS disponibles (corrélation ou absence totale),
    - classe les résultats en menaces hautes, moyennes ou simples confirmations.

    Args:
        zone_lat (float): Latitude du centre de zone à surveiller.
        zone_lon (float): Longitude du centre de zone à surveiller.
        rayon_km (float): Rayon de recherche en kilomètres.
        date_str (str): Date d'analyse au format 'YYYY-MM-DD'.

    Returns:
        str: Rapport d'alerte prêt à diffusion avec les différentes listes (haute/moyenne/info).
    """
    try:
        # 1. Vérification des données globales
        if 'df_ais_global' not in globals() or 'df_rf_global' not in globals():
            return "❌ Erreur technique : Les DataFrames 'df_ais_global' ou 'df_rf_global' ne sont pas chargés."

        # 2. Préparation et Filtrage des données RF (Unseen Labs)
        # On filtre d'abord grossièrement par carré pour la rapidité
        lat_min, lat_max = zone_lat - 2.0, zone_lat + 2.0
        lon_min, lon_max = zone_lon - 2.0, zone_lon + 2.0

        # Filtrer RF par date et bounding box
        mask_rf = (
            (df_rf_global['timestamp_utc'].astype(str).str.contains(date_str)) &
            (df_rf_global['latitude_deg'].between(lat_min, lat_max)) &
            (df_rf_global['longitude_deg'].between(lon_min, lon_max))
        )
        local_rf = df_rf_global[mask_rf].copy()

        if local_rf.empty:
            return f"✅ R.A.S. Aucune émission RF détectée dans cette zone le {date_str}."

        # Filtrer AIS par date (Orbit Sailor : 1 point par jour)
        # On s'assure d'avoir les navires actifs ce jour-là
        mask_ais = df_ais_global['datetime'].astype(str).str.contains(date_str)
        daily_ais = df_ais_global[mask_ais]

        # Listes pour le rapport
        alertes_hautes = []   # Dark Ships
        alertes_moyennes = [] # Incohérences
        confi_ais = []        # Navires corrélés (Info)

        # 3. Boucle d'analyse
        for idx, signal in local_rf.iterrows():
            # Vérification distance précise
            dist = haversine_km(zone_lat, zone_lon, signal['latitude_deg'], signal['longitude_deg'])
            if dist > rayon_km:
                continue

            # --- ANALYSE INTELLIGENTE ---

            # Récupération des données de confiance Unseen Labs
            # Note : Adapter le nom exact si 'cand_1_confidence' ou autre
            confiance = signal.get('cand_1_confidence', 0)
            candidat_mmsi = signal.get('cand_1_mmsi', None)

            # CAS 1 : Signal non corrélé par le capteur (Dark Ship probable)
            if pd.isna(candidat_mmsi) or confiance < 50: # Seuil arbitraire 50%
                alertes_hautes.append(
                    f"- Signal {signal.get('RF_Frequency_MHz', '?')} MHz à {signal['latitude_deg']:.3f}, {signal['longitude_deg']:.3f}. "
                    f"Aucun navire AIS identifié."
                )

            # CAS 2 : Le capteur propose un candidat, on vérifie dans NOTRE base AIS
            else:
                # Le capteur dit "C'est probablement le navire X"
                # Vérifions si le navire X a émis de l'AIS ce jour-là dans nos données
                navire_ais = daily_ais[daily_ais['Mmsi'] == candidat_mmsi]

                if navire_ais.empty:
                    # Le capteur RF l'a reconnu, mais il n'est pas dans les logs AIS du jour
                    nom_candidat = signal.get('cand_1_name', 'Inconnu')
                    alertes_moyennes.append(
                        f"- Signal attribué au {nom_candidat} (MMSI: {candidat_mmsi}), mais navire ABSENT des logs AIS du jour."
                    )
                else:
                    # Match confirmé
                    nom_navire = navire_ais.iloc[0]['Name']
                    confi_ais.append(nom_navire)

        # 4. Génération du rapport formaté
        nb_alerts = len(alertes_hautes) + len(alertes_moyennes)
        if nb_alerts == 0:
            return f"✅ Zone claire. {len(confi_ais)} navires identifiés et corrélés par RF."

        rapport = f"""🚨 **RAPPORT DE SURVEILLANCE : {len(alertes_hautes)} MENACES DÉTECTÉES**
📍 Zone : {zone_lat}, {zone_lon} ({rayon_km} km) | Date : {date_str}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔴 **MENACES HAUTES (DARK SHIPS)**
_Navires émettant en radio mais invisibles en AIS_
{chr(10).join(alertes_hautes) if alertes_hautes else "Aucune"}

🟠 **ANOMALIES (INCOHÉRENCES)**
_Navires détectés par RF mais muets en AIS aujourd'hui_
{chr(10).join(alertes_moyennes) if alertes_moyennes else "Aucune"}

🟢 **CORRÉLATIONS CONFIRMÉES**
_Navires dont l'émission RF correspond à la position AIS_
{', '.join(confi_ais[:5])} {'...' if len(confi_ais)>5 else ''} ({len(confi_ais)} total)
"""
        return rapport

    except Exception as e:
        return f"❌ Erreur lors de l'analyse RF: {str(e)}"

@tool
def analyser_et_cartographier_zones_blanches(
    resolution_deg: float = 1.0,
    seuil_min_heures: int = 24,
    start_date: str = None,
    end_date: str = None
) -> str:
    """
    TOOL COMPLET : Analyse les zones sans couverture AIS.
    1. Affiche une CARTE interactive (Heatmap) sur fond CLAIR.
    2. Retourne un RAPPORT textuel détaillé sur les zones aveugles.
    """
    try:
        # --- 1. PREPARATION DONNEES ---
        if 'df_ais_global' not in globals() or df_ais_global is None:
            return "❌ Données AIS non disponibles."

        df = df_ais_global.copy()
        if not pd.api.types.is_datetime64_any_dtype(df['datetime']):
            df['datetime'] = pd.to_datetime(df['datetime'], utc=True)

        # Filtre Date
        if start_date: df = df[df['datetime'] >= pd.to_datetime(start_date, utc=True)]
        if end_date: df = df[df['datetime'] <= pd.to_datetime(end_date, utc=True)]

        # --- 2. CALCUL GRILLE ---
        df['lat_bin'] = np.floor(df['Position_LatitudeDeg'] / resolution_deg) * resolution_deg
        df['lon_bin'] = np.floor(df['Position_LongitudeDeg'] / resolution_deg) * resolution_deg

        grid = df.groupby(['lat_bin', 'lon_bin'])['datetime'].max().reset_index()
        ref_date = df['datetime'].max()

        grid['silence'] = (ref_date - grid['datetime']).dt.total_seconds() / 3600.0

        blanches = grid[grid['silence'] >= seuil_min_heures].copy()

        # Centres
        blanches['lat_c'] = blanches['lat_bin'] + (resolution_deg / 2)
        blanches['lon_c'] = blanches['lon_bin'] + (resolution_deg / 2)

        if blanches.empty:
            return f"✅ Couverture excellente. Aucune zone avec > {seuil_min_heures}h de silence."

        # --- 3. GENERATION CARTE (CORRIGÉE) ---

        # A. Création avec fond de carte CLAIR par défaut (OpenStreetMap)
        m = folium.Map(
            location=[blanches['lat_c'].mean(), blanches['lon_c'].mean()],
            zoom_start=3,
            tiles='OpenStreetMap' # <--- CHANGEMENT ICI (Au lieu de CartoDB dark_matter)
        )

        # Ajout d'options de fonds de carte (Layer Control)
        folium.TileLayer('CartoDB positron', name='Fond Clair (Minimal)').add_to(m)
        folium.TileLayer('CartoDB dark_matter', name='Fond Sombre').add_to(m)

        # B. Heatmap
        heat_data = blanches[['lat_c', 'lon_c', 'silence']].values.tolist()
        HeatMap(
            heat_data,
            name="🔥 Intensité Silence",
            min_opacity=0.4,
            radius=25,
            blur=15,
            gradient={0.2: 'blue', 0.4: 'lime', 0.6: 'orange', 1: 'red'}
        ).add_to(m)

        # C. Cercles
        circles = folium.FeatureGroup(name="⭕ Zones Critiques")
        for _, r in blanches.iterrows():
            hours = int(r['silence'])
            color = '#ff0000' if hours > 100 else '#ff9900'
            folium.Circle(
                location=[r['lat_c'], r['lon_c']],
                radius=resolution_deg * 45000,
                color=color, fill=False, weight=2,
                popup=f"Silence: {hours}h",
            ).add_to(circles)
        circles.add_to(m)

        # Ajout du contrôle de couches (en haut à droite)
        folium.LayerControl().add_to(m)

        # D. AFFICHAGE DANS L'INTERFACE
        if 'actions_output' in globals():
            with actions_output:
                clear_output(wait=True)
                print(f"🗺️ CARTE GÉNÉRÉE : {len(blanches)} zones aveugles détectées.")
                display(m)
        else:
            display(m)

        # --- 4. GENERATION RAPPORT ---
        blanches = blanches.sort_values('silence', ascending=False)
        total_zones = len(grid)
        pct_loss = (len(blanches) / max(total_zones, 1)) * 100

        rpt = f"📉 **RAPPORT COUVERTURE AIS**\n"
        rpt += f"📅 Analyse au : {ref_date.strftime('%d/%m %H:%M')}\n"
        rpt += f"⚠️ **{len(blanches)} Zones Aveugles** (> {seuil_min_heures}h de silence)\n"
        rpt += f"🌍 Perte de couverture estimée : **{pct_loss:.1f}%** de la surface surveillée.\n"
        rpt += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"

        rpt += "📍 **TOP 5 DES ZONES CRITIQUES :**\n"
        for _, r in blanches.head(5).iterrows():
            rpt += f"- [{r['lat_c']:.1f}, {r['lon_c']:.1f}] : Muet depuis **{int(r['silence'])}h**\n"

        rpt += "\n🗺️ *La carte interactive (Fond Clair) a été affichée ci-dessous.* 👇"

        return rpt

    except Exception as e:
        return f"❌ Erreur technique : {str(e)}"

from langchain_core.tools import tool
import pandas as pd
import numpy as np
import math
from datetime import datetime, timedelta

# --- 1. FONCTIONS MATHÉMATIQUES & NAVIGATION ---

def _calculer_point_estime(lat, lon, vitesse_nds, cap_deg, heures_ecoulees):
    """
    Calcule la position future estimée (Dead Reckoning) sur la sphère terrestre.
    """
    R = 6371.0 # Rayon Terre km
    # Distance parcourue = Vitesse (noeuds) * 1.852 (km/nm) * Temps (h)
    dist_km = (vitesse_nds * 1.852) * heures_ecoulees

    lat_rad = math.radians(lat)
    lon_rad = math.radians(lon)
    cap_rad = math.radians(cap_deg)

    # Formules de trigonométrie sphérique
    lat_new = math.asin(math.sin(lat_rad) * math.cos(dist_km/R) +
                        math.cos(lat_rad) * math.sin(dist_km/R) * math.cos(cap_rad))

    lon_new = lon_rad + math.atan2(math.sin(cap_rad) * math.sin(dist_km/R) * math.cos(lat_rad),
                                   math.cos(dist_km/R) - math.sin(lat_rad) * math.sin(lat_new))

    return math.degrees(lat_new), math.degrees(lon_new), dist_km

def _est_sous_trajectoire_sat(lat_navire, lon_navire, row_sat, buffer_deg=2.0):
    """
    Vérifie si la position estimée tombe sous la fauchée du satellite.
    Utilise les colonnes Start/End Lat/Lon du fichier CNES.
    """
    try:
        lat_min = min(row_sat['StartLatitude'], row_sat['EndLatitude']) - buffer_deg
        lat_max = max(row_sat['StartLatitude'], row_sat['EndLatitude']) + buffer_deg
        lon_min = min(row_sat['StartLongitude'], row_sat['EndLongitude']) - buffer_deg
        lon_max = max(row_sat['StartLongitude'], row_sat['EndLongitude']) + buffer_deg

        return (lat_min <= lat_navire <= lat_max) and (lon_min <= lon_navire <= lon_max)
    except:
        return False

def _haversine_distance(lat1, lon1, lat2, lon2):
    """Distance en km."""
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

#### Investigation et OSINT

Reconstitution du scénario de disparition d'un navire, et enrichissement de sa fiche par sources ouvertes.

`investiguer_disparition_navire`, `enrichir_navire_osint`

In [ ]:
# --- 2. LE TOOL ---

@tool
def investiguer_disparition_navire(identifiant_navire: str, heures_estimees: float = 0.0) -> str:
    """
    Reconstitue la trajectoire probable d'un navire devenu silencieux et vérifie les capteurs disponibles.

    Étapes :
    - récupère la dernière position AIS connue et calcule une estime (dead reckoning) selon cap/vitesse,
    - vérifie les passages satellites CNES capables d'imager la zone pendant la période supposée,
    - parcourt les détections RF postérieures pour confirmer ou infirmer la présence du navire.

    Args:
        identifiant_navire (str): MMSI ou nom exact du navire recherché.
        heures_estimees (float): Temps écoulé depuis la perte de signal (0 = calcul automatique).

    Returns:
        str: Rapport d'investigation détaillant position estimée, couverture satellite et traces RF.
    """
    try:
        # A. VÉRIFICATIONS
        if 'df_ais_global' not in globals(): return "❌ Erreur : Données AIS non chargées."

        # B. RETROUVER LA DERNIÈRE POSITION CONNUE (LKP)
        target = str(identifiant_navire).strip().upper()
        df = df_ais_global

        # Recherche par Nom ou MMSI
        ship = df[df['Name'].str.upper() == target]
        if ship.empty and target.isdigit():
            ship = df[df['Mmsi'].astype(str) == target]

        if ship.empty:
            return f"❌ Navire '{identifiant_navire}' introuvable dans l'historique AIS."

        # On prend le dernier point connu (Last Known Position)
        lkp = ship.sort_values('datetime', ascending=False).iloc[0]

        if not isinstance(lkp['datetime'], pd.Timestamp):
            lkp_time = pd.to_datetime(lkp['datetime'])
        else:
            lkp_time = lkp['datetime']

        # C. DÉFINITION DU TEMPS ÉCOULÉ (Delta T)
        # Si l'utilisateur donne un temps ("depuis 5h"), on l'utilise.
        # Sinon, on calcule par rapport à la "fin du monde" (fin du dataset).
        if heures_estimees > 0:
            delta_h = float(heures_estimees)
            now_simulated = lkp_time + timedelta(hours=delta_h)
        else:
            dataset_end = pd.to_datetime(df['datetime']).max()
            delta_h = (dataset_end - lkp_time).total_seconds() / 3600.0
            now_simulated = dataset_end

        if delta_h < 0.5:
            return f"ℹ️ Le navire '{lkp['Name']}' est actif (Dernière pos il y a {int(delta_h*60)} min)."

        # D. CALCUL DE L'ESTIME (Où devrait-il être ?)
        # Récupération vitesse (SOG) et cap (COG)
        try:
            vitesse = float(lkp.get('SogKt', 0))
            cap = float(lkp.get('CogDegrees', 0))
        except:
            vitesse, cap = 0.0, 0.0

        # Si le navire était à l'arrêt, on suppose qu'il a pu redémarrer (vitesse min de recherche 5 nds)
        # ou qu'il a dérivé. Pour l'enquête, on projette une zone probable.
        search_speed = max(vitesse, 5.0) if vitesse < 1.0 else vitesse

        est_lat, est_lon, dist_proj = _calculer_point_estime(
            lkp['Position_LatitudeDeg'], lkp['Position_LongitudeDeg'],
            search_speed, cap, delta_h
        )

        # Rayon d'incertitude : 10% de la distance parcourue + base 20km
        rayon_zone = (dist_proj * 0.15) + 20.0

        # E. ANALYSE SATELLITAIRE (Éphémérides CNES)
        sats_passage = []
        if 'df_ephemeris_global' in globals():
            df_eph = df_ephemeris_global
            # Filtre temporel : Satellites passés ENTRE la disparition et maintenant
            # On suppose que les colonnes sont converties en datetime
            try:
                relevant_sats = df_eph[
                    (pd.to_datetime(df_eph['EndTime']) >= lkp_time) &
                    (pd.to_datetime(df_eph['StartTime']) <= now_simulated)
                ]

                for _, row_sat in relevant_sats.iterrows():
                    if _est_sous_trajectoire_sat(est_lat, est_lon, row_sat):
                        t_pass = pd.to_datetime(row_sat['StartTime']).strftime('%H:%M')
                        sats_passage.append(f"{row_sat['Source']} ({t_pass})")
            except Exception:
                pass # Erreur de format de date ou colonne manquante

        sat_status = "✅ IMAGERIE POSSIBLE" if sats_passage else "⚠️ ZONE AVEUGLE (Aucun passage)"

        # F. RECHERCHE SIGNAUX RF (Preuve de vie)
        candidats_rf = []
        if 'df_rf_global' in globals():
            df_rf = df_rf_global
            for _, rf in df_rf.iterrows():
                rf_t = pd.to_datetime(rf['timestamp_utc'])

                # Le signal doit être POSTÉRIEUR à la disparition
                if rf_t <= lkp_time: continue

                # Le signal doit être proche de la position ESTIMÉE
                d = _haversine_distance(est_lat, est_lon, rf['latitude_deg'], rf['longitude_deg'])

                if d <= rayon_zone:
                    conf = "Moyenne"
                    if d < 15: conf = "FORTE (Sur la route estimée)"

                    candidats_rf.append({
                        "dist": d,
                        "lat": rf['latitude_deg'], "lon": rf['longitude_deg'],
                        "time": rf_t,
                        "freq": rf.get('RF_Frequency_MHz', '?'),
                        "conf": conf
                    })

        # G. GÉNÉRATION DU RAPPORT FORENSIQUE
        msg = f"🕵️‍♂️ **ENQUÊTE CIBLE : {lkp['Name']}**\n"
        msg += f"🔴 Silence AIS : **{delta_h:.1f} heures**\n"
        msg += f"⚡ Vitesse dernière : {vitesse} nds | Cap : {cap}°\n"
        msg += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"

        msg += f"📍 **1. POSITION ESTIMÉE (Projection)**\n"
        msg += f"   Si le navire a gardé son cap/vitesse, il est ici :\n"
        msg += f"   🌍 **{est_lat:.4f}, {est_lon:.4f}** (+/- {rayon_zone:.0f} km)\n"
        msg += f"   _(Distance théorique parcourue : {dist_proj:.1f} km)_\n\n"

        msg += f"🛰️ **2. COUVERTURE SATELLITE (Opportunité)**\n"
        msg += f"   Statut : {sat_status}\n"
        if sats_passage:
            msg += f"   Satellites ayant survolé la zone estimée :\n"
            for s in sats_passage[:3]: msg += f"   🛰️ {s}\n"
            if len(sats_passage)>3: msg += f"   ... (+{len(sats_passage)-3})\n"

        msg += f"\n📡 **3. CORRÉLATION SIGNAUX RF**\n"
        if candidats_rf:
            candidats_rf.sort(key=lambda x: x['dist'])
            best = candidats_rf[0]
            msg += f"🚨 **MATCH DÉTECTÉ !**\n"
            msg += f"   Émetteur actif trouvé à {best['dist']:.1f} km de la position estimée.\n"
            msg += f"   📍 Pos : {best['lat']:.4f}, {best['lon']:.4f}\n"
            msg += f"   🕒 Heure : {best['time']}\n"
            msg += f"   📻 Fréq : {best['freq']} MHz | Confiance : **{best['conf']}**\n"
        else:
            msg += f"   ❌ Aucun signal RF corrélé dans le cercle d'incertitude.\n"
            if sats_passage:
                msg += "   (Le navire est probablement en silence radio total ou a changé de cap).\n"

        return msg

    except Exception as e:
        return f"❌ Erreur critique Tool Enquête : {str(e)}"


import wikipedia
!pip install wikipedia
# Configuration langue Wikipedia
wikipedia.set_lang("fr")

@tool
def enrichir_navire_osint(identification: str, ident_type: str = "NAME") -> str:
    """
    Construit une fiche OSINT en combinant la base AIS locale et les informations publiques (Wikipedia).

    Déroulé :
    - tente d'abord de confirmer l'identité dans df_ais_global (nom, MMSI, type, pavillon),
    - lance ensuite une recherche Wikipedia robuste en français pour extraire un résumé fiable,
    - assemble les éléments dans un rapport lisible avec références sources.

    Args:
        identification (str): Nom du navire ou MMSI fourni par l'utilisateur.
        ident_type (str): Type d'identifiant ('NAME' ou 'MMSI').

    Returns:
        str: Rapport OSINT mis en forme, prêt à être cité ou copié dans un briefing.
    """
    report = []
    report.append(f"🛰️ **RAPPORT OSINT : {identification.upper()}**")
    report.append("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

    # --- 1. RECHERCHE AIS LOCALE (Pour confirmer l'identité) ---
    ais_found = False
    navire_nom_complet = identification

    if 'df_ais_global' in globals():
        df = df_ais_global
        match = pd.DataFrame()

        if ident_type.upper() == "MMSI":
            match = df[df['Mmsi'].astype(str) == str(identification)]
        else:
            # Recherche partielle insensible à la casse
            match = df[df['Name'].astype(str).str.contains(identification, case=False, na=False)]

        if not match.empty:
            ais_found = True
            row = match.iloc[0]
            navire_nom_complet = row['Name'] # On récupère le vrai nom AIS

            report.append("📡 **IDENTIFICATION AIS (Confirmée)**")
            report.append(f"- 🚢 Nom : {row['Name']}")
            report.append(f"- 🆔 MMSI : {row['Mmsi']}")
            report.append(f"- 🏳️ Pavillon : {row.get('Pays', 'Inconnu')}")
            report.append(f"- 🏷️ Type : {row.get('ShipTypeLabel', 'Inconnu')}")
        else:
            report.append("⚠️ Navire non trouvé dans la base AIS locale (Recherche Web uniquement).")

    # --- 2. RECHERCHE WIKIPEDIA ROBUSTE ---
    report.append("\n📚 **ENCYCLOPÉDIE (Wikipedia)**")

    try:
        # On ajoute "navire" pour aider la recherche à désambiguïser
        # Ex: "Charles de Gaulle" -> "Charles de Gaulle (porte-avions)"
        search_query = navire_nom_complet

        # 1. On cherche des suggestions
        results = wikipedia.search(search_query)

        if not results:
            # Essai avec le mot clé "navire" si rien trouvé
            results = wikipedia.search(search_query + " navire")

        if results:
            # On prend le premier résultat pertinent
            # On évite les homonymes (ex: aéroport, général)
            best_page = results[0]

            # Si le nom est ambigu, on essaie de trouver celui qui contient "ship", "navire", "vessel"
            for res in results:
                if any(x in res.lower() for x in ['navire', 'ship', 'vessel', 'porte-avions', 'ferry', 'tanker']):
                    best_page = res
                    break

            # 2. Extraction du résumé
            page = wikipedia.page(best_page, auto_suggest=False)
            summary = page.summary[0:500] + "..." # On limite à 500 caractères

            report.append(f"🔗 **Page trouvée :** {page.title}")
            report.append(f"📝 {summary}")
            report.append(f"\nSource : {page.url}")

        else:
            report.append("❌ Aucune page Wikipedia trouvée pour ce navire.")

    except wikipedia.exceptions.DisambiguationError as e:
        report.append(f"⚠️ **Ambiguïté détectée.** Trop de résultats : {e.options[:3]}")
    except wikipedia.exceptions.PageError:
        report.append("❌ Page introuvable sur Wikipedia.")
    except Exception as e:
        report.append(f"❌ Erreur technique Wikipedia : {str(e)}")

    return "\n".join(report)

#### Estimation de destination (fusion AIS + RF)

Destination probable d'un navire suspect, en croisant sa dernière trajectoire AIS et les détections RF postérieures.

`estimate_suspect_destination_ais_rf`

In [ ]:
@tool
def estimate_suspect_destination_ais_rf(
    zone: str,
    suspect_mmsi: str,
    days: int = 3,
    rf_corr_min_conf: float = 0.5,
    projection_hours: float = 24.0
) -> Dict[str, Any]:
    """
    Estime la destination probable d'un navire suspect en corrélant AIS et RF.

    Hypothèses :
    - df_ais_global : AIS avec colonnes
        ['Mmsi', 'datetime', 'Position_LatitudeDeg', 'Position_LongitudeDeg',
         'SogKt', 'Name', 'ShipTypeLabel', 'Pays'/'Country', ...]
    - df_rf_global : RF brut (optionnel ici, on utilise surtout df_corr_global)
    - df_corr_global : rf_correlation.parquet, avec :
        ['timestamp_utc', 'latitude_deg', 'longitude_deg',
         'RF_Frequency_MHz', 'cand_1_mmsi', 'cand_1_confidence', ...]
    - ZONES_DATA : dict {nom_zone_normalisé: {"lat": ..., "lon": ..., "rayon": ...}}
    - haversine_km(lat1, lon1, lat2, lon2) déjà défini

    Paramètres :
    - zone : nom de zone ou "lat,lon" (centre) (rayon pris par défaut si non connu)
    - suspect_mmsi : MMSI du navire suspect (str ou int)
    - days : fenêtre temporelle en jours à analyser (par défaut 3)
    - rf_corr_min_conf : seuil de confiance min pour utiliser la corrélation RF–AIS
    - projection_hours : horizon de projection pour estimer la destination (24h par défaut)

    Retour :
    - dict avec :
        - status, message
        - info sur la trajectoire fusionnée
        - point projeté
        - narratif "commandant"
    """
    try:
        # === Vérifs de base sur les données ===
        if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
            return {"status": "error", "message": "Données AIS (df_ais_global) non chargées."}

        has_corr = 'df_corr_global' in globals() and df_corr_global is not None and not df_corr_global.empty

        # === A. Résolution de la zone ===
        z_clean = _norm(zone)
        z_config = None

        if z_clean in ZONES_DATA:
            z_config = ZONES_DATA[z_clean]
        else:
            for k in ZONES_DATA:
                if k in z_clean or z_clean in k:
                    z_config = ZONES_DATA[k]
                    break

        # Format "lat,lon" en fallback
        if not z_config and "," in zone:
            try:
                p = zone.replace(" ", "").split(",")
                z_config = {"lat": float(p[0]), "lon": float(p[1]), "rayon": 200.0}
            except Exception:
                pass

        if not z_config:
            return {"status": "error", "message": f"Zone '{zone}' inconnue."}

        lat_c, lon_c, rad_km = z_config['lat'], z_config['lon'], z_config['rayon']
        deg_lat = rad_km / 111.0  # approx 1° ≈ 111 km

        # === B. Fenêtre temporelle globale ===
        df_ais = df_ais_global.copy()
        if not pd.api.types.is_datetime64_any_dtype(df_ais['datetime']):
            df_ais['datetime'] = pd.to_datetime(df_ais['datetime'])

        t_ais_max = df_ais['datetime'].max()

        if has_corr:
            df_corr = df_corr_global.copy()
            if not pd.api.types.is_datetime64_any_dtype(df_corr['timestamp_utc']):
                df_corr['timestamp_utc'] = pd.to_datetime(df_corr['timestamp_utc'])
            t_corr_max = df_corr['timestamp_utc'].max()
            t_global_max = max(t_ais_max, t_corr_max)
        else:
            t_global_max = t_ais_max

        t_min = t_global_max - pd.Timedelta(days=days)

        # === C. Filtre AIS sur MMSI, temps, zone ===
        mmsi_str = str(suspect_mmsi)

        df_ais_m = df_ais[
            (df_ais['Mmsi'].astype(str) == mmsi_str) &
            (df_ais['datetime'] >= t_min)
        ].copy()

        if df_ais_m.empty:
            return {
                "status": "ok",
                "zone": zone,
                "suspect_mmsi": mmsi_str,
                "message": (
                    f"Aucune donnée AIS récente (≤ {days} jours) pour ce MMSI dans la zone considérée."
                )
            }

        # Filtre géographique grossier sur latitude (bbox)
        df_ais_m = df_ais_m[
            (df_ais_m['Position_LatitudeDeg'].between(lat_c - deg_lat, lat_c + deg_lat))
        ].copy()

        if df_ais_m.empty:
            return {
                "status": "ok",
                "zone": zone,
                "suspect_mmsi": mmsi_str,
                "message": "Navire suspect sans positions AIS dans la zone spécifiée sur la période analysée."
            }

        # BBox longitude large (approx, valable sur Méditerranée)
        # on prend le même delta pour simplifier
        df_ais_m = df_ais_m[
            (df_ais_m['Position_LongitudeDeg'].between(lon_c - deg_lat, lon_c + deg_lat))
        ].copy()

        if df_ais_m.empty:
            return {
                "status": "ok",
                "zone": zone,
                "suspect_mmsi": mmsi_str,
                "message": "Navire suspect aux abords mais en dehors du coeur de la zone spécifiée."
            }

        # === D. Trajectoire AIS : préparation ===
        df_ais_m = df_ais_m.sort_values('datetime')

        ais_track = df_ais_m[['datetime', 'Position_LatitudeDeg', 'Position_LongitudeDeg']].rename(
            columns={
                'datetime': 'timestamp',
                'Position_LatitudeDeg': 'lat',
                'Position_LongitudeDeg': 'lon'
            }
        )
        ais_track['source'] = 'AIS'

        # === E. Ajout des points RF corrélés (optionnel) ===
        rf_track = pd.DataFrame(columns=['timestamp', 'lat', 'lon', 'source'])

        if has_corr:
            # Parse confiance
            df_corr['cand_1_confidence'] = pd.to_numeric(
                df_corr.get('cand_1_confidence', np.nan),
                errors='coerce'
            )

            df_corr_m = df_corr[
                (df_corr['cand_1_mmsi'].astype(str) == mmsi_str) &
                (df_corr['timestamp_utc'] >= t_min) &
                (df_corr['cand_1_confidence'] >= rf_corr_min_conf)
            ].copy()

            if not df_corr_m.empty:
                # Filtre géo large : même zone approx
                df_corr_m = df_corr_m[
                    df_corr_m['latitude_deg'].between(lat_c - deg_lat, lat_c + deg_lat)
                ]
                df_corr_m = df_corr_m[
                    df_corr_m['longitude_deg'].between(lon_c - deg_lat, lon_c + deg_lat)
                ]

                if not df_corr_m.empty:
                    rf_track = df_corr_m[['timestamp_utc', 'latitude_deg', 'longitude_deg']].rename(
                        columns={
                            'timestamp_utc': 'timestamp',
                            'latitude_deg': 'lat',
                            'longitude_deg': 'lon'
                        }
                    )
                    rf_track['source'] = 'RF'

        # === F. Fusion AIS + RF ===
        df_track = pd.concat([ais_track, rf_track], ignore_index=True)

        df_track = df_track.dropna(subset=['timestamp', 'lat', 'lon'])
        if df_track.empty or len(df_track) < 2:
            return {
                "status": "ok",
                "zone": zone,
                "suspect_mmsi": mmsi_str,
                "message": (
                    "Trop peu de points (AIS+RF) pour estimer une trajectoire et une destination."
                )
            }

        df_track = df_track.sort_values('timestamp')

        coords = df_track[['lat', 'lon']].to_numpy()
        times = df_track['timestamp'].to_numpy()
        sources = df_track['source'].to_numpy()

        # === G. Calcul des distances, vitesses, direction ===
        distances_km = []
        speeds_knots = []

        for i in range(1, len(coords)):
            lat1, lon1 = coords[i-1]
            lat2, lon2 = coords[i]
            d_km = haversine_km(lat1, lon1, lat2, lon2)
            dt_h = (times[i] - times[i-1]).total_seconds() / 3600.0

            if dt_h > 0:
                v_knots = d_km / dt_h * 0.539957
                speeds_knots.append(v_knots)
            distances_km.append(d_km)

        total_distance_km = float(np.nansum(distances_km)) if distances_km else 0.0
        mean_speed_knots = float(np.nanmean(speeds_knots)) if speeds_knots else 0.0
        max_speed_knots = float(np.nanmax(speeds_knots)) if speeds_knots else 0.0

        # Bearer sur le dernier segment
        def _bearing_deg(lat1, lon1, lat2, lon2):
            """Cap initial en degrés entre 0 et 360."""
            phi1, phi2 = radians(lat1), radians(lat2)
            dlon = radians(lon2 - lon1)
            x = sin(dlon) * cos(phi2)
            y = cos(phi1) * sin(phi2) - sin(phi1) * cos(phi2) * cos(dlon)
            brng = atan2(x, y)
            brng_deg = (degrees(brng) + 360.0) % 360.0
            return brng_deg

        last_lat, last_lon = coords[-1]
        last_time = times[-1]
        last_source = sources[-1]

        if len(coords) >= 2:
            prev_lat, prev_lon = coords[-2]
            bearing = _bearing_deg(prev_lat, prev_lon, last_lat, last_lon)
        else:
            bearing = np.nan

        # === H. Projection du point futur ===
        def _project_point(lat, lon, bearing_deg, distance_km):
            """Projette un point sur la sphère en suivant le cap et la distance (en km)."""
            R = 6371.0  # rayon approx Terre
            br = radians(bearing_deg)
            lat1 = radians(lat)
            lon1 = radians(lon)

            d_div_R = distance_km / R

            lat2 = asin(
                sin(lat1) * cos(d_div_R) +
                cos(lat1) * sin(d_div_R) * cos(br)
            )
            lon2 = lon1 + atan2(
                sin(br) * sin(d_div_R) * cos(lat1),
                cos(d_div_R) - sin(lat1) * sin(lat2)
            )

            return (
                degrees(lat2),
                (degrees(lon2) + 540.0) % 360.0 - 180.0  # normalisation [-180, 180]
            )

        if np.isnan(bearing) or mean_speed_knots <= 0:
            projected_lat, projected_lon = last_lat, last_lon
            projection_comment = "Impossible de projeter (direction ou vitesse moyenne non déterminée)."
        else:
            mean_speed_kmh = mean_speed_knots * 1.852
            distance_projection_km = mean_speed_kmh * (projection_hours / 1.0)
            projected_lat, projected_lon = _project_point(
                last_lat, last_lon, bearing, distance_projection_km
            )
            projection_comment = (
                f"Projection sur {projection_hours:.0f}h en suivant la direction moyenne estimée."
            )

        # === I. Relation à la zone (rentre / sort / reste) ===
        dist_now = haversine_km(lat_c, lon_c, last_lat, last_lon)
        dist_future = haversine_km(lat_c, lon_c, projected_lat, projected_lon)

        if dist_future > rad_km and dist_now <= rad_km:
            zone_trend = "sortie probable de la zone"
        elif dist_future < dist_now and dist_future <= rad_km:
            zone_trend = "poursuite d'activité à l'intérieur de la zone"
        else:
            zone_trend = "trajectoire incertaine vis-à-vis des limites de la zone"

        # === J. Narratif "Commandant" ===
        vessel_name = df_ais_m['Name'].dropna()
        vessel_name = vessel_name.iloc[-1] if not vessel_name.empty else f"MMSI {mmsi_str}"
        ship_type = df_ais_m['ShipTypeLabel'].dropna()
        ship_type = ship_type.iloc[-1] if not ship_type.empty else "Type inconnu"
        country = df_ais_m['Pays'].dropna()
        if country.empty:
            country = df_ais_m['Country'].dropna()
        country = country.iloc[-1] if not country.empty else "Pavillon inconnu"

        start_time = df_track['timestamp'].min()

        narrative_lines = []

        narrative_lines.append(
            f"👨‍✈️ Commandant,\n\n"
            f"Analyse du navire suspect {vessel_name} (MMSI {mmsi_str}, {ship_type}, {country}) "
            f"dans la zone '{zone}'."
        )

        narrative_lines.append(
            f"\n🕒 Fenêtre étudiée : du {start_time} au {last_time} (UTC)\n"
            f"📡 Points utilisés : {len(df_track)} (AIS + RF corrélés)\n"
            f"📏 Distance totale parcourue estimée : ~{total_distance_km:.1f} km\n"
            f"⚡ Vitesse moyenne : ~{mean_speed_knots:.1f} nœuds "
            f"(max ~{max_speed_knots:.1f} nœuds)\n"
        )

        narrative_lines.append(
            f"📍 Dernier point connu :\n"
            f"   - Source : {last_source}\n"
            f"   - Position : {last_lat:.3f}°N, {last_lon:.3f}°E\n"
            f"   - Heure : {last_time} (UTC)\n"
        )

        if not np.isnan(bearing):
            narrative_lines.append(
                f"🧭 Direction moyenne (dernier segment) : ~{bearing:.0f}° (0=N, 90=E)\n"
            )
        else:
            narrative_lines.append(
                "🧭 Direction moyenne : non déterminée (données insuffisantes).\n"
            )

        narrative_lines.append(
            f"\n🎯 Projection de trajectoire :\n"
            f"   - {projection_comment}\n"
            f"   - Point projeté (~{projection_hours:.0f}h) : "
            f"{projected_lat:.3f}°N, {projected_lon:.3f}°E\n"
            f"   - Interprétation vis-à-vis de la zone : {zone_trend}.\n"
        )

        if has_corr and not rf_track.empty:
            narrative_lines.append(
                "📡 Corrélation RF–AIS : utilisée pour prolonger la trajectoire lorsque l'AIS "
                "est partiel ou interrompu, ce qui renforce l'estimation de destination.\n"
            )
        else:
            narrative_lines.append(
                "ℹ️ Aucune corrélation RF significative (ou données RF manquantes) n'a été utilisée "
                "dans cette estimation.\n"
            )

        narrative = "\n".join(narrative_lines)

        # === K. Résultat structuré ===
        result = {
            "status": "ok",
            "zone": zone,
            "suspect_mmsi": mmsi_str,
            "time_window": {"start": start_time, "end": last_time},
            "track_stats": {
                "num_points": int(len(df_track)),
                "total_distance_km": total_distance_km,
                "mean_speed_knots": mean_speed_knots,
                "max_speed_knots": max_speed_knots,
            },
            "last_point": {
                "lat": float(last_lat),
                "lon": float(last_lon),
                "timestamp": last_time,
                "source": last_source,
                "distance_to_zone_center_km": dist_now,
            },
            "projection": {
                "projection_hours": projection_hours,
                "projected_lat": float(projected_lat),
                "projected_lon": float(projected_lon),
                "distance_from_zone_center_km": dist_future,
                "zone_trend": zone_trend,
            },
            "used_rf_corr": bool(has_corr and not rf_track.empty),
            "narrative": narrative,
        }

        return result

    except Exception as e:
        return {
            "status": "error",
            "message": f"Erreur technique dans estimate_suspect_destination_ais_rf : {str(e)}"
        }

#### Interception et infrastructures critiques

Planification d'une interception sur zone, et exposition des infrastructures critiques (câbles, terminaux, champs offshore).

`planifier_interception_sur_zone`, `analyser_infrastructure_critique`

In [ ]:
@tool
def planifier_interception_sur_zone(
    navire_ami_mmsi: str,
    zone: str,
    vitesse_ami_max_knots: float = 20.0,
    rayon_defaut_km: float = 5.0,
) -> dict:
    """
    Peux-tu m’aider à planifier une interception sur zone en fournissant des données
    sur le navire XXX de la marine Française dans [zone spécifique] ?

    - Récupère la dernière position AIS du navire ami (MMSI navire_ami_mmsi)
    - Résout la zone (ZONES_DATA ou "lat, lon[, rayon]")
    - Construit une route maritime estimée (en Méditerranée) qui évite la terre
    - Calcule la distance de route + temps de transit
    - Génère une carte Folium avec :
        * navire ami,
        * centre de zone + cercle de rayon,
        * route maritime estimée (polyline qui ne passe pas sur les terres).
    """

    # --- Vérifs de base sur les données AIS ---
    if "df_ais_global" not in globals() or df_ais_global is None or df_ais_global.empty:
        return {
            "status": "error",
            "message": "Les données AIS (df_ais_global) ne sont pas disponibles."
        }

    df = df_ais_global.copy()
    if not pd.api.types.is_datetime64_any_dtype(df["datetime"]):
        df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
    df = df.dropna(subset=["datetime"])

    # --- 1) Dernière position du navire ami ---
    mmsi_str = str(navire_ami_mmsi).strip()
    df_ami = df[df["Mmsi"].astype(str) == mmsi_str].copy()

    if df_ami.empty:
        return {
            "status": "ok",
            "message": f"Aucun navire AIS trouvé avec le MMSI {mmsi_str}."
        }

    df_ami = df_ami.sort_values("datetime")
    row_ami = df_ami.iloc[-1]

    fr_name = row_ami.get("Name", f"MMSI {mmsi_str}")
    fr_type = row_ami.get("ShipTypeLabel", "Type inconnu")
    fr_country = row_ami.get("Country", "Pays inconnu")
    fr_lat = float(row_ami.get("Position_LatitudeDeg"))
    fr_lon = float(row_ami.get("Position_LongitudeDeg"))
    fr_sog = float(row_ami.get("SogKt") if not pd.isna(row_ami.get("SogKt")) else 0.0)
    fr_cog = float(row_ami.get("CogDeg") if not pd.isna(row_ami.get("CogDeg")) else 0.0)
    fr_time = row_ami.get("datetime")

    # --- 2) Résolution de la zone ---
    z_config = None

    if "ZONES_DATA" in globals():
        if "_norm" in globals():
            z_clean = _norm(zone)
        else:
            z_clean = zone.strip().lower()
        if z_clean in ZONES_DATA:
            z_config = ZONES_DATA[z_clean]
        else:
            for k in ZONES_DATA:
                if z_clean in k or k in z_clean:
                    z_config = ZONES_DATA[k]
                    break

    if z_config is None:
        nums = re.findall(r"[-+]?\d*\.?\d+", zone.replace(",", " "))
        if len(nums) >= 2:
            lat_c = float(nums[0])
            lon_c = float(nums[1])
            rayon_km = float(nums[2]) if len(nums) >= 3 else rayon_defaut_km
            z_config = {"lat": lat_c, "lon": lon_c, "rayon": rayon_km}

    if z_config is None:
        return {
            "status": "error",
            "message": f"Zone '{zone}' inconnue ou mal formatée (attendu : nom connu ou 'lat, lon[, rayon]')."
        }

    lat_c = z_config["lat"]
    lon_c = z_config["lon"]
    rayon_km = z_config.get("rayon", rayon_defaut_km)

    # --- 3) Route maritime estimée ---
    route_coords = build_mediterranean_route(fr_lat, fr_lon, lat_c, lon_c)
    route_dist_km = route_length_km(route_coords)

    # Grande-cercle pour info (distance “en avion”, non utilisée pour le temps)
    gc_dist_km = haversine_km(fr_lat, fr_lon, lat_c, lon_c)

    # Temps estimé en suivant la route maritime
    v_fr_kmh = max(vitesse_ami_max_knots, 1e-3) * 1.852
    time_route_h = route_dist_km / v_fr_kmh if v_fr_kmh > 0 else None

    # --- 4) Carte Folium ---
    # centre = milieu approximatif de la route
    center_lat = np.mean([p[0] for p in route_coords])
    center_lon = np.mean([p[1] for p in route_coords])

    m = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles="CartoDB positron")

    # Cercle zone
    folium.Circle(
        location=[lat_c, lon_c],
        radius=rayon_km * 1000.0,
        color="red",
        fill=True,
        fill_opacity=0.1,
        popup=f"Zone : {zone} (rayon {rayon_km} km)"
    ).add_to(m)

    # Centre zone
    folium.Marker(
        location=[lat_c, lon_c],
        popup=f"Centre zone : {zone}",
        icon=folium.Icon(color="red", icon="crosshairs", prefix="fa"),
    ).add_to(m)

    # Navire ami
    popup_ami = (
        f"{fr_name}<br>"
        f"MMSI {mmsi_str}<br>"
        f"{fr_type}, pavillon {fr_country}<br>"
        f"Pos: {fr_lat:.3f}°N, {fr_lon:.3f}°E<br>"
        f"SOG: {fr_sog:.1f} kn, COG: {fr_cog:.0f}°<br>"
        f"{fr_time} (UTC)"
    )
    folium.Marker(
        location=[fr_lat, fr_lon],
        popup=popup_ami,
        icon=folium.Icon(color="blue", icon="ship", prefix="fa"),
    ).add_to(m)

    # Route maritime estimée
    folium.PolyLine(
        locations=[(lat, lon) for (lat, lon) in route_coords],
        color="blue",
        weight=3,
        opacity=0.8,
        tooltip="Route maritime estimée (évite la terre)"
    ).add_to(m)

    # --- 5) Sauvegardes ---
    zone_slug = re.sub(r"[^a-zA-Z0-9]+", "_", zone).strip("_")
    html_path = f"interception_{mmsi_str}_{zone_slug}.html"
    png_path = None

    m.save(html_path)

    try:
        import io
        from PIL import Image
        png_data = m._to_png(5)
        img = Image.open(io.BytesIO(png_data))
        png_path = f"interception_{mmsi_str}_{zone_slug}.png"
        img.save(png_path)
    except Exception:
        png_path = None

    # --- 6) Narratif ---
    narrative = []
    narrative.append(
        f"👨‍✈️ Commandant,\n\n"
        f"Planification d'interception sur zone pour le navire **{fr_name}** "
        f"(MMSI {mmsi_str}, {fr_type}, pavillon {fr_country})."
    )
    narrative.append(
        f"\n📍 Dernière position AIS connue : ({fr_lat:.3f}°N, {fr_lon:.3f}°E) "
        f"à {fr_time} (UTC).\n"
        f"🎯 Zone demandée : *{zone}* (centre {lat_c:.3f}°N, {lon_c:.3f}°E, rayon {rayon_km:.1f} km).\n"
        f"📏 Distance 'en avion' (grande cercle) : ~{gc_dist_km:.1f} km.\n"
        f"🧭 Distance estimée en suivant une route maritime : ~{route_dist_km:.1f} km.\n"
    )
    narrative.append(
        f"🚀 Vitesse max retenue pour la planification : {vitesse_ami_max_knots:.1f} nœuds "
        f"(~{v_fr_kmh:.1f} km/h).\n"
    )

    if time_route_h is not None:
        narrative.append(
            f"⏱️ Temps estimé pour rejoindre la zone en suivant la route maritime : "
            f"~{time_route_h:.1f} heures (~{time_route_h/24:.1f} jours).\n"
        )
    else:
        narrative.append(
            "⏱️ Impossible d'estimer un temps de transit (vitesse max non valide).\n"
        )

    narrative.append(
        "\n🗺️ Une carte a été générée avec :\n"
        "   - la position actuelle du navire ami,\n"
        "   - le centre de la zone et son rayon,\n"
        "   - une route maritime estimée qui évite les côtes.\n"
        f"   ➜ Fichier HTML : `{html_path}`"
    )
    if png_path:
        narrative.append(f"\n📸 Une image statique de la route est également disponible : `{png_path}`.")

    return {
        "status": "ok",
        "mode": "positionnement_sans_cible",
        "zone": {
            "label": zone,
            "lat_center": lat_c,
            "lon_center": lon_c,
            "rayon_km": rayon_km,
        },
        "friendly_ship": {
            "mmsi": mmsi_str,
            "name": fr_name,
            "type": fr_type,
            "country": fr_country,
            "lat": fr_lat,
            "lon": fr_lon,
            "time": fr_time,
            "sog_knots": fr_sog,
            "cog_deg": fr_cog,
        },
        "interception": {
            "great_circle_distance_km": gc_dist_km,
            "route_length_km": route_dist_km,
            "time_to_center_hours": time_route_h,
            "vitesse_ami_max_knots": vitesse_ami_max_knots,
        },
        "route_coords": route_coords,
        "map_html_path": html_path,
        "map_png_path": png_path,
        "narrative": "\n".join(narrative),
    }

@tool
def analyser_infrastructure_critique(
    infrastructure: str,
    hours: int = 24,
    rayon_km: float = 30.0,
    proximite_critique_km: float = 5.0
) -> str:
    """
    Analyse rapide des navires opérant près d'une infrastructure critique
    (port, terminal, oléoduc, etc.) en combinant AIS + RF + éphémérides.

    - Localise l'infrastructure (via INFRA_DATA / ZONES_DATA ou "lat,lon[,rayon]")
    - Prend les dernières `hours` heures de données
    - Analyse :
        * Trafic AIS autour de l'infrastructure
        * Détections RF (dont RF corrélées à l'AIS si df_corr_global dispo)
        * Passages satellites (df_ephemeris_global)

    Args:
        infrastructure: nom de l'infrastructure ("port de Haifa",
                       "pipeline X", clé dans INFRA_DATA, ou "lat, lon[, rayon]").
        hours: fenêtre temporelle en heures (par défaut 24).
        rayon_km: rayon d'analyse autour de l'infrastructure (par défaut 30 km).
        proximite_critique_km: rayon "immédiat" autour de l'infrastructure
                               (par défaut 5 km) pour les alertes rapprochées.

    Retour:
        Rapport texte en français pour le commandant.
    """
    try:
        # Vérif des 3 sources
        if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
            return "❌ Les données AIS (df_ais_global) ne sont pas disponibles."
        if 'df_rf_global' not in globals() or df_rf_global is None or df_rf_global.empty:
            return "❌ Les données RF (df_rf_global) ne sont pas disponibles."
        if 'df_ephemeris_global' not in globals() or df_ephemeris_global is None or df_ephemeris_global.empty:
            eph_available = False
        else:
            eph_available = True

        # On peut (optionnellement) utiliser df_corr_global si dispo
        corr_available = (
            'df_corr_global' in globals()
            and df_corr_global is not None
            and not df_corr_global.empty
        )

        # --- 1) Résolution de l'infrastructure -> (lat_c, lon_c, rayon)
        infra_name = infrastructure.strip()
        infra_clean = _norm(infrastructure) if '_norm' in globals() else infra_name.lower()
        infra_cfg = None

        # a) INFRA_DATA (à définir côté utilisateur si besoin)
        if 'INFRA_DATA' in globals():
            if infra_clean in INFRA_DATA:
                infra_cfg = INFRA_DATA[infra_clean]
            else:
                for k, v in INFRA_DATA.items():
                    if infra_clean in k or k in infra_clean:
                        infra_cfg = v
                        break

        # b) ZONES_DATA en fallback
        if infra_cfg is None and 'ZONES_DATA' in globals():
            if infra_clean in ZONES_DATA:
                infra_cfg = ZONES_DATA[infra_clean]
            else:
                for k, v in ZONES_DATA.items():
                    if infra_clean in k or k in infra_clean:
                        infra_cfg = v
                        break

        # c) Fallback "lat, lon[, rayon]"
        if infra_cfg is None:
            nums = re.findall(r"[-+]?\d*\.?\d+", infrastructure.replace(",", " "))
            if len(nums) >= 2:
                lat_c = float(nums[0])
                lon_c = float(nums[1])
                if len(nums) >= 3:
                    rayon_km = float(nums[2])
            else:
                return (
                    f"❌ Impossible d'interpréter l'infrastructure '{infrastructure}'.\n"
                    "Utilisez soit un nom présent dans INFRA_DATA/ZONES_DATA, "
                    "soit le format 'lat, lon[, rayon_km]'."
                )
        else:
            lat_c = infra_cfg.get("lat")
            lon_c = infra_cfg.get("lon")
            if infra_cfg.get("rayon"):
                rayon_km = infra_cfg["rayon"]

        if lat_c is None or lon_c is None:
            return f"❌ Infrastructure '{infrastructure}' mal définie (lat/lon manquants)."

        # --- 2) Fenêtre temporelle globale ---
        df_ais = df_ais_global.copy()
        if not pd.api.types.is_datetime64_any_dtype(df_ais["datetime"]):
            df_ais["datetime"] = pd.to_datetime(df_ais["datetime"], utc=True, errors="coerce")
        df_ais = df_ais.dropna(subset=["datetime"])

        t_ais_max = df_ais["datetime"].max()

        df_rf = df_rf_global.copy()
        if not pd.api.types.is_datetime64_any_dtype(df_rf["timestamp_utc"]):
            df_rf["timestamp_utc"] = pd.to_datetime(df_rf["timestamp_utc"], utc=True, errors="coerce")
        df_rf = df_rf.dropna(subset=["timestamp_utc"])
        t_rf_max = df_rf["timestamp_utc"].max()

        t_max = max(t_ais_max, t_rf_max)
        t_min = t_max - pd.Timedelta(hours=hours)

        # --- 3) AIS : navires autour de l'infrastructure ---
        df_ais_w = df_ais[(df_ais["datetime"] >= t_min) & (df_ais["datetime"] <= t_max)].copy()
        df_ais_w = df_ais_w.dropna(subset=["Position_LatitudeDeg", "Position_LongitudeDeg"])

        deg_lat = rayon_km / 111.0
        deg_lon = rayon_km / 85.0  # approx

        df_ais_zone = df_ais_w[
            df_ais_w["Position_LatitudeDeg"].between(lat_c - deg_lat, lat_c + deg_lat)
            & df_ais_w["Position_LongitudeDeg"].between(lon_c - deg_lon, lon_c + deg_lon)
        ].copy()

        def _hv(lat1, lon1, lat2, lon2):
            return haversine_km(lat1, lon1, lat2, lon2)

        if not df_ais_zone.empty:
            df_ais_zone["distance_km"] = df_ais_zone.apply(
                lambda r: _hv(lat_c, lon_c, r["Position_LatitudeDeg"], r["Position_LongitudeDeg"]),
                axis=1,
            )
            df_ais_zone = df_ais_zone[df_ais_zone["distance_km"] <= rayon_km].copy()

        # Dernière position par navire dans la zone
        if not df_ais_zone.empty:
            idx_last = df_ais_zone.groupby("Mmsi")["datetime"].idxmax()
            df_ais_last = df_ais_zone.loc[idx_last].copy()
            # classification par type métier (militaire, commercial, pêche, autre)
            df_ais_last["type_navire"] = df_ais_last.apply(_classer_type_navire_ais, axis=1)
        else:
            df_ais_last = pd.DataFrame()

        nb_navires_zone = int(len(df_ais_last))
        nb_navires_proches = int(
            df_ais_last[df_ais_last["distance_km"] <= proximite_critique_km].shape[0]
        )

        type_counts = (
            df_ais_last["type_navire"].value_counts().to_dict()
            if not df_ais_last.empty else {}
        )
        type_counts_proches = (
            df_ais_last[df_ais_last["distance_km"] <= proximite_critique_km]["type_navire"]
            .value_counts()
            .to_dict()
            if not df_ais_last.empty else {}
        )

        # Top navires les plus proches
        top_navires_proches = []
        if not df_ais_last.empty:
            df_tmp = df_ais_last[df_ais_last["distance_km"] <= rayon_km].copy()
            df_tmp = df_tmp.sort_values("distance_km").head(5)
            for _, r in df_tmp.iterrows():
                top_navires_proches.append({
                    "name": r.get("Name", "Nom inconnu"),
                    "mmsi": r.get("Mmsi"),
                    "type": r.get("ShipTypeLabel", "Type inconnu"),
                    "type_metier": r.get("type_navire", "autre"),
                    "country": r.get("Country", "Pays inconnu"),
                    "distance_km": float(r.get("distance_km", np.nan)),
                })

        # --- 4) RF brutes autour de l'infrastructure ---
        df_rf_w = df_rf[
            (df_rf["timestamp_utc"] >= t_min) & (df_rf["timestamp_utc"] <= t_max)
        ].copy()
        df_rf_w = df_rf_w.dropna(subset=["latitude_deg", "longitude_deg"])

        df_rf_zone = df_rf_w[
            df_rf_w["latitude_deg"].between(lat_c - deg_lat, lat_c + deg_lat)
            & df_rf_w["longitude_deg"].between(lon_c - deg_lon, lon_c + deg_lon)
        ].copy()

        if not df_rf_zone.empty:
            df_rf_zone["distance_km"] = df_rf_zone.apply(
                lambda r: _hv(lat_c, lon_c, r["latitude_deg"], r["longitude_deg"]),
                axis=1,
            )
            df_rf_zone = df_rf_zone[df_rf_zone["distance_km"] <= rayon_km].copy()

        nb_rf_zone = int(len(df_rf_zone))
        nb_rf_proches = int(
            df_rf_zone[df_rf_zone["distance_km"] <= proximite_critique_km].shape[0]
        )

        # --- 5) RF corrélées (df_corr_global) pour détecter du "dark" ---
        dark_contacts_proches = 0
        nb_corr_total = 0
        nb_corr_fortes = 0

        if corr_available:
            df_corr = df_corr_global.copy()
            if not pd.api.types.is_datetime64_any_dtype(df_corr["timestamp_utc"]):
                df_corr["timestamp_utc"] = pd.to_datetime(df_corr["timestamp_utc"], utc=True, errors="coerce")
            df_corr = df_corr.dropna(subset=["timestamp_utc", "latitude_deg", "longitude_deg"])

            df_corr_w = df_corr[
                (df_corr["timestamp_utc"] >= t_min) & (df_corr["timestamp_utc"] <= t_max)
            ].copy()

            df_corr_w["cand_1_confidence"] = pd.to_numeric(
                df_corr_w.get("cand_1_confidence", np.nan),
                errors="coerce"
            ).fillna(0.0)

            df_corr_zone = df_corr_w[
                df_corr_w["latitude_deg"].between(lat_c - deg_lat, lat_c + deg_lat)
                & df_corr_w["longitude_deg"].between(lon_c - deg_lon, lon_c + deg_lon)
            ].copy()

            if not df_corr_zone.empty:
                df_corr_zone["distance_km"] = df_corr_zone.apply(
                    lambda r: _hv(lat_c, lon_c, r["latitude_deg"], r["longitude_deg"]),
                    axis=1,
                )
                df_corr_zone = df_corr_zone[df_corr_zone["distance_km"] <= rayon_km].copy()

            nb_corr_total = int(len(df_corr_zone))

            # seuil simple pour AIS fiable
            seuil_conf = 0.5
            df_corr_forte = df_corr_zone[df_corr_zone["cand_1_confidence"] >= seuil_conf]
            df_corr_faible = df_corr_zone[df_corr_zone["cand_1_confidence"] < seuil_conf]

            nb_corr_fortes = int(len(df_corr_forte))
            dark_contacts_proches = int(
                df_corr_faible[df_corr_faible["distance_km"] <= proximite_critique_km].shape[0]
            )
        # --- 6) Éphémérides : passages satellites récents ---
        sat_txt = ""
        if eph_available:
            df_eph = df_ephemeris_global.copy()
            if not pd.api.types.is_datetime64_any_dtype(df_eph["StartTime"]):
                df_eph["StartTime"] = pd.to_datetime(df_eph["StartTime"], utc=True, errors="coerce")
            if not pd.api.types.is_datetime64_any_dtype(df_eph["EndTime"]):
                df_eph["EndTime"] = pd.to_datetime(df_eph["EndTime"], utc=True, errors="coerce")

            df_eph = df_eph.dropna(subset=["StartTime", "EndTime"])
            df_eph_w = df_eph[
                (df_eph["EndTime"] >= t_min) & (df_eph["StartTime"] <= t_max)
            ].copy()

            if df_eph_w.empty:
                sat_txt = (
                    "Aucun passage JAPETUS récent sur la période considérée "
                    f"({hours} dernières heures)."
                )
            else:
                # distance approx au centre de l'infra = min(start, end)
                def _pass_dist(r):
                    d1 = _hv(lat_c, lon_c, r["StartLatitude"], r["StartLongitude"])
                    d2 = _hv(lat_c, lon_c, r["EndLatitude"], r["EndLongitude"])
                    return min(d1, d2)

                df_eph_w["dist_infra_km"] = df_eph_w.apply(_pass_dist, axis=1)
                df_eph_zone = df_eph_w[df_eph_w["dist_infra_km"] <= 500].copy()

                if df_eph_zone.empty:
                    sat_txt = (
                        f"{len(df_eph_w)} passage(s) JAPETUS récents, "
                        "mais assez éloignés (>500 km) de l'infrastructure."
                    )
                else:
                    nb_passes = int(len(df_eph_zone))
                    best = df_eph_zone.sort_values("dist_infra_km").iloc[0]
                    sat_txt = (
                        f"{nb_passes} passage(s) JAPETUS ont survolé ou frôlé la zone "
                        f"(≤500 km) sur les {hours} dernières heures.\n"
                        f"- Passage le plus proche : {best.get('Source','JAPETUS')} sur {best.get('Target','Zone')} "
                        f"du {best['StartTime']} au {best['EndTime']} (distance ~{best['dist_infra_km']:.1f} km)."
                    )
        else:
            sat_txt = "Les éphémérides JAPETUS ne sont pas disponibles dans cet environnement."

        # --- 7) Heuristique simple de niveau de risque ---
        # Très simpliste, juste pour le pitch.
        risk_level = "faible"
        risk_reasons = []

        if dark_contacts_proches > 0:
            risk_level = "élevé"
            risk_reasons.append(
                f"{dark_contacts_proches} contact(s) RF corrélés faible-confiance "
                f"(potentiellement 'dark') dans le rayon {proximite_critique_km} km."
            )
        elif nb_rf_proches > 0:
            risk_level = "modéré"
            risk_reasons.append(
                f"{nb_rf_proches} détection(s) RF à proximité immédiate de l'infrastructure."
            )

        if nb_navires_proches > 0:
            if risk_level == "faible":
                risk_level = "modéré"
            risk_reasons.append(
                f"{nb_navires_proches} navire(s) AIS à moins de {proximite_critique_km} km."
            )

        # --- 8) Construction du narratif pour le commandant ---
        periode_txt = f"du {t_min} au {t_max} (UTC)"
        centre_txt = f"centre ~ ({lat_c:.3f}°N, {lon_c:.3f}°E), rayon {rayon_km:.1f} km"

        txt = []
        txt.append(
            f"👨‍✈️ Commandant,\n\n"
            f"Analyse multi-capteurs autour de l'infrastructure critique **{infra_name}**\n"
            f"({centre_txt}) sur les **{hours} dernières heures** ({periode_txt}).\n"
        )

        # AIS
        txt.append("🚢 **Vue AIS autour de l'infrastructure**")
        if nb_navires_zone == 0:
            txt.append(
                f"- Aucun navire AIS n'a été détecté dans un rayon de {rayon_km:.1f} km."
            )
        else:
            txt.append(
                f"- {nb_navires_zone} navire(s) AIS avec position récente dans le rayon {rayon_km:.1f} km."
            )
            if type_counts:
                txt.append("  Répartition par type métier :")
                for t, c in type_counts.items():
                    txt.append(f"    • {t} : {c} navire(s)")
            if nb_navires_proches > 0:
                txt.append(
                    f"- {nb_navires_proches} navire(s) à moins de {proximite_critique_km:.1f} km de l'infrastructure "
                    "(zone de proximité immédiate)."
                )
                if type_counts_proches:
                    txt.append("  Types des navires très proches :")
                    for t, c in type_counts_proches.items():
                        txt.append(f"    • {t} : {c}")
            if top_navires_proches:
                txt.append("\n  Navires les plus proches :")
                for n in top_navires_proches:
                    txt.append(
                        f"    - {n['name']} (MMSI {n['mmsi']}, {n['type']}, {n['country']}) "
                        f"à ~{n['distance_km']:.1f} km ({n['type_metier']})"
                    )

        # RF brutes + corrélées
        txt.append("\n📡 **Vue RF UnseenLabs (émissions radio)**")
        txt.append(
            f"- {nb_rf_zone} détection(s) RF dans le rayon {rayon_km:.1f} km, "
            f"dont {nb_rf_proches} à moins de {proximite_critique_km:.1f} km."
        )
        if corr_available:
            txt.append(
                f"- {nb_corr_total} événement(s) RF disposant d'une corrélation RF/AIS dans la zone, "
                f"dont {nb_corr_fortes} avec AIS de confiance élevée (cand_1_confidence ≥ 0.5)."
            )
            if dark_contacts_proches > 0:
                txt.append(
                    f"- ⚠ Présence de détections RF corrélées de faible confiance "
                    f"à proximité immédiate : {dark_contacts_proches} cas susceptibles de "
                    "correspondre à des comportements 'dark' (AIS non fiable ou coupé)."
                )
        else:
            txt.append(
                "- Les corrélations RF/AIS détaillées (rf_correlation.parquet) ne sont pas disponibles ; "
                "impossible de qualifier précisément les contacts 'dark'."
            )

        # Satellites
        txt.append("\n🛰️ **Couverture satellites (JAPETUS)**")
        txt.append(sat_txt)

        # Risque
        txt.append("\n🧮 **Appréciation rapide du risque autour de l'infrastructure**")
        txt.append(f"- Niveau de risque estimé : **{risk_level.upper()}**")
        if risk_reasons:
            txt.append("  Principaux éléments pris en compte :")
            for r in risk_reasons:
                txt.append(f"    • {r}")
        else:
            txt.append(
                "  Aucun élément particulièrement préoccupant n'a été détecté sur cette période "
                "(trafic faible ou conforme, pas de contact RF anormal)."
            )

        txt.append(
            "\n💡 Cette analyse peut servir de base à une surveillance renforcée : "
            "tasking imagerie sur les prochains passages JAPETUS, suivi temps réel des navires "
            "les plus proches et des contacts RF non corrélés."
        )

        return "\n".join(txt)

    except Exception as e:
        return f"❌ Erreur dans analyser_infrastructure_critique : {e}"

#### Navigation normale et empreintes RF

Base de comportements normaux servant de référence au scoring, et base d'empreintes RF par navire.

`tool_analyse_navigation_normale`, `tool_build_rf_fingerprint_db`

In [ ]:
@tool
def tool_analyse_navigation_normale(
    zone: str,
    hours: int = 24,
    max_traces_vessels: int = 50,
    max_points_heatmap: int = 50000
) -> Dict[str, Any]:
    """
    Analyse les schémas de navigation 'normaux' dans une zone donnée en se basant
    sur les données AIS des dernières `hours` heures (par défaut 24).

    Sorties :
    - Statistiques sur :
      * nombre de navires
      * distribution des types de navires
      * distribution des statuts (Under way, Moored, At anchor, etc.)
      * statistiques de vitesse (SOG)
    - Carte Folium des routes + heatmap de densité,
      sauvegardée en HTML et, si possible, en PNG.
    """
    # Vérifications de base
    if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
        return {
            "status": "error",
            "message": "Les données AIS (df_ais_global) ne sont pas chargées."
        }

    df = df_ais_global.copy()

    # Assurer le type datetime
    if not pd.api.types.is_datetime64_any_dtype(df['datetime']):
        df['datetime'] = pd.to_datetime(df['datetime'], utc=True)

    # === 1. Fenêtre temporelle (dernieres `hours` heures) ===
    t_max = df['datetime'].max()
    t_min = t_max - pd.Timedelta(hours=hours)
    df = df[(df['datetime'] >= t_min) & (df['datetime'] <= t_max)].copy()

    if df.empty:
        return {
            "status": "ok",
            "message": f"Aucune donnée AIS dans les {hours} dernières heures.",
            "zone": zone,
            "time_window": {"start": t_min, "end": t_max}
        }

    # === 2. Sélection de la zone ===
    # - soit via ZONES_DATA
    # - soit via "lat, lon[, rayon_km]"
    z_clean = _norm(zone) if 'ZONES_DATA' in globals() and '_norm' in globals() else zone.strip().lower()
    z_config = None

    if 'ZONES_DATA' in globals():
        if z_clean in ZONES_DATA:
            z_config = ZONES_DATA[z_clean]
        else:
            # fuzzy match simple
            for k in ZONES_DATA:
                if z_clean in k or k in z_clean:
                    z_config = ZONES_DATA[k]
                    break

    # Si pas trouvé, essayer format "lat, lon[, rayon]"
    if z_config is None:
        # exemple : "36.5, 15.2, 200" => centre (lat, lon), rayon 200 km
        nums = re.findall(r"[-+]?\d*\.?\d+", zone.replace(",", " "))
        if len(nums) >= 2:
            lat_c = float(nums[0])
            lon_c = float(nums[1])
            rayon_km = float(nums[2]) if len(nums) >= 3 else 200.0
            z_config = {"lat": lat_c, "lon": lon_c, "rayon": rayon_km}

    if z_config is None:
        return {
            "status": "error",
            "message": f"Zone '{zone}' inconnue ou mal formattée.",
        }

    lat_c = z_config['lat']
    lon_c = z_config['lon']
    rayon_km = z_config['rayon']

    # Filtre géographique (approx : bounding box puis filtre haversine)
    deg_lat = rayon_km / 111.0
    deg_lon = rayon_km / 85.0  # approx en Méditerranée

    df_zone = df[
        (df['Position_LatitudeDeg'].between(lat_c - deg_lat, lat_c + deg_lat)) &
        (df['Position_LongitudeDeg'].between(lon_c - deg_lon, lon_c + deg_lon))
    ].copy()

    if df_zone.empty:
        return {
            "status": "ok",
            "message": f"Aucune donnée AIS dans la zone '{zone}' sur les {hours} dernières heures.",
            "zone": zone,
            "time_window": {"start": t_min, "end": t_max}
        }

    # Filtre haversine plus précis
    df_zone['dist_centre_km'] = df_zone.apply(
        lambda r: haversine_km(
            lat_c, lon_c,
            r['Position_LatitudeDeg'], r['Position_LongitudeDeg']
        ),
        axis=1
    )
    df_zone = df_zone[df_zone['dist_centre_km'] <= rayon_km].copy()

    if df_zone.empty:
        return {
            "status": "ok",
            "message": f"Aucune donnée AIS dans le rayon {rayon_km} km autour de '{zone}' sur les {hours} dernières heures.",
            "zone": zone,
            "time_window": {"start": t_min, "end": t_max}
        }

    # === 3. Statistiques de navigation ===
    nb_messages = int(len(df_zone))
    nb_navires = int(df_zone['Mmsi'].nunique())

    # Typologie des navires
    ship_type_col = 'ShipTypeNormalized' if 'ShipTypeNormalized' in df_zone.columns else 'ShipTypeLabel'
    ship_type_counts = (
        df_zone[ship_type_col]
        .fillna('Unknown')
        .value_counts()
        .head(15)
    )

    # Statuts de navigation
    if 'NavigationalStatusLabel' in df_zone.columns:
        nav_status_counts = (
            df_zone['NavigationalStatusLabel']
            .fillna('UNKNOWN')
            .value_counts()
        )
    else:
        nav_status_counts = pd.Series(dtype=int)

    # Statistiques de vitesse
    if 'SogKt' in df_zone.columns:
        sog = df_zone['SogKt'].clip(lower=0)
        speed_stats = {
            "mean": float(sog.mean()),
            "median": float(sog.median()),
            "p90": float(sog.quantile(0.9)),
            "p99": float(sog.quantile(0.99)),
            "max": float(sog.max())
        }
    else:
        speed_stats = {}

    # === 4. Construction de la carte des routes ===
    # Centre de la carte = centre de la zone
    m = folium.Map(location=[lat_c, lon_c], zoom_start=7, tiles="CartoDB positron")

    # Heatmap de densité
    df_heat = df_zone[['Position_LatitudeDeg', 'Position_LongitudeDeg']].dropna()
    if len(df_heat) > max_points_heatmap:
        df_heat = df_heat.sample(max_points_heatmap, random_state=42)

    heat_data = df_heat.values.tolist()
    HeatMap(heat_data, radius=8, blur=10, max_zoom=8).add_to(m)

    # Traces de navires (on limite le nombre pour rester lisible)
    mmsi_list = df_zone['Mmsi'].dropna().astype(int).unique()
    rng = np.random.default_rng(42)
    if len(mmsi_list) > max_traces_vessels:
        mmsi_sample = rng.choice(mmsi_list, size=max_traces_vessels, replace=False)
    else:
        mmsi_sample = mmsi_list

    colors = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
        "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
        "#bcbd22", "#17becf"
    ]

    for i, mmsi in enumerate(mmsi_sample):
        sub = df_zone[df_zone['Mmsi'] == mmsi].sort_values('datetime')
        if len(sub) < 2:
            continue
        coords = sub[['Position_LatitudeDeg', 'Position_LongitudeDeg']].dropna().values.tolist()
        if len(coords) < 2:
            continue

        color = colors[i % len(colors)]
        folium.PolyLine(
            locations=coords,
            color=color,
            weight=2,
            opacity=0.7,
            tooltip=f"MMSI {mmsi}"
        ).add_to(m)

    # === 5. Sauvegarde HTML + tentative PNG ===
    # slug pour nom de fichiers
    zone_slug = re.sub(r"[^a-zA-Z0-9]+", "_", zone).strip("_")
    html_path = f"nav_patterns_{zone_slug}_last{hours}h.html"
    png_path = None

    # HTML
    m.save(html_path)

    # PNG (optionnel, dépend de l'environnement)
    try:
        import io
        from PIL import Image
        png_data = m._to_png(5)  # nécessite l'environnement compatible
        img = Image.open(io.BytesIO(png_data))
        png_path = f"nav_patterns_{zone_slug}_last{hours}h.png"
        img.save(png_path)
    except Exception:
        png_path = None  # on ne bloque pas si ça échoue

    # === 6. Narratif pour le commandant ===
    ship_type_top = ship_type_counts.head(5)
    nav_top = nav_status_counts.head(5) if not nav_status_counts.empty else None

    narrative = []
    narrative.append(
        f"👨‍✈️ Commandant,\n\n"
        f"Voici une synthèse des schémas de navigation 'normaux' observés "
        f"dans la zone *{zone}* sur les {hours} dernières heures."
    )
    narrative.append(
        f"\n🕒 Fenêtre temporelle : du {t_min} au {t_max} (UTC)\n"
        f"📨 Messages AIS analysés : {nb_messages}\n"
        f"🚢 Nombre de navires uniques : {nb_navires}\n"
    )

    if speed_stats:
        narrative.append(
            f"⚡ Vitesses (SOG) :\n"
            f"   - Moyenne : {speed_stats['mean']:.1f} nœuds\n"
            f"   - Médiane : {speed_stats['median']:.1f} nœuds\n"
            f"   - 90ᵉ percentile : {speed_stats['p90']:.1f} nœuds\n"
            f"   - Vitesse max observée : {speed_stats['max']:.1f} nœuds\n"
        )

    narrative.append("\n🛳️ Types de navires les plus représentés :")
    for idx, val in ship_type_top.items():
        narrative.append(f"   - {idx} : {val} messages")

    if nav_top is not None:
        narrative.append("\n⚓ Statuts de navigation dominants :")
        for idx, val in nav_top.items():
            narrative.append(f"   - {idx} : {val} messages")

    narrative.append(
        "\n🗺️ Une carte a été générée avec :\n"
        "   - une heatmap de densité des positions AIS,\n"
        "   - des trajectoires représentatives de plusieurs navires.\n"
        "Vous pouvez l’examiner pour visualiser les routes 'normales' "
        "et repérer d’éventuelles anomalies à l’œil."
    )

    if png_path:
        narrative.append(f"\n📸 La carte est également disponible en image : `{png_path}`.")
    else:
        narrative.append(
            "\n📎 La carte est disponible en HTML (interactif). "
            "La génération PNG n'a pas été possible dans cet environnement."
        )

    narrative_text = "\n".join(narrative)

    # === 7. Résultat structuré ===
    result = {
        "status": "ok",
        "zone": zone,
        "time_window": {"start": t_min, "end": t_max},
        "nb_messages": nb_messages,
        "nb_vessels": nb_navires,
        "ship_type_stats": ship_type_counts.to_dict(),
        "nav_status_stats": nav_status_counts.to_dict() if not nav_status_counts.empty else {},
        "speed_stats": speed_stats,
        "map_html_path": html_path,
        "map_png_path": png_path,
        "narrative": narrative_text,
    }

    return result

@tool
def tool_build_rf_fingerprint_db(
    min_confidence: float = 0.7,
    min_events_per_ship: int = 3
) -> Dict[str, Any]:
    """
    Construit une base d'empreintes RF quasi-uniques par navire (MMSI)
    en utilisant UNIQUEMENT la base corrélée rf_correlation.parquet (df_corr_global).

    Colonnes utilisées (df_corr_global) :
    - timestamp_utc (datetime64[ns, UTC])
    - RF_Frequency_MHz (float64)
    - latitude_deg, longitude_deg (float64)
    - cand_1_name (object)
    - cand_1_mmsi (object)
    - cand_1_confidence (object → converti en float)
    - number_of_candidates (int64)

    L'empreinte par navire inclut :
    - stats sur la fréquence RF (moyenne, écart-type, min, max)
    - nombre d'événements RF corrélés
    - moyenne de confiance
    - nombre moyen de candidats RF/AIS (unicité du pattern)

    Retourne :
    - message texte pour le commandant
    - fingerprints_df : DataFrame par MMSI avec les caractéristiques RF
    """

    # 1) Vérification des données globales
    if 'df_corr_global' not in globals() or df_corr_global is None or df_corr_global.empty:
        return {
            "status": "error",
            "message": "Les données de corrélation RF/AIS (df_corr_global) ne sont pas chargées."
        }

    df_corr = df_corr_global.copy()

    # 2) Normalisation des types
    # timestamp_utc → datetime (UTC)
    if not pd.api.types.is_datetime64_any_dtype(df_corr['timestamp_utc']):
        df_corr['timestamp_utc'] = pd.to_datetime(df_corr['timestamp_utc'], utc=True)

    # RF_Frequency_MHz → float (normalement déjà le cas)
    df_corr['RF_Frequency_MHz'] = pd.to_numeric(df_corr['RF_Frequency_MHz'], errors='coerce')

    # cand_1_confidence → float (peut être une string, ou "uncorrelated" etc.)
    if 'cand_1_confidence' in df_corr.columns:
        df_corr['cand_1_confidence'] = pd.to_numeric(
            df_corr['cand_1_confidence'], errors='coerce'
        ).fillna(0.0)
    else:
        df_corr['cand_1_confidence'] = 0.0

    # 3) Filtrer sur les corrélations de bonne confiance
    df_corr_strong = df_corr[df_corr['cand_1_confidence'] >= min_confidence].copy()
    if df_corr_strong.empty:
        return {
            "status": "ok",
            "message": (
                f"Aucune corrélation RF/AIS avec confiance ≥ {min_confidence}. "
                "Impossible de construire des empreintes fiables."
            )
        }

    # 4) Nettoyage MMSI : garder seulement les lignes où cand_1_mmsi est renseigné
    if 'cand_1_mmsi' not in df_corr_strong.columns:
        return {
            "status": "error",
            "message": "La colonne 'cand_1_mmsi' est absente dans df_corr_global."
        }

    df_corr_strong = df_corr_strong[df_corr_strong['cand_1_mmsi'].notna()].copy()
    df_corr_strong['cand_1_mmsi'] = df_corr_strong['cand_1_mmsi'].astype(str)

    # Optionnel : exclure explicitement 'uncorrelated' si jamais ça apparaît en MMSI
    df_corr_strong = df_corr_strong[
        df_corr_strong['cand_1_mmsi'].str.lower() != 'uncorrelated'
    ]

    if df_corr_strong.empty:
        return {
            "status": "ok",
            "message": (
                "Après nettoyage des MMSI, aucune corrélation exploitable n'est disponible "
                f"avec une confiance ≥ {min_confidence}."
            )
        }

    # 5) Construction des empreintes par MMSI
    fingerprints = []

    for mmsi, group in df_corr_strong.groupby('cand_1_mmsi'):
        # On impose un minimum de détections RF corrélées pour considérer que l'empreinte est fiable
        if len(group) < min_events_per_ship:
            continue

        freqs = group['RF_Frequency_MHz'].dropna()
        confs = group['cand_1_confidence'].dropna()
        nb_candidates = group['number_of_candidates'] if 'number_of_candidates' in group.columns else pd.Series(dtype=float)

        # Lat/lon pour caractériser un peu la zone d'émission (facultatif mais utile)
        lats = group['latitude_deg'].dropna()
        lons = group['longitude_deg'].dropna()

        fingerprint = {
            "mmsi": mmsi,
            "nb_events": int(len(group)),
            # Fréquence RF
            "freq_mean": float(freqs.mean()) if not freqs.empty else None,
            "freq_std": float(freqs.std()) if not freqs.empty else None,
            "freq_min": float(freqs.min()) if not freqs.empty else None,
            "freq_max": float(freqs.max()) if not freqs.empty else None,
            # Confiance moyenne de corrélation
            "mean_confidence": float(confs.mean()) if not confs.empty else None,
            # Nombre de candidats moyens (unicité de la corrélation)
            "mean_nb_candidates": float(nb_candidates.mean()) if not nb_candidates.empty else None,
            # Extension géographique de l'émetteur (approx)
            "lat_mean": float(lats.mean()) if not lats.empty else None,
            "lon_mean": float(lons.mean()) if not lons.empty else None,
            "lat_span": float(lats.max() - lats.min()) if len(lats) > 1 else 0.0,
            "lon_span": float(lons.max() - lons.min()) if len(lons) > 1 else 0.0,
            # Pour compatibilité avec l'ancienne version (basée sur RF_VERSION_FINAL)
            "waveform_mode": None,
            "prf_mean": None,
            "pulse_duration_mean": None,
        }

        fingerprints.append(fingerprint)

    if not fingerprints:
        return {
            "status": "ok",
            "message": (
                "Aucun navire n'a suffisamment d'événements RF corrélés "
                f"(min_events_per_ship = {min_events_per_ship}). "
                "Base d'empreintes non construite."
            ),
            "matched_ratio_waveform": 0.0,
        }

    fingerprints_df = pd.DataFrame(fingerprints)

    # 6) Résumé textuel pour le commandant
    n_ships = len(fingerprints_df)
    msg = (
        f"👨‍✈️ Commandant,\n\n"
        f"La base d'empreintes RF a été construite à partir du fichier de corrélation RF/AIS "
        f"(rf_correlation.parquet), en ne retenant que les associations de confiance ≥ {min_confidence}.\n\n"
        f"📡 Nombre de navires avec empreinte RF : {n_ships}\n"
        f"📡 Nombre moyen de détections RF corrélées par navire : "
        f"{fingerprints_df['nb_events'].mean():.1f}\n"
    )

    # Exemple : top 5 navires les plus observés
    top = fingerprints_df.sort_values('nb_events', ascending=False).head(5)
    if not top.empty:
        msg += "\nNavires les plus caractérisés (par nombre de détections RF corrélées) :\n"
        for _, row in top.iterrows():
            freq_mean = row['freq_mean']
            freq_std = row['freq_std']
            if freq_mean is not None and freq_std is not None:
                msg += (
                    f" - MMSI {row['mmsi']} : {row['nb_events']} détections RF, "
                    f"fréquence ~{freq_mean:.1f} MHz (±{freq_std:.1f} MHz), "
                    f"confiance moyenne ~{row['mean_confidence']:.2f}\n"
                )
            else:
                msg += (
                    f" - MMSI {row['mmsi']} : {row['nb_events']} détections RF, "
                    f"confiance moyenne ~{row['mean_confidence']:.2f}\n"
                )

    result = {
        "status": "ok",
        "message": msg,
        "fingerprints_df": fingerprints_df,
        # Pour compat avec l'ancienne version, on indique qu'on n'a pas de waveform réel ici
        "matched_ratio_waveform": 0.0,
    }
    return result

#### Interception et infrastructures critiques — version finale

Redéfinitions enrichies des deux tools ci-dessus ; ce sont ces versions qui sont actives à l'exécution.

`planifier_interception_sur_zone`, `analyser_infrastructure_critique`

In [ ]:
@tool
def planifier_interception_sur_zone(
    navire_ami_mmsi: str,
    zone: str,
    vitesse_ami_max_knots: float = 20.0,
    rayon_defaut_km: float = 5.0,
) -> dict:
    """
    Peux-tu m’aider à planifier une interception sur zone en fournissant des données
    sur le navire XXX de la marine Française dans [zone spécifique] ?

    - Récupère la dernière position AIS du navire ami (MMSI navire_ami_mmsi)
    - Résout la zone (ZONES_DATA ou "lat, lon[, rayon]")
    - Construit une route maritime estimée (en Méditerranée) qui évite la terre
    - Calcule la distance de route + temps de transit
    - Génère une carte Folium avec :
        * navire ami,
        * centre de zone + cercle de rayon,
        * route maritime estimée (polyline qui ne passe pas sur les terres).
    """

    # --- Vérifs de base sur les données AIS ---
    if "df_ais_global" not in globals() or df_ais_global is None or df_ais_global.empty:
        return {
            "status": "error",
            "message": "Les données AIS (df_ais_global) ne sont pas disponibles."
        }

    df = df_ais_global.copy()
    if not pd.api.types.is_datetime64_any_dtype(df["datetime"]):
        df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
    df = df.dropna(subset=["datetime"])

    # --- 1) Dernière position du navire ami ---
    mmsi_str = str(navire_ami_mmsi).strip()
    df_ami = df[df["Mmsi"].astype(str) == mmsi_str].copy()

    if df_ami.empty:
        return {
            "status": "ok",
            "message": f"Aucun navire AIS trouvé avec le MMSI {mmsi_str}."
        }

    df_ami = df_ami.sort_values("datetime")
    row_ami = df_ami.iloc[-1]

    fr_name = row_ami.get("Name", f"MMSI {mmsi_str}")
    fr_type = row_ami.get("ShipTypeLabel", "Type inconnu")
    fr_country = row_ami.get("Country", "Pays inconnu")
    fr_lat = float(row_ami.get("Position_LatitudeDeg"))
    fr_lon = float(row_ami.get("Position_LongitudeDeg"))
    fr_sog = float(row_ami.get("SogKt") if not pd.isna(row_ami.get("SogKt")) else 0.0)
    fr_cog = float(row_ami.get("CogDeg") if not pd.isna(row_ami.get("CogDeg")) else 0.0)
    fr_time = row_ami.get("datetime")

    # --- 2) Résolution de la zone ---
    z_config = None

    if "ZONES_DATA" in globals():
        if "_norm" in globals():
            z_clean = _norm(zone)
        else:
            z_clean = zone.strip().lower()
        if z_clean in ZONES_DATA:
            z_config = ZONES_DATA[z_clean]
        else:
            for k in ZONES_DATA:
                if z_clean in k or k in z_clean:
                    z_config = ZONES_DATA[k]
                    break

    if z_config is None:
        nums = re.findall(r"[-+]?\d*\.?\d+", zone.replace(",", " "))
        if len(nums) >= 2:
            lat_c = float(nums[0])
            lon_c = float(nums[1])
            rayon_km = float(nums[2]) if len(nums) >= 3 else rayon_defaut_km
            z_config = {"lat": lat_c, "lon": lon_c, "rayon": rayon_km}

    if z_config is None:
        return {
            "status": "error",
            "message": f"Zone '{zone}' inconnue ou mal formatée (attendu : nom connu ou 'lat, lon[, rayon]')."
        }

    lat_c = z_config["lat"]
    lon_c = z_config["lon"]
    rayon_km = z_config.get("rayon", rayon_defaut_km)

    # --- 3) Route maritime estimée ---
    route_coords = build_mediterranean_route(fr_lat, fr_lon, lat_c, lon_c)
    route_dist_km = route_length_km(route_coords)

    # Grande-cercle pour info (distance “en avion”, non utilisée pour le temps)
    gc_dist_km = haversine_km(fr_lat, fr_lon, lat_c, lon_c)

    # Temps estimé en suivant la route maritime
    v_fr_kmh = max(vitesse_ami_max_knots, 1e-3) * 1.852
    time_route_h = route_dist_km / v_fr_kmh if v_fr_kmh > 0 else None

    # --- 4) Carte Folium ---
    # centre = milieu approximatif de la route
    center_lat = np.mean([p[0] for p in route_coords])
    center_lon = np.mean([p[1] for p in route_coords])

    m = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles="CartoDB positron")

    # Cercle zone
    folium.Circle(
        location=[lat_c, lon_c],
        radius=rayon_km * 1000.0,
        color="red",
        fill=True,
        fill_opacity=0.1,
        popup=f"Zone : {zone} (rayon {rayon_km} km)"
    ).add_to(m)

    # Centre zone
    folium.Marker(
        location=[lat_c, lon_c],
        popup=f"Centre zone : {zone}",
        icon=folium.Icon(color="red", icon="crosshairs", prefix="fa"),
    ).add_to(m)

    # Navire ami
    popup_ami = (
        f"{fr_name}<br>"
        f"MMSI {mmsi_str}<br>"
        f"{fr_type}, pavillon {fr_country}<br>"
        f"Pos: {fr_lat:.3f}°N, {fr_lon:.3f}°E<br>"
        f"SOG: {fr_sog:.1f} kn, COG: {fr_cog:.0f}°<br>"
        f"{fr_time} (UTC)"
    )
    folium.Marker(
        location=[fr_lat, fr_lon],
        popup=popup_ami,
        icon=folium.Icon(color="blue", icon="ship", prefix="fa"),
    ).add_to(m)

    # Route maritime estimée
    folium.PolyLine(
        locations=[(lat, lon) for (lat, lon) in route_coords],
        color="blue",
        weight=3,
        opacity=0.8,
        tooltip="Route maritime estimée (évite la terre)"
    ).add_to(m)

    # --- 5) Sauvegardes ---
    zone_slug = re.sub(r"[^a-zA-Z0-9]+", "_", zone).strip("_")
    html_path = f"interception_{mmsi_str}_{zone_slug}.html"
    png_path = None

    m.save(html_path)

    try:
        import io
        from PIL import Image
        png_data = m._to_png(5)
        img = Image.open(io.BytesIO(png_data))
        png_path = f"interception_{mmsi_str}_{zone_slug}.png"
        img.save(png_path)
    except Exception:
        png_path = None

    # --- 6) Narratif ---
    narrative = []
    narrative.append(
        f"👨‍✈️ Commandant,\n\n"
        f"Planification d'interception sur zone pour le navire **{fr_name}** "
        f"(MMSI {mmsi_str}, {fr_type}, pavillon {fr_country})."
    )
    narrative.append(
        f"\n📍 Dernière position AIS connue : ({fr_lat:.3f}°N, {fr_lon:.3f}°E) "
        f"à {fr_time} (UTC).\n"
        f"🎯 Zone demandée : *{zone}* (centre {lat_c:.3f}°N, {lon_c:.3f}°E, rayon {rayon_km:.1f} km).\n"
        f"📏 Distance 'en avion' (grande cercle) : ~{gc_dist_km:.1f} km.\n"
        f"🧭 Distance estimée en suivant une route maritime : ~{route_dist_km:.1f} km.\n"
    )
    narrative.append(
        f"🚀 Vitesse max retenue pour la planification : {vitesse_ami_max_knots:.1f} nœuds "
        f"(~{v_fr_kmh:.1f} km/h).\n"
    )

    if time_route_h is not None:
        narrative.append(
            f"⏱️ Temps estimé pour rejoindre la zone en suivant la route maritime : "
            f"~{time_route_h:.1f} heures (~{time_route_h/24:.1f} jours).\n"
        )
    else:
        narrative.append(
            "⏱️ Impossible d'estimer un temps de transit (vitesse max non valide).\n"
        )

    narrative.append(
        "\n🗺️ Une carte a été générée avec :\n"
        "   - la position actuelle du navire ami,\n"
        "   - le centre de la zone et son rayon,\n"
        "   - une route maritime estimée qui évite les côtes.\n"
        f"   ➜ Fichier HTML : `{html_path}`"
    )
    if png_path:
        narrative.append(f"\n📸 Une image statique de la route est également disponible : `{png_path}`.")

    return {
        "status": "ok",
        "mode": "positionnement_sans_cible",
        "zone": {
            "label": zone,
            "lat_center": lat_c,
            "lon_center": lon_c,
            "rayon_km": rayon_km,
        },
        "friendly_ship": {
            "mmsi": mmsi_str,
            "name": fr_name,
            "type": fr_type,
            "country": fr_country,
            "lat": fr_lat,
            "lon": fr_lon,
            "time": fr_time,
            "sog_knots": fr_sog,
            "cog_deg": fr_cog,
        },
        "interception": {
            "great_circle_distance_km": gc_dist_km,
            "route_length_km": route_dist_km,
            "time_to_center_hours": time_route_h,
            "vitesse_ami_max_knots": vitesse_ami_max_knots,
        },
        "route_coords": route_coords,
        "map_html_path": html_path,
        "map_png_path": png_path,
        "narrative": "\n".join(narrative),
    }

@tool
def analyser_infrastructure_critique(
    infrastructure: str,
    hours: int = 24,
    rayon_km: float = 30.0,
    proximite_critique_km: float = 5.0
) -> str:
    """
    Analyse rapide des navires opérant près d'une infrastructure critique
    (port, terminal, oléoduc, etc.) en combinant AIS + RF + éphémérides.

    Dark vessel (dans ce tool) =
      - navire connu qui coupe l'AIS mais continue d'émettre en RF dans la zone
      - contact RF sans corrélation AIS fiable (émetteur inconnu)

    - Localise l'infrastructure (via INFRA_DATA / ZONES_DATA ou "lat,lon[,rayon]")
    - Prend les dernières `hours` heures de données
    - Analyse :
        * Trafic AIS autour de l'infrastructure
        * Détections RF (connues vs inconnues, dark vs non-dark)
        * Passages satellites (df_ephemeris_global)
    """
    try:
        # Vérif des 3 sources
        if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
            return "❌ Les données AIS (df_ais_global) ne sont pas disponibles."
        if 'df_rf_global' not in globals() or df_rf_global is None or df_rf_global.empty:
            return "❌ Les données RF (df_rf_global) ne sont pas disponibles."
        if 'df_ephemeris_global' not in globals() or df_ephemeris_global is None or df_ephemeris_global.empty:
            eph_available = False
        else:
            eph_available = True

        corr_available = (
            'df_corr_global' in globals()
            and df_corr_global is not None
            and not df_corr_global.empty
        )

        # --- 1) Résolution de l'infrastructure -> (lat_c, lon_c, rayon)
        infra_name = infrastructure.strip()
        infra_clean = _norm(infrastructure) if '_norm' in globals() else infra_name.lower()
        infra_cfg = None

        if 'INFRA_DATA' in globals():
            if infra_clean in INFRA_DATA:
                infra_cfg = INFRA_DATA[infra_clean]
            else:
                for k, v in INFRA_DATA.items():
                    if infra_clean in k or k in infra_clean:
                        infra_cfg = v
                        break

        if infra_cfg is None and 'ZONES_DATA' in globals():
            if infra_clean in ZONES_DATA:
                infra_cfg = ZONES_DATA[infra_clean]
            else:
                for k, v in ZONES_DATA.items():
                    if infra_clean in k or k in infra_clean:
                        infra_cfg = v
                        break

        if infra_cfg is None:
            nums = re.findall(r"[-+]?\d*\.?\d+", infrastructure.replace(",", " "))
            if len(nums) >= 2:
                lat_c = float(nums[0])
                lon_c = float(nums[1])
                if len(nums) >= 3:
                    rayon_km = float(nums[2])
            else:
                return (
                    f"❌ Impossible d'interpréter l'infrastructure '{infrastructure}'.\n"
                    "Utilisez soit un nom présent dans INFRA_DATA/ZONES_DATA, "
                    "soit le format 'lat, lon[, rayon_km]'."
                )
        else:
            lat_c = infra_cfg.get("lat")
            lon_c = infra_cfg.get("lon")
            if infra_cfg.get("rayon"):
                rayon_km = infra_cfg["rayon"]

        if lat_c is None or lon_c is None:
            return f"❌ Infrastructure '{infrastructure}' mal définie (lat/lon manquants)."

        # --- 2) Fenêtre temporelle globale ---
        df_ais = df_ais_global.copy()
        if not pd.api.types.is_datetime64_any_dtype(df_ais["datetime"]):
            df_ais["datetime"] = pd.to_datetime(df_ais["datetime"], utc=True, errors="coerce")
        df_ais = df_ais.dropna(subset=["datetime"])
        t_ais_max = df_ais["datetime"].max()

        df_rf = df_rf_global.copy()
        if not pd.api.types.is_datetime64_any_dtype(df_rf["timestamp_utc"]):
            df_rf["timestamp_utc"] = pd.to_datetime(df_rf["timestamp_utc"], utc=True, errors="coerce")
        df_rf = df_rf.dropna(subset=["timestamp_utc"])
        t_rf_max = df_rf["timestamp_utc"].max()

        t_max = max(t_ais_max, t_rf_max)
        t_min = t_max - pd.Timedelta(hours=hours)

        # --- 3) AIS : navires autour de l'infrastructure ---
        df_ais_w = df_ais[(df_ais["datetime"] >= t_min) & (df_ais["datetime"] <= t_max)].copy()
        df_ais_w = df_ais_w.dropna(subset=["Position_LatitudeDeg", "Position_LongitudeDeg"])

        deg_lat = rayon_km / 111.0
        deg_lon = rayon_km / 85.0  # approx en Med

        df_ais_zone = df_ais_w[
            df_ais_w["Position_LatitudeDeg"].between(lat_c - deg_lat, lat_c + deg_lat)
            & df_ais_w["Position_LongitudeDeg"].between(lon_c - deg_lon, lon_c + deg_lon)
        ].copy()

        def _hv(lat1, lon1, lat2, lon2):
            return haversine_km(lat1, lon1, lat2, lon2)

        if not df_ais_zone.empty:
            df_ais_zone["distance_km"] = df_ais_zone.apply(
                lambda r: _hv(lat_c, lon_c, r["Position_LatitudeDeg"], r["Position_LongitudeDeg"]),
                axis=1,
            )
            df_ais_zone = df_ais_zone[df_ais_zone["distance_km"] <= rayon_km].copy()

        if not df_ais_zone.empty:
            idx_last = df_ais_zone.groupby("Mmsi")["datetime"].idxmax()
            df_ais_last = df_ais_zone.loc[idx_last].copy()
            df_ais_last["type_navire"] = df_ais_last.apply(_classer_type_navire_ais, axis=1)
        else:
            df_ais_last = pd.DataFrame()

        nb_navires_zone = int(len(df_ais_last))
        nb_navires_proches = int(
            df_ais_last[df_ais_last["distance_km"] <= proximite_critique_km].shape[0]
        )

        type_counts = df_ais_last["type_navire"].value_counts().to_dict() if not df_ais_last.empty else {}
        type_counts_proches = (
            df_ais_last[df_ais_last["distance_km"] <= proximite_critique_km]["type_navire"]
            .value_counts()
            .to_dict()
            if not df_ais_last.empty else {}
        )

        top_navires_proches = []
        if not df_ais_last.empty:
            df_tmp = df_ais_last[df_ais_last["distance_km"] <= rayon_km].copy()
            df_tmp = df_tmp.sort_values("distance_km").head(5)
            for _, r in df_tmp.iterrows():
                top_navires_proches.append({
                    "name": r.get("Name", "Nom inconnu"),
                    "mmsi": r.get("Mmsi"),
                    "type": r.get("ShipTypeLabel", "Type inconnu"),
                    "type_metier": r.get("type_navire", "autre"),
                    "country": r.get("Country", "Pays inconnu"),
                    "distance_km": float(r.get("distance_km", np.nan)),
                })

        # --- 4) RF brutes autour de l'infrastructure ---
        df_rf_w = df_rf[
            (df_rf["timestamp_utc"] >= t_min) & (df_rf["timestamp_utc"] <= t_max)
        ].copy()
        df_rf_w = df_rf_w.dropna(subset=["latitude_deg", "longitude_deg"])

        df_rf_zone = df_rf_w[
            df_rf_w["latitude_deg"].between(lat_c - deg_lat, lat_c + deg_lat)
            & df_rf_w["longitude_deg"].between(lon_c - deg_lon, lon_c + deg_lon)
        ].copy()

        if not df_rf_zone.empty:
            df_rf_zone["distance_km"] = df_rf_zone.apply(
                lambda r: _hv(lat_c, lon_c, r["latitude_deg"], r["longitude_deg"]),
                axis=1,
            )
            df_rf_zone = df_rf_zone[df_rf_zone["distance_km"] <= rayon_km].copy()

        nb_rf_zone = int(len(df_rf_zone))
        nb_rf_proches = int(
            df_rf_zone[df_rf_zone["distance_km"] <= proximite_critique_km].shape[0]
        )

        # --- 5) RF corrélées & dark vessels ---
        dark_ships_mmsi = set()
        nb_dark_ships_zone = 0
        rf_dark_events_proches = 0
        rf_unknown_events_zone = 0
        rf_unknown_events_proches = 0
        nb_corr_total = 0
        nb_corr_known = 0
        nb_corr_unknown = 0

        if corr_available:
            df_corr = df_corr_global.copy()
            if not pd.api.types.is_datetime64_any_dtype(df_corr["timestamp_utc"]):
                df_corr["timestamp_utc"] = pd.to_datetime(df_corr["timestamp_utc"], utc=True, errors="coerce")
            df_corr = df_corr.dropna(subset=["timestamp_utc", "latitude_deg", "longitude_deg"])

            df_corr_w = df_corr[
                (df_corr["timestamp_utc"] >= t_min) & (df_corr["timestamp_utc"] <= t_max)
            ].copy()

            df_corr_w["cand_1_confidence"] = pd.to_numeric(
                df_corr_w.get("cand_1_confidence", np.nan),
                errors="coerce"
            ).fillna(0.0)

            df_corr_zone = df_corr_w[
                df_corr_w["latitude_deg"].between(lat_c - deg_lat, lat_c + deg_lat)
                & df_corr_w["longitude_deg"].between(lon_c - deg_lon, lon_c + deg_lon)
            ].copy()

            if not df_corr_zone.empty:
                df_corr_zone["distance_km"] = df_corr_zone.apply(
                    lambda r: _hv(lat_c, lon_c, r["latitude_deg"], r["longitude_deg"]),
                    axis=1,
                )
                df_corr_zone = df_corr_zone[df_corr_zone["distance_km"] <= rayon_km].copy()

            nb_corr_total = int(len(df_corr_zone))

            if nb_corr_total > 0:
                seuil_conf = 0.5
                df_corr_zone["cand_1_mmsi_str"] = df_corr_zone.get("cand_1_mmsi", "").astype(str).str.strip()

                # RF bien corrélées à un navire connu
                df_corr_known = df_corr_zone[
                    (df_corr_zone["cand_1_confidence"] >= seuil_conf)
                    & (df_corr_zone["cand_1_mmsi_str"] != "")
                    & (df_corr_zone["cand_1_mmsi_str"].str.lower() != "uncorrelated")
                ].copy()

                # RF non attribuées (émetteurs inconnus / corrélation faible)
                df_corr_unknown = df_corr_zone[~df_corr_zone.index.isin(df_corr_known.index)].copy()

                nb_corr_known = int(len(df_corr_known))
                nb_corr_unknown = int(len(df_corr_unknown))
                rf_unknown_events_zone = nb_corr_unknown

                # MMSI AIS présents dans la zone (fenêtre temporelle)
                ais_mmsi_zone = set(df_ais_zone["Mmsi"].astype(str).unique()) if not df_ais_zone.empty else set()
                known_mmsi_corr = set(df_corr_known["cand_1_mmsi_str"].unique()) if not df_corr_known.empty else set()

                # Navires "dark" = détectés en RF (corrélés) mais absents AIS dans la zone
                dark_ships_mmsi = known_mmsi_corr - ais_mmsi_zone
                nb_dark_ships_zone = len(dark_ships_mmsi)

                if nb_dark_ships_zone > 0:
                    rf_dark_events_proches = int(
                        df_corr_known[
                            (df_corr_known["cand_1_mmsi_str"].isin(dark_ships_mmsi))
                            & (df_corr_known["distance_km"] <= proximite_critique_km)
                        ].shape[0]
                    )

                rf_unknown_events_proches = int(
                    df_corr_unknown[df_corr_unknown["distance_km"] <= proximite_critique_km].shape[0]
                )

        # --- 6) Éphémérides : passages satellites récents ---
        sat_txt = ""
        if eph_available:
            df_eph = df_ephemeris_global.copy()
            if not pd.api.types.is_datetime64_any_dtype(df_eph["StartTime"]):
                df_eph["StartTime"] = pd.to_datetime(df_eph["StartTime"], utc=True, errors="coerce")
            if not pd.api.types.is_datetime64_any_dtype(df_eph["EndTime"]):
                df_eph["EndTime"] = pd.to_datetime(df_eph["EndTime"], utc=True, errors="coerce")

            df_eph = df_eph.dropna(subset=["StartTime", "EndTime"])
            df_eph_w = df_eph[
                (df_eph["EndTime"] >= t_min) & (df_eph["StartTime"] <= t_max)
            ].copy()

            if df_eph_w.empty:
                sat_txt = (
                    "Aucun passage JAPETUS récent sur la période considérée "
                    f"({hours} dernières heures)."
                )
            else:
                def _pass_dist(r):
                    d1 = _hv(lat_c, lon_c, r["StartLatitude"], r["StartLongitude"])
                    d2 = _hv(lat_c, lon_c, r["EndLatitude"], r["EndLongitude"])
                    return min(d1, d2)

                df_eph_w["dist_infra_km"] = df_eph_w.apply(_pass_dist, axis=1)
                df_eph_zone = df_eph_w[df_eph_w["dist_infra_km"] <= 500].copy()

                if df_eph_zone.empty:
                    sat_txt = (
                        f"{len(df_eph_w)} passage(s) JAPETUS récents, "
                        "mais assez éloignés (>500 km) de l'infrastructure."
                    )
                else:
                    nb_passes = int(len(df_eph_zone))
                    best = df_eph_zone.sort_values("dist_infra_km").iloc[0]
                    sat_txt = (
                        f"{nb_passes} passage(s) JAPETUS ont survolé ou frôlé la zone "
                        f"(≤500 km) sur les {hours} dernières heures.\n"
                        f"- Passage le plus proche : {best.get('Source','JAPETUS')} sur {best.get('Target','Zone')} "
                        f"du {best['StartTime']} au {best['EndTime']} (distance ~{best['dist_infra_km']:.1f} km)."
                    )
        else:
            sat_txt = "Les éphémérides JAPETUS ne sont pas disponibles dans cet environnement."

        # --- 7) Heuristique de niveau de risque ---
        risk_level = "faible"
        risk_reasons = []

        if (nb_dark_ships_zone > 0) or (rf_dark_events_proches > 0) or (rf_unknown_events_proches > 0):
            risk_level = "élevé"
            if nb_dark_ships_zone > 0:
                risk_reasons.append(
                    f"{nb_dark_ships_zone} navire(s) semblent en mode 'dark' : présents en RF mais absents AIS "
                    f"dans un rayon de {rayon_km:.1f} km autour de l'infrastructure."
                )
            if rf_dark_events_proches > 0:
                risk_reasons.append(
                    f"{rf_dark_events_proches} détection(s) RF de ces navires 'dark' "
                    f"dans la proximité immédiate (≤ {proximite_critique_km:.1f} km)."
                )
            if rf_unknown_events_proches > 0:
                risk_reasons.append(
                    f"{rf_unknown_events_proches} détection(s) RF non corrélées à l'AIS "
                    f"dans la proximité immédiate (émetteurs inconnus)."
                )
        elif nb_rf_proches > 0:
            risk_level = "modéré"
            risk_reasons.append(
                f"{nb_rf_proches} détection(s) RF dans la zone de proximité immédiate "
                f"(≤ {proximite_critique_km:.1f} km)."
            )

        if nb_navires_proches > 0 and risk_level == "faible":
            risk_level = "modéré"
            risk_reasons.append(
                f"{nb_navires_proches} navire(s) AIS à moins de {proximite_critique_km:.1f} km de l'infrastructure."
            )

        # --- 8) Narratif pour le commandant ---
        periode_txt = f"du {t_min} au {t_max} (UTC)"
        centre_txt = f"centre ~ ({lat_c:.3f}°N, {lon_c:.3f}°E), rayon {rayon_km:.1f} km"

        txt = []
        txt.append(
            f"👨‍✈️ Commandant,\n\n"
            f"Analyse multi-capteurs autour de l'infrastructure critique **{infra_name}**\n"
            f"({centre_txt}) sur les **{hours} dernières heures** ({periode_txt}).\n"
        )

        # AIS
        txt.append("🚢 **Vue AIS autour de l'infrastructure**")
        if nb_navires_zone == 0:
            txt.append(
                f"- Aucun navire AIS n'a été détecté dans un rayon de {rayon_km:.1f} km."
            )
        else:
            txt.append(
                f"- {nb_navires_zone} navire(s) AIS avec position récente dans le rayon {rayon_km:.1f} km."
            )
            if type_counts:
                txt.append("  Répartition par type métier :")
                for t, c in type_counts.items():
                    txt.append(f"    • {t} : {c} navire(s)")
            if nb_navires_proches > 0:
                txt.append(
                    f"- {nb_navires_proches} navire(s) à moins de {proximite_critique_km:.1f} km de l'infrastructure "
                    "(zone de proximité immédiate)."
                )
                if type_counts_proches:
                    txt.append("  Types des navires très proches :")
                    for t, c in type_counts_proches.items():
                        txt.append(f"    • {t} : {c}")
            if top_navires_proches:
                txt.append("\n  Navires les plus proches :")
                for n in top_navires_proches:
                    txt.append(
                        f"    - {n['name']} (MMSI {n['mmsi']}, {n['type']}, {n['country']}) "
                        f"à ~{n['distance_km']:.1f} km ({n['type_metier']})"
                    )

        # RF
        txt.append("\n📡 **Vue RF UnseenLabs (émissions radio)**")
        txt.append(
            f"- {nb_rf_zone} détection(s) RF dans le rayon {rayon_km:.1f} km, "
            f"dont {nb_rf_proches} à moins de {proximite_critique_km:.1f} km."
        )
        if corr_available:
            txt.append(
                f"- {nb_corr_total} événement(s) RF disposent d'une tentative de corrélation RF/AIS dans la zone."
            )
            txt.append(
                f"  Parmi eux : {nb_corr_known} bien corrélés à des navires connus, "
                f"{nb_corr_unknown} détection(s) RF non attribuées de façon fiable (émetteurs inconnus)."
            )
            if nb_dark_ships_zone > 0:
                txt.append(
                    f"- ⚠ {nb_dark_ships_zone} navire(s) semblent en **mode 'dark'** : présents en RF mais "
                    "sans AIS récent dans la zone analysée."
                )
            if rf_unknown_events_zone > 0:
                txt.append(
                    f"- ⚠ {rf_unknown_events_zone} détection(s) RF non corrélées à un MMSI fiable "
                    "(contacts potentiellement inconnus)."
                )
            if (rf_dark_events_proches > 0) or (rf_unknown_events_proches > 0):
                txt.append(
                    f"- Dans la zone de proximité immédiate (≤ {proximite_critique_km:.1f} km) : "
                    f"{rf_dark_events_proches} émission(s) de navires 'dark' et "
                    f"{rf_unknown_events_proches} émission(s) RF d'origine inconnue."
                )
        else:
            txt.append(
                "- Les corrélations RF/AIS détaillées (rf_correlation.parquet) ne sont pas disponibles ; "
                "impossible de distinguer précisément navires connus / inconnus / dark."
            )

        # Satellites
        txt.append("\n🛰️ **Couverture satellites (JAPETUS)**")
        txt.append(sat_txt)

        # Risque
        txt.append("\n🧮 **Appréciation rapide du risque autour de l'infrastructure**")
        txt.append(f"- Niveau de risque estimé : **{risk_level.upper()}**")
        if risk_reasons:
            txt.append("  Principaux éléments pris en compte :")
            for r in risk_reasons:
                txt.append(f"    • {r}")
        else:
            txt.append(
                "  Aucun élément particulièrement préoccupant n'a été détecté sur cette période "
                "(trafic faible ou conforme, pas de contact RF anormal)."
            )

        txt.append(
            "\n💡 Cette analyse peut servir de base à une surveillance renforcée : "
            "tasking imagerie sur les prochains passages JAPETUS, suivi temps réel des navires "
            "les plus proches et des contacts RF non identifiés ou 'dark'."
        )

        return "\n".join(txt)

    except Exception as e:
        return f"❌ Erreur dans analyser_infrastructure_critique : {e}"

#### Signatures RF non standard

Émissions RF dont la signature ne correspond à aucun profil de navire attendu dans la zone — le marqueur le plus direct d'un navire masquant son identité.

`analyser_signatures_rf_non_standard`

In [ ]:
@tool
def analyser_signatures_rf_non_standard(
    zone: str,
    hours: int = 24,
    rayon_km: float = 100.0,
    min_conf_corr: float = 0.5,
    min_events_per_ship: int = 3
) -> Dict[str, Any]:
    """
    Analyse RF : génère une liste de navires ayant des signatures RF « non standard »
    dans une zone donnée pour inspection prioritaire.

    Utilise principalement df_corr_global (rf_correlation.parquet) :
      - timestamp_utc (datetime64)
      - RF_Frequency_MHz (float64)
      - latitude_deg, longitude_deg (float64)
      - cand_1_mmsi (str)
      - cand_1_confidence (float)
      - number_of_candidates (int, optionnel)

    Complète avec df_ais_global si disponible pour enrichir (Nom, type, pavillon).

    Heuristiques « non standard » inspirées du monde réel :
      - Fréquence moyenne hors bande statistique de la zone (outlier de fréquence)
      - Bande fréquentielle atypique pour un navire civil (hors VHF / X / S classiques)
      - Spectre RF très large (écart-type de fréquence élevé → émission large bande)
      - Beaucoup de corrélations de faible confiance dans la zone
      - Activité RF intense (beaucoup d’événements par heure) par rapport aux autres navires
      - Présence RF significative avec peu ou pas d'AIS récent (candidat « dark-like »)

    Paramètres :
      - zone : nom de zone (clé ZONES_DATA/INFRA_DATA) ou "lat,lon[,rayon]".
      - hours : fenêtre temporelle (heures) remontant depuis la dernière détection RF.
      - rayon_km : rayon d'analyse autour du centre de zone.
      - min_conf_corr : seuil de confiance RF→AIS considéré comme « bonne corrélation ».
      - min_events_per_ship : nb min de détections RF pour caractériser un navire.

    Retourne :
      dict avec :
        - status, message éventuel
        - zone, center, time_window
        - nb_signaux_rf_zone, nb_navires_corrélés
        - nb_navires_suspects
        - suspects : liste de dict (mmsi, nom, type, pavillon, stats RF, flags, priorité)
        - nb_contacts_rf_sans_mmsi
        - narrative : texte en français pour le commandant
    """
    import pandas as pd
    import numpy as np
    import math
    import re
    from typing import Dict, Any, List

    try:
        # -------------------------
        # 0) Vérifications de base
        # -------------------------
        if 'df_corr_global' not in globals() or df_corr_global is None or df_corr_global.empty:
            return {
                "status": "error",
                "message": (
                    "Les données de corrélation RF/AIS (df_corr_global) ne sont pas disponibles. "
                    "Impossible d’identifier les navires à partir des signatures RF."
                )
            }

        ais_available = (
            'df_ais_global' in globals()
            and df_ais_global is not None
            and not df_ais_global.empty
        )

        df_corr = df_corr_global.copy()

        # Normalisation timestamp
        if not pd.api.types.is_datetime64_any_dtype(
            df_corr.get("timestamp_utc", pd.Series(dtype="datetime64[ns]"))
        ):
            df_corr["timestamp_utc"] = pd.to_datetime(
                df_corr["timestamp_utc"], utc=True, errors="coerce"
            )
        df_corr = df_corr.dropna(subset=["timestamp_utc"])

        if df_corr.empty:
            return {
                "status": "error",
                "message": "Aucune donnée de corrélation RF/AIS exploitable (timestamps invalides)."
            }

        # Normalisation confiance et fréquence
        df_corr["cand_1_confidence"] = pd.to_numeric(
            df_corr.get("cand_1_confidence", np.nan), errors="coerce"
        ).fillna(0.0)
        df_corr["RF_Frequency_MHz"] = pd.to_numeric(
            df_corr.get("RF_Frequency_MHz", np.nan), errors="coerce"
        )

        # Fonctions utilitaires
        if 'haversine_km' in globals():
            hv = haversine_km  # type: ignore
        else:
            def hv(lat1, lon1, lat2, lon2):
                R = 6371.0
                phi1, phi2 = math.radians(lat1), math.radians(lat2)
                dphi = math.radians(lat2 - lat1)
                dlambda = math.radians(lon2 - lon1)
                a = (math.sin(dphi / 2) ** 2 +
                     math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2)
                c = 2 * math.asin(math.sqrt(a))
                return R * c

        def _normalize_name(s: str) -> str:
            if '_norm' in globals():
                return _norm(s)  # type: ignore
            return re.sub(r"\s+", " ", s.strip().lower())

        # Classification simple de bande RF (monde réel simplifié)
        def _band_label(freq_mhz: float) -> str:
            if pd.isna(freq_mhz):
                return "unknown"
            if 156 <= freq_mhz <= 174:
                return "VHF_marine"
            if 2900 <= freq_mhz <= 3100:
                return "S_band_radar"
            if 9200 <= freq_mhz <= 9500:
                return "X_band_radar"
            return "other_band"

        # Détection navire "militaire" vs "civil" à partir du type AIS
        def _is_military_type(ship_type_label: str) -> bool:
            if not ship_type_label:
                return False
            st = ship_type_label.lower()
            return any(
                kw in st
                for kw in ["military", "navy", "patrol", "warship"]
            )

        # -------------------------
        # 1) Résolution de la zone
        # -------------------------
        z_clean = _normalize_name(zone)
        lat_c = lon_c = None

        infra_cfg = None
        if 'INFRA_DATA' in globals():
            if z_clean in INFRA_DATA:
                infra_cfg = INFRA_DATA[z_clean]
            else:
                for k, v in INFRA_DATA.items():
                    if z_clean in k or k in z_clean:
                        infra_cfg = v
                        break

        zone_cfg = None
        if 'ZONES_DATA' in globals():
            if z_clean in ZONES_DATA:
                zone_cfg = ZONES_DATA[z_clean]
            else:
                for k, v in ZONES_DATA.items():
                    if z_clean in k or k in z_clean:
                        zone_cfg = v
                        break

        cfg = infra_cfg or zone_cfg

        if cfg is None:
            nums = re.findall(r"[-+]?\d*\.?\d+", zone.replace(",", " "))
            if len(nums) >= 2:
                lat_c = float(nums[0])
                lon_c = float(nums[1])
                if len(nums) >= 3:
                    rayon_km = float(nums[2])
            else:
                return {
                    "status": "error",
                    "message": (
                        f"Impossible de résoudre la zone '{zone}'. "
                        "Utilisez un nom connu (ZONES_DATA/INFRA_DATA) ou 'lat, lon[, rayon_km]'."
                    )
                }
        else:
            lat_c = cfg.get("lat")
            lon_c = cfg.get("lon")
            if cfg.get("rayon"):
                rayon_km = cfg["rayon"]

        if lat_c is None or lon_c is None:
            return {
                "status": "error",
                "message": f"Zone '{zone}' mal définie (lat/lon manquants)."
            }

        # -------------------------
        # 2) Fenêtre temporelle
        # -------------------------
        t_max = df_corr["timestamp_utc"].max()
        t_min = t_max - pd.Timedelta(hours=int(hours))

        df_corr_w = df_corr[
            (df_corr["timestamp_utc"] >= t_min) &
            (df_corr["timestamp_utc"] <= t_max)
        ].copy()

        if df_corr_w.empty:
            return {
                "status": "ok",
                "zone": zone,
                "center": {"lat": lat_c, "lon": lon_c, "rayon_km": rayon_km},
                "time_window": {"start": t_min, "end": t_max},
                "nb_signaux_rf_zone": 0,
                "nb_navires_corrélés": 0,
                "nb_navires_suspects": 0,
                "suspects": [],
                "nb_contacts_rf_sans_mmsi": 0,
                "narrative": (
                    "👨‍✈️ Commandant,\n\n"
                    f"Aucune détection RF corrélée (rf_correlation) dans la zone '{zone}' "
                    f"sur les {hours} dernières heures. "
                    "Impossible d’identifier des signatures RF non standard.\n"
                )
            }

        # -------------------------
        # 3) Filtre géographique
        # -------------------------
        deg_lat = rayon_km / 111.0
        deg_lon = rayon_km / 85.0  # approx en Méditerranée

        df_corr_zone = df_corr_w[
            df_corr_w["latitude_deg"].between(lat_c - deg_lat, lat_c + deg_lat) &
            df_corr_w["longitude_deg"].between(lon_c - deg_lon, lon_c + deg_lon)
        ].copy()

        if df_corr_zone.empty:
            return {
                "status": "ok",
                "zone": zone,
                "center": {"lat": lat_c, "lon": lon_c, "rayon_km": rayon_km},
                "time_window": {"start": t_min, "end": t_max},
                "nb_signaux_rf_zone": 0,
                "nb_navires_corrélés": 0,
                "nb_navires_suspects": 0,
                "suspects": [],
                "nb_contacts_rf_sans_mmsi": 0,
                "narrative": (
                    "👨‍✈️ Commandant,\n\n"
                    f"Aucune détection RF corrélée aux navires dans le rayon de {rayon_km:.1f} km "
                    f"autour de la zone '{zone}' sur les {hours} dernières heures.\n"
                )
            }

        df_corr_zone["distance_km"] = df_corr_zone.apply(
            lambda r: hv(lat_c, lon_c, r["latitude_deg"], r["longitude_deg"]),
            axis=1
        )
        df_corr_zone = df_corr_zone[df_corr_zone["distance_km"] <= rayon_km].copy()

        if df_corr_zone.empty:
            return {
                "status": "ok",
                "zone": zone,
                "center": {"lat": lat_c, "lon": lon_c, "rayon_km": rayon_km},
                "time_window": {"start": t_min, "end": t_max},
                "nb_signaux_rf_zone": 0,
                "nb_navires_corrélés": 0,
                "nb_navires_suspects": 0,
                "suspects": [],
                "nb_contacts_rf_sans_mmsi": 0,
                "narrative": (
                    "👨‍✈️ Commandant,\n\n"
                    f"Aucune détection RF corrélée dans le rayon exact de {rayon_km:.1f} km "
                    f"autour de '{zone}' sur les {hours} dernières heures.\n"
                )
            }

        # -------------------------
        # 4) Préparation des données
        # -------------------------
        # Contacts sans MMSI (RF anonymes / uncorrelated)
        df_anonymous = df_corr_zone[
            df_corr_zone["cand_1_mmsi"].isna() |
            (df_corr_zone["cand_1_mmsi"].astype(str).str.lower() == "uncorrelated")
        ]
        nb_contacts_rf_sans_mmsi = int(len(df_anonymous))

        # Navires avec MMSI identifié
        df_corr_ships = df_corr_zone[
            df_corr_zone["cand_1_mmsi"].notna() &
            (df_corr_zone["cand_1_mmsi"].astype(str).str.lower() != "uncorrelated")
        ].copy()

        if df_corr_ships.empty:
            narrative = (
                "👨‍✈️ Commandant,\n\n"
                f"Des émissions RF ont été détectées dans la zone '{zone}', "
                "mais aucune corrélation fiable avec un MMSI n'a été trouvée. "
                "Il s'agit probablement de contacts anonymes ou mal corrélés. "
                "Aucun navire spécifique ne peut être priorisé pour inspection "
                "sur la seule base des données RF actuelles.\n"
            )
            return {
                "status": "ok",
                "zone": zone,
                "center": {"lat": lat_c, "lon": lon_c, "rayon_km": rayon_km},
                "time_window": {"start": t_min, "end": t_max},
                "nb_signaux_rf_zone": int(len(df_corr_zone)),
                "nb_navires_corrélés": 0,
                "nb_navires_suspects": 0,
                "suspects": [],
                "nb_contacts_rf_sans_mmsi": nb_contacts_rf_sans_mmsi,
                "narrative": narrative
            }

        # -------------------------
        # 5) Baselines statistiques RF (zone)
        # -------------------------
        df_good = df_corr_ships[df_corr_ships["cand_1_confidence"] >= min_conf_corr].copy()
        freq_series = df_good["RF_Frequency_MHz"].dropna()

        if freq_series.empty:
            freq_low_th = freq_high_th = None
        else:
            q25 = freq_series.quantile(0.25)
            q75 = freq_series.quantile(0.75)
            iqr = max(q75 - q25, 1e-3)
            freq_low_th = q25 - 1.5 * iqr
            freq_high_th = q75 + 1.5 * iqr

        # Seuil pour "spectre large"
        std_list = []
        for mmsi, grp in df_good.groupby("cand_1_mmsi"):
            if len(grp) >= min_events_per_ship:
                std_val = grp["RF_Frequency_MHz"].std()
                if not pd.isna(std_val):
                    std_list.append(std_val)
        if std_list:
            std_series = pd.Series(std_list)
            std_th = max(std_series.quantile(0.75), 5.0)
        else:
            std_th = 10.0  # fallback

        # Baseline d'intensité d'activité RF (événements par heure)
        events_per_hour_all = []
        for mmsi, grp in df_corr_ships.groupby("cand_1_mmsi"):
            times = grp["timestamp_utc"].sort_values()
            if len(times) >= 2:
                duration_h = (times.iloc[-1] - times.iloc[0]).total_seconds() / 3600.0
                if duration_h > 0:
                    events_per_hour_all.append(len(times) / duration_h)
        if events_per_hour_all:
            eph_series = pd.Series(events_per_hour_all)
            eph_th = max(eph_series.quantile(0.75), 1.0)
        else:
            eph_th = 2.0  # fallback

        # -------------------------
        # 5bis) Préparation AIS (enrichissement)
        # -------------------------
        ais_info_cache: Dict[str, Dict[str, Any]] = {}
        if ais_available:
            df_ais = df_ais_global.copy()
            if not pd.api.types.is_datetime64_any_dtype(
                df_ais.get("datetime", pd.Series(dtype="datetime64[ns]"))
            ):
                df_ais["datetime"] = pd.to_datetime(
                    df_ais["datetime"], utc=True, errors="coerce"
                )
            df_ais = df_ais.dropna(subset=["datetime"])

            def _get_ais_info(mmsi_str: str) -> Dict[str, Any]:
                if mmsi_str in ais_info_cache:
                    return ais_info_cache[mmsi_str]
                subset = df_ais[df_ais["Mmsi"].astype(str) == mmsi_str]
                if subset.empty:
                    info = {
                        "name": f"MMSI {mmsi_str}",
                        "ship_type": "Type inconnu",
                        "country": "Pavillon inconnu",
                        "is_military": False
                    }
                else:
                    row = subset.sort_values("datetime").iloc[-1]
                    name = row.get("Name", None) or f"MMSI {mmsi_str}"
                    ship_type = row.get("ShipTypeLabel", None) or "Type inconnu"
                    country = (
                        row.get("Pays", None)
                        or row.get("Country", None)
                        or "Pavillon inconnu"
                    )
                    info = {
                        "name": str(name),
                        "ship_type": str(ship_type),
                        "country": str(country),
                        "is_military": _is_military_type(str(ship_type))
                    }
                ais_info_cache[mmsi_str] = info
                return info
        else:
            def _get_ais_info(mmsi_str: str) -> Dict[str, Any]:
                return {
                    "name": f"MMSI {mmsi_str}",
                    "ship_type": "Type inconnu",
                    "country": "Pavillon inconnu",
                    "is_military": False
                }

        # -------------------------
        # 6) Analyse par navire
        # -------------------------
        suspects: List[Dict[str, Any]] = []
        nb_navires_corr = 0

        for mmsi, grp in df_corr_ships.groupby("cand_1_mmsi"):
            mmsi_str = str(mmsi)
            nb_events = len(grp)
            if nb_events < min_events_per_ship:
                continue

            nb_navires_corr += 1

            freqs = grp["RF_Frequency_MHz"].dropna()
            if freqs.empty:
                continue

            freq_mean = float(freqs.mean())
            freq_std = float(freqs.std()) if len(freqs) > 1 else 0.0
            freq_min = float(freqs.min())
            freq_max = float(freqs.max())

            confs = grp["cand_1_confidence"]
            nb_high_conf = int((confs >= min_conf_corr).sum())
            nb_low_conf = int(nb_events - nb_high_conf)
            ratio_low = nb_low_conf / nb_events if nb_events > 0 else 0.0

            lats = grp["latitude_deg"].dropna()
            lons = grp["longitude_deg"].dropna()
            if not lats.empty and not lons.empty:
                lat_mean = float(lats.mean())
                lon_mean = float(lons.mean())
                dist_mean = hv(lat_c, lon_c, lat_mean, lon_mean)
            else:
                lat_mean = lon_mean = None
                dist_mean = None

            # Intensité RF (événements par heure)
            times = grp["timestamp_utc"].sort_values()
            if len(times) >= 2:
                duration_h = (times.iloc[-1] - times.iloc[0]).total_seconds() / 3600.0
                events_per_hour = nb_events / duration_h if duration_h > 0 else nb_events
            else:
                duration_h = 0.0
                events_per_hour = nb_events

            # Bande dominante
            band_labels = freqs.apply(_band_label)
            dominant_band = band_labels.value_counts().idxmax() if not band_labels.empty else "unknown"

            ais_info = _get_ais_info(mmsi_str)
            is_military = ais_info["is_military"]

            # Flags « non standard »
            flags = []

            # 1) Fréquence moyenne statistiquement atypique dans la zone
            if freq_low_th is not None and freq_high_th is not None:
                if freq_mean < freq_low_th or freq_mean > freq_high_th:
                    flags.append("frequence_moyenne_hors_bande_zone")

            # 2) Bande atypique pour un navire supposé civil
            if dominant_band == "other_band" and not is_military:
                flags.append("bande_atypique_pour_navire_civil")

            # 3) Spectre large
            if freq_std >= std_th:
                flags.append("spectre_large")

            # 4) Beaucoup de corrélations de faible confiance
            if ratio_low > 0.5:
                flags.append("correlations_faible_confiance")

            # 5) Activité RF intense
            if events_per_hour >= eph_th:
                flags.append("activite_rf_intense")

            # 6) Dark-like : RF mais pas d'AIS récent
            is_dark_like = False
            if ais_available:
                df_ais_ship = df_ais[df_ais["Mmsi"].astype(str) == mmsi_str]  # type: ignore
                df_ais_ship = df_ais_ship[
                    (df_ais_ship["datetime"] >= t_min) &
                    (df_ais_ship["datetime"] <= t_max)
                ]
                if df_ais_ship.empty and nb_events >= min_events_per_ship:
                    is_dark_like = True
                    flags.append("rf_sans_ais_recent")

            # Si aucun flag → navire RF « normal »
            if not flags:
                continue

            # Score de priorité
            score = 0
            if "rf_sans_ais_recent" in flags:
                score += 4
            if "frequence_moyenne_hors_bande_zone" in flags:
                score += 3
            if "bande_atypique_pour_navire_civil" in flags:
                score += 3
            if "spectre_large" in flags:
                score += 2
            if "activite_rf_intense" in flags:
                score += 2
            if "correlations_faible_confiance" in flags:
                score += 1
            if dist_mean is not None and dist_mean <= rayon_km / 2:
                score += 1  # plus proche → plus prioritaire

            if score >= 7:
                priorite = "HAUTE"
            elif score >= 3:
                priorite = "MOYENNE"
            else:
                priorite = "FAIBLE"

            suspects.append({
                "mmsi": mmsi_str,
                "name": ais_info["name"],
                "ship_type": ais_info["ship_type"],
                "country": ais_info["country"],
                "is_military": bool(is_military),
                "nb_rf_events": int(nb_events),
                "nb_high_conf_events": nb_high_conf,
                "nb_low_conf_events": nb_low_conf,
                "events_per_hour": float(events_per_hour),
                "freq_mean_mhz": freq_mean,
                "freq_std_mhz": freq_std,
                "freq_min_mhz": freq_min,
                "freq_max_mhz": freq_max,
                "dominant_band": dominant_band,
                "lat_mean": lat_mean,
                "lon_mean": lon_mean,
                "distance_mean_km": dist_mean,
                "flags": flags,
                "priority": priorite,
                "priority_score": score,
            })

        suspects = sorted(
            suspects,
            key=lambda s: (-s["priority_score"], s["distance_mean_km"] if s["distance_mean_km"] is not None else 1e9)
        )

        nb_navires_suspects = len(suspects)

        # -------------------------
        # 7) Narratif commandant
        # -------------------------
        periode_txt = f"du {t_min} au {t_max} (UTC)"
        centre_txt = f"centre ~ ({lat_c:.3f}°N, {lon_c:.3f}°E), rayon {rayon_km:.1f} km"

        lignes: List[str] = []
        lignes.append(
            f"👨‍✈️ Commandant,\n\n"
            f"Analyse des signatures RF dans la zone **{zone}** ({centre_txt}) "
            f"sur les **{hours} dernières heures** ({periode_txt}).\n"
        )

        lignes.append(
            f"📡 **Volume de détections RF corrélées (rf_correlation)**\n"
            f"- Total de détections RF dans la zone : {int(len(df_corr_zone))}\n"
            f"- Navires avec MMSI identifié : {nb_navires_corr}\n"
            f"- Contacts RF sans MMSI fiable (anonymes/uncorrelated) : {nb_contacts_rf_sans_mmsi}\n"
        )

        if nb_navires_suspects == 0:
            lignes.append(
                "\n✅ Aucun navire ne présente de signature RF clairement non standard "
                "selon les heuristiques actuelles (bande de fréquence, spectre, activité, liaison avec l’AIS). "
                "Les émissions observées sont compatibles avec une activité maritime normale.\n"
            )
        else:
            lignes.append(
                f"\n⚠️ **Navires à signatures RF non standard identifiés : {nb_navires_suspects}**\n"
                "Ces navires sont proposés pour inspection prioritaire (profil RF atypique, activité suspecte ou comportement dark-like).\n"
            )
            top_display = min(5, nb_navires_suspects)
            lignes.append(
                f"🧾 Détail des {top_display} navire(s) les plus prioritaires :"
            )
            for s in suspects[:top_display]:
                flags_txt = ", ".join(s["flags"])
                dist_txt = (
                    f"{s['distance_mean_km']:.1f} km"
                    if s["distance_mean_km"] is not None else "distance inconnue"
                )
                band_txt = s["dominant_band"].replace("_", " ")
                lignes.append(
                    f"\n- {s['name']} (MMSI {s['mmsi']}, {s['ship_type']}, {s['country']})\n"
                    f"  • Priorité : **{s['priority']}** (score {s['priority_score']})\n"
                    f"  • Événements RF : {s['nb_rf_events']} (dont {s['nb_high_conf_events']} de haute confiance, "
                    f"≈{s['events_per_hour']:.1f} évènement(s)/heure)\n"
                    f"  • Bande RF dominante : {band_txt}\n"
                    f"  • Bande RF : ~{s['freq_mean_mhz']:.1f} MHz "
                    f"(min {s['freq_min_mhz']:.1f}, max {s['freq_max_mhz']:.1f}, σ≈{s['freq_std_mhz']:.1f} MHz)\n"
                    f"  • Position moyenne : {dist_txt} du centre de zone\n"
                    f"  • Motifs de suspicion : {flags_txt}\n"
                )

            lignes.append(
                "\n💡 Recommandation :\n"
                "- Inspecter en priorité les navires marqués **HAUTE priorité**, en particulier ceux "
                "présentant à la fois des bandes RF atypiques pour des navires civils, une activité RF intense "
                "et une absence d’AIS récent (« dark-like »).\n"
                "- Surveiller également les contacts RF anonymes (sans MMSI) s’ils se concentrent à proximité "
                "d’infrastructures critiques ou de zones d’intérêt.\n"
            )

        narrative = "\n".join(lignes)

        # -------------------------
        # 8) Résultat structuré
        # -------------------------
        result = {
            "status": "ok",
            "zone": zone,
            "center": {"lat": lat_c, "lon": lon_c, "rayon_km": rayon_km},
            "time_window": {"start": t_min, "end": t_max},
            "nb_signaux_rf_zone": int(len(df_corr_zone)),
            "nb_navires_corrélés": int(nb_navires_corr),
            "nb_navires_suspects": int(nb_navires_suspects),
            "suspects": suspects,
            "nb_contacts_rf_sans_mmsi": nb_contacts_rf_sans_mmsi,
            "narrative": narrative,
        }
        return result

    except Exception as e:
        return {
            "status": "error",
            "message": f"Erreur dans analyser_signatures_rf_non_standard : {e}"
        }

@tool
def analyser_signatures_rf_non_standard(
    zone: str,
    hours: int = 24,
    rayon_km: float = 100.0,
    min_conf_corr: float = 0.5,
    min_events_per_ship: int = 3
) -> Dict[str, Any]:
    """
    Analyse RF : génère une liste de navires ayant des signatures RF « non standard »
    dans une zone donnée pour inspection prioritaire.

    Utilise principalement df_corr_global (rf_correlation.parquet) :
      - timestamp_utc (datetime64)
      - RF_Frequency_MHz (float64)
      - latitude_deg, longitude_deg (float64)
      - cand_1_mmsi (str)
      - cand_1_confidence (float)
      - number_of_candidates (int, optionnel)

    Complète avec df_ais_global si disponible pour enrichir (Nom, type, pavillon).

    Heuristiques « non standard » inspirées du monde réel :
      - Fréquence moyenne hors bande statistique de la zone (outlier de fréquence)
      - Bande fréquentielle atypique pour un navire civil (hors VHF / X / S classiques)
      - Spectre RF très large (écart-type de fréquence élevé → émission large bande)
      - Beaucoup de corrélations de faible confiance dans la zone
      - Activité RF intense (beaucoup d’événements par heure) par rapport aux autres navires
      - Présence RF significative avec peu ou pas d'AIS récent (candidat « dark-like »)

    Paramètres :
      - zone : nom de zone (clé ZONES_DATA/INFRA_DATA) ou "lat,lon[,rayon]".
      - hours : fenêtre temporelle (heures) remontant depuis la dernière détection RF.
      - rayon_km : rayon d'analyse autour du centre de zone.
      - min_conf_corr : seuil de confiance RF→AIS considéré comme « bonne corrélation ».
      - min_events_per_ship : nb min de détections RF pour caractériser un navire.

    Retourne :
      dict avec :
        - status, message éventuel
        - zone, center, time_window
        - nb_signaux_rf_zone, nb_navires_corrélés
        - nb_navires_suspects
        - suspects : liste de dict (mmsi, nom, type, pavillon, stats RF, flags, priorité)
        - nb_contacts_rf_sans_mmsi
        - narrative : texte en français pour le commandant
    """
    import pandas as pd
    import numpy as np
    import math
    import re
    from typing import Dict, Any, List

    try:
        # -------------------------
        # 0) Vérifications de base
        # -------------------------
        if 'df_corr_global' not in globals() or df_corr_global is None or df_corr_global.empty:
            return {
                "status": "error",
                "message": (
                    "Les données de corrélation RF/AIS (df_corr_global) ne sont pas disponibles. "
                    "Impossible d’identifier les navires à partir des signatures RF."
                )
            }

        ais_available = (
            'df_ais_global' in globals()
            and df_ais_global is not None
            and not df_ais_global.empty
        )

        df_corr = df_corr_global.copy()

        # Normalisation timestamp
        if not pd.api.types.is_datetime64_any_dtype(
            df_corr.get("timestamp_utc", pd.Series(dtype="datetime64[ns]"))
        ):
            df_corr["timestamp_utc"] = pd.to_datetime(
                df_corr["timestamp_utc"], utc=True, errors="coerce"
            )
        df_corr = df_corr.dropna(subset=["timestamp_utc"])

        if df_corr.empty:
            return {
                "status": "error",
                "message": "Aucune donnée de corrélation RF/AIS exploitable (timestamps invalides)."
            }

        # Normalisation confiance et fréquence
        df_corr["cand_1_confidence"] = pd.to_numeric(
            df_corr.get("cand_1_confidence", np.nan), errors="coerce"
        ).fillna(0.0)
        df_corr["RF_Frequency_MHz"] = pd.to_numeric(
            df_corr.get("RF_Frequency_MHz", np.nan), errors="coerce"
        )

        # Fonctions utilitaires
        if 'haversine_km' in globals():
            hv = haversine_km  # type: ignore
        else:
            def hv(lat1, lon1, lat2, lon2):
                R = 6371.0
                phi1, phi2 = math.radians(lat1), math.radians(lat2)
                dphi = math.radians(lat2 - lat1)
                dlambda = math.radians(lon2 - lon1)
                a = (math.sin(dphi / 2) ** 2 +
                     math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2)
                c = 2 * math.asin(math.sqrt(a))
                return R * c

        def _normalize_name(s: str) -> str:
            if '_norm' in globals():
                return _norm(s)  # type: ignore
            return re.sub(r"\s+", " ", s.strip().lower())

        # Classification simple de bande RF (monde réel simplifié)
        def _band_label(freq_mhz: float) -> str:
            if pd.isna(freq_mhz):
                return "unknown"
            if 156 <= freq_mhz <= 174:
                return "VHF_marine"
            if 2900 <= freq_mhz <= 3100:
                return "S_band_radar"
            if 9200 <= freq_mhz <= 9500:
                return "X_band_radar"
            return "other_band"

        # Détection navire "militaire" vs "civil" à partir du type AIS
        def _is_military_type(ship_type_label: str) -> bool:
            if not ship_type_label:
                return False
            st = ship_type_label.lower()
            return any(
                kw in st
                for kw in ["military", "navy", "patrol", "warship"]
            )

        # -------------------------
        # 1) Résolution de la zone
        # -------------------------
        z_clean = _normalize_name(zone)
        lat_c = lon_c = None

        infra_cfg = None
        if 'INFRA_DATA' in globals():
            if z_clean in INFRA_DATA:
                infra_cfg = INFRA_DATA[z_clean]
            else:
                for k, v in INFRA_DATA.items():
                    if z_clean in k or k in z_clean:
                        infra_cfg = v
                        break

        zone_cfg = None
        if 'ZONES_DATA' in globals():
            if z_clean in ZONES_DATA:
                zone_cfg = ZONES_DATA[z_clean]
            else:
                for k, v in ZONES_DATA.items():
                    if z_clean in k or k in z_clean:
                        zone_cfg = v
                        break

        cfg = infra_cfg or zone_cfg

        if cfg is None:
            nums = re.findall(r"[-+]?\d*\.?\d+", zone.replace(",", " "))
            if len(nums) >= 2:
                lat_c = float(nums[0])
                lon_c = float(nums[1])
                if len(nums) >= 3:
                    rayon_km = float(nums[2])
            else:
                return {
                    "status": "error",
                    "message": (
                        f"Impossible de résoudre la zone '{zone}'. "
                        "Utilisez un nom connu (ZONES_DATA/INFRA_DATA) ou 'lat, lon[, rayon_km]'."
                    )
                }
        else:
            lat_c = cfg.get("lat")
            lon_c = cfg.get("lon")
            if cfg.get("rayon"):
                rayon_km = cfg["rayon"]

        if lat_c is None or lon_c is None:
            return {
                "status": "error",
                "message": f"Zone '{zone}' mal définie (lat/lon manquants)."
            }

        # -------------------------
        # 2) Fenêtre temporelle
        # -------------------------
        t_max = df_corr["timestamp_utc"].max()
        t_min = t_max - pd.Timedelta(hours=int(hours))

        df_corr_w = df_corr[
            (df_corr["timestamp_utc"] >= t_min) &
            (df_corr["timestamp_utc"] <= t_max)
        ].copy()

        if df_corr_w.empty:
            return {
                "status": "ok",
                "zone": zone,
                "center": {"lat": lat_c, "lon": lon_c, "rayon_km": rayon_km},
                "time_window": {"start": t_min, "end": t_max},
                "nb_signaux_rf_zone": 0,
                "nb_navires_corrélés": 0,
                "nb_navires_suspects": 0,
                "suspects": [],
                "nb_contacts_rf_sans_mmsi": 0,
                "narrative": (
                    "👨‍✈️ Commandant,\n\n"
                    f"Aucune détection RF corrélée (rf_correlation) dans la zone '{zone}' "
                    f"sur les {hours} dernières heures. "
                    "Impossible d’identifier des signatures RF non standard.\n"
                )
            }

        # -------------------------
        # 3) Filtre géographique
        # -------------------------
        deg_lat = rayon_km / 111.0
        deg_lon = rayon_km / 85.0  # approx en Méditerranée

        df_corr_zone = df_corr_w[
            df_corr_w["latitude_deg"].between(lat_c - deg_lat, lat_c + deg_lat) &
            df_corr_w["longitude_deg"].between(lon_c - deg_lon, lon_c + deg_lon)
        ].copy()

        if df_corr_zone.empty:
            return {
                "status": "ok",
                "zone": zone,
                "center": {"lat": lat_c, "lon": lon_c, "rayon_km": rayon_km},
                "time_window": {"start": t_min, "end": t_max},
                "nb_signaux_rf_zone": 0,
                "nb_navires_corrélés": 0,
                "nb_navires_suspects": 0,
                "suspects": [],
                "nb_contacts_rf_sans_mmsi": 0,
                "narrative": (
                    "👨‍✈️ Commandant,\n\n"
                    f"Aucune détection RF corrélée aux navires dans le rayon de {rayon_km:.1f} km "
                    f"autour de la zone '{zone}' sur les {hours} dernières heures.\n"
                )
            }

        df_corr_zone["distance_km"] = df_corr_zone.apply(
            lambda r: hv(lat_c, lon_c, r["latitude_deg"], r["longitude_deg"]),
            axis=1
        )
        df_corr_zone = df_corr_zone[df_corr_zone["distance_km"] <= rayon_km].copy()

        if df_corr_zone.empty:
            return {
                "status": "ok",
                "zone": zone,
                "center": {"lat": lat_c, "lon": lon_c, "rayon_km": rayon_km},
                "time_window": {"start": t_min, "end": t_max},
                "nb_signaux_rf_zone": 0,
                "nb_navires_corrélés": 0,
                "nb_navires_suspects": 0,
                "suspects": [],
                "nb_contacts_rf_sans_mmsi": 0,
                "narrative": (
                    "👨‍✈️ Commandant,\n\n"
                    f"Aucune détection RF corrélée dans le rayon exact de {rayon_km:.1f} km "
                    f"autour de '{zone}' sur les {hours} dernières heures.\n"
                )
            }

        # -------------------------
        # 4) Préparation des données
        # -------------------------
        # Contacts sans MMSI (RF anonymes / uncorrelated)
        df_anonymous = df_corr_zone[
            df_corr_zone["cand_1_mmsi"].isna() |
            (df_corr_zone["cand_1_mmsi"].astype(str).str.lower() == "uncorrelated")
        ]
        nb_contacts_rf_sans_mmsi = int(len(df_anonymous))

        # Navires avec MMSI identifié
        df_corr_ships = df_corr_zone[
            df_corr_zone["cand_1_mmsi"].notna() &
            (df_corr_zone["cand_1_mmsi"].astype(str).str.lower() != "uncorrelated")
        ].copy()

        if df_corr_ships.empty:
            narrative = (
                "👨‍✈️ Commandant,\n\n"
                f"Des émissions RF ont été détectées dans la zone '{zone}', "
                "mais aucune corrélation fiable avec un MMSI n'a été trouvée. "
                "Il s'agit probablement de contacts anonymes ou mal corrélés. "
                "Aucun navire spécifique ne peut être priorisé pour inspection "
                "sur la seule base des données RF actuelles.\n"
            )
            return {
                "status": "ok",
                "zone": zone,
                "center": {"lat": lat_c, "lon": lon_c, "rayon_km": rayon_km},
                "time_window": {"start": t_min, "end": t_max},
                "nb_signaux_rf_zone": int(len(df_corr_zone)),
                "nb_navires_corrélés": 0,
                "nb_navires_suspects": 0,
                "suspects": [],
                "nb_contacts_rf_sans_mmsi": nb_contacts_rf_sans_mmsi,
                "narrative": narrative
            }

        # -------------------------
        # 5) Baselines statistiques RF (zone)
        # -------------------------
        df_good = df_corr_ships[df_corr_ships["cand_1_confidence"] >= min_conf_corr].copy()
        freq_series = df_good["RF_Frequency_MHz"].dropna()

        if freq_series.empty:
            freq_low_th = freq_high_th = None
        else:
            q25 = freq_series.quantile(0.25)
            q75 = freq_series.quantile(0.75)
            iqr = max(q75 - q25, 1e-3)
            freq_low_th = q25 - 1.5 * iqr
            freq_high_th = q75 + 1.5 * iqr

        # Seuil pour "spectre large"
        std_list = []
        for mmsi, grp in df_good.groupby("cand_1_mmsi"):
            if len(grp) >= min_events_per_ship:
                std_val = grp["RF_Frequency_MHz"].std()
                if not pd.isna(std_val):
                    std_list.append(std_val)
        if std_list:
            std_series = pd.Series(std_list)
            std_th = max(std_series.quantile(0.75), 5.0)
        else:
            std_th = 10.0  # fallback

        # Baseline d'intensité d'activité RF (événements par heure)
        events_per_hour_all = []
        for mmsi, grp in df_corr_ships.groupby("cand_1_mmsi"):
            times = grp["timestamp_utc"].sort_values()
            if len(times) >= 2:
                duration_h = (times.iloc[-1] - times.iloc[0]).total_seconds() / 3600.0
                if duration_h > 0:
                    events_per_hour_all.append(len(times) / duration_h)
        if events_per_hour_all:
            eph_series = pd.Series(events_per_hour_all)
            eph_th = max(eph_series.quantile(0.75), 1.0)
        else:
            eph_th = 2.0  # fallback

        # -------------------------
        # 5bis) Préparation AIS (enrichissement)
        # -------------------------
        ais_info_cache: Dict[str, Dict[str, Any]] = {}
        if ais_available:
            df_ais = df_ais_global.copy()
            if not pd.api.types.is_datetime64_any_dtype(
                df_ais.get("datetime", pd.Series(dtype="datetime64[ns]"))
            ):
                df_ais["datetime"] = pd.to_datetime(
                    df_ais["datetime"], utc=True, errors="coerce"
                )
            df_ais = df_ais.dropna(subset=["datetime"])

            def _get_ais_info(mmsi_str: str) -> Dict[str, Any]:
                if mmsi_str in ais_info_cache:
                    return ais_info_cache[mmsi_str]
                subset = df_ais[df_ais["Mmsi"].astype(str) == mmsi_str]
                if subset.empty:
                    info = {
                        "name": f"MMSI {mmsi_str}",
                        "ship_type": "Type inconnu",
                        "country": "Pavillon inconnu",
                        "is_military": False
                    }
                else:
                    row = subset.sort_values("datetime").iloc[-1]
                    name = row.get("Name", None) or f"MMSI {mmsi_str}"
                    ship_type = row.get("ShipTypeLabel", None) or "Type inconnu"
                    country = (
                        row.get("Pays", None)
                        or row.get("Country", None)
                        or "Pavillon inconnu"
                    )
                    info = {
                        "name": str(name),
                        "ship_type": str(ship_type),
                        "country": str(country),
                        "is_military": _is_military_type(str(ship_type))
                    }
                ais_info_cache[mmsi_str] = info
                return info
        else:
            def _get_ais_info(mmsi_str: str) -> Dict[str, Any]:
                return {
                    "name": f"MMSI {mmsi_str}",
                    "ship_type": "Type inconnu",
                    "country": "Pavillon inconnu",
                    "is_military": False
                }

        # -------------------------
        # 6) Analyse par navire
        # -------------------------
        suspects: List[Dict[str, Any]] = []
        nb_navires_corr = 0

        for mmsi, grp in df_corr_ships.groupby("cand_1_mmsi"):
            mmsi_str = str(mmsi)
            nb_events = len(grp)
            if nb_events < min_events_per_ship:
                continue

            nb_navires_corr += 1

            freqs = grp["RF_Frequency_MHz"].dropna()
            if freqs.empty:
                continue

            freq_mean = float(freqs.mean())
            freq_std = float(freqs.std()) if len(freqs) > 1 else 0.0
            freq_min = float(freqs.min())
            freq_max = float(freqs.max())

            confs = grp["cand_1_confidence"]
            nb_high_conf = int((confs >= min_conf_corr).sum())
            nb_low_conf = int(nb_events - nb_high_conf)
            ratio_low = nb_low_conf / nb_events if nb_events > 0 else 0.0

            lats = grp["latitude_deg"].dropna()
            lons = grp["longitude_deg"].dropna()
            if not lats.empty and not lons.empty:
                lat_mean = float(lats.mean())
                lon_mean = float(lons.mean())
                dist_mean = hv(lat_c, lon_c, lat_mean, lon_mean)
            else:
                lat_mean = lon_mean = None
                dist_mean = None

            # Intensité RF (événements par heure)
            times = grp["timestamp_utc"].sort_values()
            if len(times) >= 2:
                duration_h = (times.iloc[-1] - times.iloc[0]).total_seconds() / 3600.0
                events_per_hour = nb_events / duration_h if duration_h > 0 else nb_events
            else:
                duration_h = 0.0
                events_per_hour = nb_events

            # Bande dominante
            band_labels = freqs.apply(_band_label)
            dominant_band = band_labels.value_counts().idxmax() if not band_labels.empty else "unknown"

            ais_info = _get_ais_info(mmsi_str)
            is_military = ais_info["is_military"]

            # Flags « non standard »
            flags = []

            # 1) Fréquence moyenne statistiquement atypique dans la zone
            if freq_low_th is not None and freq_high_th is not None:
                if freq_mean < freq_low_th or freq_mean > freq_high_th:
                    flags.append("frequence_moyenne_hors_bande_zone")

            # 2) Bande atypique pour un navire supposé civil
            if dominant_band == "other_band" and not is_military:
                flags.append("bande_atypique_pour_navire_civil")

            # 3) Spectre large
            if freq_std >= std_th:
                flags.append("spectre_large")

            # 4) Beaucoup de corrélations de faible confiance
            if ratio_low > 0.5:
                flags.append("correlations_faible_confiance")

            # 5) Activité RF intense
            if events_per_hour >= eph_th:
                flags.append("activite_rf_intense")

            # 6) Dark-like : RF mais pas d'AIS récent
            is_dark_like = False
            if ais_available:
                df_ais_ship = df_ais[df_ais["Mmsi"].astype(str) == mmsi_str]  # type: ignore
                df_ais_ship = df_ais_ship[
                    (df_ais_ship["datetime"] >= t_min) &
                    (df_ais_ship["datetime"] <= t_max)
                ]
                if df_ais_ship.empty and nb_events >= min_events_per_ship:
                    is_dark_like = True
                    flags.append("rf_sans_ais_recent")

            # Si aucun flag → navire RF « normal »
            if not flags:
                continue

            # Score de priorité
            score = 0
            if "rf_sans_ais_recent" in flags:
                score += 4
            if "frequence_moyenne_hors_bande_zone" in flags:
                score += 3
            if "bande_atypique_pour_navire_civil" in flags:
                score += 3
            if "spectre_large" in flags:
                score += 2
            if "activite_rf_intense" in flags:
                score += 2
            if "correlations_faible_confiance" in flags:
                score += 1
            if dist_mean is not None and dist_mean <= rayon_km / 2:
                score += 1  # plus proche → plus prioritaire

            if score >= 7:
                priorite = "HAUTE"
            elif score >= 3:
                priorite = "MOYENNE"
            else:
                priorite = "FAIBLE"

            suspects.append({
                "mmsi": mmsi_str,
                "name": ais_info["name"],
                "ship_type": ais_info["ship_type"],
                "country": ais_info["country"],
                "is_military": bool(is_military),
                "nb_rf_events": int(nb_events),
                "nb_high_conf_events": nb_high_conf,
                "nb_low_conf_events": nb_low_conf,
                "events_per_hour": float(events_per_hour),
                "freq_mean_mhz": freq_mean,
                "freq_std_mhz": freq_std,
                "freq_min_mhz": freq_min,
                "freq_max_mhz": freq_max,
                "dominant_band": dominant_band,
                "lat_mean": lat_mean,
                "lon_mean": lon_mean,
                "distance_mean_km": dist_mean,
                "flags": flags,
                "priority": priorite,
                "priority_score": score,
            })

        suspects = sorted(
            suspects,
            key=lambda s: (-s["priority_score"], s["distance_mean_km"] if s["distance_mean_km"] is not None else 1e9)
        )

        nb_navires_suspects = len(suspects)

        # -------------------------
        # 7) Narratif commandant
        # -------------------------
        periode_txt = f"du {t_min} au {t_max} (UTC)"
        centre_txt = f"centre ~ ({lat_c:.3f}°N, {lon_c:.3f}°E), rayon {rayon_km:.1f} km"

        lignes: List[str] = []
        lignes.append(
            f"👨‍✈️ Commandant,\n\n"
            f"Analyse des signatures RF dans la zone **{zone}** ({centre_txt}) "
            f"sur les **{hours} dernières heures** ({periode_txt}).\n"
        )

        lignes.append(
            f"📡 **Volume de détections RF corrélées (rf_correlation)**\n"
            f"- Total de détections RF dans la zone : {int(len(df_corr_zone))}\n"
            f"- Navires avec MMSI identifié : {nb_navires_corr}\n"
            f"- Contacts RF sans MMSI fiable (anonymes/uncorrelated) : {nb_contacts_rf_sans_mmsi}\n"
        )

        if nb_navires_suspects == 0:
            lignes.append(
                "\n✅ Aucun navire ne présente de signature RF clairement non standard "
                "selon les heuristiques actuelles (bande de fréquence, spectre, activité, liaison avec l’AIS). "
                "Les émissions observées sont compatibles avec une activité maritime normale.\n"
            )
        else:
            lignes.append(
                f"\n⚠️ **Navires à signatures RF non standard identifiés : {nb_navires_suspects}**\n"
                "Ces navires sont proposés pour inspection prioritaire (profil RF atypique, activité suspecte ou comportement dark-like).\n"
            )
            top_display = min(5, nb_navires_suspects)
            lignes.append(
                f"🧾 Détail des {top_display} navire(s) les plus prioritaires :"
            )
            for s in suspects[:top_display]:
                flags_txt = ", ".join(s["flags"])
                dist_txt = (
                    f"{s['distance_mean_km']:.1f} km"
                    if s["distance_mean_km"] is not None else "distance inconnue"
                )
                band_txt = s["dominant_band"].replace("_", " ")
                lignes.append(
                    f"\n- {s['name']} (MMSI {s['mmsi']}, {s['ship_type']}, {s['country']})\n"
                    f"  • Priorité : **{s['priority']}** (score {s['priority_score']})\n"
                    f"  • Événements RF : {s['nb_rf_events']} (dont {s['nb_high_conf_events']} de haute confiance, "
                    f"≈{s['events_per_hour']:.1f} évènement(s)/heure)\n"
                    f"  • Bande RF dominante : {band_txt}\n"
                    f"  • Bande RF : ~{s['freq_mean_mhz']:.1f} MHz "
                    f"(min {s['freq_min_mhz']:.1f}, max {s['freq_max_mhz']:.1f}, σ≈{s['freq_std_mhz']:.1f} MHz)\n"
                    f"  • Position moyenne : {dist_txt} du centre de zone\n"
                    f"  • Motifs de suspicion : {flags_txt}\n"
                )

            lignes.append(
                "\n💡 Recommandation :\n"
                "- Inspecter en priorité les navires marqués **HAUTE priorité**, en particulier ceux "
                "présentant à la fois des bandes RF atypiques pour des navires civils, une activité RF intense "
                "et une absence d’AIS récent (« dark-like »).\n"
                "- Surveiller également les contacts RF anonymes (sans MMSI) s’ils se concentrent à proximité "
                "d’infrastructures critiques ou de zones d’intérêt.\n"
            )

        narrative = "\n".join(lignes)

        # -------------------------
        # 8) Résultat structuré
        # -------------------------
        result = {
            "status": "ok",
            "zone": zone,
            "center": {"lat": lat_c, "lon": lon_c, "rayon_km": rayon_km},
            "time_window": {"start": t_min, "end": t_max},
            "nb_signaux_rf_zone": int(len(df_corr_zone)),
            "nb_navires_corrélés": int(nb_navires_corr),
            "nb_navires_suspects": int(nb_navires_suspects),
            "suspects": suspects,
            "nb_contacts_rf_sans_mmsi": nb_contacts_rf_sans_mmsi,
            "narrative": narrative,
        }
        return result

    except Exception as e:
        return {
            "status": "error",
            "message": f"Erreur dans analyser_signatures_rf_non_standard : {e}"
        }

#### Empreintes RF (version finale) et registre des tools

Dernière version de la base d'empreintes RF, tool d'exemple `rechercher_navire` fourni dans le starter pack, et registre final des tools exposés à l'agent.

`tool_build_rf_fingerprint_db`, `rechercher_navire`

In [ ]:
@tool
def tool_build_rf_fingerprint_db(
    min_confidence: float = 0.7,
    min_events_per_ship: int = 3,
    excel_path: str = "rf_fingerprints_db.xlsx",
) -> Dict[str, Any]:
    """
    Construit une base d'empreintes RF quasi-uniques par navire (MMSI)
    en utilisant UNIQUEMENT la base corrélée rf_correlation.parquet (df_corr_global),
    puis l'exporte dans un fichier Excel.

    Colonnes utilisées (df_corr_global) :
    - timestamp_utc (datetime64[ns, UTC])
    - RF_Frequency_MHz (float64)
    - latitude_deg, longitude_deg (float64)
    - cand_1_name (object)
    - cand_1_mmsi (object)
    - cand_1_confidence (object → converti en float)
    - number_of_candidates (int64)

    L'empreinte par navire inclut :
    - stats sur la fréquence RF (moyenne, écart-type, min, max)
    - nombre d'événements RF corrélés
    - moyenne de confiance
    - nombre moyen de candidats RF/AIS (unicité du pattern)

    Exporte :
    - un fichier Excel `excel_path` contenant la table des empreintes

    Retourne :
    - status, message
    - fingerprints_df : DataFrame par MMSI avec les caractéristiques RF
    - excel_path : chemin du fichier Excel créé (ou None si échec)
    """

    # 1) Vérification des données globales
    if 'df_corr_global' not in globals() or df_corr_global is None or df_corr_global.empty:
        return {
            "status": "error",
            "message": "Les données de corrélation RF/AIS (df_corr_global) ne sont pas chargées."
        }

    df_corr = df_corr_global.copy()

    # 2) Normalisation des types
    if not pd.api.types.is_datetime64_any_dtype(df_corr['timestamp_utc']):
        df_corr['timestamp_utc'] = pd.to_datetime(df_corr['timestamp_utc'], utc=True)

    df_corr['RF_Frequency_MHz'] = pd.to_numeric(df_corr['RF_Frequency_MHz'], errors='coerce')

    if 'cand_1_confidence' in df_corr.columns:
        df_corr['cand_1_confidence'] = pd.to_numeric(
            df_corr['cand_1_confidence'], errors='coerce'
        ).fillna(0.0)
    else:
        df_corr['cand_1_confidence'] = 0.0

    # 3) Filtrer sur les corrélations de bonne confiance
    df_corr_strong = df_corr[df_corr['cand_1_confidence'] >= min_confidence].copy()
    if df_corr_strong.empty:
        return {
            "status": "ok",
            "message": (
                f"Aucune corrélation RF/AIS avec confiance ≥ {min_confidence}. "
                "Impossible de construire des empreintes fiables."
            )
        }

    # 4) Nettoyage MMSI
    if 'cand_1_mmsi' not in df_corr_strong.columns:
        return {
            "status": "error",
            "message": "La colonne 'cand_1_mmsi' est absente dans df_corr_global."
        }

    df_corr_strong = df_corr_strong[df_corr_strong['cand_1_mmsi'].notna()].copy()
    df_corr_strong['cand_1_mmsi'] = df_corr_strong['cand_1_mmsi'].astype(str)

    df_corr_strong = df_corr_strong[
        df_corr_strong['cand_1_mmsi'].str.lower() != 'uncorrelated'
    ]

    if df_corr_strong.empty:
        return {
            "status": "ok",
            "message": (
                "Après nettoyage des MMSI, aucune corrélation exploitable n'est disponible "
                f"avec une confiance ≥ {min_confidence}."
            )
        }

    # 5) Construction des empreintes par MMSI
    fingerprints = []

    for mmsi, group in df_corr_strong.groupby('cand_1_mmsi'):
        if len(group) < min_events_per_ship:
            continue

        freqs = group['RF_Frequency_MHz'].dropna()
        confs = group['cand_1_confidence'].dropna()
        nb_candidates = group['number_of_candidates'] if 'number_of_candidates' in group.columns else pd.Series(dtype=float)

        lats = group['latitude_deg'].dropna()
        lons = group['longitude_deg'].dropna()

        fingerprint = {
            "mmsi": mmsi,
            "nb_events": int(len(group)),
            "freq_mean": float(freqs.mean()) if not freqs.empty else None,
            "freq_std": float(freqs.std()) if not freqs.empty else None,
            "freq_min": float(freqs.min()) if not freqs.empty else None,
            "freq_max": float(freqs.max()) if not freqs.empty else None,
            "mean_confidence": float(confs.mean()) if not confs.empty else None,
            "mean_nb_candidates": float(nb_candidates.mean()) if not nb_candidates.empty else None,
            "lat_mean": float(lats.mean()) if not lats.empty else None,
            "lon_mean": float(lons.mean()) if not lons.empty else None,
            "lat_span": float(lats.max() - lats.min()) if len(lats) > 1 else 0.0,
            "lon_span": float(lons.max() - lons.min()) if len(lons) > 1 else 0.0,
            "waveform_mode": None,
            "prf_mean": None,
            "pulse_duration_mean": None,
        }

        fingerprints.append(fingerprint)

    if not fingerprints:
        return {
            "status": "ok",
            "message": (
                "Aucun navire n'a suffisamment d'événements RF corrélés "
                f"(min_events_per_ship = {min_events_per_ship}). "
                "Base d'empreintes non construite."
            ),
            "matched_ratio_waveform": 0.0,
            "excel_path": None,
        }

    fingerprints_df = pd.DataFrame(fingerprints)

    # 6) Export Excel
    excel_file_path = excel_path
    excel_msg = ""
    try:
        fingerprints_df.to_excel(excel_file_path, index=False)
        excel_msg = f"\n💾 La base d'empreintes RF a été exportée dans le fichier : `{excel_file_path}`."
    except Exception as e:
        excel_file_path = None
        excel_msg = f"\n⚠️ Échec de l'export Excel : {e}"

    # 7) Résumé textuel
    n_ships = len(fingerprints_df)
    msg = (
        f"👨‍✈️ Commandant,\n\n"
        f"La base d'empreintes RF a été construite à partir du fichier de corrélation RF/AIS "
        f"(rf_correlation.parquet), en ne retenant que les associations de confiance ≥ {min_confidence}.\n\n"
        f"📡 Nombre de navires avec empreinte RF : {n_ships}\n"
        f"📡 Nombre moyen de détections RF corrélées par navire : "
        f"{fingerprints_df['nb_events'].mean():.1f}\n"
    )

    top = fingerprints_df.sort_values('nb_events', ascending=False).head(5)
    if not top.empty:
        msg += "\nNavires les plus caractérisés (par nombre de détections RF corrélées) :\n"
        for _, row in top.iterrows():
            freq_mean = row['freq_mean']
            freq_std = row['freq_std']
            if freq_mean is not None and freq_std is not None:
                msg += (
                    f" - MMSI {row['mmsi']} : {row['nb_events']} détections RF, "
                    f"fréquence ~{freq_mean:.1f} MHz (±{freq_std:.1f} MHz), "
                    f"confiance moyenne ~{row['mean_confidence']:.2f}\n"
                )
            else:
                msg += (
                    f" - MMSI {row['mmsi']} : {row['nb_events']} détections RF, "
                    f"confiance moyenne ~{row['mean_confidence']:.2f}\n"
                )

    msg += excel_msg

    result = {
        "status": "ok",
        "message": msg,
        "fingerprints_df": fingerprints_df,
        "matched_ratio_waveform": 0.0,
        "excel_path": excel_file_path,
    }
    return result

@tool
def rechercher_navire(mmsi: str) -> str:
    """
    Récupère la fiche synthétique d'un navire à partir de son MMSI.

    Ce tool :
    - vérifie que les données AIS sont chargées,
    - extrait la dernière position et les caractéristiques principales du MMSI demandé,
    - formate la réponse dans un bloc texte destiné à l'amiral/utilisateur.

    Args:
        mmsi (str): Identifiant MMSI du navire (chaîne ou nombre converti en texte).

    Returns:
        str: Rapport court contenant nom, type, pavillon, position, vitesse et cap.
    """
    try:
        # Vérifier que les données AIS sont disponibles
        if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
            return "❌ Aucune donnée AIS disponible"

        # Convertir le MMSI en string pour la recherche
        mmsi_str = str(mmsi).strip()

        # Rechercher le navire dans les données AIS
        vessel_row = df_ais_global[df_ais_global['Mmsi'].astype(str) == mmsi_str]

        if vessel_row.empty:
            return f"❌ Aucun navire trouvé avec le MMSI {mmsi_str}"

        # Prendre la dernière position du navire
        vessel_row = vessel_row.sort_values('datetime').tail(1)
        row = vessel_row.iloc[0]

        # Extraire les informations du navire
        vessel_name = row.get('Name', 'Sans nom')
        vessel_type = row.get('ShipTypeNormalized', 'Non déclaré')
        vessel_country = row.get('Country', 'Non identifié')
        lat = row.get('Position_LatitudeDeg', 'N/A')
        lon = row.get('Position_LongitudeDeg', 'N/A')
        speed = row.get('SogKt', 'N/A')
        course = row.get('CogDeg', 'N/A')
        last_update = row.get('datetime', 'N/A')

        # Formater la réponse
        result = f"""📍 **Navire trouvé**
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🚢 **Nom:** {vessel_name}
🔢 **MMSI:** {mmsi_str}
🏷️ **Type:** {vessel_type}
🏳️ **Pavillon:** {vessel_country}
📍 **Position:** {lat}°N, {lon}°E
⚡ **Vitesse:** {speed} nœuds
🧭 **Cap:** {course}°
🕐 **Dernière position:** {last_update}"""

        return result

    except Exception as e:
        return f"❌ Erreur lors de la recherche: {str(e)}"

# === LISTE DES TOOLS POUR L'AGENT ===
# Liste complète des 15 tools disponibles pour l'agent IA maritime
tools = [
    rechercher_navire,
    rechercher_position_navire,
    analyser_presence_strategique,
    rechercher_navires_zone_combinee,
    rechercher_image_satellite_navire,
    tool_dark_vessel_behavior,
    estimate_suspect_destination_ais_rf,
    tool_build_rf_fingerprint_db,
    tool_analyse_navigation_normale,
    analyser_infrastructure_critique,
    planifier_interception_sur_zone,
    generer_rapport_pdf,

    analyser_comportement_avance,
    analyser_profil_vitesse,
    analyser_signatures_rf_non_standard,
    generer_rapport_anomalies_navigation,
    analyser_tendances_trafic,
    analyser_menaces_zone,
    investiguer_disparition_navire,
    enrichir_navire_osint,
    filtrer_navires_par_type_zone,
    analyser_signatures_rf_non_standard,
    analyser_et_cartographier_zones_blanches,
    predire_passages_satellites

    ]

print("✅ Tool d'exemple créé: rechercher_navire")
print(f"📦 {len(tools)} tool(s) disponible(s) pour l'agent")
print("\n💡 Liste complète des tools:")
for i, tool in enumerate(tools, 1):
    print(f"   {i}. {tool.name}")

**Configuration de l'agent LangChain simple - Fonctions _make_llm et _make_agent pour créer un agent avec outils via OpenRouter**

In [ ]:
# 🤖 AGENT MODERNE (LANGGRAPH)
!pip install langchain-openai
!pip install langchain
!pip install python-docx
!pip install reportlab
import os
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent


# 1. DÉFINITION DU PROMPT SYSTÈME (C'est ici que vous le spécialisez)
system_prompt = """Tu es un assistant maritime expert nommé "posAIdon".

TES RESPONSABILITÉS :
- Analyser les données AIS et répondre aux questions sur les navires.
- Utiliser SYSTÉMATIQUEMENT les tools si on te demande une info précise (position, destination, vitesse, cap, ETA, etc.).
- Si tu ne trouves pas l'information, le dire clairement sans inventer.

HIÉRARCHIE DE DÉCISION (PROCESSUS INTERNE) :
1. Lire et analyser la requête de l’utilisateur pour déterminer ce qui est effectivement demandé.
2. Décider si l’usage d’un tool est nécessaire.
   - Utilise un tool lorsqu’une donnée factuelle ou mesurable est demandée
     (ex : position, dernière mise à jour, vitesse, cap, destination, ETA, MMSI, IMO).
   - Ne PAS utiliser un tool si la question relève d’une explication générale, d’un contexte,
     d’une interprétation ou d’une analyse ne nécessitant pas de données en temps réel.
3. Si un tool est nécessaire, l’appeler immédiatement.
   - Si plusieurs tools sont nécessaires, les utiliser dans l’ordre logique permettant d’obtenir
     l’information cible.
4. Une fois les résultats obtenus, formuler une réponse claire en langage humain.

RÈGLES D’ERREURS ET DE SÉCURITÉ :

1. **Absence de données / Tool « not found »**
   - Si un tool renvoie « not found », « empty », « null », ou toute réponse indiquant que
     l’information n’existe pas ou n’a pas été trouvée, tu dois le signaler clairement
     à l’utilisateur.

2. **Interdiction d’inventer**
   - Tu ne dois JAMAIS inventer une position, une route, une vitesse, une destination,
     une ETA, un MMSI ou toute autre donnée.
   - Si l’information n’est pas accessible via un tool, tu t’abstiens totalement de
     fournir une valeur fictive.

3. **Messages d’erreurs standardisés**
   - Les réponses d’erreur doivent rester professionnelles, concises et conformes au style naval.
   - Elles doivent indiquer explicitement lorsqu’un tool n’a rien retourné.

GESTION DES DEMANDES AMBIGUËS :

1. **Interdiction de deviner**
   - Tu ne dois jamais sélectionner un navire au hasard.
   - Aucun guess basé sur un nom partiel, une zone géographique ou un contexte implicite.

2. **Clarification obligatoire avant tool**
   - Tant que les informations nécessaires à un appel tool fiable ne sont pas fournies,
     tu dois rester en mode clarification et ne déclencher aucun tool.

TON STYLE :
- Professionnel, concis et précis (style officier de marine).
- Utilise des emojis maritimes : ⚓, 🚢, 📍, 🌊.
- Commence tes réponses importantes par "👨‍✈️ Commandant,".
"""
checkpointer = MemorySaver()
# 2. LLM (Via OpenRouter)
llm = ChatOpenAI(
    model="x-ai/grok-4-fast",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.7
)



# 3. CRÉATION DE L'AGENT

agent_executor = create_react_agent(llm, tools,prompt=system_prompt, checkpointer=checkpointer)
globals()['agent'] = agent_executor
print("✅ Agent LangGraph créé avec succès !")

✅ Agent LangGraph créé avec succès !


**Fonction ask_naval_assistant - Création de l'AgentExecutor et exécution de l'agent IA maritime avec gestion des erreurs**

In [ ]:
# 🤖 FONCTION PRINCIPALE POUR INTERROGER L'AGENT IA
# Cette cellule contient la fonction ask_naval_assistant qui communique directement
# avec l'agent LangChain. Elle crée un AgentExecutor qui exécute l'agent avec les tools
# disponibles. L'agent peut appeler les tools pour obtenir des informations et
# construire une réponse à partir de la question de l'utilisateur.


from langchain_core.messages import BaseMessage
THREAD_ID = "naval-session-1"
def ask_naval_assistant(question: str, thread_id: str = THREAD_ID):
    """
    Interroge l'agent IA maritime .
    """
    try:
        # 1. Vérifications de base
        if 'agent' not in globals():
            return {"output": "❌ Agent non configuré. Exécutez la cellule de l'agent."}

        agent_executor = globals()['agent']
        print(f"🚀 Question: {question}")

        # 2. Exécution de l'agent
        # LangGraph attend une entrée sous forme de 'messages'
        # Note: On utilise invoke() standard
        config = {"configurable": {"thread_id": thread_id}}
        response = agent_executor.invoke({"messages": [("user", question)]}, config=config)

        print("✅ Réponse brute reçue")

        # 3. Extraction de la réponse
        output = ""


        if isinstance(response, dict) and "messages" in response:
            last_message = response["messages"][-1]
            # Vérifier si c'est bien une réponse de l'IA
            output = last_message.content


        elif isinstance(response, dict) and "output" in response:
            output = response["output"]


        elif isinstance(response, str):
            output = response

        # 4. Vérification du contenu final
        if not output:
            print("⚠️ Le contenu de la réponse est vide.")
            return {"output": "⚠️ L'agent a traité la demande mais n'a retourné aucun texte."}

        return {"output": output}

    except Exception as e:
        # Gestion des erreurs (inchangée mais propre)
        error_msg = str(e)
        print(f"❌ Erreur ask_naval_assistant: {error_msg}")

        # Feedback utilisateur amélioré
        if "400" in error_msg:
            return {"output": "❌ Erreur de format (400). Reformulez votre question."}

        import traceback
        traceback.print_exc()

        return {"output": f"❌ Erreur technique: {error_msg[:100]}..."}



**Handler de chat simple - Fonction handle_chat pour gérer l'envoi de questions à l'agent IA et afficher les réponses**

In [ ]:
# 💬 HANDLER DE CHAT POUR L'INTERFACE
# Cette cellule contient la fonction handle_chat qui est appelée quand l'utilisateur clique sur le bouton "Envoyer" dans l'interface de chat.
# Elle récupère la question de l'utilisateur, appelle ask_naval_assistant pour obtenir la réponse de l'agent IA,
# puis met à jour l'affichage et l'historique des discussions.

import datetime as dt

# handler de chat pour l'interface
def handle_chat(button):
    """
    Gère l'envoi d'une question au chatbot.

    Cette fonction est appelée quand l'utilisateur clique sur le bouton "Envoyer"
    dans l'interface de chat. Elle envoie la question à l'agent IA et affiche la réponse.

    Args:
        button: Le bouton qui a déclenché l'action (non utilisé mais requis par ipywidgets)
    """
    try:
        # Récupérer la question de l'utilisateur
        user_input = chat_input.value.strip()

        if not user_input:
            chat_output.value = "<div style='padding:10px; color:orange;'>⚠️ Veuillez saisir une question</div>"
            return

        # Afficher le message de l'utilisateur
        chat_output.value = f"""
        <div style='padding:12px; background:#f0f8ff; border-radius:8px; border-left:3px solid #2196f3;'>
            <strong style='color:#01579b;'>💭 Vous:</strong>
            <span style='color:#0277bd;'>{user_input}</span>
        </div>
        <div style='padding:12px; background:#fff9e6; border-radius:8px; margin-top:10px; border-left:3px solid #ffc107;'>
            <strong style='color:#e65100; font-weight:bold;'>🤖 Assistant:</strong>
            <span style='color:#e65100; font-weight:600;'>Réflexion en cours...</span>
        </div>
        """

        # initialiser l'historique global si necessaire
        if 'chat_history' not in globals():
            globals()['chat_history'] = []

        # Ajouter la question à l'historique
        globals()['chat_history'].append({
            'timestamp': dt.datetime.now().strftime('%H:%M:%S'),
            'user': user_input,
            'assistant': None
        })

        # appeler l'agent IA
        # L'agent reçoit la question et peut utiliser les tools pour répondre.
        try:
            response = ask_naval_assistant(user_input)

            if response is None:
                assistant_response = "❌ Erreur: Réponse None de l'assistant"
            elif isinstance(response, dict):
                assistant_response = response.get('output', '❌ Pas de output dans la réponse')
                if not assistant_response or assistant_response == "":
                    assistant_response = "⚠️ Réponse vide de l'assistant"
            else:
                assistant_response = str(response)

            print(f"✅ Réponse finale reçue")

        except Exception as agent_error:
            print(f"❌ Erreur agent: {agent_error}")
            import traceback
            traceback.print_exc()
            assistant_response = f"❌ Erreur: {str(agent_error)[:200]}"

        if globals()['chat_history']:
            globals()['chat_history'][-1]['assistant'] = assistant_response

        # maj de l'historique dans l'interface
        if 'history_output' in globals():
            history_html = ""
            for entry in globals()['chat_history']:
                history_html += f"""
                <div style='padding:12px; margin-bottom:10px; background:#f8f9fa; border-radius:8px; border-left:3px solid #2196f3;'>
                    <div style='font-size:11px; color:#666; margin-bottom:5px;'>{entry['timestamp']}</div>
                    <div style='padding:8px; background:#e3f2fd; border-radius:5px; margin-bottom:8px;'>
                        <strong style='color:#01579b;'>💭 Vous:</strong>
                        <span style='color:#0277bd;'>{entry['user']}</span>
                    </div>
                """
                if entry['assistant']:
                    history_html += f"""
                    <div style='padding:8px; background:#e8f5e9; border-radius:5px;'>
                        <strong style='color:#1b5e20;'>🤖 Assistant:</strong>
                        <div style='color:#2e7d32; white-space:pre-line; margin-top:5px;'>{entry['assistant']}</div>
                    </div>
                    """
                history_html += "</div>"

            if history_html:
                history_output.value = history_html
            else:
                history_output.value = """
                <div style='padding:14px; background:#f8f9fa; border-radius:8px; color:#495057;
                            font-size:13px; border: 1px solid #e9ecef;'>
                    Aucun échange pour l'instant.
                </div>
                """

        # Afficher la réponse formatée
        chat_output.value = f"""
        <div style='padding:15px; background:#f8f9fa; border-radius:8px; border-left:4px solid #667eea;
                    box-shadow:0 2px 8px rgba(0,0,0,0.08);'>
            <div style='font-weight:bold; margin-bottom:12px; color:#495057; display:flex; align-items:center; gap:8px;'>
                <span style='font-size:20px;'>🤖</span>
                <span>Assistant Maritime - {dt.datetime.now().strftime('%H:%M:%S')}</span>
            </div>
            <div style='padding:12px; background:#f0f8ff; border-radius:8px; border-left:3px solid #2196f3; margin-bottom:10px;'>
                <strong style='color:#01579b;'>💭 Vous:</strong>
                <span style='color:#0277bd;'>{user_input}</span>
            </div>
            <div style='padding:12px; background:#e8f5e9; border-radius:8px; border-left:3px solid #4caf50;'>
                <strong style='color:#1b5e20;'>🤖 Assistant:</strong>
                <div style='color:#2e7d32; white-space:pre-line; line-height:1.6; margin-top:8px;'>
                    {assistant_response}
                </div>
            </div>
        </div>
        """

        # Effacer le champ de saisie
        chat_input.value = ""

    except Exception as e:
        error_msg = f"❌ Erreur système: {str(e)}"
        chat_output.value = f"""
        <div style='padding:15px; background:#ffebee; border-radius:8px; border-left:4px solid #d32f2f;'>
            <div style='color:#c62828; font-weight:600;'>{error_msg}</div>
        </div>
        """
        print(f"Erreur handle_chat: {e}")
        import traceback
        traceback.print_exc()

# Configurer le handler du bouton de chat
if 'chat_button' in globals():
    chat_button.on_click(handle_chat)
    print("✅ Handler de chat configuré")

✅ Handler de chat configuré


**Gestion de la mémoire, historique du chat**

In [ ]:
def append_chat_history(user_text=None, ai_text=None):
    """Gère l'historique des messages de chat"""
    global chat_memory_lc

    try:
        if user_text is not None:
            chat_memory_lc.append({"role": "user", "content": user_text})
        if ai_text is not None:
            chat_memory_lc.append({"role": "assistant", "content": ai_text})

        # Limiter la taille de la mémoire
        if len(chat_memory_lc) > 24:  # 12 tours de conversation
            chat_memory_lc = chat_memory_lc[-24:]

    except Exception as e:
        print(f"⚠️ Erreur append_chat_history: {e}")

# Variables de chat
last_question, last_answer = (None, None)

**Classe MultiVesselTimelineMap - Visualisation simultanée des trajectoires de plusieurs navires avec couleurs distinctes et contrôle timeline**

In [ ]:
class MultiVesselTimelineMap:
    """Timeline pour afficher plusieurs bateaux simultanément avec des couleurs différentes"""

    def __init__(self):
        self.vessels_history = {}
        self.color_map = {}
        self.timeline_map = None

    def create_multi_vessel_timeline(self, mmsi_list, date, start_time='00:00', end_time='23:59'):
        """
        Crée une carte avec timeline pour plusieurs bateaux

        Args:
            mmsi_list: Liste des MMSI à afficher
            date: Date à afficher
            start_time: Heure de début
            end_time: Heure de fin

        Returns:
            Objet folium.Map
        """
        try:
            # Palette de couleurs
            color_palette = [
                '#ff0000',
                '#00ff00',
                '#0000ff',
                '#ffff00',
                '#ff00ff',
                '#00ffff',
                '#ff8800',
                '#8800ff',
                '#00ff88',
                '#ff0088'
            ]

            # Charger les données pour tous les bateaux
            self.vessels_history = {}
            self.color_map = {}
            all_lats = []
            all_lons = []

            print(f"🔄 Chargement des trajectoires pour {len(mmsi_list)} bateau(x)...")

            for i, mmsi in enumerate(mmsi_list):
                vessel_history = self.load_vessel_data(mmsi, date, start_time, end_time)

                if vessel_history:
                    self.vessels_history[str(mmsi)] = vessel_history
                    self.color_map[str(mmsi)] = color_palette[i % len(color_palette)]

                    # Collecter toutes les coordonnées pour centrer la carte
                    for pos in vessel_history:
                        all_lats.append(pos['lat'])
                        all_lons.append(pos['lon'])

                    print(f"  ✅ {len(vessel_history)} positions chargées pour MMSI {mmsi}")
                else:
                    print(f"  ⚠️ Aucune donnée pour MMSI {mmsi}")

            if not self.vessels_history:
                print("❌ Aucune donnée de trajectoire disponible")
                return None

            # Calculer le centre et le zoom
            if all_lats and all_lons:
                center_lat = sum(all_lats) / len(all_lats)
                center_lon = sum(all_lons) / len(all_lons)
                lat_range = max(all_lats) - min(all_lats)
                lon_range = max(all_lons) - min(all_lons)
                max_range = max(lat_range, lon_range)

                if max_range < 0.01:
                    zoom = 13
                elif max_range < 0.1:
                    zoom = 10
                elif max_range < 1:
                    zoom = 8
                else:
                    zoom = 6
            else:
                center_lat, center_lon = (38.5, 15.0)
                zoom = 5

            # Créer la carte
            m = folium.Map(
                location=[center_lat, center_lon],
                zoom_start=zoom,
                tiles='CartoDB dark_matter',
                prefer_canvas=True
            )

            # Ajouter les couches de données
            add_maritime_layers(m)

            # Ajouter les trajectoires pour chaque bateau
            vessel_names = {}
            for mmsi, history in self.vessels_history.items():
                color = self.color_map[mmsi]
                trace_coords = [[p['lat'], p['lon']] for p in history]

                # Obtenir le nom du bateau
                if 'df_ais_global' in globals() and df_ais_global is not None:
                    vessel_row = df_ais_global[df_ais_global['Mmsi'].astype(str) == mmsi]
                    if not vessel_row.empty:
                        vessel_name = vessel_row.iloc[0]['Name'] or f'MMSI {mmsi}'
                    else:
                        vessel_name = f'MMSI {mmsi}'
                else:
                    vessel_name = f'MMSI {mmsi}'

                vessel_names[mmsi] = vessel_name

                # Tracer la trajectoire
                if len(trace_coords) > 1:
                    folium.PolyLine(
                        trace_coords,
                        color=color,
                        weight=3,
                        opacity=0.8,
                        popup=f'Trajectoire de {vessel_name} ({len(trace_coords)} points)'
                    ).add_to(m)

                # Marqueurs de début
                if trace_coords:
                    folium.CircleMarker(
                        location=trace_coords[0],
                        radius=8,
                        popup=f'🏁 Départ {vessel_name}',
                        color=color,
                        fillColor=color,
                        fillOpacity=0.9
                    ).add_to(m)

                    # Marqueur de fin
                    if len(trace_coords) > 1:
                        folium.CircleMarker(
                            location=trace_coords[-1],
                            radius=8,
                            popup=f'🏁 Arrivée {vessel_name}',
                            color=color,
                            fillColor=color,
                            fillOpacity=0.6
                        ).add_to(m)

                    # Marqueur de position actuelle (animé)
                    current_pos = trace_coords[0]
                    folium.CircleMarker(
                        location=current_pos,
                        radius=15,
                        popup=f'🚢 {vessel_name} (Position actuelle)',
                        color='#ffffff',
                        weight=3,
                        fillColor=color,
                        fillOpacity=1.0
                    ).add_to(m)

            # Ajout de la légende des couleurs
            legend_html = self._create_legend_html(vessel_names)
            m.get_root().html.add_child(folium.Element(legend_html))

            # Ajout du contrôle de la timeline
            timeline_html = self.create_multi_vessel_timeline_control()
            m.get_root().html.add_child(folium.Element(timeline_html))

            folium.LayerControl().add_to(m)

            self.timeline_map = m
            print(f"✅ Timeline multi-bateaux créée avec {len(self.vessels_history)} bateau(x)")
            return m

        except Exception as e:
            print(f"❌ Erreur création timeline multi-bateaux: {e}")
            import traceback
            traceback.print_exc()
            return None

    def _create_legend_html(self, vessel_names):
        """Crée une légende HTML pour les bateaux"""
        legend_items = []
        for mmsi, name in vessel_names.items():
            color = self.color_map[mmsi]
            legend_items.append(f"""
                <div style="margin: 5px 0;">
                    <span style="display: inline-block; width: 20px; height: 20px;
                          background: {color}; border-radius: 50%; margin-right: 8px;
                          vertical-align: middle;"></span>
                    <span style="color: white; vertical-align: middle;">{name}</span>
                </div>
            """)

        return f"""
        <div style="position: fixed;
                    top: 60px;
                    right: 10px;
                    background: rgba(0, 0, 0, 0.8);
                    padding: 15px;
                    border-radius: 8px;
                    z-index: 1000;
                    max-width: 250px;
                    border: 2px solid #00ff41;">
            <div style="color: #00ff41; font-weight: bold; margin-bottom: 10px;
                       font-family: 'Segoe UI', sans-serif;">
                🚢 Bateaux Tracés
            </div>
            {''.join(legend_items)}
        </div>
        """

    def create_multi_vessel_timeline_control(self):
        """Crée le contrôle timeline pour plusieurs bateaux"""
        # Préparation des données de tous les bateaux
        all_timeline_data = {}
        for mmsi, history in self.vessels_history.items():
            timeline_data = []
            for i, record in enumerate(history):
                timeline_data.append({
                    'index': i,
                    'timestamp': record['ts'],
                    'datetime': datetime.fromtimestamp(record['ts']).strftime('%Y-%m-%d %H:%M:%S'),
                    'lat': record['lat'],
                    'lon': record['lon'],
                    'speed': record.get('sog', 0),
                    'course': record.get('cog', 0)
                })
            all_timeline_data[mmsi] = timeline_data

        # Compter le nombre total de positions
        total_positions = sum(len(data) for data in all_timeline_data.values())

        timeline_html = f'''
        <style>
        .multi-timeline-container {{
            position: absolute;
            bottom: 20px;
            left: 20px;
            right: 20px;
            background: rgba(0, 0, 0, 0.9);
            border-radius: 10px;
            padding: 15px;
            z-index: 1000;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            border: 2px solid #00ff41;
            box-shadow: 0 0 20px rgba(0, 255, 65, 0.3);
        }}

        .multi-timeline-header {{
            color: #00ff41;
            font-size: 14px;
            font-weight: bold;
            margin-bottom: 10px;
            text-align: center;
            text-shadow: 0 0 10px rgba(0, 255, 65, 0.5);
        }}

        .timeline-info {{
            color: #ccc;
            font-size: 12px;
            margin-top: 10px;
        }}
        </style>

        <div class="multi-timeline-container">
            <div class="multi-timeline-header">
                🚢 Timeline Multi-Bateaux - {len(self.vessels_history)} bateau(x) - {total_positions} positions
            </div>
            <div class="timeline-info">
                ℹ️ Utilisez la souris pour naviguer sur la carte et voir les trajectoires complètes
            </div>
        </div>
        '''

        return timeline_html

    def load_vessel_data(self, mmsi, date, start_time, end_time):
        """Charge les données historiques du navire"""
        try:
            if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
                return []

            # Préparation de la fenêtre temporelle
            selected_date = pd.to_datetime(date).date() if isinstance(date, str) else date
            start_hour, start_min = map(int, start_time.split(':'))
            end_hour, end_min = map(int, end_time.split(':'))

            start_datetime = datetime.combine(selected_date, dt.time(start_hour, start_min))
            end_datetime = datetime.combine(selected_date, dt.time(end_hour, end_min))

            mmsi_str = str(mmsi)

            # Filtrage des données du navire
            vessel_data = df_ais_global[df_ais_global['Mmsi'].astype(str) == mmsi_str].copy()

            if vessel_data.empty:
                return []

            # Convertir datetime si nécessaire
            if not pd.api.types.is_datetime64_any_dtype(vessel_data['datetime']):
                vessel_data['datetime'] = pd.to_datetime(vessel_data['datetime'])

            # Filtrage par date et heure
            vessel_data = vessel_data[
                (vessel_data['datetime'] >= start_datetime) &
                (vessel_data['datetime'] <= end_datetime)
            ].sort_values('datetime')

            if vessel_data.empty:
                return []

            # Conversion au format attendu
            vessel_history = []
            for _, row in vessel_data.iterrows():
                try:
                    record = {
                        'ts': row['datetime'].timestamp(),
                        'lat': float(row['Position_LatitudeDeg']),
                        'lon': float(row['Position_LongitudeDeg']),
                        'sog': float(row.get('SogKt', 0)),
                        'cog': float(row.get('CogDeg', 0))
                    }
                    vessel_history.append(record)
                except Exception:
                    continue

            return vessel_history

        except Exception as e:
            print(f"❌ Erreur chargement données pour MMSI {mmsi}: {e}")
            return []

# Instance globale
multi_vessel_timeline = MultiVesselTimelineMap()

print("✅ Classe MultiVesselTimelineMap créée")


✅ Classe MultiVesselTimelineMap créée


# Construction de l'interface utilisateur interactive ( Map , Timeline , Filtres , Cache ... )

**Classe OptimizedMapCache - Système de cache multi-niveaux (DataFrames, HTML, métriques) avec TTL et gestion mémoire**

In [ ]:
import pickle
from functools import lru_cache

class OptimizedMapCache:
    """Cache multi-niveaux pour rendus instantanés de carte"""

    def __init__(self, max_size=100):
        self.max_size = max_size

        # 🎯 Niveau 1: Cache des DataFrames filtrés (ultra-rapide)
        self.df_cache = {}  # Clé: hash des filtres → DataFrame filtré

        # 🗺️ Niveau 2: Cache des cartes HTML complètes (instantané)
        self.html_cache = {}  # Clé: hash complet → HTML de la carte

        # 📊 Niveau 3: Cache des métriques pré-calculées
        self.metrics_cache = {}  # Clé: MMSI/date → métriques

        # 📈 Statistiques
        self.stats = {
            'df_hits': 0,
            'df_misses': 0,
            'html_hits': 0,
            'html_misses': 0,
            'total_saves': 0
        }

        # Timestamps pour invalidation intelligente
        self.timestamps = {}
        self.ttl_seconds = 3600  # 1 heure par défaut

    def _generate_filter_key(self, vessel_type='Tous', country='Tous', selected_date=None):
        """Génère une clé unique pour les filtres"""
        key_string = f"{vessel_type}_{country}_{selected_date}"
        return hashlib.md5(key_string.encode()).hexdigest()[:16]

    def _generate_map_key(self, df_hash, selected_mmsi=None, zoom=None):
        """Génère une clé unique pour une carte complète"""
        key_string = f"{df_hash}_{selected_mmsi}_{zoom}"
        return hashlib.md5(key_string.encode()).hexdigest()[:16]

    def _is_cache_valid(self, key):
        """Vérifie si le cache est encore valide"""
        if key not in self.timestamps:
            return False

        age = datetime.now() - self.timestamps[key]
        return age.total_seconds() < self.ttl_seconds

    # niveau 1 : Cache DataFrame
    def get_filtered_df(self, vessel_type='Tous', country='Tous', selected_date=None):
        """Récupère un DataFrame filtré depuis le cache"""
        key = self._generate_filter_key(vessel_type, country, selected_date)

        if key in self.df_cache and self._is_cache_valid(key):
            self.stats['df_hits'] += 1
            # print(f"⚡ DataFrame CACHE HIT - Instantané ! (Type: {vessel_type}, Pays: {country})")
            return self.df_cache[key].copy(), key

        self.stats['df_misses'] += 1
        return None, key

    def put_filtered_df(self, key, df):
        """Stocke un DataFrame filtré dans le cache"""
        # Limiter la taille du cache
        if len(self.df_cache) >= self.max_size:
            # Supprimer les entrées les plus anciennes
            oldest_key = min(self.timestamps.keys(), key=lambda k: self.timestamps[k])
            if oldest_key in self.df_cache:
                del self.df_cache[oldest_key]
                del self.timestamps[oldest_key]

        self.df_cache[key] = df.copy()
        self.timestamps[key] = datetime.now()
        self.stats['total_saves'] += 1
        # print(f"💾 DataFrame mis en cache ({len(df)} lignes)")

    # niveau 2 : Cache HTML de carte
    def get_map_html(self, df_hash, selected_mmsi=None):
        """Récupère le HTML d'une carte depuis le cache"""
        key = self._generate_map_key(df_hash, selected_mmsi)

        if key in self.html_cache and self._is_cache_valid(key):
            self.stats['html_hits'] += 1
            # print(f"🗺️ CARTE HTML CACHE HIT - Affichage INSTANTANÉ ! (MMSI: {selected_mmsi})")
            return self.html_cache[key]

        self.stats['html_misses'] += 1
        return None

    def put_map_html(self, df_hash, html_content, selected_mmsi=None):
        """Stocke le HTML d'une carte dans le cache"""
        key = self._generate_map_key(df_hash, selected_mmsi)

        # Limiter la taille du cache HTML (plus lourd)
        if len(self.html_cache) >= self.max_size // 2:
            oldest_key = min(
                [k for k in self.timestamps.keys() if k in self.html_cache],
                key=lambda k: self.timestamps[k],
                default=None
            )
            if oldest_key and oldest_key in self.html_cache:
                del self.html_cache[oldest_key]

        self.html_cache[key] = html_content
        self.timestamps[key] = datetime.now()
        # print(f"💾 Carte HTML mise en cache")

    # niveau 3: Cache métriques
    @lru_cache(maxsize=500)
    def get_vessel_metrics(self, mmsi_tuple):
        """Cache LRU pour les métriques de navires (ultra-rapide)"""
        # Cette méthode utilise le cache LRU natif de Python
        return None

    def clear_all(self):
        """Vide tous les caches"""
        self.df_cache.clear()
        self.html_cache.clear()
        self.timestamps.clear()
        self.get_vessel_metrics.cache_clear()
        print("🗑️ Tous les caches vidés")

    def clear_old_entries(self):
        """Supprime les entrées expirées"""
        now = datetime.now()
        expired_keys = [
            key for key, timestamp in self.timestamps.items()
            if (now - timestamp).total_seconds() > self.ttl_seconds
        ]

        for key in expired_keys:
            if key in self.df_cache:
                del self.df_cache[key]
            if key in self.html_cache:
                del self.html_cache[key]
            del self.timestamps[key]

        if expired_keys:
            print(f"🧹 {len(expired_keys)} entrées expirées supprimées")

    def get_stats(self):
        """Retourne les statistiques du cache"""
        total_df = self.stats['df_hits'] + self.stats['df_misses']
        total_html = self.stats['html_hits'] + self.stats['html_misses']

        df_hit_rate = (self.stats['df_hits'] / total_df * 100) if total_df > 0 else 0
        html_hit_rate = (self.stats['html_hits'] / total_html * 100) if total_html > 0 else 0

        return {
            'df_cache_size': len(self.df_cache),
            'html_cache_size': len(self.html_cache),
            'df_hit_rate': round(df_hit_rate, 2),
            'html_hit_rate': round(html_hit_rate, 2),
            'total_saves': self.stats['total_saves'],
            'memory_usage_mb': round(self._estimate_memory_usage() / 1024 / 1024, 2)
        }

    def _estimate_memory_usage(self):
        """Estime l'utilisation mémoire du cache"""
        total_bytes = 0

        # DataFrame cache
        for df in self.df_cache.values():
            total_bytes += df.memory_usage(deep=True).sum()

        # HTML cache (estimation)
        for html in self.html_cache.values():
            total_bytes += len(html.encode('utf-8'))

        return total_bytes

# Instance globale du cache optimisé
optimized_map_cache = OptimizedMapCache(max_size=100)

cache_clear_btn = widgets.Button(
    description='🗑️ Vider le Cache',
    button_style='warning',
    layout=widgets.Layout(width='150px')
)

def clear_cache_click(button):
    """Vide complètement le cache"""
    try:
        optimized_map_cache.clear_all()
        cache_clear_btn.button_style = 'success'
        cache_clear_btn.description = '✅ Cache Vidé'

        # Réinitialiser le bouton après 2 secondes
        import threading
        def reset_button():
            time.sleep(2)
            cache_clear_btn.button_style = 'warning'
            cache_clear_btn.description = '🗑️ Vider le Cache'

        threading.Thread(target=reset_button, daemon=True).start()
    except Exception as e:
        print(f"❌ Erreur lors du vidage du cache: {e}")

cache_clear_btn.on_click(clear_cache_click)

**Classe AnimatedVesselTracker et interface - Animation complète des trajectoires avec TimestampedGeoJson, marqueurs date/heure, et contrôles timeline interactifs**

In [ ]:
class AnimatedVesselTracker:
    """Système d'animation complète pour UN ou PLUSIEURS navires avec timeline player"""

    def __init__(self, cache=None):
        self.cache = cache or optimized_map_cache
        self.vessels_history = {}  # Dict: mmsi -> history
        self.animation_map = None
        self.color_palette = [
            '#ff0000',
            '#00ff00',
            '#0000ff',
            '#ffff00',
            '#ff00ff',
            '#00ffff',
            '#ff8800',
            '#8800ff',
            '#00ff88',
            '#ff0088'
        ]

    def create_animated_map(self, mmsi_list):
        """
        Crée une carte animée pour un ou plusieurs navires avec timeline player

        Args:
            mmsi_list: str (single MMSI) or list of str (multiple MMSIs)

        Returns:
            folium.Map with animation
        """
        try:
            # Convertir en liste si c'est une chaîne unique
            if isinstance(mmsi_list, str):
                mmsi_list = [mmsi_list]
            elif not isinstance(mmsi_list, list):
                mmsi_list = [str(mmsi_list)]

            print(f"🎬 Création animation pour {len(mmsi_list)} bateau(x)")

            # Charger l'historique de chaque navire
            all_lats = []
            all_lons = []
            all_dates = []
            self.vessels_history = {}

            for idx, mmsi in enumerate(mmsi_list):
                mmsi_str = str(mmsi)
                print(f"   📡 Chargement navire {idx+1}/{len(mmsi_list)}: MMSI {mmsi_str}")

                vessel_history = self._load_full_history(mmsi_str)

                if vessel_history:
                    self.vessels_history[mmsi_str] = {
                        'history': vessel_history,
                        'color': self.color_palette[idx % len(self.color_palette)],
                        'name': vessel_history[0].get('name', f'MMSI {mmsi_str}')
                    }

                    # Collecter coordonnées pour centrage
                    for pos in vessel_history:
                        all_lats.append(pos['lat'])
                        all_lons.append(pos['lon'])
                        all_dates.append(pos['datetime'])

                    print(f"      ✅ {len(vessel_history)} positions chargées")
                else:
                    print(f"      ⚠️ Aucune donnée pour MMSI {mmsi_str}")

            if not self.vessels_history:
                print("❌ Aucune donnée disponible")
                return None

            # Calculer centre et zoom
            center_lat = sum(all_lats) / len(all_lats)
            center_lon = sum(all_lons) / len(all_lons)
            lat_range = max(all_lats) - min(all_lats)
            lon_range = max(all_lons) - min(all_lons)
            max_range = max(lat_range, lon_range)

            if max_range < 0.01:
                zoom = 13
            elif max_range < 0.1:
                zoom = 10
            elif max_range < 1:
                zoom = 8
            else:
                zoom = 6

            # Créer la carte
            m = folium.Map(
                location=[center_lat, center_lon],
                zoom_start=zoom,
                tiles='CartoDB dark_matter',
                prefer_canvas=True
            )

            # Ajout des couches EMODnet
            add_maritime_layers(m)

            # ajout de tous les points de positions avec date/heure
            for mmsi_str, vessel_data in self.vessels_history.items():
                history = vessel_data['history']
                color = vessel_data['color']
                name = vessel_data['name']

                print(f"   📍 Ajout de {len(history)} marqueurs de position pour {name}")

                # Ajouter un marqueur pour chaque position
                for i, point in enumerate(history):
                    # Créer un petit cercle pour chaque position
                    folium.CircleMarker(
                        location=[point['lat'], point['lon']],
                        radius=3,
                        popup=folium.Popup(f"""
                            <div style='font-family:Arial; min-width:200px; color:#1a1a1a;'>
                                <div style='background:{color}; color:#ffffff; padding:8px; margin:-9px -9px 8px -9px;
                                           border-radius:5px 5px 0 0; font-weight:bold;'>
                                    {name} - Position {i+1}/{len(history)}
                                </div>
                                <table style='width:100%; font-size:12px; margin-top:5px;'>
                                    <tr><td><b>Position:</b></td><td>{i+1}/{len(history)}</td></tr>
                                    <tr><td><b>Date:</b></td><td>{point['datetime'].strftime('%d/%m/%Y')}</td></tr>
                                    <tr><td><b>Heure:</b></td><td>{point['datetime'].strftime('%H:%M:%S')}</td></tr>
                                    <tr><td><b>Coordonnées:</b></td><td>{point['lat']:.4f}°N<br>{point['lon']:.4f}°E</td></tr>
                                    <tr><td><b>Vitesse:</b></td><td>{point.get('sog', 0):.1f} kts</td></tr>
                                    <tr><td><b>Cap:</b></td><td>{point.get('cog', 0):.0f}°</td></tr>
                                </table>
                            </div>
                        """, max_width=250),
                        tooltip=f"{name} - {point['datetime'].strftime('%d/%m/%Y %H:%M')}",
                        color=color,
                        fillColor=color,
                        fillOpacity=0.6,
                        weight=1
                    ).add_to(m)

                # Tracé complet (ligne de trajectoire)
                trace_coords = [[p['lat'], p['lon']] for p in history]
                if len(trace_coords) > 1:
                    folium.PolyLine(
                        trace_coords,
                        color=color,
                        weight=3,
                        opacity=0.7,
                        popup=f'<b>{name}</b><br>{len(trace_coords)} positions',
                        tooltip=f'Trajectoire {name}'
                    ).add_to(m)

                # Marqueur départ
                folium.Marker(
                    location=[history[0]['lat'], history[0]['lon']],
                    popup=f"""
                        <div style='font-family:Arial; color:#1a1a1a; min-width:200px;'>
                            <div style='background:{color}; color:#ffffff; padding:8px; margin:-9px -9px 8px -9px;
                                       border-radius:5px 5px 0 0; font-weight:bold;'>
                                🏁 DÉPART - {name}
                            </div>
                            <table style='width:100%; font-size:12px; margin-top:5px;'>
                                <tr><td><b>Date:</b></td><td>{history[0]['datetime'].strftime('%d/%m/%Y %H:%M')}</td></tr>
                                <tr><td><b>Position:</b></td><td>{history[0]['lat']:.4f}°N<br>{history[0]['lon']:.4f}°E</td></tr>
                            </table>
                        </div>
                    """,
                    icon=folium.Icon(color='green', icon='play', prefix='fa')
                ).add_to(m)

                # Marqueur arrivée
                folium.Marker(
                    location=[history[-1]['lat'], history[-1]['lon']],
                    popup=f"""
                        <div style='font-family:Arial; color:#1a1a1a; min-width:200px;'>
                            <div style='background:{color}; color:#ffffff; padding:8px; margin:-9px -9px 8px -9px;
                                       border-radius:5px 5px 0 0; font-weight:bold;'>
                                🏁 ARRIVÉE - {name}
                            </div>
                            <table style='width:100%; font-size:12px; margin-top:5px;'>
                                <tr><td><b>Date:</b></td><td>{history[-1]['datetime'].strftime('%d/%m/%Y %H:%M')}</td></tr>
                                <tr><td><b>Position:</b></td><td>{history[-1]['lat']:.4f}°N<br>{history[-1]['lon']:.4f}°E</td></tr>
                            </table>
                        </div>
                    """,
                    icon=folium.Icon(color='red', icon='stop', prefix='fa')
                ).add_to(m)

            # ajouter le timeline player
            all_features = []
            for mmsi_str, vessel_data in self.vessels_history.items():
                history = vessel_data['history']
                color = vessel_data['color']
                name = vessel_data['name']

                for point in history:
                    feature = {
                        'type': 'Feature',
                        'geometry': {
                            'type': 'Point',
                            'coordinates': [point['lon'], point['lat']]
                        },
                        'properties': {
                            'time': point['datetime'].isoformat(),
                            'popup': f"{name}<br>{point['datetime'].strftime('%d/%m/%Y %H:%M')}",
                            'icon': 'circle',
                            'iconstyle': {
                                'fillColor': color,
                                'fillOpacity': 1.0,
                                'stroke': 'true',
                                'radius': 8,
                                'weight': 2,
                                'color': '#ffffff'
                            }
                        }
                    }
                    all_features.append(feature)

            # Ajouter le plugin TimestampedGeoJson pour le player
            plugins.TimestampedGeoJson(
                {
                    'type': 'FeatureCollection',
                    'features': all_features
                },
                period='PT1M',  # 1 minute entre les points
                duration='PT30S',  # Afficher pendant 30 secondes
                add_last_point=True,
                auto_play=False,
                loop=True,
                max_speed=20,
                loop_button=True,
                date_options='DD/MM/YYYY HH:mm',
                time_slider_drag_update=True,
            ).add_to(m)

            first_date = min(all_dates)
            last_date = max(all_dates)
            total_days = (last_date - first_date).days + 1
            total_positions = sum(len(v['history']) for v in self.vessels_history.values())

            # Limiter la liste des navires affichés (max 3 pour économiser l'espace)
            vessel_list_html = ''
            vessel_count = 0
            for mmsi_str, vessel_data in self.vessels_history.items():
                if vessel_count >= 3:
                    remaining = len(self.vessels_history) - vessel_count
                    vessel_list_html += f'<span style="color:#666; font-size:11px;">... et {remaining} autre(s)</span><br>'
                    break
                vessel_list_html += f'''
                    <span style="display:inline-block; width:10px; height:10px; background:{vessel_data['color']};
                                 border-radius:50%; margin-right:5px; border:1px solid #333;"></span>
                    <span style="font-size:11px;">{vessel_data['name']}</span><br>
                '''
                vessel_count += 1

            title_html = f'''
            <div style="position: fixed; top: 10px; left: 50px; width: 380px; max-height: 280px;
                        background-color: rgba(255, 255, 255, 0.95); border:2px solid #00ff41;
                        border-radius: 8px; z-index:9999; padding: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.2);
                        font-family: 'Segoe UI', Arial, sans-serif; overflow-y: auto; overflow-x: hidden;">
                <h4 style="margin:0 0 6px 0; color: #1a1a1a; font-size:14px; font-weight:bold;">
                    🎬 ANIMATION ITINÉRAIRE COMPLET
                </h4>
                <div style="color:#2c2c2c; font-size:12px; line-height:1.5;">
                    <div style="margin-bottom:4px;">
                        <b>Navires:</b> {len(self.vessels_history)}<br>
                        {vessel_list_html}
                    </div>
                    <div style="margin-bottom:4px; font-size:11px;">
                        <b>Période:</b> {first_date.strftime('%d/%m/%Y')} → {last_date.strftime('%d/%m/%Y')}<br>
                        <b>Durée:</b> {total_days} jour(s)<br>
                        <b>Positions:</b> {total_positions}<br>
                        <b>Distance:</b> {self._calculate_total_distance():.1f} NM
                    </div>
                    <div style="background:#e3f2fd; padding:5px; border-radius:4px; border-left:3px solid #1976d2; font-size:10px; margin-top:4px;">
                        <b>🎮 Contrôles Timeline:</b><br>
                        • <b>▶️ Play</b> pour démarrer<br>
                        • <b>⏸️ Pause</b> pour arrêter<br>
                        • <b>🔄 Loop</b> pour répéter<br>
                        • Ajustez la vitesse<br>
                        • Cliquez sur les points
                    </div>
                </div>
            </div>
            '''
            m.get_root().html.add_child(folium.Element(title_html))

            # Ajout du contrôle des couches
            folium.LayerControl(position='topright').add_to(m)

            self.animation_map = m
            print(f"✅ Animation créée: {len(self.vessels_history)} navire(s), {total_positions} positions")
            print(f"🎮 Timeline player activé avec contrôles interactifs")
            return m

        except Exception as e:
            print(f"❌ Erreur création animation: {e}")
            import traceback
            traceback.print_exc()
            return None

    def _load_full_history(self, mmsi):
        """Charge TOUT l'historique d'un navire"""
        try:
            if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
                return []

            mmsi_str = str(mmsi)
            vessel_data = df_ais_global[df_ais_global['Mmsi'].astype(str) == mmsi_str].copy()

            if vessel_data.empty:
                return []

            # Convertir datetime
            if not pd.api.types.is_datetime64_any_dtype(vessel_data['datetime']):
                vessel_data['datetime'] = pd.to_datetime(vessel_data['datetime'])

            # Trier et filtrer
            vessel_data = vessel_data.sort_values('datetime')
            vessel_data = vessel_data[
                (vessel_data['Position_LatitudeDeg'].notna()) &
                (vessel_data['Position_LongitudeDeg'].notna()) &
                (vessel_data['Position_LatitudeDeg'] != 0) &
                (vessel_data['Position_LongitudeDeg'] != 0)
            ]

            # récupérer le nom du navire
            vessel_name = None
            for _, row in vessel_data.iterrows():
                name = str(row.get('Name', '')).strip()
                if name and name.lower() not in ['unknown', 'nan', 'none', '']:
                    vessel_name = name
                    break

            if not vessel_name:
                last_name = str(vessel_data.iloc[-1].get('Name', '')).strip() if not vessel_data.empty else ''
                if last_name and last_name.lower() not in ['unknown', 'nan', 'none', '']:
                    vessel_name = last_name
                else:
                    vessel_name = f'MMSI {mmsi_str}'


            # Convertir au format requis
            history = []
            for _, row in vessel_data.iterrows():
                try:
                    history.append({
                        'datetime': row['datetime'].to_pydatetime(),
                        'lat': float(row['Position_LatitudeDeg']),
                        'lon': float(row['Position_LongitudeDeg']),
                        'sog': float(row.get('SogKt', 0)) if pd.notna(row.get('SogKt')) else 0,
                        'cog': float(row.get('CogDegrees', 0)) if pd.notna(row.get('CogDegrees')) else 0,
                        'name': vessel_name  # Utiliser le nom corrigé pour toutes les positions
                    })
                except:
                    continue

            return history

        except Exception as e:
            print(f"⚠️ Erreur chargement historique MMSI {mmsi}: {e}")
            return []


    def _calculate_total_distance(self):
        """Calcule la distance totale pour tous les navires"""
        from math import radians, sin, cos, sqrt, atan2

        total_distance = 0
        for vessel_data in self.vessels_history.values():
            history = vessel_data['history']
            for i in range(1, len(history)):
                lat1 = radians(history[i-1]['lat'])
                lon1 = radians(history[i-1]['lon'])
                lat2 = radians(history[i]['lat'])
                lon2 = radians(history[i]['lon'])

                dlat = lat2 - lat1
                dlon = lon2 - lon1
                a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
                c = 2 * atan2(sqrt(a), sqrt(1-a))

                distance_nm = (6371000 * c) / 1852.0
                total_distance += distance_nm

        return total_distance

# Créeation de l'instance globale
animated_tracker = AnimatedVesselTracker()

# interface utilisateur

animation_status = widgets.HTML(
    value="""
    <div style='padding:12px; background:#e8f4f8; border-radius:8px; border-left:4px solid #0288d1; color:#01579b; font-weight:500;'>
        <strong>ℹ️ Animation Itinéraire Complet</strong><br>
        Sélectionnez un ou plusieurs bateaux puis cliquez sur le bouton
    </div>
    """,
    layout=widgets.Layout(width='100%')
)

animation_create_btn = widgets.Button(
    description='🎬 Créer Animation',
    button_style='success',
    tooltip='Crée l\'animation complète pour tous les bateaux sélectionnés',
    layout=widgets.Layout(width='200px', height='40px'),
    style={'button_color': '#4caf50', 'font_weight': 'bold'}
)

def exit_animation(button):
    """Quitte l'animation et revient à la carte principale"""
    try:
        global selected_mmsi_list, last_selected_mmsi

        selected_mmsi_to_keep = None
        if selected_mmsi_list:
            selected_mmsi_to_keep = selected_mmsi_list[0] if len(selected_mmsi_list) == 1 else None
        elif last_selected_mmsi:
            selected_mmsi_to_keep = last_selected_mmsi

        if 'refresh_ui_from_stream' in globals():
            selected_date = None
            vessel_type = 'Tous'
            country = 'Tous'

            if 'date_selector' in globals():
                selected_date = date_selector.value
            if 'filter_type_dropdown' in globals():
                vessel_type = filter_type_dropdown.value
            if 'filter_country_dropdown' in globals():
                country = filter_country_dropdown.value

            refresh_ui_from_stream(
                selected_mmsi=selected_mmsi_to_keep,
                selected_date=selected_date,
                vessel_type=vessel_type,
                country=country,
                force_refresh=False
            )
        else:
            # Fallback: réafficher la carte de base manuellement
            with map_output:
                clear_output(wait=True)
                if 'df_ais_global' in globals() and df_ais_global is not None and not df_ais_global.empty:
                    df_last_positions = get_latest_vessel_positions()
                    df_valid = df_last_positions[
                        (df_last_positions['Position_LatitudeDeg'].notna()) &
                        (df_last_positions['Position_LongitudeDeg'].notna()) &
                        (df_last_positions['Position_LatitudeDeg'] != 0) &
                        (df_last_positions['Position_LongitudeDeg'] != 0)
                    ].copy()

                    live_df = pd.DataFrame({
                        'mmsi': df_valid['Mmsi'].astype(str),
                        'boat_name': df_valid['Name'].fillna('Sans nom'),
                        'latitude': df_valid['Position_LatitudeDeg'],
                        'longitude': df_valid['Position_LongitudeDeg'],
                        'boat_type': df_valid['ShipTypeLabel'].fillna('Non déclaré'),
                        'flag_country': df_valid['Pays'].fillna('Inconnu')
                    })
                    display(create_boat_map(live_df, selected_mmsi=selected_mmsi_to_keep, selected_mmsi_list=selected_mmsi_list if selected_mmsi_list else None))

        animation_status.value = """
        <div style='padding:12px; background:#e8f5e9; border-radius:8px; border-left:4px solid #4caf50; color:#1b5e20; font-weight:500;'>
            ✅ <strong>Retour à la carte principale</strong><br>
            L'animation a été quittée, vous pouvez créer une nouvelle animation
        </div>
        """

        print("✅ Animation quittée - Retour à la carte principale")

    except Exception as e:
        animation_status.value = f"""
        <div style='padding:12px; background:#ffebee; border-radius:8px; border-left:4px solid #d32f2f; color:#c62828; font-weight:500;'>
            ❌ <strong>Erreur:</strong> {str(e)[:100]}
        </div>
        """
        print(f"Erreur quitter animation: {e}")
        import traceback
        traceback.print_exc()

animation_exit_btn = widgets.Button(
    description='🔙 Quitter Animation',
    button_style='warning',
    tooltip='Quitte l\'animation et revient à la carte principale',
    layout=widgets.Layout(width='200px', height='40px'),
    style={'button_color': '#ff9800', 'font_weight': 'bold'}
)

animation_exit_btn.on_click(exit_animation)

def create_full_animation(button):
    """Crée l'animation pour tous les bateaux sélectionnés"""
    global selected_mmsi_list

    # Utiliser selected_mmsi_list si disponible, sinon last_selected_mmsi
    if selected_mmsi_list:
        mmsi_to_animate = selected_mmsi_list
        print(f"🎬 Animation de {len(mmsi_to_animate)} bateau(x) sélectionné(s)")
    elif last_selected_mmsi:
        mmsi_to_animate = [last_selected_mmsi]
        print(f"🎬 Animation d'1 bateau sélectionné")
    else:
        animation_status.value = """
        <div style='padding:12px; background:#ffebee; border-radius:8px; border-left:4px solid #d32f2f; color:#c62828; font-weight:500;'>
            ⚠️ <strong>Aucun bateau sélectionné</strong><br>
            Sélectionnez un ou plusieurs bateaux depuis la liste ci-dessus
        </div>
        """
        return

    if len(mmsi_to_animate) > 10:
        animation_status.value = f"""
        <div style='padding:12px; background:#fff3e0; border-radius:8px; border-left:4px solid #f57c00; color:#e65100; font-weight:500;'>
            ⚠️ <strong>Trop de bateaux sélectionnés ({len(mmsi_to_animate)})</strong><br>
            Pour des performances optimales, sélectionnez maximum 10 bateaux
        </div>
        """
        return

    animation_status.value = """
    <div style='padding:12px; background:#e3f2fd; border-radius:8px; border-left:4px solid #1976d2; color:#0d47a1; font-weight:500;'>
        🔄 <strong>Chargement en cours...</strong><br>
        Création de l'animation, veuillez patienter
    </div>
    """

    try:
        start_time = time.time()

        # Créer l'animation
        animated_map = animated_tracker.create_animated_map(mmsi_to_animate)

        load_time = time.time() - start_time

        if animated_map:
            with map_output:
                clear_output(wait=True)
                display(animated_map)


            # Stats
            total_positions = sum(len(v['history']) for v in animated_tracker.vessels_history.values())
            vessel_names = [v['name'] for v in animated_tracker.vessels_history.values()]

            animation_status.value = f"""
            <div style='padding:12px; background:#e8f5e9; border-radius:8px; border-left:4px solid #4caf50; color:#1b5e20; font-weight:500;'>
                ✅ <strong>Animation créée avec succès !</strong><br>
                🚢 <strong>Navires:</strong> {len(mmsi_to_animate)} ({', '.join(vessel_names[:3])}{'...' if len(vessel_names) > 3 else ''})<br>
                📊 <strong>Positions totales:</strong> {total_positions} (avec marqueurs date/heure)<br>
                🗺️ <strong>Distance totale:</strong> {animated_tracker._calculate_total_distance():.1f} NM<br>
                ⏱️ <strong>Temps de chargement:</strong> {load_time:.2f}s<br>
                <br>
                <strong>🎮 Timeline Player:</strong><br>
                • Utilisez les contrôles en bas de la carte<br>
                • <b>▶️ Play</b> pour démarrer l'animation<br>
                • <b>⏸️ Pause</b> pour arrêter<br>
                • <b>🔄 Loop</b> pour répéter<br>
                • Ajustez la vitesse avec le curseur<br>
                • Cliquez sur les points pour voir date/heure<br>
                <br>
                <strong>🗺️ Couches disponibles:</strong><br>
                • Bathymétrie, câbles sous-marins, routes maritimes<br>
                • Température de surface, courants marins<br>
                • Utilisez le contrôle des couches (en haut à droite)
            </div>
            """
        else:
            animation_status.value = """
            <div style='padding:12px; background:#fff3e0; border-radius:8px; border-left:4px solid #f57c00; color:#e65100; font-weight:500;'>
                ⚠️ <strong>Aucune donnée trouvée</strong><br>
                Les bateaux sélectionnés n'ont pas d'historique disponible
            </div>
            """

    except Exception as e:
        animation_status.value = f"""
        <div style='padding:12px; background:#ffebee; border-radius:8px; border-left:4px solid #d32f2f; color:#c62828; font-weight:500;'>
            ❌ <strong>Erreur:</strong> {str(e)[:100]}<br>
            Veuillez réessayer ou sélectionner d'autres bateaux
        </div>
        """
        print(f"Erreur détaillée: {e}")
        import traceback
        traceback.print_exc()

animation_create_btn.on_click(create_full_animation)

# Interface complète
animated_timeline_interface = widgets.VBox([
    widgets.HTML("""
        <h4 style='margin:10px 0 5px 0; color:#00bfff; font-weight:600;'>
            🎬 Animation Itinéraire Complet
        </h4>
    """),
    widgets.HTML("""
        <p style='margin:5px 0; color:#00bfff; font-size:13px; line-height:1.5; font-weight:500;'>
            Visualisez l'itinéraire complet d'un ou plusieurs navires avec timeline player interactif.<br>
            <b>Fonctionnalités:</b> Points de position avec date/heure, trajectoires colorées, couches EMODnet, contrôles timeline
        </p>
    """),
    widgets.HBox([
        animation_create_btn,
        animation_exit_btn
    ], layout=widgets.Layout(justify_content='flex-start', align_items='center', margin='10px 0', gap='10px')),
    animation_status
], layout=widgets.Layout(
    border='2px solid #4caf50',
    border_radius='8px',
    padding='15px',
    margin='10px 0',
    background_color='#ffffff'
))

**Interface de filtrage interactive - Création des widgets ipywidgets (type, pays, date) et handlers (apply_vessel_filters, reset_vessel_filters, show_all_vessels, on_date_filter_apply)**

In [ ]:
from datetime import datetime, date

def get_filter_options_from_data():
    """Récupère les options de filtrage depuis df_ais_global"""
    types_set = {'Tous'}
    countries_set = {'Tous'}

    if 'df_ais_global' in globals() and df_ais_global is not None and not df_ais_global.empty:
        if 'ShipTypeLabel' in df_ais_global.columns:
            # Liste des types principaux valides (sans les variantes Hazardous, Other, etc.)
            valid_types = {
                'Fishing',  # Pêche
                'Towing',  # Remorquage
                'Dredging or underwater ops',  # Dragage
                'Diving ops',  # Plongée
                'Military ops',  # Militaire
                'Sailing',  # Voilier
                'Pleasure Craft',  # Plaisance
                'High speed craft (HSC)',  # Navire à grande vitesse
                'Pilot Vessel',  # Pilote
                'Search and Rescue vessel',  # Recherche et sauvetage
                'Tug',  # Remorqueur
                'Port Tender',  # Navire portuaire
                'Anti-pollution equipment',  # Anti-pollution
                'Law Enforcement',  # Application de la loi
                'Medical Transport',  # Transport médical
                'Passenger',  # Passager
                'Cargo',  # Cargo
                'Tanker',  # Pétrolier
                'Wing in ground (WIG)',  # Aéroglisseur
            }

            # Récupérer tous les types uniques
            all_types = df_ais_global['ShipTypeLabel'].dropna().unique()

            # Filtrer pour ne garder que les types principaux
            for ship_type in all_types:
                ship_type_str = str(ship_type).strip()
                # Vérifier si c'est un type principal (sans les suffixes Hazardous, Other, etc.)
                if ship_type_str in valid_types:
                    types_set.add(ship_type_str)
                # Ou si c'est un type qui commence par un type principal (ex: "Cargo" au lieu de "Cargo - Hazardous category A")
                else:
                    for valid_type in valid_types:
                        if ship_type_str.startswith(valid_type):
                            # Prendre seulement le type principal
                            types_set.add(valid_type)
                            break

        if 'Pays' in df_ais_global.columns:
            countries_set.update(df_ais_global['Pays'].dropna().unique())

    return {
        'types': sorted(list(types_set)),
        'countries': sorted(list(countries_set))
    }

# Widgets de filtrage
filter_options = get_filter_options_from_data()

filter_type_dropdown = widgets.Dropdown(
    options=filter_options['types'],
    value='Tous',
    description='⚓ Type:',
    style={
        'description_width': '70px',
        'description_color': '#667eea'
    },
    layout=widgets.Layout(
        width='200px',
        border='2px solid #667eea',
        border_radius='8px',
        box_shadow='0 2px 8px rgba(102, 126, 234, 0.12)'
    )
)

filter_country_dropdown = widgets.Dropdown(
    options=filter_options['countries'],
    value='Tous',
    description='🌍 Pays:',
    style={
        'description_width': '70px',
        'description_color': '#667eea'
    },
    layout=widgets.Layout(
        width='200px',
        border='2px solid #667eea',
        border_radius='8px',
        box_shadow='0 2px 8px rgba(102, 126, 234, 0.12)'
    )
)

date_selector = widgets.Dropdown(
    options=[('Toutes les dates', None)],
    value=None,
    description='📅 Date:',
    style={
        'description_width': '70px',
        'description_color': '#667eea'
    },
    layout=widgets.Layout(
        width='220px',
        border='2px solid #667eea',
        border_radius='8px',
        box_shadow='0 2px 8px rgba(102, 126, 234, 0.12)'
    )
)

# Boutons
filter_apply_btn = widgets.Button(
    description='✅ Appliquer',
    button_style='success',
    icon='check',
    layout=widgets.Layout(
        width='130px',
        height='35px',
        border_radius='8px',
        box_shadow='0 3px 10px rgba(40, 167, 69, 0.2)'
    ),
    style={'button_color': '#28a745', 'font_weight': 'bold'}
)

filter_reset_btn = widgets.Button(
    description='🔄 Reset',
    button_style='warning',
    icon='refresh',
    layout=widgets.Layout(
        width='110px',
        height='35px',
        border_radius='8px',
        box_shadow='0 3px 10px rgba(255, 193, 7, 0.2)'
    ),
    style={'button_color': '#ffc107', 'font_weight': 'bold'}
)

filter_show_all_btn = widgets.Button(
    description='🌍 Tous',
    button_style='info',
    icon='globe',
    layout=widgets.Layout(
        width='110px',
        height='35px',
        border_radius='8px',
        box_shadow='0 3px 10px rgba(23, 162, 184, 0.2)'
    ),
    style={'button_color': '#17a2b8', 'font_weight': 'bold'}
)

date_apply_btn = widgets.Button(
    description='📅 Filtrer',
    button_style='primary',
    icon='calendar',
    layout=widgets.Layout(
        width='120px',
        height='35px',
        border_radius='8px',
        box_shadow='0 3px 10px rgba(102, 126, 234, 0.2)'
    ),
    style={'button_color': '#667eea', 'font_weight': 'bold'}
)

# bouton toggle pour les éphémérides
ephemeris_toggle_btn = widgets.ToggleButton(
    value=False,
    description='🛰️ Satellites',
    button_style='info',
    icon='satellite',
    tooltip='Afficher/masquer les trajectoires satellites',
    layout=widgets.Layout(
        width='140px',
        height='35px',
        border_radius='8px',
        box_shadow='0 3px 10px rgba(23, 162, 184, 0.2)'
    ),
    style={'button_color': '#17a2b8', 'font_weight': 'bold'}
)

# Variable globale pour suivre l'état des éphémérides
show_ephemeris_global = False

def toggle_ephemeris(change):
    """Active/désactive l'affichage des trajectoires satellites"""
    global show_ephemeris_global

    show_ephemeris_global = change['new']

    # Message de chargement
    if change['new']:
        ephemeris_toggle_btn.description = '🛰️ Satellites ON'
        ephemeris_toggle_btn.button_style = 'success'
        ephemeris_toggle_btn.style.button_color = '#28a745'

        # Message de chargement visible
        filter_status.value = """
        <div style='padding:14px 18px; background: linear-gradient(135deg, #fff3cd 0%, #fffbf0 100%);
                    border-radius: 10px; border-left: 4px solid #ffc107;
                    box-shadow: 0 3px 10px rgba(255, 193, 7, 0.15); border: 1px solid #ffecb5;'>
            <div style='display: flex; align-items: center; gap: 10px; color: #664d03; font-weight: 600;'>
                <span style='font-size: 20px; animation: spin 2s linear infinite;'>⏳</span>
                <div style='flex: 1;'>
                    <div style='font-size: 14px; margin-bottom: 4px;'>
                        <strong>🛰️ Chargement des trajectoires satellites...</strong>
                    </div>
                    <div style='font-size: 12px; opacity: 0.9; line-height: 1.5;'>
                        ⏱️ <strong>Veuillez patienter</strong> - Le chargement peut prendre 30 à 60 secondes<br>
                        📊 Traitement de nombreuses données (navires + RF + satellites)<br>
                        💡 <em>Sur Google Colab, le rendu de grandes cartes peut être lent</em>
                    </div>
                </div>
            </div>
        </div>
        <style>
        @keyframes spin {
            from { transform: rotate(0deg); }
            to { transform: rotate(360deg); }
        }
        </style>
        """
        print("🛰️ Affichage des trajectoires satellites activé - Chargement en cours...")
    else:
        ephemeris_toggle_btn.description = '🛰️ Satellites'
        ephemeris_toggle_btn.button_style = 'info'
        ephemeris_toggle_btn.style.button_color = '#17a2b8'

        # Message de chargement pour la désactivation
        filter_status.value = """
        <div style='padding:14px 18px; background: linear-gradient(135deg, #fff3cd 0%, #fffbf0 100%);
                    border-radius: 10px; border-left: 4px solid #ffc107;
                    box-shadow: 0 3px 10px rgba(255, 193, 7, 0.15); border: 1px solid #ffecb5;'>
            <div style='display: flex; align-items: center; gap: 10px; color: #664d03; font-weight: 600;'>
                <span style='font-size: 20px; animation: spin 2s linear infinite;'>⏳</span>
                <div style='flex: 1;'>
                    <div style='font-size: 14px; margin-bottom: 4px;'>
                        <strong>🔄 Masquage des trajectoires satellites...</strong>
                    </div>
                    <div style='font-size: 12px; opacity: 0.9;'>
                        ⏱️ <strong>Veuillez patienter</strong> - Rafraîchissement de la carte en cours
                    </div>
                </div>
            </div>
        </div>
        <style>
        @keyframes spin {
            from { transform: rotate(0deg); }
            to { transform: rotate(360deg); }
        }
        </style>
        """
        print("🛰️ Affichage des trajectoires satellites désactivé - Rafraîchissement en cours...")

    # Rafraîchir la carte avec le nouvel état
    try:
        if 'refresh_ui_from_stream' in globals():
            # Récupérer les filtres actuels
            selected_date = None
            vessel_type = 'Tous'
            country = 'Tous'
            selected_mmsi = None

            if 'date_selector' in globals():
                selected_date = date_selector.value
            if 'filter_type_dropdown' in globals():
                vessel_type = filter_type_dropdown.value
            if 'filter_country_dropdown' in globals():
                country = filter_country_dropdown.value
            if 'last_selected_mmsi' in globals():
                selected_mmsi = last_selected_mmsi
            elif 'selected_mmsi_list' in globals() and selected_mmsi_list:
                selected_mmsi = selected_mmsi_list[0] if len(selected_mmsi_list) == 1 else None

            print(f"🔄 Rafraîchissement de la carte avec éphémérides {'activés' if show_ephemeris_global else 'désactivés'}...")

            # Appeler refresh_ui_from_stream avec force_refresh=True pour forcer le rafraîchissement
            refresh_ui_from_stream(
                selected_mmsi=selected_mmsi,
                selected_date=selected_date,
                vessel_type=vessel_type,
                country=country,
                force_refresh=True
            )

            if show_ephemeris_global:
                filter_status.value = """
                <div style='padding:14px 18px; background: linear-gradient(135deg, #d1e7dd 0%, #d4edda 100%);
                            border-radius: 10px; border-left: 4px solid #198754;
                            box-shadow: 0 3px 10px rgba(25, 135, 84, 0.15); border: 1px solid #badbcc;'>
                    <div style='color: #0f5132; font-weight: 600;'>
                        <div style='display: flex; align-items: center; gap: 10px; margin-bottom: 8px;'>
                            <span style='font-size: 18px;'>✅</span>
                            <span style='font-size: 14px;'>Trajectoires satellites affichées!</span>
                        </div>
                        <div style='font-size: 12px; padding-left: 28px; line-height: 1.6; font-weight: 500;'>
                            🛰️ Les trajectoires satellites sont maintenant visibles sur la carte<br>
                            💡 <em>Si la carte apparaît floutée, relancez la cellule d'affichage de l'interface</em>
                        </div>
                    </div>
                </div>
                """
            else:
                filter_status.value = """
                <div style='padding:14px 18px; background: linear-gradient(135deg, #d1e7dd 0%, #d4edda 100%);
                            border-radius: 10px; border-left: 4px solid #198754;
                            box-shadow: 0 3px 10px rgba(25, 135, 84, 0.15); border: 1px solid #badbcc;'>
                    <div style='color: #0f5132; font-weight: 600;'>
                        <div style='display: flex; align-items: center; gap: 10px; margin-bottom: 8px;'>
                            <span style='font-size: 18px;'>✅</span>
                            <span style='font-size: 14px;'>Trajectoires satellites masquées!</span>
                        </div>
                        <div style='font-size: 12px; padding-left: 28px; font-weight: 500;'>
                            🗺️ La carte a été rafraîchie sans les trajectoires satellites
                        </div>
                    </div>
                </div>
                """
        else:
            filter_status.value = """
            <div style='padding:14px 18px; background: linear-gradient(135deg, #f8d7da 0%, #ffebee 100%);
                        border-radius: 10px; border-left: 4px solid #dc3545;
                        box-shadow: 0 3px 10px rgba(220, 53, 69, 0.15); border: 1px solid #f5c2c7;'>
                <div style='display: flex; align-items: center; gap: 10px; color: #842029; font-weight: 600;'>
                    <span style='font-size: 18px;'>❌</span>
                    <span style='font-size: 13px;'>Fonction refresh_ui_from_stream non disponible</span>
                </div>
            </div>
            """
    except Exception as e:
        filter_status.value = f"""
        <div style='padding:14px 18px; background: linear-gradient(135deg, #f8d7da 0%, #ffebee 100%);
                    border-radius: 10px; border-left: 4px solid #dc3545;
                    box-shadow: 0 3px 10px rgba(220, 53, 69, 0.15); border: 1px solid #f5c2c7;'>
            <div style='display: flex; align-items: center; gap: 10px; color: #842029; font-weight: 600;'>
                <span style='font-size: 18px;'>❌</span>
                <span style='font-size: 13px;'>Erreur: {str(e)[:60]}</span>
            </div>
        </div>
        """
        print(f"❌ Erreur lors du rafraîchissement: {e}")
        import traceback
        traceback.print_exc()

# Connecter l'observer au toggle button
ephemeris_toggle_btn.observe(toggle_ephemeris, names='value')

# Widget de statut
filter_status = widgets.HTML(
    value="""
    <div style='padding:14px 18px;
                background: linear-gradient(135deg, #f8f9fa 0%, #e9ecef 100%);
                border-radius: 10px;
                border-left: 4px solid #6c757d;
                box-shadow: 0 3px 10px rgba(108, 117, 125, 0.12);
                border: 1px solid #dee2e6;'>
        <div style='display: flex; align-items: center; gap: 10px; color: #495057; font-weight: 600;'>
            <span style='font-size: 18px;'>ℹ️</span>
            <span style='font-size: 13px;'>Sélectionnez des filtres et cliquez sur "Appliquer"</span>
        </div>
    </div>
    """,
    layout=widgets.Layout(width='100%', margin='10px 0 0 0')
)

def apply_vessel_filters(button):
    """Applique les filtres de type et pays"""
    global selected_mmsi_list, last_selected_mmsi

    vessel_type = filter_type_dropdown.value
    country = filter_country_dropdown.value
    selected_date = date_selector.value

    if vessel_type != 'Tous' or country != 'Tous' or selected_date is not None:
        # Réinitialiser la sélection de navires
        selected_mmsi_list = []
        last_selected_mmsi = None

        # Réinitialiser le widget de sélection
        if 'boat_selector' in globals():
            boat_selector.value = ()

        # Mettre à jour l'affichage de sélection
        if 'boat_selection_info' in globals():
            boat_selection_info.value = """
            <div style='padding:12px 15px; background:#e8f4f8; border-radius:8px;
                        border-left:4px solid #0288d1; color:#01579b; font-size:13px;
                        margin:10px 0; box-shadow: 0 2px 6px rgba(2, 136, 209, 0.2);'>
                ℹ️ <strong>Sélection de navire réinitialisée</strong> - Filtres appliqués
            </div>
            """

        if 'weather_output' in globals():
            weather_output.value = """
            <div style='padding: 15px; background: #e8f4f8; border-radius: 8px; border-left: 4px solid #0288d1;
                        color: #01579b; font-weight: 500;'>
                <strong>🌤️ Météo</strong><br>
                Sélectionnez et validez un ou plusieurs bateaux
            </div>
            """
        if 'location_output' in globals():
            location_output.value = """
            <div style='padding: 15px; background: #e8f5e9; border-radius: 8px; border-left: 4px solid #4caf50;
                        color: #1b5e20; font-weight: 500;'>
                <strong>🌍 Localisation</strong><br>
                Sélectionnez et validez un ou plusieurs bateaux
            </div>
            """

        print("🔄 Sélection de navire réinitialisée (filtres appliqués)")

    filter_status.value = """
    <div style='padding:14px 18px; background: linear-gradient(135deg, #cfe2ff 0%, #e3f2fd 100%);
                border-radius: 10px; border-left: 4px solid #0d6efd;
                box-shadow: 0 3px 10px rgba(13, 110, 253, 0.15); border: 1px solid #b6d4fe;'>
        <div style='display: flex; align-items: center; gap: 10px; color: #084298; font-weight: 600;'>
            <span style='font-size: 18px;'>🔄</span>
            <span style='font-size: 13px;'>Application des filtres en cours...</span>
        </div>
    </div>
    """

    try:
        if 'refresh_ui_from_stream' in globals():
            refresh_ui_from_stream(
                selected_mmsi=None,  # Pas de navire sélectionné quand on applique des filtres
                vessel_type=vessel_type,
                country=country,
                selected_date=selected_date
            )

            if 'df_ais_global' in globals() and df_ais_global is not None:
                filtered_df = df_ais_global.copy()

                if vessel_type != 'Tous':
                    filtered_df = filtered_df[filtered_df['ShipTypeLabel'] == vessel_type]

                if country != 'Tous':
                    filtered_df = filtered_df[filtered_df['Pays'] == country]

                if selected_date is not None:
                    filtered_df['date_only'] = pd.to_datetime(filtered_df['datetime']).dt.date
                    filtered_df = filtered_df[filtered_df['date_only'] == selected_date]

                result_count = filtered_df['Mmsi'].nunique()
            else:
                result_count = 0

            filter_status.value = f"""
            <div style='padding:14px 18px; background: linear-gradient(135deg, #d1e7dd 0%, #d4edda 100%);
                        border-radius: 10px; border-left: 4px solid #198754;
                        box-shadow: 0 3px 10px rgba(25, 135, 84, 0.15); border: 1px solid #badbcc;'>
                <div style='color: #0f5132; font-weight: 600;'>
                    <div style='display: flex; align-items: center; gap: 10px; margin-bottom: 8px;'>
                        <span style='font-size: 18px;'>✅</span>
                        <span style='font-size: 14px;'>Filtres appliqués avec succès!</span>
                    </div>
                    <div style='font-size: 12px; padding-left: 28px; line-height: 1.6; font-weight: 500;'>
                        📊 <strong>{result_count} navire(s)</strong> trouvé(s)<br>
                        🎯 Type: <strong>{vessel_type}</strong> | Pays: <strong>{country}</strong> | Date: <strong>{selected_date or 'Toutes'}</strong><br>
                        ℹ️ Sélection de navire réinitialisée
                    </div>
                </div>
            </div>
            """
        else:
            filter_status.value = """
            <div style='padding:14px 18px; background: linear-gradient(135deg, #f8d7da 0%, #ffebee 100%);
                        border-radius: 10px; border-left: 4px solid #dc3545;
                        box-shadow: 0 3px 10px rgba(220, 53, 69, 0.15); border: 1px solid #f5c2c7;'>
                <div style='display: flex; align-items: center; gap: 10px; color: #842029; font-weight: 600;'>
                    <span style='font-size: 18px;'>❌</span>
                    <span style='font-size: 13px;'>Fonction refresh_ui_from_stream non disponible</span>
                </div>
            </div>
            """

    except Exception as e:
        filter_status.value = f"""
        <div style='padding:14px 18px; background: linear-gradient(135deg, #f8d7da 0%, #ffebee 100%);
                    border-radius: 10px; border-left: 4px solid #dc3545;
                    box-shadow: 0 3px 10px rgba(220, 53, 69, 0.15); border: 1px solid #f5c2c7;'>
            <div style='display: flex; align-items: center; gap: 10px; color: #842029; font-weight: 600;'>
                <span style='font-size: 18px;'>❌</span>
                <span style='font-size: 13px;'>Erreur: {str(e)[:60]}</span>
            </div>
        </div>
        """
        print(f"Erreur détaillée: {e}")

def reset_vessel_filters(button):
    """Remet les filtres à zéro"""
    try:
        filter_type_dropdown.value = 'Tous'
        filter_country_dropdown.value = 'Tous'
        date_selector.value = None

        filter_status.value = """
        <div style='padding:14px 18px; background: linear-gradient(135deg, #fff3cd 0%, #fffbf0 100%);
                    border-radius: 10px; border-left: 4px solid #ffc107;
                    box-shadow: 0 3px 10px rgba(255, 193, 7, 0.15); border: 1px solid #ffecb5;'>
            <div style='display: flex; align-items: center; gap: 10px; color: #664d03; font-weight: 600;'>
                <span style='font-size: 18px;'>🔄</span>
                <span style='font-size: 13px;'>Remise à zéro des filtres...</span>
            </div>
        </div>
        """

        if 'refresh_ui_from_stream' in globals():
            # Garder la sélection de navire si elle existe
            selected_mmsi = None
            if 'last_selected_mmsi' in globals() and last_selected_mmsi:
                selected_mmsi = last_selected_mmsi
            elif 'selected_mmsi_list' in globals() and selected_mmsi_list:
                selected_mmsi = selected_mmsi_list[0] if len(selected_mmsi_list) == 1 else None

            refresh_ui_from_stream(selected_mmsi=selected_mmsi)

            if 'df_ais_global' in globals() and df_ais_global is not None and not df_ais_global.empty:
                total_count = df_ais_global['Mmsi'].nunique()
            else:
                total_count = 0

            filter_status.value = f"""
            <div style='padding:14px 18px; background: linear-gradient(135deg, #d1e7dd 0%, #d4edda 100%);
                        border-radius: 10px; border-left: 4px solid #198754;
                        box-shadow: 0 3px 10px rgba(25, 135, 84, 0.15); border: 1px solid #badbcc;'>
                <div style='color: #0f5132; font-weight: 600;'>
                    <div style='display: flex; align-items: center; gap: 10px; margin-bottom: 8px;'>
                        <span style='font-size: 18px;'>✅</span>
                        <span style='font-size: 14px;'>Filtres remis à zéro!</span>
                    </div>
                    <div style='font-size: 12px; padding-left: 28px; font-weight: 500;'>
                        📊 <strong>{total_count} navire(s)</strong> affichés<br>
                        🎯 Aucun filtre appliqué
                    </div>
                </div>
            </div>
            """
        else:
            filter_status.value = """
            <div style='padding:14px 18px; background: linear-gradient(135deg, #f8d7da 0%, #ffebee 100%);
                        border-radius: 10px; border-left: 4px solid #dc3545;
                        box-shadow: 0 3px 10px rgba(220, 53, 69, 0.15); border: 1px solid #f5c2c7;'>
                <div style='display: flex; align-items: center; gap: 10px; color: #842029; font-weight: 600;'>
                    <span style='font-size: 18px;'>❌</span>
                    <span style='font-size: 13px;'>Fonction refresh_ui_from_stream non disponible</span>
                </div>
            </div>
            """

    except Exception as e:
        filter_status.value = f"""
        <div style='padding:14px 18px; background: linear-gradient(135deg, #f8d7da 0%, #ffebee 100%);
                    border-radius: 10px; border-left: 4px solid #dc3545;
                    box-shadow: 0 3px 10px rgba(220, 53, 69, 0.15); border: 1px solid #f5c2c7;'>
            <div style='display: flex; align-items: center; gap: 10px; color: #842029; font-weight: 600;'>
                <span style='font-size: 18px;'>❌</span>
                <span style='font-size: 13px;'>Erreur: {str(e)[:60]}</span>
            </div>
        </div>
        """
        print(f"Erreur détaillée: {e}")

def show_all_vessels(button):
    """Affiche tous les navires sans filtre"""
    try:
        filter_status.value = """
        <div style='padding:14px 18px; background: linear-gradient(135deg, #cfe2ff 0%, #e3f2fd 100%);
                    border-radius: 10px; border-left: 4px solid #0d6efd;
                    box-shadow: 0 3px 10px rgba(13, 110, 253, 0.15); border: 1px solid #b6d4fe;'>
            <div style='display: flex; align-items: center; gap: 10px; color: #084298; font-weight: 600;'>
                <span style='font-size: 18px;'>🔄</span>
                <span style='font-size: 13px;'>Affichage de tous les navires...</span>
            </div>
        </div>
        """

        filter_type_dropdown.value = 'Tous'
        filter_country_dropdown.value = 'Tous'
        date_selector.value = None

        if 'refresh_ui_from_stream' in globals():
            refresh_ui_from_stream()

            if 'df_ais_global' in globals() and df_ais_global is not None and not df_ais_global.empty:
                total_count = df_ais_global['Mmsi'].nunique()
            else:
                total_count = 0

            filter_status.value = f"""
            <div style='padding:14px 18px; background: linear-gradient(135deg, #d1e7dd 0%, #d4edda 100%);
                        border-radius: 10px; border-left: 4px solid #198754;
                        box-shadow: 0 3px 10px rgba(25, 135, 84, 0.15); border: 1px solid #badbcc;'>
                <div style='color: #0f5132; font-weight: 600;'>
                    <div style='display: flex; align-items: center; gap: 10px; margin-bottom: 8px;'>
                        <span style='font-size: 18px;'>✅</span>
                        <span style='font-size: 14px;'>Tous les navires affichés!</span>
                    </div>
                    <div style='font-size: 12px; padding-left: 28px; font-weight: 500;'>
                        📊 <strong>{total_count} navire(s)</strong> visibles<br>
                        🎯 Aucun filtre appliqué
                    </div>
                </div>
            </div>
            """
        else:
            filter_status.value = """
            <div style='padding:14px 18px; background: linear-gradient(135deg, #f8d7da 0%, #ffebee 100%);
                        border-radius: 10px; border-left: 4px solid #dc3545;
                        box-shadow: 0 3px 10px rgba(220, 53, 69, 0.15); border: 1px solid #f5c2c7;'>
                <div style='display: flex; align-items: center; gap: 10px; color: #842029; font-weight: 600;'>
                    <span style='font-size: 18px;'>❌</span>
                    <span style='font-size: 13px;'>Fonction refresh_ui_from_stream non disponible</span>
                </div>
            </div>
            """

    except Exception as e:
        filter_status.value = f"""
        <div style='padding:14px 18px; background: linear-gradient(135deg, #f8d7da 0%, #ffebee 100%);
                    border-radius: 10px; border-left: 4px solid #dc3545;
                    box-shadow: 0 3px 10px rgba(220, 53, 69, 0.15); border: 1px solid #f5c2c7;'>
            <div style='display: flex; align-items: center; gap: 10px; color: #842029; font-weight: 600;'>
                <span style='font-size: 18px;'>❌</span>
                <span style='font-size: 13px;'>Erreur: {str(e)[:60]}</span>
            </div>
        </div>
        """
        print(f"Erreur détaillée: {e}")

def update_filter_options():
    """Met à jour les options des widgets de filtrage depuis df_ais_global"""
    try:
        new_options = get_filter_options_from_data()

        filter_type_dropdown.options = new_options['types']
        filter_country_dropdown.options = new_options['countries']

        print("✅ Options de filtrage mises à jour depuis df_ais_global")
        print(f"   • Types disponibles: {len(new_options['types'])-1}")
        print(f"   • Pays disponibles: {len(new_options['countries'])-1}")

    except Exception as e:
        print(f"❌ Erreur mise à jour options: {e}")

def populate_date_selector():
    """Remplit le dropdown de dates avec les dates disponibles"""
    if 'df_ais_global' not in globals() or df_ais_global is None or df_ais_global.empty:
        date_selector.options = [('Aucune donnée', None)]
        return

    df_ais_global['date_only'] = pd.to_datetime(df_ais_global['datetime']).dt.date
    unique_dates = sorted(df_ais_global['date_only'].unique())

    date_options = [('Toutes les dates', None)]
    for date_val in unique_dates:
        date_str = date_val.strftime('%d/%m/%Y')
        date_options.append((date_str, date_val))

    date_selector.options = date_options
    print(f'📅 {len(unique_dates)} dates disponibles dans le dataset')

def on_date_filter_apply(button):
    """Handler pour appliquer le filtre de date"""
    global selected_mmsi_list, last_selected_mmsi

    selected_date = date_selector.value

    if selected_date is not None:
        # Réinitialiser la sélection de navires
        selected_mmsi_list = []
        last_selected_mmsi = None

        # Réinitialiser le widget de sélection
        if 'boat_selector' in globals():
            boat_selector.value = ()

        # Mettre à jour l'affichage de sélection
        if 'boat_selection_info' in globals():
            boat_selection_info.value = """
            <div style='padding:12px 15px; background:#e8f4f8; border-radius:8px;
                        border-left:4px solid #0288d1; color:#01579b; font-size:13px;
                        margin:10px 0; box-shadow: 0 2px 6px rgba(2, 136, 209, 0.2);'>
                ℹ️ <strong>Sélection de navire réinitialisée</strong> - Filtre de date appliqué
            </div>
            """

        if 'weather_output' in globals():
            weather_output.value = """
            <div style='padding: 15px; background: #e8f4f8; border-radius: 8px; border-left: 4px solid #0288d1;
                        color: #01579b; font-weight: 500;'>
                <strong>🌤️ Météo</strong><br>
                Sélectionnez et validez un ou plusieurs bateaux
            </div>
            """
        if 'location_output' in globals():
            location_output.value = """
            <div style='padding: 15px; background: #e8f5e9; border-radius: 8px; border-left: 4px solid #4caf50;
                        color: #1b5e20; font-weight: 500;'>
                <strong>🌍 Localisation</strong><br>
                Sélectionnez et validez un ou plusieurs bateaux
            </div>
            """

        print("🔄 Sélection de navire réinitialisée (filtre de date appliqué)")

    if selected_date is None:
        print('📅 Affichage de toutes les dates')
        refresh_ui_from_stream(selected_mmsi=None)  # Pas de navire sélectionné
    else:
        print(f'📅 Filtrage par date: {selected_date}')
        refresh_ui_from_stream(selected_mmsi=None, selected_date=selected_date)  # Pas de navire sélectionné

# Connecter les boutons
filter_apply_btn.on_click(apply_vessel_filters)
filter_reset_btn.on_click(reset_vessel_filters)
filter_show_all_btn.on_click(show_all_vessels)
date_apply_btn.on_click(on_date_filter_apply)

# Mettre à jour les options au chargement
update_filter_options()
populate_date_selector()

# Interface de filtrage
vessel_filter_controls = widgets.VBox([
    # Header Type/Pays
    widgets.HTML("""
        <div style='background: linear-gradient(135deg, #e3f2fd 0%, #f0f8ff 100%);
                    padding: 12px 18px; border-radius: 10px; margin-bottom: 12px;
                    border: 2px solid #90caf9; box-shadow: 0 4px 12px rgba(33, 150, 243, 0.15);'>
            <div style='display: flex; align-items: center; gap: 10px;'>
                <span style='font-size: 20px;'>🎛️</span>
                <h4 style='color:#0277bd; font-weight:700; margin:0; font-size:14px;'>
                    Filtres de Navires
                </h4>
            </div>
        </div>
    """),
    widgets.HBox([
        filter_type_dropdown,
        filter_country_dropdown,
        filter_apply_btn,
        filter_reset_btn,
        filter_show_all_btn,
        ephemeris_toggle_btn
    ], layout=widgets.Layout(margin='0 0 15px 0', flex_flow='row wrap', justify_content='flex-start')),

    # Header Date
    widgets.HTML("""
        <div style='background: linear-gradient(135deg, #e3f2fd 0%, #f0f8ff 100%);
                    padding: 12px 18px; border-radius: 10px; margin-bottom: 12px;
                    border: 2px solid #90caf9; box-shadow: 0 4px 12px rgba(33, 150, 243, 0.15);'>
            <div style='display: flex; align-items: center; gap: 10px;'>
                <span style='font-size: 20px;'>📅</span>
                <h4 style='color:#0277bd; font-weight:700; margin:0; font-size:14px;'>
                    Filtrage par Date
                </h4>
            </div>
        </div>
    """),
    widgets.HBox([
        date_selector,
        date_apply_btn
    ], layout=widgets.Layout(margin='0 0 12px 0')),

    filter_status
], layout=widgets.Layout(
    border='2px solid #e9ecef',
    border_radius='12px',
    padding='18px',
    margin='10px 0',
    background_color='#ffffff',
    box_shadow='0 4px 15px rgba(0,0,0,0.06)'
))

✅ Options de filtrage mises à jour depuis df_ais_global
   • Types disponibles: 17
   • Pays disponibles: 105
📅 8 dates disponibles dans le dataset


**Interface principale - Création du layout principal (main_layout_with_filtering) avec header, sélection multi-navires, carte de tracking, widgets d'informations, chatbot IA et historique**

In [ ]:
header = widgets.HTML("""
<div style='text-align: center; padding: 25px 20px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 50%, #f093fb 100%);
            color: white; border-radius: 12px; margin-bottom: 20px;
            box-shadow: 0 8px 20px rgba(102, 126, 234, 0.35);
            position: relative; overflow: hidden;'>

    <!-- Decorative circle -->
    <div style='position: absolute; top: -30px; right: -30px; width: 120px; height: 120px;
                background: rgba(255,255,255,0.08); border-radius: 50%;'></div>

    <!-- Content -->
    <div style='position: relative; z-index: 1;'>
        <!-- Main Title -->
        <h1 style='margin: 0 0 8px 0; font-size: 2.5em; font-weight: 800;
                    text-shadow: 0 3px 15px rgba(0,0,0,0.3);
                    letter-spacing: 0.5px;'>
            Naval assistant - Assistant Naval augmenté par IA agentique
        </h1>

        <!-- Subtitle -->
        <p style='margin: 5px 0 18px 0; font-size: 1.1em; opacity: 0.95;
                  font-weight: 400; text-shadow: 0 2px 6px rgba(0,0,0,0.2);'>
            Surveillance Maritime en Temps Réel • Méditerranée Focus
        </p>

        <!-- Badges Row -->
        <div style='display: flex; justify-content: center; gap: 12px; flex-wrap: wrap;
                    margin-bottom: 15px;'>

            <!-- Stats Badges -->
            <div style='background: rgba(255,255,255,0.2); padding: 6px 14px; border-radius: 20px;
                        border: 1px solid rgba(255,255,255,0.3); font-size: 12px; font-weight: 600;'>
                🗺️ 35+ Couches
            </div>

            <div style='background: rgba(255,255,255,0.2); padding: 6px 14px; border-radius: 20px;
                        border: 1px solid rgba(255,255,255,0.3); font-size: 12px; font-weight: 600;'>
                📊 8.7M+ Positions
            </div>

            <div style='background: rgba(255,255,255,0.2); padding: 6px 14px; border-radius: 20px;
                        border: 1px solid rgba(255,255,255,0.3); font-size: 12px; font-weight: 600;'>
                🚢 Multi-Track 10x
            </div>

            <div style='background: rgba(255,255,255,0.2); padding: 6px 14px; border-radius: 20px;
                        border: 1px solid rgba(255,255,255,0.3); font-size: 12px; font-weight: 600;'>
                ⚡ Real-Time
            </div>

            <!-- GitHub Badge 1 -->
            <a href='https://github.com/Metimer' target='_blank' style='text-decoration: none;'>
                <div style='background: rgba(0,0,0,0.25); padding: 6px 14px; border-radius: 20px;
                            border: 1px solid rgba(255,255,255,0.3); font-size: 12px; font-weight: 600;
                            display: inline-flex; align-items: center; gap: 6px; color: white;'>
                    <svg style='width: 14px; height: 14px; fill: white;' viewBox='0 0 16 16'>
                        <path d='M8 0C3.58 0 0 3.58 0 8c0 3.54 2.29 6.53 5.47 7.59.4.07.55-.17.55-.38 0-.19-.01-.82-.01-1.49-2.01.37-2.53-.49-2.69-.94-.09-.23-.48-.94-.82-1.13-.28-.15-.68-.52-.01-.53.63-.01 1.08.58 1.23.82.72 1.21 1.87.87 2.33.66.07-.52.28-.87.51-1.07-1.78-.2-3.64-.89-3.64-3.95 0-.87.31-1.59.82-2.15-.08-.2-.36-1.02.08-2.12 0 0 .67-.21 2.2.82.64-.18 1.32-.27 2-.27.68 0 1.36.09 2 .27 1.53-1.04 2.2-.82 2.2-.82.44 1.1.16 1.92.08 2.12.51.56.82 1.27.82 2.15 0 3.07-1.87 3.75-3.65 3.95.29.25.54.73.54 1.48 0 1.07-.01 1.93-.01 2.2 0 .21.15.46.55.38A8.013 8.013 0 0016 8c0-4.42-3.58-8-8-8z'/>
                    </svg>
                    Metimer
                </div>
            </a>

            <!-- GitHub Badge 2 -->
            <a href='https://github.com/Chakibprojects' target='_blank' style='text-decoration: none;'>
                <div style='background: rgba(0,0,0,0.25); padding: 6px 14px; border-radius: 20px;
                            border: 1px solid rgba(255,255,255,0.3); font-size: 12px; font-weight: 600;
                            display: inline-flex; align-items: center; gap: 6px; color: white;'>
                    <svg style='width: 14px; height: 14px; fill: white;' viewBox='0 0 16 16'>
                        <path d='M8 0C3.58 0 0 3.58 0 8c0 3.54 2.29 6.53 5.47 7.59.4.07.55-.17.55-.38 0-.19-.01-.82-.01-1.49-2.01.37-2.53-.49-2.69-.94-.09-.23-.48-.94-.82-1.13-.28-.15-.68-.52-.01-.53.63-.01 1.08.58 1.23.82.72 1.21 1.87.87 2.33.66.07-.52.28-.87.51-1.07-1.78-.2-3.64-.89-3.64-3.95 0-.87.31-1.59.82-2.15-.08-.2-.36-1.02.08-2.12 0 0 .67-.21 2.2.82.64-.18 1.32-.27 2-.27.68 0 1.36.09 2 .27 1.53-1.04 2.2-.82 2.2-.82.44 1.1.16 1.92.08 2.12.51.56.82 1.27.82 2.15 0 3.07-1.87 3.75-3.65 3.95.29.25.54.73.54 1.48 0 1.07-.01 1.93-.01 2.2 0 .21.15.46.55.38A8.013 8.013 0 0016 8c0-4.42-3.58-8-8-8z'/>
                    </svg>
                    Chakibprojects
                </div>
            </a>
        </div>

        <!-- Tech Stack -->
        <div style='padding-top: 12px; border-top: 1px solid rgba(255,255,255,0.25);
                    font-size: 11px; opacity: 0.85; font-weight: 500;'>
            🐍 Python • 📊 Pandas • 🗺️ Folium • ⚡ EMODnet • 🤖 LangChain
        </div>
    </div>
</div>
""")

if 'history_output' not in globals():
    history_output = widgets.HTML(value="""
        <div style='padding:14px; background:#f8f9fa; border-radius:8px; color:#495057;
                    font-size:13px; border: 1px solid #e9ecef;'>
            Aucun échange pour l'instant.
        </div>
    """)

if 'clear_history_button' not in globals():
    clear_history_button = widgets.Button(
        description="🗑️ Effacer l'historique",
        button_style='warning',
        icon='trash',
        layout=widgets.Layout(width='200px')
    )

def clear_history(button):
    if 'chat_history' in globals():
        globals()['chat_history'] = []

    # Mettre à jour l'affichage
    history_output.value = """
        <div style='padding:14px; background:#f8f9fa; border-radius:8px; color:#495057;
                    font-size:13px; border: 1px solid #e9ecef;'>
            Historique effacé.
        </div>
    """

    print("🗑️ Historique du chat effacé")

clear_history_button.on_click(clear_history)

if 'chat_input' not in globals():
    chat_input = widgets.Text(
        placeholder='Posez une question sur les bateaux...',
        description='Question:',
        layout=widgets.Layout(width='100%'),
        style={'description_width': 'initial'}
    )

if 'chat_button' not in globals():
    chat_button = widgets.Button(description='Envoyer', button_style='primary', icon='paper-plane')

if 'chat_output' not in globals():
    chat_output = widgets.HTML(value="""
        <div style='padding: 14px; background: linear-gradient(135deg, #fff9e6 0%, #fffbf0 100%);
                    border-radius: 10px; color: #5d4037; font-weight: 500;
                    border-left: 4px solid #ffc107; box_shadow: 0 2px 8px rgba(0,0,0,0.08);'>
            💬 <strong>Chatbot prêt!</strong> Posez vos questions sur les bateaux.
        </div>
    """)

if 'actions_output' not in globals():
    actions_output = widgets.Output()

if 'feedback_panel' not in globals():
    feedback_panel = widgets.HTML('')


# Interface principale
main_layout_with_filtering = widgets.VBox([
    header,

    widgets.HTML("""
        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 50%, #f093fb 100%);
                    padding: 15px 20px; border-radius: 12px; margin: 15px 0 10px 0;
                    box-shadow: 0 6px 18px rgba(102, 126, 234, 0.3);
                    position: relative; overflow: hidden;'>
            <div style='position: absolute; top: -15px; right: -15px; width: 80px; height: 80px;
                        background: rgba(255,255,255,0.1); border-radius: 50%;'></div>
            <h2 style='color:#ffffff; font-weight:700; margin:0; font-size:18px; position: relative; z-index: 1;
                       text-shadow: 0 2px 6px rgba(0,0,0,0.2);'>
                🎯 Sélection des bateaux
            </h2>
        </div>
    """),

    # Info box
    widgets.HTML("""
        <div style='padding:12px 15px;
                    background: linear-gradient(135deg, #e3f2fd 0%, #e8f4f8 100%);
                    border-radius: 8px; margin-bottom: 12px;
                    border-left: 4px solid #2196f3; color: #01579b; font-size: 12px;
                    box-shadow: 0 2px 8px rgba(33, 150, 243, 0.12);'>
            <strong>💡 Astuce:</strong> Maintenez <kbd style='background:#fff; padding:2px 6px;
            border-radius:3px; border:1px solid #90caf9; font-size:11px; font-weight:600;'>Ctrl</kbd>
            pour multi-sélection, puis validez
        </div>
    """),

    boat_selector,
    boat_selection_info,
    widgets.HBox([boat_validate_btn, boat_deselect_btn], layout=widgets.Layout(margin='10px 0 15px 0', gap='10px')),

    # Layout principal
    widgets.HBox([
        widgets.VBox([
            # Carte header
            widgets.HTML("""
                <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 50%, #f093fb 100%);
                            padding: 15px 20px; border-radius: 12px; margin: 15px 0 10px 0;
                            box-shadow: 0 6px 18px rgba(102, 126, 234, 0.3);
                            position: relative; overflow: hidden;'>
                    <div style='position: absolute; top: -15px; right: -15px; width: 80px; height: 80px;
                                background: rgba(255,255,255,0.1); border-radius: 50%;'></div>
                    <h2 style='color:#ffffff; font-weight:700; margin:0; font-size:18px; position: relative; z-index: 1;
                               text-shadow: 0 2px 6px rgba(0,0,0,0.2);'>
                        🗺️ Carte de tracking
                    </h2>
                </div>
            """),
            vessel_filter_controls,
            map_output,
            cache_clear_btn,
            animated_timeline_interface
        ], layout=widgets.Layout(width='60%')),

        widgets.VBox([
            # Info header
            widgets.HTML("""
                <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 50%, #f093fb 100%);
                            padding: 15px 20px; border-radius: 12px; margin: 15px 0 10px 0;
                            box-shadow: 0 6px 18px rgba(102, 126, 234, 0.3);
                            position: relative; overflow: hidden;'>
                    <div style='position: absolute; top: -15px; right: -15px; width: 80px; height: 80px;
                                background: rgba(255,255,255,0.1); border-radius: 50%;'></div>
                    <h2 style='color:#ffffff; font-weight:700; margin:0; font-size:18px; position: relative; z-index: 1;
                               text-shadow: 0 2px 6px rgba(0,0,0,0.2);'>
                        📊 Informations
                    </h2>
                </div>
            """),
            weather_output,
            widgets.HTML('<div style="margin:12px 0;"></div>'),
            location_output
        ], layout=widgets.Layout(width='40%', padding='0 0 0 20px'))
    ]),

    # Section Chatbot
    widgets.HTML("""
        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 50%, #f093fb 100%);
                    padding: 15px 20px; border-radius: 12px; margin: 25px 0 10px 0;
                    box-shadow: 0 6px 18px rgba(102, 126, 234, 0.3);
                    position: relative; overflow: hidden;'>
            <div style='position: absolute; top: -15px; right: -15px; width: 80px; height: 80px;
                        background: rgba(255,255,255,0.1); border-radius: 50%;'></div>
            <h2 style='color:#ffffff; font-weight:700; margin:0; font-size:18px; position: relative; z-index: 1;
                       text-shadow: 0 2px 6px rgba(0,0,0,0.2);'>
                🤖 Assistant IA Maritime
            </h2>
        </div>
    """),
    widgets.VBox([
        widgets.HTML("""
            <h3 style='color:#667eea; font-weight:600; margin:0 0 8px 0; font-size:15px;'>
                💬 Nouvelle question
            </h3>
        """),
        widgets.HBox([chat_input, chat_button], layout=widgets.Layout(width='100%')),
        chat_output,
        actions_output,
        feedback_panel
    ], layout=widgets.Layout(margin='0 0 15px 0')),

    # Historique
    widgets.VBox([
        widgets.HTML("""
            <h3 style='color:#667eea; font-weight:600; margin:0 0 10px 0; font-size:15px;'>
                📜 Historique des discussions
            </h3>
        """),
        widgets.HBox([clear_history_button if 'clear_history_button' in globals() else widgets.HTML('')]),
        history_output if 'history_output' in globals() else widgets.HTML('')
    ], layout=widgets.Layout(
        border='2px solid #e9ecef',
        border_radius='10px',
        padding='15px',
        background_color='#ffffff',
        box_shadow='0 3px 12px rgba(0,0,0,0.06)'
    ))
], layout=widgets.Layout(padding='15px'))

**Refresh intelligent de l'interface**

In [ ]:
print("📊 Vérification des données et de l'état de l'interface...")

if 'df_ais_global' in globals() and df_ais_global is not None and not df_ais_global.empty:
    print(f'✅ {len(df_ais_global)} lignes de données parquet disponibles')
    print(f'📝 Les données sont issues du fichier Parquet chargé')

    if 'Mmsi' in df_ais_global.columns:
        unique_vessels = df_ais_global['Mmsi'].nunique()
        print(f'🚢 {unique_vessels} navires uniques dans les données')

    # Vérifier si l'interface a déjà été initialisée
    if 'map_output' in globals() and hasattr(map_output, 'value'):
        # Vérifier si la carte existe déjà
        print('ℹ️ Interface déjà initialisée - Rafraîchissement optionnel')
        print('💡 Utilisez les filtres ou la sélection de navires pour mettre à jour la carte')
    else:
        print("🔄 Initialisation de l'interface...")
        print("⏱️  Veuillez patienter - Le chargement peut prendre 30 à 60 secondes...")

        if 'refresh_ui_from_stream' in globals():
            try:
                refresh_ui_from_stream(force_refresh=False)
                print('✅ Interface initialisée avec succès')
            except Exception as e:
                print(f'⚠️ Erreur lors de l\'initialisation: {e}')
                print('💡 Vous pouvez relancer cette cellule ou utiliser les filtres')
        else:
            print('⚠️ Fonction refresh_ui_from_stream non disponible')
            print('💡 Exécutez les cellules précédentes pour initialiser l\'interface')
else:
    print('❌ Aucune donnée AIS disponible (df_ais_global)')
    print('💡 Assurez-vous d\'avoir chargé les données Parquet dans les cellules précédentes')

📊 Vérification des données et de l'état de l'interface...
✅ 17170692 lignes de données parquet disponibles
📝 Les données sont issues du fichier Parquet chargé
🚢 19462 navires uniques dans les données
🔄 Initialisation de l'interface...
⏱️  Veuillez patienter - Le chargement peut prendre 30 à 60 secondes...
🔄 Calcul des dernières positions...
✅ 19459 positions calculées en 18.74s
💾 Cache SET pour d36ff015... (1 entrées)
⚠️ Limitation à 100 navires (sur 19459 disponibles)
🗺️ Création de la carte...
✅ 35 couches maritimes organisées par catégories
📍 100 navires - mode direct (sans clustering)
📡 Traitement de 2293 points RF...
✅ 2293 points RF ajoutés à la carte
✅ Carte créée: 100 navires, 📍 sans clustering
⏱️ Temps total: 23.35s
✅ Interface initialisée avec succès


**Affichage de l'interface complète - Vérification de tous les composants requis (main_layout_with_filtering, widgets, etc.) et affichage final du tableau de bord interactif**

In [ ]:
# Vérifier que tous les composants sont disponibles
required_components = [
    'main_layout_with_filtering',
    'vessel_filter_controls',
    'boat_selector',
    'map_output',
    'weather_output',
    'location_output',
    'chat_input',
    'chat_button',
    'chat_output'
]

missing_components = []
for component in required_components:
    if component not in globals():
        missing_components.append(component)

if missing_components:
    print("❌ Composants manquants:")
    for component in missing_components:
        print(f"   • {component}")
    print()
    print("🔧 Veuillez exécuter les cellules précédentes pour créer tous les composants nécessaires.")

# Afficher l'interface complète
display(main_layout_with_filtering)